<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/02_Preprocessing_Encoding_%26_Data_Splits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# NOTEBOOK 02 — PREPROCESSING, ENCODING & DATA SPLITS
# ==================================================================================================
#
# Project:
#   SPP-GAN — Privacy-Preserving Synthetic Tabular Data Generation with GANs
#
# Notebook Purpose:
#   This notebook converts the validated raw datasets from Notebook 01 into reproducible,
#   leakage-controlled datasets suitable for downstream statistical, generative,
#   privacy-preserving, and machine-learning utility experiments.
#
# Processing stages:
#   1. Load validated dataset registry
#   2. Normalize recognized missing-value markers
#   3. Validate feature types
#   4. Validate target variables
#   5. Create immutable original row identifiers
#   6. Create stratified train/validation/test splits
#   7. Verify split integrity
#   8. Fit preprocessing objects on TRAINING DATA ONLY
#   9. Transform train/validation/test datasets
#  10. Save native and encoded datasets
#  11. Save preprocessing objects, schemas, feature mappings and manifests
#  12. Reload saved artifacts and verify reproducibility
#
# IMPORTANT METHODOLOGICAL BOUNDARY:
#   This notebook does NOT:
#       - train GANs
#       - generate synthetic data
#       - perform statistical baseline experiments
#       - perform privacy accounting
#       - evaluate synthetic-data quality
#       - remove rows based on outlier detection
#       - perform target-based feature selection
#
# Leakage-control principle:
#   Any learned preprocessing parameter MUST be fitted using TRAINING DATA ONLY.
#
# ==================================================================================================

print("=" * 100)
print("NOTEBOOK 02 — PREPROCESSING, ENCODING & DATA SPLITS")
print("=" * 100)

print()
print("✓ Notebook scope initialized.")
print("✓ Raw-data validation boundary inherited from Notebook 01.")
print("✓ Leakage-controlled preprocessing will be enforced.")

NOTEBOOK 02 — PREPROCESSING, ENCODING & DATA SPLITS

✓ Notebook scope initialized.
✓ Raw-data validation boundary inherited from Notebook 01.
✓ Leakage-controlled preprocessing will be enforced.


In [2]:
# ==================================================================================================
# 2. IMPORT LIBRARIES
# ==================================================================================================

import os
import sys
import json
import hashlib
import pickle
import warnings
import platform
import time

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.impute import SimpleImputer

import joblib

warnings.filterwarnings("ignore")

print("=" * 100)
print("LIBRARY IMPORT STATUS")
print("=" * 100)

print(f"✓ Python       : {platform.python_version()}")
print(f"✓ NumPy        : {np.__version__}")
print(f"✓ Pandas       : {pd.__version__}")
print(f"✓ Scikit-learn: imported")
print(f"✓ Joblib       : {joblib.__version__}")

print()
print("✓ All required libraries imported successfully.")

LIBRARY IMPORT STATUS
✓ Python       : 3.13.15
✓ NumPy        : 2.1.3
✓ Pandas       : 2.2.3
✓ Scikit-learn: imported
✓ Joblib       : 1.5.3

✓ All required libraries imported successfully.


In [3]:
# ==================================================================================================
# 3. LOAD NOTEBOOK 00 CONFIGURATION
# ==================================================================================================

print("=" * 100)
print("3. LOAD NOTEBOOK 00 CONFIGURATION")
print("=" * 100)

from pathlib import Path


# --------------------------------------------------------------------------------------------------
# 3.1 Google Drive
# --------------------------------------------------------------------------------------------------

if "google.colab" in sys.modules:

    from google.colab import drive

    DRIVE_MOUNT_POINT = Path("/content/drive")
    DRIVE_MYDRIVE = DRIVE_MOUNT_POINT / "MyDrive"

    if not DRIVE_MYDRIVE.exists():

        print()
        print("Mounting Google Drive...")

        drive.mount(
            str(DRIVE_MOUNT_POINT),
            force_remount=False
        )

    if not DRIVE_MYDRIVE.exists():

        raise RuntimeError(
            "Google Drive MyDrive could not be verified."
        )

    print("✓ Google Drive verified.")

else:

    DRIVE_MOUNT_POINT = Path("/content/drive")
    DRIVE_MYDRIVE = DRIVE_MOUNT_POINT / "MyDrive"

    print("⚠ Non-Colab environment detected.")


# --------------------------------------------------------------------------------------------------
# 3.2 Resolve project root
# --------------------------------------------------------------------------------------------------

if "PROJECT_ROOT" in globals():

    PROJECT_ROOT = Path(PROJECT_ROOT)

else:

    PROJECT_ROOT = DRIVE_MYDRIVE / "SPP_GAN_Research"


PROJECT_ROOT = PROJECT_ROOT.resolve()


# --------------------------------------------------------------------------------------------------
# 3.3 Resolve directory registry
# --------------------------------------------------------------------------------------------------

if "DIRECTORIES" not in globals():

    DIRECTORIES = {
        "data": PROJECT_ROOT / "data",
        "raw_data": PROJECT_ROOT / "data" / "raw",
        "processed_data": PROJECT_ROOT / "data" / "processed",

        "models": PROJECT_ROOT / "models",
        "tvae_models": PROJECT_ROOT / "models" / "tvae",
        "ctgan_models": PROJECT_ROOT / "models" / "ctgan",
        "dp_ctgan_models": PROJECT_ROOT / "models" / "dp_ctgan",
        "spp_gan_models": PROJECT_ROOT / "models" / "spp_gan",

        "synthetic_data": PROJECT_ROOT / "synthetic_data",
        "tvae_synthetic": PROJECT_ROOT / "synthetic_data" / "tvae",
        "ctgan_synthetic": PROJECT_ROOT / "synthetic_data" / "ctgan",
        "dp_ctgan_synthetic": PROJECT_ROOT / "synthetic_data" / "dp_ctgan",
        "spp_gan_synthetic": PROJECT_ROOT / "synthetic_data" / "spp_gan",

        "results": PROJECT_ROOT / "results",
        "statistical_results": PROJECT_ROOT / "results" / "statistical",
        "ml_results": PROJECT_ROOT / "results" / "machine_learning",
        "privacy_results": PROJECT_ROOT / "results" / "privacy",
        "utility_results": PROJECT_ROOT / "results" / "utility",
        "comparative_results": PROJECT_ROOT / "results" / "comparative",
        "raw_validation_results": PROJECT_ROOT / "results" / "raw_validation",

        "logs": PROJECT_ROOT / "logs",
        "checkpoints": PROJECT_ROOT / "checkpoints",
        "manifests": PROJECT_ROOT / "manifests",

        "figures": PROJECT_ROOT / "paper" / "figures",
        "tables": PROJECT_ROOT / "paper" / "tables",

        "config": PROJECT_ROOT / "config",
    }


DIRECTORIES = {
    key: Path(value)
    for key, value in DIRECTORIES.items()
}


# --------------------------------------------------------------------------------------------------
# 3.4 Configuration constants
# --------------------------------------------------------------------------------------------------

MASTER_SEED = 2025

print()
print(f"Project root : {PROJECT_ROOT}")
print(f"Master seed  : {MASTER_SEED}")

print()
print("✓ Notebook 00 configuration context resolved.")

3. LOAD NOTEBOOK 00 CONFIGURATION

Mounting Google Drive...
Mounted at /content/drive
✓ Google Drive verified.

Project root : /content/drive/MyDrive/SPP_GAN_Research
Master seed  : 2025

✓ Notebook 00 configuration context resolved.


In [4]:
# ==================================================================================================
# SECTION 4 — LOAD NOTEBOOK 01 VALIDATED REGISTRY
# ==================================================================================================

print("=" * 100)
print("4. LOAD NOTEBOOK 01 VALIDATED REGISTRY")
print("=" * 100)

from pathlib import Path
import pandas as pd
import json
import re


# --------------------------------------------------------------------------------------------------
# 4.1 VERIFY PROJECT ROOT
# --------------------------------------------------------------------------------------------------

if "PROJECT_ROOT" not in globals():
    raise RuntimeError(
        "PROJECT_ROOT is not available. "
        "Execute Section 3 — Load Notebook 00 Configuration first."
    )

PROJECT_ROOT = Path(PROJECT_ROOT)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_VALIDATION_DIR = PROJECT_ROOT / "results" / "raw_validation"

print(f"PROJECT_ROOT       : {PROJECT_ROOT}")
print(f"RAW_DATA_DIR       : {RAW_DATA_DIR}")
print(f"RAW_VALIDATION_DIR : {RAW_VALIDATION_DIR}")

if not PROJECT_ROOT.exists():
    raise RuntimeError(
        f"PROJECT_ROOT does not exist:\n{PROJECT_ROOT}"
    )

if not RAW_DATA_DIR.exists():
    raise RuntimeError(
        f"Raw data directory does not exist:\n{RAW_DATA_DIR}"
    )

if not RAW_VALIDATION_DIR.exists():
    raise RuntimeError(
        f"Notebook 01 validation directory does not exist:\n"
        f"{RAW_VALIDATION_DIR}"
    )

print("✓ Project root verified")
print("✓ Raw data directory verified")
print("✓ Notebook 01 validation directory verified")


# --------------------------------------------------------------------------------------------------
# 4.2 CANONICAL DATASET LIST
# --------------------------------------------------------------------------------------------------

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

print("\nRequired datasets:")

for dataset_id in DATASET_IDS:
    print(f"  - {dataset_id}")


# --------------------------------------------------------------------------------------------------
# 4.3 CANONICAL TARGET POLICY
# --------------------------------------------------------------------------------------------------

TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

print("\nTarget policy:")

for dataset_id in DATASET_IDS:
    print(
        f"  {dataset_id:20s} -> {TARGET_COLUMNS[dataset_id]}"
    )


# --------------------------------------------------------------------------------------------------
# 4.4 CANONICAL IDENTIFIER POLICY
# --------------------------------------------------------------------------------------------------

IDENTIFIER_COLUMNS = {
    "adult_income": [],
    "bank_marketing": [],
    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
    ],
}

print("\nIdentifier policy:")

for dataset_id in DATASET_IDS:
    print(
        f"  {dataset_id:20s} -> {IDENTIFIER_COLUMNS[dataset_id]}"
    )


# --------------------------------------------------------------------------------------------------
# 4.5 NOTEBOOK 01 VALIDATION ARTIFACTS
# --------------------------------------------------------------------------------------------------

STRUCTURAL_DIR = RAW_VALIDATION_DIR / "structural"
FEATURE_DIR = RAW_VALIDATION_DIR / "feature_inventory"
MISSINGNESS_DIR = RAW_VALIDATION_DIR / "missingness"

STRUCTURAL_REPORT_PATH = (
    STRUCTURAL_DIR / "structural_validation_report.csv"
)

DATASET_VALIDATION_REPORT_PATH = (
    STRUCTURAL_DIR / "dataset_validation_report.csv"
)

FEATURE_INVENTORY_PATH = (
    FEATURE_DIR / "feature_inventory.csv"
)

MISSINGNESS_REPORT_PATH = (
    MISSINGNESS_DIR / "raw_missingness_report.csv"
)

print("\nNotebook 01 artifacts:")

artifact_paths = {
    "Structural validation": STRUCTURAL_REPORT_PATH,
    "Dataset validation": DATASET_VALIDATION_REPORT_PATH,
    "Feature inventory": FEATURE_INVENTORY_PATH,
    "Missingness report": MISSINGNESS_REPORT_PATH,
}

for name, path in artifact_paths.items():

    if path.exists():
        print(f"  ✓ {name}: {path}")
    else:
        print(f"  - {name}: not found")


# --------------------------------------------------------------------------------------------------
# 4.6 LOAD AVAILABLE VALIDATION REPORTS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_01_REPORTS = {}

for name, path in artifact_paths.items():

    if path.exists():

        try:

            NOTEBOOK_01_REPORTS[name] = pd.read_csv(
                path,
                low_memory=False
            )

            print(
                f"✓ Loaded {name}: "
                f"{NOTEBOOK_01_REPORTS[name].shape}"
            )

        except Exception as exc:

            raise RuntimeError(
                f"Failed to load Notebook 01 artifact:\n"
                f"{path}\n"
                f"Error: {repr(exc)}"
            ) from exc


# --------------------------------------------------------------------------------------------------
# 4.7 DISCOVER RAW CSV FILES
# --------------------------------------------------------------------------------------------------

raw_csv_files = sorted(
    RAW_DATA_DIR.rglob("*.csv")
)

raw_csv_files = [
    path
    for path in raw_csv_files
    if path.is_file()
]

print(
    f"\nCSV files discovered under raw data directory: "
    f"{len(raw_csv_files)}"
)

for path in raw_csv_files:
    print(f"  - {path}")


# --------------------------------------------------------------------------------------------------
# 4.8 NORMALIZE FILENAMES
# --------------------------------------------------------------------------------------------------

def normalize_name(value):

    value = str(value).lower()

    value = re.sub(
        r"[^a-z0-9]+",
        "_",
        value
    )

    return value.strip("_")


normalized_raw_files = {
    path: normalize_name(path.stem)
    for path in raw_csv_files
}


# --------------------------------------------------------------------------------------------------
# 4.9 DATASET FILE PATTERNS
# --------------------------------------------------------------------------------------------------

DATASET_FILE_PATTERNS = {

    "adult_income": [
        "adult_income",
        "adult-income",
        "adult",
    ],

    "bank_marketing": [
        "bank_marketing",
        "bank-marketing",
        "bankmarketing",
        "bank",
    ],

    "diabetes_130us": [
        "diabetes_130us",
        "diabetes-130us",
        "diabetes",
    ],
}


# --------------------------------------------------------------------------------------------------
# 4.10 RESOLVE RAW DATASET PATHS
# --------------------------------------------------------------------------------------------------

RAW_DATASET_PATHS = {}

for dataset_id in DATASET_IDS:

    patterns = DATASET_FILE_PATTERNS[dataset_id]

    candidates = []

    for path, normalized_filename in normalized_raw_files.items():

        for pattern in patterns:

            normalized_pattern = normalize_name(pattern)

            if normalized_pattern in normalized_filename:

                candidates.append(path)
                break

    candidates = sorted(
        set(candidates)
    )

    print(
        f"\nCandidates for {dataset_id}:"
    )

    for candidate in candidates:
        print(f"  - {candidate}")

    if len(candidates) == 0:

        raise RuntimeError(
            f"No raw CSV file could be identified for "
            f"dataset '{dataset_id}'.\n"
            f"Raw data directory: {RAW_DATA_DIR}"
        )

    if len(candidates) == 1:

        selected_path = candidates[0]

    else:

        exact_matches = [
            path
            for path in candidates
            if normalize_name(path.stem)
            == normalize_name(dataset_id)
        ]

        if len(exact_matches) == 1:

            selected_path = exact_matches[0]

        else:

            raise RuntimeError(
                f"Multiple raw-file candidates were found for "
                f"dataset '{dataset_id}'.\n\n"
                + "\n".join(
                    f"  - {path}"
                    for path in candidates
                )
                + "\n\n"
                + "Notebook 02 will not guess between files."
            )

    RAW_DATASET_PATHS[dataset_id] = selected_path

    print(
        f"✓ Selected: {selected_path}"
    )


# --------------------------------------------------------------------------------------------------
# 4.11 LOAD RAW DATASETS WITH PARSING VALIDATION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("4.11 LOAD RAW DATASETS WITH PARSING VALIDATION")
print("=" * 100)


def load_validated_csv(
    file_path,
    dataset_id,
    expected_columns,
):
    """
    Load a CSV using controlled delimiter detection.

    The loader first attempts the standard comma separator.
    If the result contains only one column, alternative delimiters
    are tested.

    The resulting dataframe must contain the expected number of columns.
    """

    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"Dataset file does not exist:\n{file_path}"
        )

    expected_n_columns = len(expected_columns)

    # ----------------------------------------------------------------------------------------------
    # Candidate separators
    # ----------------------------------------------------------------------------------------------

    candidate_separators = [
        ",",
        ";",
        "\t",
        "|",
    ]

    successful_attempts = []

    # ----------------------------------------------------------------------------------------------
    # Try candidate separators
    # ----------------------------------------------------------------------------------------------

    for separator in candidate_separators:

        try:

            df_candidate = pd.read_csv(
                file_path,
                sep=separator,
                low_memory=False
            )

            n_columns = len(df_candidate.columns)

            successful_attempts.append({
                "separator": separator,
                "columns": n_columns,
                "dataframe": df_candidate,
            })

        except Exception as exc:

            print(
                f"  Separator {repr(separator)} failed: "
                f"{repr(exc)}"
            )


    # ----------------------------------------------------------------------------------------------
    # Prefer exact expected column count
    # ----------------------------------------------------------------------------------------------

    exact_matches = [
        attempt
        for attempt in successful_attempts
        if attempt["columns"] == expected_n_columns
    ]


    if len(exact_matches) == 1:

        selected_attempt = exact_matches[0]

    elif len(exact_matches) > 1:

        # If multiple delimiters produce the same expected number
        # of columns, prefer comma, then semicolon.
        separator_priority = {
            ",": 0,
            ";": 1,
            "\t": 2,
            "|": 3,
        }

        exact_matches = sorted(
            exact_matches,
            key=lambda x: separator_priority.get(
                x["separator"],
                99
            )
        )

        selected_attempt = exact_matches[0]

    else:

        # No separator produced the expected dimensionality.
        diagnostics = "\n".join(
            [
                f"    separator={repr(a['separator'])} "
                f"-> {a['columns']} columns"
                for a in successful_attempts
            ]
        )

        raise RuntimeError(
            f"Could not correctly parse dataset '{dataset_id}'.\n\n"
            f"Expected columns : {expected_n_columns}\n"
            f"Expected names   : {expected_columns}\n\n"
            f"Parsing attempts:\n"
            f"{diagnostics}\n\n"
            f"File:\n{file_path}"
        )


    # ----------------------------------------------------------------------------------------------
    # Extract selected dataframe
    # ----------------------------------------------------------------------------------------------

    df = selected_attempt["dataframe"]
    selected_separator = selected_attempt["separator"]


    # ----------------------------------------------------------------------------------------------
    # Normalize column names
    # ----------------------------------------------------------------------------------------------

    df.columns = [
        str(column).strip()
        for column in df.columns
    ]


    # ----------------------------------------------------------------------------------------------
    # Validate exact column count
    # ----------------------------------------------------------------------------------------------

    if len(df.columns) != expected_n_columns:

        raise RuntimeError(
            f"Column-count validation failed for '{dataset_id}'.\n"
            f"Expected : {expected_n_columns}\n"
            f"Actual   : {len(df.columns)}\n"
            f"Separator: {repr(selected_separator)}"
        )


    # ----------------------------------------------------------------------------------------------
    # Validate expected columns when available
    # ----------------------------------------------------------------------------------------------

    missing_expected_columns = [
        column
        for column in expected_columns
        if column not in df.columns
    ]

    if missing_expected_columns:

        raise RuntimeError(
            f"Expected column names are missing for '{dataset_id}'.\n"
            f"Missing : {missing_expected_columns}\n"
            f"Actual  : {list(df.columns)}\n"
            f"Separator used: {repr(selected_separator)}"
        )


    # ----------------------------------------------------------------------------------------------
    # Return validated dataframe and parser information
    # ----------------------------------------------------------------------------------------------

    parser_info = {
        "separator": selected_separator,
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
    }

    return df, parser_info


# ==================================================================================================
# EXPECTED COLUMN DEFINITIONS
# ==================================================================================================

EXPECTED_COLUMNS = {

    "adult_income": [
        "age",
        "workclass",
        "fnlwgt",
        "education",
        "education_num",
        "marital_status",
        "occupation",
        "relationship",
        "race",
        "sex",
        "capital_gain",
        "capital_loss",
        "hours_per_week",
        "native_country",
        "income",
    ],

    "bank_marketing": [
        "age",
        "job",
        "marital",
        "education",
        "default",
        "balance",
        "housing",
        "loan",
        "contact",
        "day",
        "month",
        "duration",
        "campaign",
        "pdays",
        "previous",
        "poutcome",
        "y",
    ],

    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
        "race",
        "gender",
        "age",
        "weight",
        "admission_type_id",
        "discharge_disposition_id",
        "admission_source_id",
        "time_in_hospital",
        "payer_code",
        "medical_specialty",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "diag_1",
        "diag_2",
        "diag_3",
        "number_diagnoses",
        "max_glu_serum",
        "A1Cresult",
        "metformin",
        "repaglinide",
        "nateglinide",
        "chlorpropamide",
        "glimepiride",
        "acetohexamide",
        "glipizide",
        "glyburide",
        "tolbutamide",
        "pioglitazone",
        "rosiglitazone",
        "acarbose",
        "miglitol",
        "troglitazone",
        "tolazamide",
        "examide",
        "citoglipton",
        "insulin",
        "glyburide-metformin",
        "glipizide-metformin",
        "glimepiride-pioglitazone",
        "metformin-rosiglitazone",
        "metformin-pioglitazone",
        "change",
        "diabetesMed",
        "readmitted",
    ],
}


# ==================================================================================================
# LOAD ALL DATASETS
# ==================================================================================================

RAW_DATASETS = {}
DATASET_PARSING_INFO = {}

for dataset_id in DATASET_IDS:

    raw_path = RAW_DATASET_PATHS[dataset_id]

    print("\n" + "-" * 100)
    print(f"Dataset : {dataset_id}")
    print(f"File    : {raw_path}")

    df, parser_info = load_validated_csv(
        file_path=raw_path,
        dataset_id=dataset_id,
        expected_columns=EXPECTED_COLUMNS[dataset_id],
    )

    RAW_DATASETS[dataset_id] = df
    DATASET_PARSING_INFO[dataset_id] = parser_info

    print(
        f"✓ Separator detected : "
        f"{repr(parser_info['separator'])}"
    )

    print(
        f"✓ Shape              : "
        f"{parser_info['rows']:,} × "
        f"{parser_info['columns']}"
    )

    print(
        f"✓ Column schema       : validated"
    )


# ==================================================================================================
# FINAL PARSING SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("DATASET PARSING SUMMARY")
print("=" * 100)

for dataset_id in DATASET_IDS:

    info = DATASET_PARSING_INFO[dataset_id]

    print(
        f"{dataset_id:20s} | "
        f"separator={repr(info['separator']):6s} | "
        f"{info['rows']:>8,} rows × "
        f"{info['columns']:>2} columns"
    )

print("\n✓ All datasets loaded with validated parsing")


# --------------------------------------------------------------------------------------------------
# 4.12 EXPECTED DATASET SHAPES
# --------------------------------------------------------------------------------------------------

EXPECTED_SHAPES = {

    "adult_income": (
        48842,
        15,
    ),

    "bank_marketing": (
        45211,
        17,
    ),

    "diabetes_130us": (
        101766,
        50,
    ),
}


# --------------------------------------------------------------------------------------------------
# 4.13 VALIDATE DATASET SHAPES
# --------------------------------------------------------------------------------------------------

print("\nDataset shape validation:")

for dataset_id in DATASET_IDS:

    actual_shape = RAW_DATASETS[dataset_id].shape
    expected_shape = EXPECTED_SHAPES[dataset_id]

    if actual_shape != expected_shape:

        raise RuntimeError(
            f"Dataset shape mismatch.\n"
            f"Dataset  : {dataset_id}\n"
            f"Expected : {expected_shape}\n"
            f"Actual   : {actual_shape}\n\n"
            "Check the raw file and CSV parsing configuration "
            "before continuing."
        )

    print(
        f"  ✓ {dataset_id:20s} "
        f"{actual_shape[0]:>8,} × "
        f"{actual_shape[1]:>3}"
    )


# --------------------------------------------------------------------------------------------------
# 4.14 VALIDATE TARGET COLUMNS
# --------------------------------------------------------------------------------------------------

print("\nTarget-column validation:")

for dataset_id in DATASET_IDS:

    df = RAW_DATASETS[dataset_id]

    target_column = TARGET_COLUMNS[dataset_id]

    if target_column not in df.columns:

        raise RuntimeError(
            f"Target column not found.\n"
            f"Dataset : {dataset_id}\n"
            f"Target  : {target_column}\n"
            f"Columns : {list(df.columns)}"
        )

    unique_values = df[target_column].nunique(
        dropna=False
    )

    print(
        f"  ✓ {dataset_id:20s} "
        f"target={target_column} "
        f"| unique={unique_values}"
    )


# --------------------------------------------------------------------------------------------------
# 4.15 VALIDATE IDENTIFIER COLUMNS
# --------------------------------------------------------------------------------------------------

print("\nIdentifier-column validation:")

for dataset_id in DATASET_IDS:

    df = RAW_DATASETS[dataset_id]

    identifiers = IDENTIFIER_COLUMNS[dataset_id]

    missing_identifiers = [
        column
        for column in identifiers
        if column not in df.columns
    ]

    if missing_identifiers:

        raise RuntimeError(
            f"Identifier column(s) not found.\n"
            f"Dataset : {dataset_id}\n"
            f"Missing : {missing_identifiers}"
        )

    print(
        f"  ✓ {dataset_id:20s} "
        f"identifiers={identifiers}"
    )


# --------------------------------------------------------------------------------------------------
# 4.16 CONSTRUCT CANONICAL VALIDATED REGISTRY
# --------------------------------------------------------------------------------------------------

registry_records = []

for dataset_id in DATASET_IDS:

    df = RAW_DATASETS[dataset_id]

    registry_records.append({

        "dataset_id": dataset_id,

        "raw_path": str(
            RAW_DATASET_PATHS[dataset_id]
        ),

        "rows": int(
            df.shape[0]
        ),

        "columns": int(
            df.shape[1]
        ),

        "target_column": TARGET_COLUMNS[dataset_id],

        "identifier_columns": json.dumps(
            IDENTIFIER_COLUMNS[dataset_id]
        ),

        "validated_by_notebook_01": True,

        "reconstructed_by_notebook_02": True,
    })


CANONICAL_DATASET_REGISTRY_DF = pd.DataFrame(
    registry_records
)


# --------------------------------------------------------------------------------------------------
# 4.17 SAVE RECONSTRUCTED REGISTRY
# --------------------------------------------------------------------------------------------------

RECONSTRUCTED_REGISTRY_PATH = (
    RAW_VALIDATION_DIR
    / "validated_dataset_registry.csv"
)

CANONICAL_DATASET_REGISTRY_DF.to_csv(
    RECONSTRUCTED_REGISTRY_PATH,
    index=False
)

print(
    "\n✓ Reconstructed validated registry saved:"
)

print(
    f"  {RECONSTRUCTED_REGISTRY_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 4.18 RELOAD REGISTRY
# --------------------------------------------------------------------------------------------------

if not RECONSTRUCTED_REGISTRY_PATH.exists():

    raise RuntimeError(
        "The reconstructed validated registry was not saved successfully."
    )

RELOADED_REGISTRY = pd.read_csv(
    RECONSTRUCTED_REGISTRY_PATH
)

required_columns = [
    "dataset_id",
    "raw_path",
    "rows",
    "columns",
    "target_column",
    "identifier_columns",
]

missing_columns = [
    column
    for column in required_columns
    if column not in RELOADED_REGISTRY.columns
]

if missing_columns:

    raise RuntimeError(
        "Reconstructed registry is missing required columns:\n"
        f"{missing_columns}"
    )

print(
    "✓ Reconstructed registry reloaded successfully"
)

print(
    f"✓ Registry rows: {len(RELOADED_REGISTRY)}"
)


# --------------------------------------------------------------------------------------------------
# 4.19 FINAL RUNTIME VERIFICATION
# --------------------------------------------------------------------------------------------------

required_objects = [
    "DATASET_IDS",
    "RAW_DATASETS",
    "RAW_DATASET_PATHS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
    "CANONICAL_DATASET_REGISTRY_DF",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Required Section 4 runtime objects are missing:\n"
        f"{missing_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 4.20 SECTION SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 4 VERIFICATION")
print("=" * 100)

print(
    f"Dataset count       : {len(DATASET_IDS)}"
)

print(
    f"Raw datasets loaded : {len(RAW_DATASETS)}"
)

print(
    f"Registry rows        : "
    f"{len(CANONICAL_DATASET_REGISTRY_DF)}"
)

print("\nValidated datasets:")

for dataset_id in DATASET_IDS:

    df = RAW_DATASETS[dataset_id]

    print(
        f"  {dataset_id:20s} | "
        f"{len(df):>8,} rows | "
        f"{len(df.columns):>3} columns | "
        f"target={TARGET_COLUMNS[dataset_id]}"
    )

print("\n" + "=" * 100)
print("✓ SECTION 4 — LOAD NOTEBOOK 01 VALIDATED REGISTRY : PASS")
print("=" * 100)

4. LOAD NOTEBOOK 01 VALIDATED REGISTRY
PROJECT_ROOT       : /content/drive/MyDrive/SPP_GAN_Research
RAW_DATA_DIR       : /content/drive/MyDrive/SPP_GAN_Research/data/raw
RAW_VALIDATION_DIR : /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation
✓ Project root verified
✓ Raw data directory verified
✓ Notebook 01 validation directory verified

Required datasets:
  - adult_income
  - bank_marketing
  - diabetes_130us

Target policy:
  adult_income         -> income
  bank_marketing       -> y
  diabetes_130us       -> readmitted

Identifier policy:
  adult_income         -> []
  bank_marketing       -> []
  diabetes_130us       -> ['encounter_id', 'patient_nbr']

Notebook 01 artifacts:
  ✓ Structural validation: /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/structural/structural_validation_report.csv
  ✓ Dataset validation: /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/structural/dataset_validation_report.csv
  ✓ Feature inventory: /content/drive

In [5]:
# ==================================================================================================
# 5. CREATE NOTEBOOK 02 DIRECTORIES
# ==================================================================================================

print("=" * 100)
print("5. CREATE NOTEBOOK 02 DIRECTORIES")
print("=" * 100)


NB02_DIRECTORIES = {

    "processed_root":
        PROJECT_ROOT / "data" / "processed" / "notebook_02",

    "native":
        PROJECT_ROOT / "data" / "processed" / "notebook_02" / "native",

    "encoded":
        PROJECT_ROOT / "data" / "processed" / "notebook_02" / "encoded",

    "splits":
        PROJECT_ROOT / "data" / "processed" / "notebook_02" / "splits",

    "preprocessors":
        PROJECT_ROOT / "data" / "processed" / "notebook_02" / "preprocessors",

    "schemas":
        PROJECT_ROOT / "data" / "processed" / "notebook_02" / "schemas",

    "feature_mapping":
        PROJECT_ROOT / "data" / "processed" / "notebook_02" / "feature_mapping",

    "manifests":
        PROJECT_ROOT / "results" / "raw_validation" / "notebook_02_manifests",
}


for directory in NB02_DIRECTORIES.values():

    Path(directory).mkdir(
        parents=True,
        exist_ok=True
    )


print()

for name, directory in NB02_DIRECTORIES.items():

    print(
        f"✓ {name:<20} → {directory}"
    )


print()
print("✓ Notebook 02 directories ready.")

5. CREATE NOTEBOOK 02 DIRECTORIES

✓ processed_root       → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ native               → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native
✓ encoded              → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/encoded
✓ splits               → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/splits
✓ preprocessors        → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors
✓ schemas              → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas
✓ feature_mapping      → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/feature_mapping
✓ manifests            → /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests

✓ Notebook 02 directories ready.


In [6]:
# ==================================================================================================
# 6. LOAD SPLIT CONFIGURATION
# ==================================================================================================

print("=" * 100)
print("6. LOAD SPLIT CONFIGURATION")
print("=" * 100)


SPLIT_CONFIG = {

    "train_size": 0.70,
    "validation_size": 0.15,
    "test_size": 0.15,

    "random_state": MASTER_SEED,

    "stratify": True,

    "shuffle": True,
}


# --------------------------------------------------------------------------------------------------
# Validate split configuration
# --------------------------------------------------------------------------------------------------

split_total = (
    SPLIT_CONFIG["train_size"]
    + SPLIT_CONFIG["validation_size"]
    + SPLIT_CONFIG["test_size"]
)


if not np.isclose(split_total, 1.0):

    raise ValueError(
        f"Train/validation/test proportions must sum to 1.0. "
        f"Current value: {split_total}"
    )


if SPLIT_CONFIG["train_size"] <= 0:
    raise ValueError("Training proportion must be > 0.")


if SPLIT_CONFIG["validation_size"] <= 0:
    raise ValueError("Validation proportion must be > 0.")


if SPLIT_CONFIG["test_size"] <= 0:
    raise ValueError("Test proportion must be > 0.")


print()
print(f"Train fraction      : {SPLIT_CONFIG['train_size']:.2f}")
print(f"Validation fraction : {SPLIT_CONFIG['validation_size']:.2f}")
print(f"Test fraction       : {SPLIT_CONFIG['test_size']:.2f}")
print(f"Random seed         : {SPLIT_CONFIG['random_state']}")
print(f"Stratification      : {SPLIT_CONFIG['stratify']}")
print(f"Shuffle             : {SPLIT_CONFIG['shuffle']}")

print()
print("✓ Split configuration validated.")

6. LOAD SPLIT CONFIGURATION

Train fraction      : 0.70
Validation fraction : 0.15
Test fraction       : 0.15
Random seed         : 2025
Stratification      : True
Shuffle             : True

✓ Split configuration validated.


In [7]:
# ==================================================================================================
# 7. DEFINE MISSING-VALUE POLICY
# ==================================================================================================

print("=" * 100)
print("7. DEFINE MISSING-VALUE POLICY")
print("=" * 100)


MISSING_VALUE_POLICY = {

    # Values that are universally interpreted as missing.
    "global_missing_markers": [
        "",
        "NA",
        "N/A",
        "na",
        "n/a",
        "NaN",
        "nan",
        "NULL",
        "null",
        "None",
        "none",
    ],

    # Dataset-specific missing markers.
    #
    # Adult Income uses '?' as an unknown/missing marker.
    # Diabetes 130-US uses '?' extensively for missing categorical fields.
    #
    # Bank Marketing's 'unknown' is retained as an observed category.
    "dataset_specific_markers": {

        "adult_income": [
            "?"
        ],

        "bank_marketing": [
            # Deliberately empty.
            # 'unknown' is treated as an observed category.
        ],

        "diabetes_130us": [
            "?"
        ],
    },

    # Missing-value policy after normalization.
    "numeric_imputation": "median",

    "categorical_imputation": "most_frequent",

    "unknown_categories": "preserve",

    "target_missing_policy": "fail",
}


print()
print("Global missing markers:")
print(
    MISSING_VALUE_POLICY["global_missing_markers"]
)

print()
print("Dataset-specific markers:")

for dataset_id, markers in (
    MISSING_VALUE_POLICY["dataset_specific_markers"]
).items():

    print(
        f"  {dataset_id:<20}: {markers}"
    )

print()
print(
    "✓ Missing-value policy defined."
)
print(
    "✓ Legitimate 'unknown' categories are preserved."
)

7. DEFINE MISSING-VALUE POLICY

Global missing markers:
['', 'NA', 'N/A', 'na', 'n/a', 'NaN', 'nan', 'NULL', 'null', 'None', 'none']

Dataset-specific markers:
  adult_income        : ['?']
  bank_marketing      : []
  diabetes_130us      : ['?']

✓ Missing-value policy defined.
✓ Legitimate 'unknown' categories are preserved.


In [8]:
# ==================================================================================================
# 8. DEFINE IDENTIFIER / TARGET POLICY
# ==================================================================================================

print("=" * 100)
print("8. DEFINE IDENTIFIER / TARGET POLICY")
print("=" * 100)


IDENTIFIER_POLICY = {

    # Explicit research identifiers.
    #
    # These are excluded from modeling/preprocessing features because they
    # represent record-level identity rather than useful generative structure.
    "explicit_identifiers": {

        "adult_income": [],

        "bank_marketing": [],

        "diabetes_130us": [
            "encounter_id",
            "patient_nbr",
        ],
    },

    # Target variables are retained as analytical variables.
    "retain_target": True,

    # Targets are used for stratification.
    "stratify_on_target": True,

    # Create an immutable internal row ID before splitting.
    "create_original_row_id": True,

    # Original row ID must never become a model feature.
    "exclude_original_row_id_from_model": True,

    # Do not automatically remove heuristic identifier candidates.
    "heuristic_identifiers_are_flags_only": True,
}


# --------------------------------------------------------------------------------------------------
# Validate explicit identifier columns
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    df = RAW_DATASETS[dataset_id]

    for column in IDENTIFIER_POLICY["explicit_identifiers"].get(
        dataset_id,
        []
    ):

        if column not in df.columns:

            raise ValueError(
                f"Configured identifier '{column}' not found in "
                f"dataset '{dataset_id}'."
            )


print()
for dataset_id in DATASET_IDS:

    print(
        f"{dataset_id:<20} "
        f"Target = {TARGET_COLUMNS[dataset_id]!r} | "
        f"Excluded identifiers = "
        f"{IDENTIFIER_POLICY['explicit_identifiers'].get(dataset_id, [])}"
    )


print()
print("✓ Identifier and target policy validated.")

8. DEFINE IDENTIFIER / TARGET POLICY

adult_income         Target = 'income' | Excluded identifiers = []
bank_marketing       Target = 'y' | Excluded identifiers = []
diabetes_130us       Target = 'readmitted' | Excluded identifiers = ['encounter_id', 'patient_nbr']

✓ Identifier and target policy validated.


In [9]:
# ==================================================================================================
# 9. NORMALIZE MISSING VALUES
# ==================================================================================================

print("=" * 100)
print("9. NORMALIZE MISSING VALUES")
print("=" * 100)


NORMALIZED_DATASETS = {}

MISSING_NORMALIZATION_REPORT = []


for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    source_df = RAW_DATASETS[dataset_id]

    # Copy so RAW_DATASETS remains untouched.
    df = source_df.copy(deep=True)

    markers = set(
        MISSING_VALUE_POLICY["global_missing_markers"]
        + MISSING_VALUE_POLICY["dataset_specific_markers"].get(
            dataset_id,
            []
        )
    )

    before_missing = int(
        df.isna().sum().sum()
    )

    # ----------------------------------------------------------------------------------------------
    # Normalize object/string columns only.
    # ----------------------------------------------------------------------------------------------

    for column in df.columns:

        if (
            pd.api.types.is_object_dtype(df[column])
            or pd.api.types.is_string_dtype(df[column])
        ):

            # Strip surrounding whitespace without converting legitimate
            # internal content.
            df[column] = df[column].map(
                lambda x: x.strip()
                if isinstance(x, str)
                else x
            )

            df[column] = df[column].replace(
                list(markers),
                np.nan
            )

    after_missing = int(
        df.isna().sum().sum()
    )

    newly_detected = after_missing - before_missing

    NORMALIZED_DATASETS[dataset_id] = df

    MISSING_NORMALIZATION_REPORT.append(
        {
            "dataset_id": dataset_id,
            "rows": len(df),
            "columns": len(df.columns),
            "missing_before": before_missing,
            "missing_after": after_missing,
            "newly_normalized_missing": newly_detected,
        }
    )

    print(
        f"Missing before : {before_missing:,}"
    )

    print(
        f"Missing after  : {after_missing:,}"
    )

    print(
        f"Newly normalized : {newly_detected:,}"
    )


MISSING_NORMALIZATION_DF = pd.DataFrame(
    MISSING_NORMALIZATION_REPORT
)


print()
print("✓ Missing-value markers normalized.")
print("✓ No imputation performed.")
print("✓ Original RAW_DATASETS remain unchanged.")

9. NORMALIZE MISSING VALUES

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Missing before : 6,465
Missing after  : 6,465
Newly normalized : 0

----------------------------------------------------------------------------------------------------
Dataset: bank_marketing
----------------------------------------------------------------------------------------------------
Missing before : 0
Missing after  : 0
Newly normalized : 0

----------------------------------------------------------------------------------------------------
Dataset: diabetes_130us
----------------------------------------------------------------------------------------------------
Missing before : 181,168
Missing after  : 374,017
Newly normalized : 192,849

✓ Missing-value markers normalized.
✓ No imputation performed.
✓ Original RAW_DATASETS re

In [10]:
# ==============================================================================
# SECTION 10 — VALIDATE FEATURE TYPES
# ==============================================================================

print("=" * 100)
print("10. VALIDATE FEATURE TYPES")
print("=" * 100)

import pandas as pd
import numpy as np


# ==============================================================================
# 10.1 — REQUIRED OBJECT VALIDATION
# ==============================================================================

REQUIRED_OBJECTS = [
    "RAW_DATASETS",
    "DATASET_IDS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
]

_missing_objects = [
    obj
    for obj in REQUIRED_OBJECTS
    if obj not in globals()
]

if _missing_objects:
    raise RuntimeError(
        "Required Notebook 02 objects are missing:\n"
        f"{_missing_objects}\n\n"
        "Please execute the preceding Notebook 02 sections first."
    )

print("[PASS] Required Notebook 02 objects are available.")


# ==============================================================================
# 10.2 — INITIALIZE FEATURE-TYPE CONTAINERS
# ==============================================================================

FEATURE_TYPE_REPORTS = {}
FEATURE_TYPE_SUMMARY = {}

print("[PASS] Feature-type validation containers initialized.")


# ==============================================================================
# 10.3 — DATASET-WISE FEATURE-TYPE VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    # --------------------------------------------------------------------------
    # Load Dataset
    # --------------------------------------------------------------------------

    if dataset_id not in RAW_DATASETS:
        raise RuntimeError(
            f"[FAIL] RAW_DATASETS does not contain '{dataset_id}'."
        )

    df = RAW_DATASETS[dataset_id]

    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            f"[FAIL] RAW_DATASETS['{dataset_id}'] is not a pandas DataFrame."
        )

    # --------------------------------------------------------------------------
    # Retrieve Target and Identifier Configuration
    # --------------------------------------------------------------------------

    target_column = TARGET_COLUMNS.get(dataset_id)

    if target_column is None:
        raise RuntimeError(
            f"[FAIL] Target column is not defined for dataset '{dataset_id}'."
        )

    identifier_columns = IDENTIFIER_COLUMNS.get(dataset_id, [])

    # Ensure identifier configuration is a list
    if identifier_columns is None:
        identifier_columns = []

    identifier_columns = list(identifier_columns)

    # --------------------------------------------------------------------------
    # Validate Target Column
    # --------------------------------------------------------------------------

    if target_column not in df.columns:

        raise RuntimeError(
            f"[FAIL] Target column '{target_column}' "
            f"not found in dataset '{dataset_id}'.\n"
            f"Available columns:\n{list(df.columns)}"
        )

    # --------------------------------------------------------------------------
    # Validate Identifier Columns
    # --------------------------------------------------------------------------

    missing_identifiers = [
        column
        for column in identifier_columns
        if column not in df.columns
    ]

    if missing_identifiers:

        raise RuntimeError(
            f"[FAIL] Identifier columns missing from "
            f"'{dataset_id}': {missing_identifiers}"
        )

    # --------------------------------------------------------------------------
    # Detect Duplicate Column Names
    # --------------------------------------------------------------------------

    duplicate_columns = df.columns[
        df.columns.duplicated()
    ].tolist()

    if duplicate_columns:

        raise RuntimeError(
            f"[FAIL] Duplicate column names detected in "
            f"'{dataset_id}': {duplicate_columns}"
        )

    # --------------------------------------------------------------------------
    # Build Feature-Type Records
    # --------------------------------------------------------------------------

    records = []

    for column in df.columns:

        series = df[column]

        # ----------------------------------------------------------------------
        # Feature Role
        # ----------------------------------------------------------------------

        is_target = column == target_column

        is_identifier = column in identifier_columns

        # ----------------------------------------------------------------------
        # Raw Pandas Data Type
        # ----------------------------------------------------------------------

        pandas_dtype = str(series.dtype)

        # ----------------------------------------------------------------------
        # Semantic Feature Type
        # ----------------------------------------------------------------------
        #
        # Priority:
        #
        # 1. Identifier
        # 2. Target
        # 3. Numeric
        # 4. Categorical
        #
        # The target is intentionally treated as categorical because the
        # synthetic-data pipeline must preserve the joint feature-target
        # distribution and the targets in these datasets are categorical.
        #
        # Identifiers are retained only for audit purposes and excluded from
        # model-training / synthetic-data generation.
        # ----------------------------------------------------------------------

        if is_identifier:

            semantic_type = "identifier"

        elif is_target:

            semantic_type = "categorical"

        elif pd.api.types.is_bool_dtype(series):

            semantic_type = "categorical"

        elif pd.api.types.is_numeric_dtype(series):

            semantic_type = "numeric"

        else:

            semantic_type = "categorical"

        # ----------------------------------------------------------------------
        # Cardinality
        # ----------------------------------------------------------------------

        n_unique = int(
            series.nunique(dropna=True)
        )

        # ----------------------------------------------------------------------
        # Missingness
        # ----------------------------------------------------------------------

        n_missing = int(
            series.isna().sum()
        )

        missing_rate = (
            float(n_missing / len(series))
            if len(series) > 0
            else np.nan
        )

        # ----------------------------------------------------------------------
        # Record
        # ----------------------------------------------------------------------

        records.append({
            "dataset_id": dataset_id,
            "column": column,
            "pandas_dtype": pandas_dtype,
            "semantic_type": semantic_type,
            "is_target": bool(is_target),
            "is_identifier": bool(is_identifier),
            "n_unique": n_unique,
            "n_missing": n_missing,
            "missing_rate": missing_rate,
        })

    # ==============================================================================
    # 10.4 — CREATE DATASET FEATURE-TYPE REPORT
    # ==============================================================================

    feature_report = pd.DataFrame(records)

    FEATURE_TYPE_REPORTS[dataset_id] = feature_report

    # ==============================================================================
    # 10.5 — DERIVE FEATURE COUNTS
    # ==============================================================================

    # Modeling features exclude identifiers.
    modeling_features = feature_report[
        ~feature_report["is_identifier"].astype(bool)
    ].copy()

    numeric_features = modeling_features[
        modeling_features["semantic_type"] == "numeric"
    ].copy()

    categorical_features = modeling_features[
        modeling_features["semantic_type"] == "categorical"
    ].copy()

    identifier_features = feature_report[
        feature_report["is_identifier"].astype(bool)
    ].copy()

    target_features = feature_report[
        feature_report["is_target"].astype(bool)
    ].copy()

    # ==============================================================================
    # 10.6 — CREATE DATASET SUMMARY
    # ==============================================================================

    target_series = df[target_column]

    FEATURE_TYPE_SUMMARY[dataset_id] = {

        "rows":
            int(len(df)),

        "total_columns":
            int(len(df.columns)),

        "modeling_columns":
            int(len(modeling_features)),

        "numeric_columns":
            int(len(numeric_features)),

        "categorical_columns":
            int(len(categorical_features)),

        "identifier_columns":
            int(len(identifier_features)),

        "target_column":
            target_column,

        "target_dtype":
            str(target_series.dtype),

        "target_unique_values":
            int(target_series.nunique(dropna=True)),

        "target_missing_values":
            int(target_series.isna().sum()),

        "target_missing_rate":
            float(
                target_series.isna().mean()
            ),
    }

    # ==============================================================================
    # 10.7 — DISPLAY DATASET SUMMARY
    # ==============================================================================

    print(
        f"Rows                    : {len(df):,}"
    )

    print(
        f"Total columns           : {len(df.columns)}"
    )

    print(
        f"Modeling columns        : {len(modeling_features)}"
    )

    print(
        f"Numeric features        : {len(numeric_features)}"
    )

    print(
        f"Categorical features    : {len(categorical_features)}"
    )

    print(
        f"Identifier columns      : {len(identifier_features)}"
    )

    print(
        f"Target                  : {target_column}"
    )

    print(
        f"Target unique values    : "
        f"{target_series.nunique(dropna=True):,}"
    )

    print(
        f"Target missing values   : "
        f"{target_series.isna().sum():,}"
    )

    # ==============================================================================
    # 10.8 — VALIDATION GATES
    # ==============================================================================

    # --------------------------------------------------------------------------
    # Gate 1 — Number of records
    # --------------------------------------------------------------------------

    assert len(feature_report) == len(df.columns), (
        f"Feature report column count mismatch for {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 2 — Every column must be represented
    # --------------------------------------------------------------------------

    assert set(feature_report["column"]) == set(df.columns), (
        f"Feature report does not contain exactly the columns "
        f"of {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 3 — Target must exist
    # --------------------------------------------------------------------------

    assert target_column in feature_report["column"].values, (
        f"Target column '{target_column}' missing from feature report."
    )

    # --------------------------------------------------------------------------
    # Gate 4 — Exactly one target
    # --------------------------------------------------------------------------

    assert int(
        feature_report["is_target"].sum()
    ) == 1, (
        f"Expected exactly one target for {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 5 — Target must be categorical
    # --------------------------------------------------------------------------

    target_row = feature_report[
        feature_report["column"] == target_column
    ].iloc[0]

    assert target_row["semantic_type"] == "categorical", (
        f"Target '{target_column}' must be classified as categorical."
    )

    # --------------------------------------------------------------------------
    # Gate 6 — Identifier count
    # --------------------------------------------------------------------------

    assert int(
        feature_report["is_identifier"].sum()
    ) == len(identifier_columns), (
        f"Identifier count mismatch for {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 7 — Identifier classification
    # --------------------------------------------------------------------------

    for identifier in identifier_columns:

        identifier_row = feature_report[
            feature_report["column"] == identifier
        ].iloc[0]

        # IMPORTANT:
        # Use bool(...) rather than `is True` because Pandas may return
        # numpy.bool_ instead of the native Python bool object.
        assert bool(
            identifier_row["is_identifier"]
        ) is True, (
            f"Column '{identifier}' was not marked as an identifier."
        )

        assert identifier_row["semantic_type"] == "identifier", (
            f"Column '{identifier}' must have semantic type 'identifier'."
        )

    # --------------------------------------------------------------------------
    # Gate 8 — Identifiers must not be modeling features
    # --------------------------------------------------------------------------

    modeling_identifier_count = int(
        modeling_features["is_identifier"].astype(bool).sum()
    )

    assert modeling_identifier_count == 0, (
        f"Identifier leakage detected in modeling features "
        f"for {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 9 — Every column must have a semantic type
    # --------------------------------------------------------------------------

    assert feature_report["semantic_type"].notna().all(), (
        f"Missing semantic feature type detected in {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 10 — Only allowed semantic types
    # --------------------------------------------------------------------------

    allowed_semantic_types = {
        "numeric",
        "categorical",
        "identifier",
    }

    observed_semantic_types = set(
        feature_report["semantic_type"].dropna().unique()
    )

    assert observed_semantic_types.issubset(
        allowed_semantic_types
    ), (
        f"Unexpected semantic feature types in {dataset_id}: "
        f"{observed_semantic_types - allowed_semantic_types}"
    )

    # --------------------------------------------------------------------------
    # Gate 11 — Modeling feature accounting
    # --------------------------------------------------------------------------

    expected_modeling_columns = (
        len(df.columns) - len(identifier_columns)
    )

    assert len(modeling_features) == expected_modeling_columns, (
        f"Modeling feature count mismatch for {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 12 — Numeric + categorical accounting
    # --------------------------------------------------------------------------

    assert (
        len(numeric_features) +
        len(categorical_features)
        ==
        len(modeling_features)
    ), (
        f"Numeric/categorical feature accounting mismatch "
        f"for {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 13 — No column can simultaneously be target and identifier
    # --------------------------------------------------------------------------

    overlapping_roles = feature_report[
        feature_report["is_target"].astype(bool)
        &
        feature_report["is_identifier"].astype(bool)
    ]

    assert len(overlapping_roles) == 0, (
        f"Target/identifier role overlap detected in {dataset_id}: "
        f"{overlapping_roles['column'].tolist()}"
    )

    # --------------------------------------------------------------------------
    # Dataset validation passed
    # --------------------------------------------------------------------------

    print(
        "\n[PASS] Feature-type validation completed."
    )


# ==============================================================================
# 10.9 — COMBINE ALL FEATURE-TYPE REPORTS
# ==============================================================================

FEATURE_TYPE_DF = pd.concat(
    FEATURE_TYPE_REPORTS.values(),
    ignore_index=True
)


# ==============================================================================
# 10.10 — CREATE FEATURE-TYPE SUMMARY DATAFRAME
# ==============================================================================

FEATURE_TYPE_SUMMARY_DF = (
    pd.DataFrame.from_dict(
        FEATURE_TYPE_SUMMARY,
        orient="index"
    )
    .reset_index()
    .rename(
        columns={"index": "dataset_id"}
    )
)


# ==============================================================================
# 10.11 — GLOBAL VALIDATION
# ==============================================================================

print("\n" + "=" * 100)
print("GLOBAL FEATURE-TYPE VALIDATION")
print("=" * 100)

# --------------------------------------------------------------------------
# Expected datasets
# --------------------------------------------------------------------------

assert set(FEATURE_TYPE_REPORTS.keys()) == set(DATASET_IDS), (
    "Feature-type reports do not cover exactly the configured datasets."
)

# --------------------------------------------------------------------------
# Expected total feature records
# --------------------------------------------------------------------------

expected_total_columns = sum(
    len(RAW_DATASETS[dataset_id].columns)
    for dataset_id in DATASET_IDS
)

assert len(FEATURE_TYPE_DF) == expected_total_columns, (
    "Combined feature-type report has an unexpected number of rows."
)

# --------------------------------------------------------------------------
# Validate dataset IDs
# --------------------------------------------------------------------------

assert set(
    FEATURE_TYPE_DF["dataset_id"].unique()
) == set(DATASET_IDS), (
    "Combined feature-type report contains unexpected dataset IDs."
)

# --------------------------------------------------------------------------
# Validate unique dataset-column combinations
# --------------------------------------------------------------------------

duplicate_feature_records = FEATURE_TYPE_DF[
    FEATURE_TYPE_DF.duplicated(
        subset=["dataset_id", "column"],
        keep=False
    )
]

assert duplicate_feature_records.empty, (
    "Duplicate dataset-column records detected in FEATURE_TYPE_DF."
)

print(
    "[PASS] Global feature-type validation passed."
)


# ==============================================================================
# 10.12 — DISPLAY FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 10 SUMMARY")
print("=" * 100)

display(
    FEATURE_TYPE_SUMMARY_DF[
        [
            "dataset_id",
            "rows",
            "total_columns",
            "modeling_columns",
            "numeric_columns",
            "categorical_columns",
            "identifier_columns",
            "target_column",
            "target_dtype",
            "target_unique_values",
            "target_missing_values",
        ]
    ]
)


# ==============================================================================
# 10.13 — SEMANTIC TYPE DISTRIBUTION
# ==============================================================================

print("\n" + "-" * 100)
print("SEMANTIC FEATURE-TYPE DISTRIBUTION")
print("-" * 100)

semantic_distribution = (
    FEATURE_TYPE_DF
    .groupby(
        ["dataset_id", "semantic_type"]
    )
    .size()
    .unstack(fill_value=0)
)

display(semantic_distribution)


# ==============================================================================
# 10.14 — TARGET VALIDATION SUMMARY
# ==============================================================================

print("\n" + "-" * 100)
print("TARGET VALIDATION SUMMARY")
print("-" * 100)

target_validation_summary = (
    FEATURE_TYPE_DF[
        FEATURE_TYPE_DF["is_target"].astype(bool)
    ][
        [
            "dataset_id",
            "column",
            "pandas_dtype",
            "semantic_type",
            "n_unique",
            "n_missing",
            "missing_rate",
        ]
    ]
    .reset_index(drop=True)
)

display(target_validation_summary)


# ==============================================================================
# 10.15 — IDENTIFIER VALIDATION SUMMARY
# ==============================================================================

print("\n" + "-" * 100)
print("IDENTIFIER VALIDATION SUMMARY")
print("-" * 100)

identifier_validation_summary = (
    FEATURE_TYPE_DF[
        FEATURE_TYPE_DF["is_identifier"].astype(bool)
    ][
        [
            "dataset_id",
            "column",
            "pandas_dtype",
            "semantic_type",
            "n_unique",
            "n_missing",
            "missing_rate",
        ]
    ]
    .reset_index(drop=True)
)

if identifier_validation_summary.empty:

    print("No identifier columns are configured.")

else:

    display(identifier_validation_summary)


# ==============================================================================
# 10.16 — FINAL SECTION STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 10 COMPLETE")
print("=" * 100)

print(
    f"Datasets validated       : {len(DATASET_IDS)}"
)

print(
    f"Total feature records    : {len(FEATURE_TYPE_DF):,}"
)

print(
    f"Modeling feature records : "
    f"{int((~FEATURE_TYPE_DF['is_identifier'].astype(bool)).sum()):,}"
)

print(
    f"Identifier records       : "
    f"{int(FEATURE_TYPE_DF['is_identifier'].astype(bool).sum()):,}"
)

print(
    f"Target records           : "
    f"{int(FEATURE_TYPE_DF['is_target'].astype(bool).sum()):,}"
)

print("\n[PASS] SECTION 10 — VALIDATE FEATURE TYPES")
print("[PASS] No identifier leakage detected.")
print("[PASS] Target columns validated.")
print("[PASS] Numeric/categorical feature accounting validated.")
print("[PASS] Global feature-type validation passed.")

10. VALIDATE FEATURE TYPES
[PASS] Required Notebook 02 objects are available.
[PASS] Feature-type validation containers initialized.

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Rows                    : 48,842
Total columns           : 15
Modeling columns        : 15
Numeric features        : 6
Categorical features    : 9
Identifier columns      : 0
Target                  : income
Target unique values    : 2
Target missing values   : 0

[PASS] Feature-type validation completed.

----------------------------------------------------------------------------------------------------
Dataset: bank_marketing
----------------------------------------------------------------------------------------------------
Rows                    : 45,211
Total columns           : 17
Modeling columns        : 17
Numeric features 

,dataset_id,rows,total_columns,modeling_columns,numeric_columns,categorical_columns,identifier_columns,target_column,target_dtype,target_unique_values,target_missing_values
0,adult_income,48842,15,15,6,9,0,income,object,2,0
1,bank_marketing,45211,17,17,7,10,0,y,object,2,0
2,diabetes_130us,101766,50,48,11,37,2,readmitted,object,3,0



----------------------------------------------------------------------------------------------------
SEMANTIC FEATURE-TYPE DISTRIBUTION
----------------------------------------------------------------------------------------------------


semantic_type,categorical,identifier,numeric
dataset_id,,,
adult_income,9,0,6
bank_marketing,10,0,7
diabetes_130us,37,2,11



----------------------------------------------------------------------------------------------------
TARGET VALIDATION SUMMARY
----------------------------------------------------------------------------------------------------


,dataset_id,column,pandas_dtype,semantic_type,n_unique,n_missing,missing_rate
0,adult_income,income,object,categorical,2,0,0.0
1,bank_marketing,y,object,categorical,2,0,0.0
2,diabetes_130us,readmitted,object,categorical,3,0,0.0



----------------------------------------------------------------------------------------------------
IDENTIFIER VALIDATION SUMMARY
----------------------------------------------------------------------------------------------------


,dataset_id,column,pandas_dtype,semantic_type,n_unique,n_missing,missing_rate
0,diabetes_130us,encounter_id,int64,identifier,101766,0,0.0
1,diabetes_130us,patient_nbr,int64,identifier,71518,0,0.0



SECTION 10 COMPLETE
Datasets validated       : 3
Total feature records    : 82
Modeling feature records : 80
Identifier records       : 2
Target records           : 3

[PASS] SECTION 10 — VALIDATE FEATURE TYPES
[PASS] No identifier leakage detected.
[PASS] Target columns validated.
[PASS] Numeric/categorical feature accounting validated.
[PASS] Global feature-type validation passed.


In [11]:
# ==================================================================================================
# 11. VALIDATE TARGET
# ==================================================================================================

print("=" * 100)
print("11. VALIDATE TARGET")
print("=" * 100)


TARGET_VALIDATION_REPORT = []


for dataset_id in DATASET_IDS:

    df = NORMALIZED_DATASETS[dataset_id]

    target = TARGET_COLUMNS[dataset_id]

    # ----------------------------------------------------------------------------------------------
    # Existence
    # ----------------------------------------------------------------------------------------------

    if target not in df.columns:

        raise ValueError(
            f"Target '{target}' does not exist in '{dataset_id}'."
        )


    target_series = df[target]


    # ----------------------------------------------------------------------------------------------
    # Missing target
    # ----------------------------------------------------------------------------------------------

    missing_count = int(
        target_series.isna().sum()
    )

    if missing_count > 0:

        if (
            MISSING_VALUE_POLICY["target_missing_policy"]
            == "fail"
        ):

            raise ValueError(
                f"Target '{target}' in '{dataset_id}' contains "
                f"{missing_count:,} missing values."
            )


    # ----------------------------------------------------------------------------------------------
    # Unique values
    # ----------------------------------------------------------------------------------------------

    unique_values = (
        target_series
        .dropna()
        .unique()
        .tolist()
    )


    if len(unique_values) < 2:

        raise ValueError(
            f"Target '{target}' in '{dataset_id}' has fewer than "
            f"two observed classes."
        )


    TARGET_VALIDATION_REPORT.append(
        {
            "dataset_id": dataset_id,
            "target_column": target,
            "dtype": str(target_series.dtype),
            "n_rows": len(df),
            "missing_count": missing_count,
            "missing_rate": float(
                missing_count / len(df)
            ),
            "n_unique": len(unique_values),
            "target_classes": json.dumps(
                [str(x) for x in unique_values]
            ),
            "validation_status": "PASS",
        }
    )

    print()
    print(
        f"✓ {dataset_id:<20} "
        f"Target = {target!r} | "
        f"Classes = {len(unique_values)} | "
        f"Missing = {missing_count:,}"
    )


TARGET_VALIDATION_DF = pd.DataFrame(
    TARGET_VALIDATION_REPORT
)


print()
print("✓ All targets validated successfully.")

11. VALIDATE TARGET

✓ adult_income         Target = 'income' | Classes = 2 | Missing = 0

✓ bank_marketing       Target = 'y' | Classes = 2 | Missing = 0

✓ diabetes_130us       Target = 'readmitted' | Classes = 3 | Missing = 0

✓ All targets validated successfully.


In [12]:
# ==================================================================================================
# 12. CREATE ORIGINAL ROW IDS
# ==================================================================================================

print("=" * 100)
print("12. CREATE ORIGINAL ROW IDS")
print("=" * 100)


DATASETS_WITH_ROW_IDS = {}


for dataset_id in DATASET_IDS:

    df = NORMALIZED_DATASETS[dataset_id].copy(
        deep=True
    )

    # Preserve original row order explicitly.
    original_row_ids = np.arange(
        len(df),
        dtype=np.int64
    )

    # Internal provenance identifier.
    df.insert(
        0,
        "__original_row_id__",
        original_row_ids
    )

    DATASETS_WITH_ROW_IDS[dataset_id] = df

    print(
        f"✓ {dataset_id:<20} "
        f"{len(df):,} original row IDs created."
    )


print()
print("✓ Original row IDs created.")
print("✓ Original row IDs are provenance metadata only.")
print("✓ Original row IDs will not be used as model features.")

12. CREATE ORIGINAL ROW IDS
✓ adult_income         48,842 original row IDs created.
✓ bank_marketing       45,211 original row IDs created.
✓ diabetes_130us       101,766 original row IDs created.

✓ Original row IDs created.
✓ Original row IDs are provenance metadata only.
✓ Original row IDs will not be used as model features.


In [13]:
# ==================================================================================================
# 13. STRATIFIED TRAIN / VALIDATION / TEST SPLIT
# ==================================================================================================

print("=" * 100)
print("13. STRATIFIED TRAIN / VALIDATION / TEST SPLIT")
print("=" * 100)

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split


# ==================================================================================================
# 13.1 REQUIRED OBJECT VALIDATION
# ==================================================================================================

required_objects = [
    "DATASET_IDS",
    "RAW_DATASETS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 13 cannot proceed.\n"
        f"Missing required objects: {missing_objects}"
    )

print("✓ Required objects detected.")


# ==================================================================================================
# 13.2 SPLIT CONFIGURATION
# ==================================================================================================

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

if not np.isclose(
    TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION,
    1.0
):
    raise RuntimeError(
        "Train/validation/test fractions must sum to 1.0."
    )

# Use the canonical project seed.
if "RANDOM_SEED" in globals():
    SPLIT_RANDOM_SEED = int(RANDOM_SEED)
elif "MASTER_SEED" in globals():
    SPLIT_RANDOM_SEED = int(MASTER_SEED)
else:
    SPLIT_RANDOM_SEED = 2025

print(f"✓ Train fraction      : {TRAIN_FRACTION:.2f}")
print(f"✓ Validation fraction : {VALIDATION_FRACTION:.2f}")
print(f"✓ Test fraction       : {TEST_FRACTION:.2f}")
print(f"✓ Random seed         : {SPLIT_RANDOM_SEED}")


# ==================================================================================================
# 13.3 CANONICAL PROVENANCE COLUMN
# ==================================================================================================

PROVENANCE_COLUMN = "__original_row_id__"

print(f"✓ Provenance column   : {PROVENANCE_COLUMN}")


# ==================================================================================================
# 13.4 INITIALIZE OUTPUT CONTAINERS
# ==================================================================================================

TRAIN_DATASETS = {}
VALIDATION_DATASETS = {}
TEST_DATASETS = {}

SPLIT_MANIFESTS = {}
SPLIT_SUMMARY = []

print("✓ Split containers initialized.")


# ==================================================================================================
# 13.5 CREATE STRATIFIED SPLITS
# ==================================================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    # ----------------------------------------------------------------------------------------------
    # Load raw dataset
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in RAW_DATASETS:
        raise RuntimeError(
            f"{dataset_id}: RAW_DATASETS entry not found."
        )

    df = RAW_DATASETS[dataset_id].copy()

    target = TARGET_COLUMNS[dataset_id]
    identifier_columns = IDENTIFIER_COLUMNS.get(dataset_id, [])

    print(f"Original rows            : {len(df):,}")
    print(f"Original columns         : {len(df.columns):,}")
    print(f"Target                   : {target}")
    print(f"Identifier columns       : {identifier_columns}")


    # ----------------------------------------------------------------------------------------------
    # Validate target
    # ----------------------------------------------------------------------------------------------

    if target not in df.columns:
        raise RuntimeError(
            f"{dataset_id}: target column '{target}' not found."
        )

    if df[target].isna().any():
        raise RuntimeError(
            f"{dataset_id}: target contains missing values. "
            "Stratified splitting cannot proceed safely."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate identifiers
    # ----------------------------------------------------------------------------------------------

    missing_identifiers = [
        col for col in identifier_columns
        if col not in df.columns
    ]

    if missing_identifiers:
        raise RuntimeError(
            f"{dataset_id}: configured identifier columns not found: "
            f"{missing_identifiers}"
        )


    # ----------------------------------------------------------------------------------------------
    # Create deterministic original-row provenance ID
    #
    # IMPORTANT:
    # This is created BEFORE splitting so every observation retains its
    # exact relationship with the original raw dataset.
    # ----------------------------------------------------------------------------------------------

    if PROVENANCE_COLUMN in df.columns:
        raise RuntimeError(
            f"{dataset_id}: provenance column '{PROVENANCE_COLUMN}' "
            "already exists in the raw dataset."
        )

    df[PROVENANCE_COLUMN] = np.arange(
        len(df),
        dtype=np.int64
    )

    if not df[PROVENANCE_COLUMN].is_unique:
        raise RuntimeError(
            f"{dataset_id}: generated provenance IDs are not unique."
        )

    if df[PROVENANCE_COLUMN].isna().any():
        raise RuntimeError(
            f"{dataset_id}: generated provenance IDs contain missing values."
        )


    # ----------------------------------------------------------------------------------------------
    # First split:
    # 70% train
    # 30% temporary
    # ----------------------------------------------------------------------------------------------

    train_df, temp_df = train_test_split(
        df,
        test_size=(VALIDATION_FRACTION + TEST_FRACTION),
        random_state=SPLIT_RANDOM_SEED,
        stratify=df[target],
    )


    # ----------------------------------------------------------------------------------------------
    # Second split:
    # Split temporary 50/50 into validation and test.
    #
    # Because validation and test are both 15% of the original data,
    # each receives 50% of the temporary 30%.
    # ----------------------------------------------------------------------------------------------

    validation_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=SPLIT_RANDOM_SEED,
        stratify=temp_df[target],
    )


    # ----------------------------------------------------------------------------------------------
    # Reset indexes only.
    #
    # The original row provenance remains untouched.
    # ----------------------------------------------------------------------------------------------

    train_df = train_df.reset_index(drop=True)
    validation_df = validation_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)


    # ----------------------------------------------------------------------------------------------
    # Validate provenance
    # ----------------------------------------------------------------------------------------------

    for split_name, split_df in [
        ("train", train_df),
        ("validation", validation_df),
        ("test", test_df),
    ]:

        if PROVENANCE_COLUMN not in split_df.columns:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                f"provenance column '{PROVENANCE_COLUMN}' "
                "was not preserved."
            )

        if split_df[PROVENANCE_COLUMN].isna().any():
            raise RuntimeError(
                f"{dataset_id} | {split_name}: provenance contains NaN."
            )

        if not split_df[PROVENANCE_COLUMN].is_unique:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: provenance IDs "
                "are not unique."
            )


    # ----------------------------------------------------------------------------------------------
    # Validate split disjointness
    # ----------------------------------------------------------------------------------------------

    train_ids = set(train_df[PROVENANCE_COLUMN].tolist())
    validation_ids = set(validation_df[PROVENANCE_COLUMN].tolist())
    test_ids = set(test_df[PROVENANCE_COLUMN].tolist())

    if train_ids.intersection(validation_ids):
        raise RuntimeError(
            f"{dataset_id}: train and validation provenance overlap."
        )

    if train_ids.intersection(test_ids):
        raise RuntimeError(
            f"{dataset_id}: train and test provenance overlap."
        )

    if validation_ids.intersection(test_ids):
        raise RuntimeError(
            f"{dataset_id}: validation and test provenance overlap."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate complete coverage
    # ----------------------------------------------------------------------------------------------

    combined_ids = (
        train_ids
        | validation_ids
        | test_ids
    )

    expected_ids = set(
        df[PROVENANCE_COLUMN].tolist()
    )

    if combined_ids != expected_ids:
        missing_ids = expected_ids - combined_ids
        extra_ids = combined_ids - expected_ids

        raise RuntimeError(
            f"{dataset_id}: split provenance coverage mismatch.\n"
            f"Missing IDs: {len(missing_ids)}\n"
            f"Extra IDs  : {len(extra_ids)}"
        )


    # ----------------------------------------------------------------------------------------------
    # Validate row counts
    # ----------------------------------------------------------------------------------------------

    total_rows = len(df)

    if (
        len(train_df)
        + len(validation_df)
        + len(test_df)
        != total_rows
    ):
        raise RuntimeError(
            f"{dataset_id}: split row counts do not sum to original rows."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate target distribution
    # ----------------------------------------------------------------------------------------------

    train_target_distribution = (
        train_df[target]
        .value_counts(normalize=True, dropna=False)
        .sort_index()
    )

    validation_target_distribution = (
        validation_df[target]
        .value_counts(normalize=True, dropna=False)
        .sort_index()
    )

    test_target_distribution = (
        test_df[target]
        .value_counts(normalize=True, dropna=False)
        .sort_index()
    )

    all_target_classes = sorted(
        set(train_target_distribution.index)
        | set(validation_target_distribution.index)
        | set(test_target_distribution.index)
    )

    for class_value in all_target_classes:

        train_pct = float(
            train_target_distribution.get(class_value, 0.0)
        )

        validation_pct = float(
            validation_target_distribution.get(class_value, 0.0)
        )

        test_pct = float(
            test_target_distribution.get(class_value, 0.0)
        )

        if (
            train_pct == 0.0
            or validation_pct == 0.0
            or test_pct == 0.0
        ):
            raise RuntimeError(
                f"{dataset_id}: target class '{class_value}' "
                "is absent from at least one split."
            )


    # ----------------------------------------------------------------------------------------------
    # Store canonical split datasets
    # ----------------------------------------------------------------------------------------------

    TRAIN_DATASETS[dataset_id] = train_df
    VALIDATION_DATASETS[dataset_id] = validation_df
    TEST_DATASETS[dataset_id] = test_df


    # ----------------------------------------------------------------------------------------------
    # Create split manifest
    # ----------------------------------------------------------------------------------------------

    split_manifest = pd.concat(
        [
            train_df[[PROVENANCE_COLUMN]].assign(
                split="train"
            ),
            validation_df[[PROVENANCE_COLUMN]].assign(
                split="validation"
            ),
            test_df[[PROVENANCE_COLUMN]].assign(
                split="test"
            ),
        ],
        ignore_index=True,
    )

    if len(split_manifest) != total_rows:
        raise RuntimeError(
            f"{dataset_id}: split manifest row count mismatch."
        )

    if not split_manifest[PROVENANCE_COLUMN].is_unique:
        raise RuntimeError(
            f"{dataset_id}: split manifest provenance is not unique."
        )

    SPLIT_MANIFESTS[dataset_id] = split_manifest


    # ----------------------------------------------------------------------------------------------
    # Summary record
    # ----------------------------------------------------------------------------------------------

    SPLIT_SUMMARY.append(
        {
            "dataset_id": dataset_id,
            "original_rows": total_rows,
            "original_columns": len(df.columns) - 1,
            "train_rows": len(train_df),
            "validation_rows": len(validation_df),
            "test_rows": len(test_df),
            "train_fraction": len(train_df) / total_rows,
            "validation_fraction": len(validation_df) / total_rows,
            "test_fraction": len(test_df) / total_rows,
            "target_column": target,
            "identifier_count": len(identifier_columns),
            "provenance_column": PROVENANCE_COLUMN,
            "provenance_unique": True,
            "split_disjoint": True,
            "complete_coverage": True,
        }
    )


    # ----------------------------------------------------------------------------------------------
    # Display
    # ----------------------------------------------------------------------------------------------

    print(f"Train rows               : {len(train_df):,}")
    print(f"Validation rows          : {len(validation_df):,}")
    print(f"Test rows                : {len(test_df):,}")

    print(
        f"Train fraction           : "
        f"{len(train_df) / total_rows:.6f}"
    )

    print(
        f"Validation fraction      : "
        f"{len(validation_df) / total_rows:.6f}"
    )

    print(
        f"Test fraction            : "
        f"{len(test_df) / total_rows:.6f}"
    )

    print(
        f"Provenance column        : "
        f"{PROVENANCE_COLUMN}"
    )

    print(
        f"Provenance coverage      : "
        f"{len(combined_ids):,}/{total_rows:,}"
    )

    print("✓ Split provenance       : PASS")
    print("✓ Split disjointness     : PASS")
    print("✓ Complete coverage      : PASS")
    print("✓ Target stratification  : PASS")


# ==================================================================================================
# 13.6 SUMMARY DATAFRAME
# ==================================================================================================

SPLIT_SUMMARY_DF = pd.DataFrame(SPLIT_SUMMARY)


# ==================================================================================================
# 13.7 GLOBAL VALIDATION
# ==================================================================================================

if set(TRAIN_DATASETS.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "TRAIN_DATASETS dataset coverage mismatch."
    )

if set(VALIDATION_DATASETS.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "VALIDATION_DATASETS dataset coverage mismatch."
    )

if set(TEST_DATASETS.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "TEST_DATASETS dataset coverage mismatch."
    )

if set(SPLIT_MANIFESTS.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "SPLIT_MANIFESTS dataset coverage mismatch."
    )


# ==================================================================================================
# 13.8 FINAL OUTPUT
# ==================================================================================================

print("\n" + "=" * 100)
print("SECTION 13 COMPLETE")
print("=" * 100)

print("\nSplit Summary:")
display(SPLIT_SUMMARY_DF)

print("\nCanonical split objects created:")
print("  ✓ TRAIN_DATASETS")
print("  ✓ VALIDATION_DATASETS")
print("  ✓ TEST_DATASETS")
print("  ✓ SPLIT_MANIFESTS")
print("  ✓ SPLIT_SUMMARY_DF")

print("\nProvenance policy:")
print(f"  ✓ Canonical column: {PROVENANCE_COLUMN}")
print("  ✓ Created before splitting")
print("  ✓ Unique within each dataset")
print("  ✓ Train/validation/test mutually exclusive")
print("  ✓ Complete original-row coverage")

print("\nSTATUS: PASS")

13. STRATIFIED TRAIN / VALIDATION / TEST SPLIT
✓ Required objects detected.
✓ Train fraction      : 0.70
✓ Validation fraction : 0.15
✓ Test fraction       : 0.15
✓ Random seed         : 2025
✓ Provenance column   : __original_row_id__
✓ Split containers initialized.

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Original rows            : 48,842
Original columns         : 15
Target                   : income
Identifier columns       : []
Train rows               : 34,189
Validation rows          : 7,326
Test rows                : 7,327
Train fraction           : 0.699992
Validation fraction      : 0.149994
Test fraction            : 0.150014
Provenance column        : __original_row_id__
Provenance coverage      : 48,842/48,842
✓ Split provenance       : PASS
✓ Split disjointness     : PASS
✓ Complete coverage

,dataset_id,original_rows,original_columns,train_rows,validation_rows,test_rows,train_fraction,validation_fraction,test_fraction,target_column,identifier_count,provenance_column,provenance_unique,split_disjoint,complete_coverage
0,adult_income,48842,15,34189,7326,7327,0.699992,0.149994,0.150014,income,0,__original_row_id__,True,True,True
1,bank_marketing,45211,17,31647,6782,6782,0.699985,0.150008,0.150008,y,0,__original_row_id__,True,True,True
2,diabetes_130us,101766,50,71236,15265,15265,0.699998,0.150001,0.150001,readmitted,2,__original_row_id__,True,True,True



Canonical split objects created:
  ✓ TRAIN_DATASETS
  ✓ VALIDATION_DATASETS
  ✓ TEST_DATASETS
  ✓ SPLIT_MANIFESTS
  ✓ SPLIT_SUMMARY_DF

Provenance policy:
  ✓ Canonical column: __original_row_id__
  ✓ Created before splitting
  ✓ Unique within each dataset
  ✓ Train/validation/test mutually exclusive
  ✓ Complete original-row coverage

STATUS: PASS


In [15]:
# ==================================================================================================
# 14. VERIFY SPLIT INTEGRITY
# ==================================================================================================

print("=" * 100)
print("14. VERIFY SPLIT INTEGRITY")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Required canonical objects
# --------------------------------------------------------------------------------------------------

REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "DATASETS_WITH_ROW_IDS",
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
]

missing_objects = [
    obj for obj in REQUIRED_OBJECTS
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required canonical Notebook 02 objects are missing:\n"
        + "\n".join(f"  - {obj}" for obj in missing_objects)
    )


# --------------------------------------------------------------------------------------------------
# Initialize results
# --------------------------------------------------------------------------------------------------

SPLIT_INTEGRITY_RESULTS = []


# --------------------------------------------------------------------------------------------------
# Dataset-by-dataset integrity verification
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    # ----------------------------------------------------------------------------------------------
    # Load canonical datasets
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in DATASETS_WITH_ROW_IDS:
        raise RuntimeError(
            f"{dataset_id}: missing from DATASETS_WITH_ROW_IDS."
        )

    if dataset_id not in TRAIN_DATASETS:
        raise RuntimeError(
            f"{dataset_id}: missing from TRAIN_DATASETS."
        )

    if dataset_id not in VALIDATION_DATASETS:
        raise RuntimeError(
            f"{dataset_id}: missing from VALIDATION_DATASETS."
        )

    if dataset_id not in TEST_DATASETS:
        raise RuntimeError(
            f"{dataset_id}: missing from TEST_DATASETS."
        )


    df = DATASETS_WITH_ROW_IDS[dataset_id]

    train_df = TRAIN_DATASETS[dataset_id]
    validation_df = VALIDATION_DATASETS[dataset_id]
    test_df = TEST_DATASETS[dataset_id]


    # ----------------------------------------------------------------------------------------------
    # Validate provenance column presence
    # ----------------------------------------------------------------------------------------------

    PROVENANCE_COLUMN = "__original_row_id__"

    required_frames = {
        "original": df,
        "train": train_df,
        "validation": validation_df,
        "test": test_df,
    }

    missing_provenance = []

    for frame_name, frame in required_frames.items():

        if PROVENANCE_COLUMN not in frame.columns:
            missing_provenance.append(frame_name)

    if missing_provenance:

        raise RuntimeError(
            f"{dataset_id}: provenance column "
            f"'{PROVENANCE_COLUMN}' missing from "
            f"{', '.join(missing_provenance)} dataset(s)."
        )


    # ----------------------------------------------------------------------------------------------
    # Extract authoritative provenance IDs
    # ----------------------------------------------------------------------------------------------

    original_ids = set(
        df[PROVENANCE_COLUMN]
    )

    train_ids = set(
        train_df[PROVENANCE_COLUMN]
    )

    validation_ids = set(
        validation_df[PROVENANCE_COLUMN]
    )

    test_ids = set(
        test_df[PROVENANCE_COLUMN]
    )


    # ----------------------------------------------------------------------------------------------
    # Check provenance uniqueness within each split
    # ----------------------------------------------------------------------------------------------

    original_provenance_unique = (
        df[PROVENANCE_COLUMN].is_unique
    )

    train_provenance_unique = (
        train_df[PROVENANCE_COLUMN].is_unique
    )

    validation_provenance_unique = (
        validation_df[PROVENANCE_COLUMN].is_unique
    )

    test_provenance_unique = (
        test_df[PROVENANCE_COLUMN].is_unique
    )

    provenance_unique = (
        original_provenance_unique
        and train_provenance_unique
        and validation_provenance_unique
        and test_provenance_unique
    )


    # ----------------------------------------------------------------------------------------------
    # No overlap between splits
    # ----------------------------------------------------------------------------------------------

    train_validation_overlap = (
        train_ids & validation_ids
    )

    train_test_overlap = (
        train_ids & test_ids
    )

    validation_test_overlap = (
        validation_ids & test_ids
    )


    no_overlap = (
        len(train_validation_overlap) == 0
        and len(train_test_overlap) == 0
        and len(validation_test_overlap) == 0
    )


    # ----------------------------------------------------------------------------------------------
    # Complete coverage
    # ----------------------------------------------------------------------------------------------

    combined_ids = (
        train_ids
        | validation_ids
        | test_ids
    )

    complete_coverage = (
        combined_ids == original_ids
    )


    # ----------------------------------------------------------------------------------------------
    # Row-count preservation
    # ----------------------------------------------------------------------------------------------

    row_count_correct = (
        len(train_df)
        + len(validation_df)
        + len(test_df)
        == len(df)
    )


    # ----------------------------------------------------------------------------------------------
    # Provenance count consistency
    # ----------------------------------------------------------------------------------------------

    provenance_count_correct = (
        len(original_ids) == len(df)
        and len(train_ids) == len(train_df)
        and len(validation_ids) == len(validation_df)
        and len(test_ids) == len(test_df)
    )


    # ----------------------------------------------------------------------------------------------
    # Final dataset integrity status
    # ----------------------------------------------------------------------------------------------

    integrity_pass = (
        provenance_unique
        and no_overlap
        and complete_coverage
        and row_count_correct
        and provenance_count_correct
    )


    # ----------------------------------------------------------------------------------------------
    # Store results
    # ----------------------------------------------------------------------------------------------

    SPLIT_INTEGRITY_RESULTS.append(
        {
            "dataset_id": dataset_id,

            "original_rows": len(df),
            "train_rows": len(train_df),
            "validation_rows": len(validation_df),
            "test_rows": len(test_df),

            "original_provenance_unique":
                original_provenance_unique,

            "train_provenance_unique":
                train_provenance_unique,

            "validation_provenance_unique":
                validation_provenance_unique,

            "test_provenance_unique":
                test_provenance_unique,

            "provenance_unique":
                provenance_unique,

            "train_validation_overlap":
                len(train_validation_overlap),

            "train_test_overlap":
                len(train_test_overlap),

            "validation_test_overlap":
                len(validation_test_overlap),

            "no_overlap":
                no_overlap,

            "combined_unique_provenance_ids":
                len(combined_ids),

            "original_unique_provenance_ids":
                len(original_ids),

            "complete_coverage":
                complete_coverage,

            "row_count_correct":
                row_count_correct,

            "provenance_count_correct":
                provenance_count_correct,

            "integrity_status":
                (
                    "PASS"
                    if integrity_pass
                    else "FAIL"
                ),
        }
    )


    # ----------------------------------------------------------------------------------------------
    # Console reporting
    # ----------------------------------------------------------------------------------------------

    print()
    print(
        f"{dataset_id:<20} "
        f"Integrity = "
        f"{'PASS' if integrity_pass else 'FAIL'}"
    )

    print(
        f"  Original rows            : {len(df)}"
    )

    print(
        f"  Train rows               : {len(train_df)}"
    )

    print(
        f"  Validation rows          : {len(validation_df)}"
    )

    print(
        f"  Test rows                : {len(test_df)}"
    )

    print(
        f"  Provenance unique        : "
        f"{provenance_unique}"
    )

    print(
        f"  Train/Validation overlap : "
        f"{len(train_validation_overlap)}"
    )

    print(
        f"  Train/Test overlap       : "
        f"{len(train_test_overlap)}"
    )

    print(
        f"  Validation/Test overlap  : "
        f"{len(validation_test_overlap)}"
    )

    print(
        f"  Complete coverage       : "
        f"{complete_coverage}"
    )

    print(
        f"  Row count correct       : "
        f"{row_count_correct}"
    )


# --------------------------------------------------------------------------------------------------
# Create canonical integrity DataFrame
# --------------------------------------------------------------------------------------------------

SPLIT_INTEGRITY_DF = pd.DataFrame(
    SPLIT_INTEGRITY_RESULTS
)


# --------------------------------------------------------------------------------------------------
# Global integrity gate
# --------------------------------------------------------------------------------------------------

if SPLIT_INTEGRITY_DF.empty:
    raise RuntimeError(
        "SPLIT_INTEGRITY_DF is empty. "
        "No split integrity results were generated."
    )


if not (
    SPLIT_INTEGRITY_DF["integrity_status"] == "PASS"
).all():

    failed_datasets = SPLIT_INTEGRITY_DF.loc[
        SPLIT_INTEGRITY_DF["integrity_status"] != "PASS",
        "dataset_id"
    ].tolist()

    raise RuntimeError(
        "One or more dataset splits failed integrity validation.\n"
        f"Failed datasets: {failed_datasets}"
    )


# --------------------------------------------------------------------------------------------------
# Final PASS
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("✓ ALL TRAIN / VALIDATION / TEST SPLITS PASSED INTEGRITY VALIDATION")
print("=" * 100)
print()
print(
    f"Datasets verified : {len(SPLIT_INTEGRITY_DF)}"
)

print(
    f"Total original rows verified : "
    f"{SPLIT_INTEGRITY_DF['original_rows'].sum():,}"
)

print()
print("Canonical provenance column:")
print(f"  {PROVENANCE_COLUMN}")

print()
print("Integrity checks enforced:")
print("  ✓ Provenance uniqueness")
print("  ✓ Train/validation disjointness")
print("  ✓ Train/test disjointness")
print("  ✓ Validation/test disjointness")
print("  ✓ Complete provenance coverage")
print("  ✓ Row-count preservation")
print("  ✓ Provenance-count consistency")
print("=" * 100)

14. VERIFY SPLIT INTEGRITY

adult_income         Integrity = PASS
  Original rows            : 48842
  Train rows               : 34189
  Validation rows          : 7326
  Test rows                : 7327
  Provenance unique        : True
  Train/Validation overlap : 0
  Train/Test overlap       : 0
  Validation/Test overlap  : 0
  Complete coverage       : True
  Row count correct       : True

bank_marketing       Integrity = PASS
  Original rows            : 45211
  Train rows               : 31647
  Validation rows          : 6782
  Test rows                : 6782
  Provenance unique        : True
  Train/Validation overlap : 0
  Train/Test overlap       : 0
  Validation/Test overlap  : 0
  Complete coverage       : True
  Row count correct       : True

diabetes_130us       Integrity = PASS
  Original rows            : 101766
  Train rows               : 71236
  Validation rows          : 15265
  Test rows                : 15265
  Provenance unique        : True
  Train/Validation 

In [40]:
# ==================================================================================================
# 15. PREPARE TRAINING PREPROCESSING DATA
# ==================================================================================================

print("=" * 100)
print("15. PREPARE TRAINING PREPROCESSING DATA")
print("=" * 100)

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------------------
# Required Objects
# --------------------------------------------------------------------------------------------------

_REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_DATASETS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
]

for _obj in _REQUIRED_OBJECTS:
    if _obj not in globals():
        raise RuntimeError(
            f"Required object '{_obj}' not found. "
            "Execute the preceding Notebook 02 sections first."
        )

PROVENANCE_COLUMN = "__original_row_id__"

# --------------------------------------------------------------------------------------------------
# Reset Section Outputs
# --------------------------------------------------------------------------------------------------

TRAIN_PREPROCESSING_DATA = {}
TRAIN_PREPROCESSING_COLUMNS = {}
TRAIN_PREPROCESSING_METADATA = {}
TRAIN_PREPROCESSING_SUMMARY = []

# --------------------------------------------------------------------------------------------------
# Dataset-Level Preparation
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("-" * 100)
    print(f"Dataset : {dataset_id}")

    # ----------------------------------------------------------------------------------------------
    # Load Training Dataset
    # ----------------------------------------------------------------------------------------------

    train_df = TRAIN_DATASETS[dataset_id].copy()

    target = TARGET_COLUMNS[dataset_id]
    identifier_columns = list(IDENTIFIER_COLUMNS.get(dataset_id, []))

    # ----------------------------------------------------------------------------------------------
    # Validate Required Columns
    # ----------------------------------------------------------------------------------------------

    if PROVENANCE_COLUMN not in train_df.columns:
        raise RuntimeError(
            f"{dataset_id}: provenance column '{PROVENANCE_COLUMN}' "
            "is missing from training dataset."
        )

    if target not in train_df.columns:
        raise RuntimeError(
            f"{dataset_id}: target column '{target}' "
            "is missing from training dataset."
        )

    missing_identifiers = [
        col for col in identifier_columns
        if col not in train_df.columns
    ]

    if missing_identifiers:
        raise RuntimeError(
            f"{dataset_id}: configured identifier columns missing from "
            f"training dataset: {missing_identifiers}"
        )

    # ----------------------------------------------------------------------------------------------
    # GENERATIVE / MODELING DATASET COLUMNS
    #
    # Target is retained here because the downstream generative model
    # evaluates the joint distribution of usable variables and target.
    #
    # Explicit identifiers and provenance are excluded.
    # ----------------------------------------------------------------------------------------------

    generative_columns = [
        col
        for col in train_df.columns
        if col != PROVENANCE_COLUMN
        and col not in identifier_columns
    ]

    # ----------------------------------------------------------------------------------------------
    # PREPROCESSOR INPUT COLUMNS
    #
    # IMPORTANT:
    # The target is NOT an input to the generic feature preprocessor.
    #
    # The target remains available in TRAIN_DATASETS and is retained
    # for downstream generative/native dataset construction.
    # ----------------------------------------------------------------------------------------------

    preprocessing_columns = [
        col
        for col in generative_columns
        if col != target
    ]

    # ----------------------------------------------------------------------------------------------
    # Policy Validation
    # ----------------------------------------------------------------------------------------------

    if target not in generative_columns:
        raise RuntimeError(
            f"{dataset_id}: target '{target}' was incorrectly excluded "
            "from the generative/modeling dataset."
        )

    if target in preprocessing_columns:
        raise RuntimeError(
            f"{dataset_id}: target '{target}' must not be included "
            "in generic preprocessing input."
        )

    forbidden_columns = set(identifier_columns + [PROVENANCE_COLUMN])

    forbidden_in_preprocessing = [
        col for col in preprocessing_columns
        if col in forbidden_columns
    ]

    if forbidden_in_preprocessing:
        raise RuntimeError(
            f"{dataset_id}: forbidden columns found in preprocessing input: "
            f"{forbidden_in_preprocessing}"
        )

    if not preprocessing_columns:
        raise RuntimeError(
            f"{dataset_id}: no feature columns remain after excluding "
            "target, identifiers, and provenance."
        )

    # ----------------------------------------------------------------------------------------------
    # Create Training-Only Preprocessing Data
    # ----------------------------------------------------------------------------------------------

    preprocessing_df = train_df[preprocessing_columns].copy()

    # ----------------------------------------------------------------------------------------------
    # Determine Feature Types
    # ----------------------------------------------------------------------------------------------

    numeric_columns = []
    categorical_columns = []

    for col in preprocessing_columns:

        series = preprocessing_df[col]

        if pd.api.types.is_bool_dtype(series):
            categorical_columns.append(col)

        elif pd.api.types.is_numeric_dtype(series):
            numeric_columns.append(col)

        else:
            categorical_columns.append(col)

    # ----------------------------------------------------------------------------------------------
    # Feature-Type Integrity
    # ----------------------------------------------------------------------------------------------

    if set(numeric_columns).intersection(categorical_columns):
        raise RuntimeError(
            f"{dataset_id}: feature appears in both numeric and categorical "
            "feature lists."
        )

    classified_columns = set(numeric_columns + categorical_columns)

    if classified_columns != set(preprocessing_columns):
        unclassified = sorted(
            set(preprocessing_columns) - classified_columns
        )

        raise RuntimeError(
            f"{dataset_id}: unclassified preprocessing columns: "
            f"{unclassified}"
        )

    # ----------------------------------------------------------------------------------------------
    # Store Objects
    # ----------------------------------------------------------------------------------------------

    TRAIN_PREPROCESSING_DATA[dataset_id] = preprocessing_df

    TRAIN_PREPROCESSING_COLUMNS[dataset_id] = {
        # Columns used by the generic preprocessor
        "all_columns": preprocessing_columns,

        # Explicit feature groups
        "numeric_columns": numeric_columns,
        "categorical_columns": categorical_columns,

        # Target retained outside the generic feature transformer
        "target_column": target,

        # Explicit exclusions
        "identifier_columns": identifier_columns,
        "provenance_column": PROVENANCE_COLUMN,

        # Generative/native dataset schema
        "generative_columns": generative_columns,

        # Populated later by Section 16/17
        "transformed_columns": [],
    }

    TRAIN_PREPROCESSING_METADATA[dataset_id] = {
        "dataset_id": dataset_id,
        "training_rows": int(len(preprocessing_df)),

        # Generic preprocessor input
        "input_columns": preprocessing_columns,

        # Feature typing
        "numeric_columns": numeric_columns,
        "categorical_columns": categorical_columns,

        # Target policy
        "target_column": target,
        "target_retained_in_training_dataset": True,
        "target_excluded_from_preprocessor_input": True,

        # Identifier/provenance policy
        "identifier_columns": identifier_columns,
        "provenance_column": PROVENANCE_COLUMN,
        "provenance_excluded_from_modeling": True,
        "identifiers_excluded_from_modeling": True,

        # Generative/native schema
        "generative_columns": generative_columns,

        # Fitting policy
        "fit_dataset": "train_only",
        "fit_scope": "training_features_only",
    }

    # ----------------------------------------------------------------------------------------------
    # Summary
    # ----------------------------------------------------------------------------------------------

    TRAIN_PREPROCESSING_SUMMARY.append({
        "dataset_id": dataset_id,
        "training_rows": int(len(train_df)),
        "generative_columns": int(len(generative_columns)),
        "preprocessing_features": int(len(preprocessing_columns)),
        "numeric_features": int(len(numeric_columns)),
        "categorical_features": int(len(categorical_columns)),
        "target": target,
        "target_in_generative_columns": target in generative_columns,
        "target_in_preprocessing_input": target in preprocessing_columns,
        "identifiers_excluded": all(
            col not in preprocessing_columns
            for col in identifier_columns
        ),
        "provenance_excluded": PROVENANCE_COLUMN not in preprocessing_columns,
    })

# --------------------------------------------------------------------------------------------------
# Create Summary DataFrame
# --------------------------------------------------------------------------------------------------

TRAIN_PREPROCESSING_SUMMARY_DF = pd.DataFrame(
    TRAIN_PREPROCESSING_SUMMARY
)

# --------------------------------------------------------------------------------------------------
# Global Integrity Validation
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    target = TARGET_COLUMNS[dataset_id]
    identifier_columns = list(IDENTIFIER_COLUMNS.get(dataset_id, []))

    preprocessing_columns = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]["all_columns"]

    generative_columns = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]["generative_columns"]

    # Target must remain available to generative/native pipeline
    if target not in generative_columns:
        raise RuntimeError(
            f"{dataset_id}: target was lost from generative columns."
        )

    # Target must NOT enter generic preprocessing
    if target in preprocessing_columns:
        raise RuntimeError(
            f"{dataset_id}: target '{target}' is present in "
            "TRAIN_PREPROCESSING_COLUMNS['all_columns']."
        )

    # Identifiers must be excluded
    for col in identifier_columns:
        if col in preprocessing_columns:
            raise RuntimeError(
                f"{dataset_id}: identifier '{col}' entered preprocessing input."
            )

    # Provenance must be excluded
    if PROVENANCE_COLUMN in preprocessing_columns:
        raise RuntimeError(
            f"{dataset_id}: provenance column entered preprocessing input."
        )

    # Training preprocessing data must match declared input schema
    actual_columns = list(
        TRAIN_PREPROCESSING_DATA[dataset_id].columns
    )

    if actual_columns != preprocessing_columns:
        raise RuntimeError(
            f"{dataset_id}: TRAIN_PREPROCESSING_DATA columns do not "
            "match TRAIN_PREPROCESSING_COLUMNS['all_columns']."
        )

# --------------------------------------------------------------------------------------------------
# Display Summary
# --------------------------------------------------------------------------------------------------

print("\nTRAINING PREPROCESSING SUMMARY")
display(TRAIN_PREPROCESSING_SUMMARY_DF)

print("\n" + "=" * 100)
print("SECTION 15 STATUS : PASS")
print("=" * 100)

print(
    "\nPolicy:"
    "\n  ✓ Target retained in TRAIN_DATASETS / generative schema"
    "\n  ✓ Target excluded from generic preprocessing input"
    "\n  ✓ Explicit identifiers excluded"
    "\n  ✓ Original row ID excluded"
    "\n  ✓ Preprocessing input contains features only"
    "\n  ✓ Preprocessing data derived from training split only"
    "\n  ✓ Feature typing is explicit and complete"
)

15. PREPARE TRAINING PREPROCESSING DATA
----------------------------------------------------------------------------------------------------
Dataset : adult_income
----------------------------------------------------------------------------------------------------
Dataset : bank_marketing
----------------------------------------------------------------------------------------------------
Dataset : diabetes_130us

TRAINING PREPROCESSING SUMMARY


,dataset_id,training_rows,generative_columns,preprocessing_features,numeric_features,categorical_features,target,target_in_generative_columns,target_in_preprocessing_input,identifiers_excluded,provenance_excluded
0,adult_income,34189,15,14,6,8,income,True,False,True,True
1,bank_marketing,31647,17,16,7,9,y,True,False,True,True
2,diabetes_130us,71236,48,47,11,36,readmitted,True,False,True,True



SECTION 15 STATUS : PASS

Policy:
  ✓ Target retained in TRAIN_DATASETS / generative schema
  ✓ Target excluded from generic preprocessing input
  ✓ Explicit identifiers excluded
  ✓ Original row ID excluded
  ✓ Preprocessing input contains features only
  ✓ Preprocessing data derived from training split only
  ✓ Feature typing is explicit and complete


In [41]:
# ==============================================================================
# SECTION 16 — FIT TRAINING-ONLY PREPROCESSOR
# ==============================================================================

print("=" * 100)
print("16. FIT TRAINING-ONLY PREPROCESSOR")
print("=" * 100)

import inspect
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ==============================================================================
# 16.1 — REQUIRED OBJECT VALIDATION
# ==============================================================================

REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_DATASETS",
    "TRAIN_PREPROCESSING_DATA",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
]

_missing_objects = [
    obj for obj in REQUIRED_OBJECTS
    if obj not in globals()
]

if _missing_objects:
    raise RuntimeError(
        "Required objects for Section 16 are missing:\n"
        f"{_missing_objects}\n\n"
        "Execute Sections 1–15 before running Section 16."
    )

PROVENANCE_COLUMN = "__original_row_id__"

print("[PASS] Required Section 16 objects are available.")


# ==============================================================================
# 16.2 — INITIALIZE PREPROCESSOR CONTAINERS
# ==============================================================================

TRAIN_PREPROCESSORS = {}
PREPROCESSOR_METADATA = {}

print("[PASS] Preprocessor containers initialized.")


# ==============================================================================
# 16.3 — ONE-HOT ENCODER COMPATIBILITY
# ==============================================================================

_OHE_PARAMETERS = inspect.signature(
    OneHotEncoder
).parameters

if "sparse_output" in _OHE_PARAMETERS:

    OHE_CONFIG = {
        "handle_unknown": "ignore",
        "sparse_output": False,
        "dtype": np.float32,
    }

    OHE_SPARSE_PARAMETER = "sparse_output"

else:

    OHE_CONFIG = {
        "handle_unknown": "ignore",
        "sparse": False,
        "dtype": np.float32,
    }

    OHE_SPARSE_PARAMETER = "sparse"

print(
    f"[INFO] OneHotEncoder configuration: "
    f"{OHE_SPARSE_PARAMETER}=False, "
    f"handle_unknown='ignore'"
)


# ==============================================================================
# 16.4 — DATASET-WISE TRAINING-ONLY FITTING
# ==============================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    # ==========================================================================
    # 16.4.1 — RETRIEVE TRAINING DATA
    # ==========================================================================

    if dataset_id not in TRAIN_DATASETS:
        raise RuntimeError(
            f"[FAIL] Training dataset '{dataset_id}' is not available."
        )

    train_df = TRAIN_DATASETS[dataset_id].copy()

    if not isinstance(train_df, pd.DataFrame):
        raise TypeError(
            f"[FAIL] TRAIN_DATASETS['{dataset_id}'] "
            "is not a pandas DataFrame."
        )

    # ==========================================================================
    # 16.4.2 — RETRIEVE SECTION 15 PREPROCESSING CONFIGURATION
    # ==========================================================================

    if dataset_id not in TRAIN_PREPROCESSING_DATA:
        raise RuntimeError(
            f"[FAIL] TRAIN_PREPROCESSING_DATA for '{dataset_id}' "
            "is unavailable.\nRun Section 15 first."
        )

    if dataset_id not in TRAIN_PREPROCESSING_COLUMNS:
        raise RuntimeError(
            f"[FAIL] TRAIN_PREPROCESSING_COLUMNS for '{dataset_id}' "
            "is unavailable.\nRun Section 15 first."
        )

    preprocessing_config = (
        TRAIN_PREPROCESSING_COLUMNS[dataset_id]
    )

    # ==========================================================================
    # 16.4.3 — RETRIEVE FEATURE-ONLY PREPROCESSING SCHEMA
    # ==========================================================================

    model_columns = list(
        preprocessing_config["all_columns"]
    )

    numeric_columns = list(
        preprocessing_config["numeric_columns"]
    )

    categorical_columns = list(
        preprocessing_config["categorical_columns"]
    )

    target_column = preprocessing_config[
        "target_column"
    ]

    identifier_columns = list(
        preprocessing_config["identifier_columns"]
    )

    generative_columns = list(
        preprocessing_config["generative_columns"]
    )

    # ==========================================================================
    # 16.4.4 — CRITICAL TARGET POLICY VALIDATION
    # ==========================================================================

    # Target MUST remain in generative/native schema.
    if target_column not in generative_columns:
        raise RuntimeError(
            f"[FAIL] Target '{target_column}' is missing from "
            f"generative columns for '{dataset_id}'."
        )

    # Target MUST NOT enter generic feature preprocessing.
    if target_column in model_columns:
        raise RuntimeError(
            f"[FAIL] Target leakage detected for '{dataset_id}': "
            f"'{target_column}' is present in preprocessor input."
        )

    if target_column in numeric_columns:
        raise RuntimeError(
            f"[FAIL] Target '{target_column}' incorrectly classified "
            f"as numeric preprocessing input for '{dataset_id}'."
        )

    if target_column in categorical_columns:
        raise RuntimeError(
            f"[FAIL] Target '{target_column}' incorrectly classified "
            f"as categorical preprocessing input for '{dataset_id}'."
        )

    # ==========================================================================
    # 16.4.5 — IDENTIFIER / PROVENANCE EXCLUSION VALIDATION
    # ==========================================================================

    leaked_identifiers = [
        column
        for column in identifier_columns
        if column in model_columns
    ]

    if leaked_identifiers:
        raise RuntimeError(
            f"[FAIL] Identifier leakage detected for "
            f"'{dataset_id}': {leaked_identifiers}"
        )

    if PROVENANCE_COLUMN in model_columns:
        raise RuntimeError(
            f"[FAIL] Provenance leakage detected for "
            f"'{dataset_id}': {PROVENANCE_COLUMN}"
        )

    # ==========================================================================
    # 16.4.6 — VERIFY PREPROCESSING DATA
    # ==========================================================================

    preprocessing_df = (
        TRAIN_PREPROCESSING_DATA[dataset_id].copy()
    )

    if list(preprocessing_df.columns) != model_columns:
        raise RuntimeError(
            f"[FAIL] TRAIN_PREPROCESSING_DATA column schema mismatch "
            f"for '{dataset_id}'.\n"
            f"Expected: {model_columns}\n"
            f"Actual  : {list(preprocessing_df.columns)}"
        )

    if target_column in preprocessing_df.columns:
        raise RuntimeError(
            f"[FAIL] Target '{target_column}' is present in "
            f"TRAIN_PREPROCESSING_DATA for '{dataset_id}'."
        )

    if PROVENANCE_COLUMN in preprocessing_df.columns:
        raise RuntimeError(
            f"[FAIL] Provenance column is present in "
            f"TRAIN_PREPROCESSING_DATA for '{dataset_id}'."
        )

    # ==========================================================================
    # 16.4.7 — VERIFY ALL PREPROCESSOR INPUT COLUMNS EXIST
    # ==========================================================================

    missing_model_columns = [
        column
        for column in model_columns
        if column not in preprocessing_df.columns
    ]

    if missing_model_columns:
        raise RuntimeError(
            f"[FAIL] Preprocessing columns missing from training "
            f"data for '{dataset_id}': {missing_model_columns}"
        )

    # ==========================================================================
    # 16.4.8 — VERIFY FEATURE ACCOUNTING
    # ==========================================================================

    if (
        len(numeric_columns)
        + len(categorical_columns)
        != len(model_columns)
    ):
        raise RuntimeError(
            f"[FAIL] Numeric/categorical feature accounting mismatch "
            f"for '{dataset_id}'.\n"
            f"Preprocessing columns : {len(model_columns)}\n"
            f"Numeric columns      : {len(numeric_columns)}\n"
            f"Categorical columns  : {len(categorical_columns)}"
        )

    # ==========================================================================
    # 16.4.9 — VERIFY FEATURE-TYPE OVERLAP
    # ==========================================================================

    feature_overlap = (
        set(numeric_columns)
        & set(categorical_columns)
    )

    if feature_overlap:
        raise RuntimeError(
            f"[FAIL] Feature-type overlap detected for "
            f"'{dataset_id}': {sorted(feature_overlap)}"
        )

    # ==========================================================================
    # 16.4.10 — VERIFY FEATURE-TYPE COMPLETENESS
    # ==========================================================================

    classified_columns = set(
        numeric_columns + categorical_columns
    )

    if classified_columns != set(model_columns):

        unclassified_columns = sorted(
            set(model_columns) - classified_columns
        )

        unexpected_columns = sorted(
            classified_columns - set(model_columns)
        )

        raise RuntimeError(
            f"[FAIL] Feature classification mismatch for "
            f"'{dataset_id}'.\n"
            f"Unclassified : {unclassified_columns}\n"
            f"Unexpected   : {unexpected_columns}"
        )

    # ==========================================================================
    # 16.4.11 — CREATE NUMERIC PIPELINE
    # ==========================================================================

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    # ==========================================================================
    # 16.4.12 — CREATE CATEGORICAL PIPELINE
    # ==========================================================================

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                ),
            ),
            (
                "encoder",
                OneHotEncoder(
                    **OHE_CONFIG
                ),
            ),
        ]
    )

    # ==========================================================================
    # 16.4.13 — CREATE COLUMN TRANSFORMER
    # ==========================================================================

    transformers = []

    if numeric_columns:
        transformers.append(
            (
                "numeric",
                numeric_pipeline,
                numeric_columns,
            )
        )

    if categorical_columns:
        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns,
            )
        )

    if not transformers:
        raise RuntimeError(
            f"[FAIL] No preprocessing transformers created "
            f"for '{dataset_id}'."
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )

    # ==========================================================================
    # 16.4.14 — FIT USING TRAINING FEATURES ONLY
    # ==========================================================================

    print(
        f"Fitting feature preprocessor on "
        f"{len(preprocessing_df):,} training rows..."
    )

    with warnings.catch_warnings():

        warnings.simplefilter("ignore")

        preprocessor.fit(
            preprocessing_df
        )

    print(
        "[PASS] Preprocessor fitted using training features only."
    )

    # ==========================================================================
    # 16.4.15 — VERIFY PREPROCESSOR INPUT SCHEMA
    # ==========================================================================

    if not hasattr(
        preprocessor,
        "feature_names_in_"
    ):
        raise RuntimeError(
            f"[FAIL] Fitted preprocessor for '{dataset_id}' "
            "does not expose feature_names_in_."
        )

    fitted_input_columns = list(
        preprocessor.feature_names_in_
    )

    if fitted_input_columns != model_columns:
        raise RuntimeError(
            f"[FAIL] Fitted preprocessor input schema mismatch "
            f"for '{dataset_id}'.\n"
            f"Expected: {model_columns}\n"
            f"Fitted  : {fitted_input_columns}"
        )

    # Explicit target / identifier / provenance checks
    forbidden_input_columns = set(
        [target_column, PROVENANCE_COLUMN]
        + identifier_columns
    )

    leaked_fitted_columns = [
        column
        for column in fitted_input_columns
        if column in forbidden_input_columns
    ]

    if leaked_fitted_columns:
        raise RuntimeError(
            f"[FAIL] Forbidden columns entered fitted preprocessor "
            f"for '{dataset_id}': {leaked_fitted_columns}"
        )

    # ==========================================================================
    # 16.4.16 — DETERMINE TRANSFORMED FEATURE NAMES
    # ==========================================================================

    try:

        transformed_feature_names = (
            preprocessor
            .get_feature_names_out()
            .tolist()
        )

    except Exception as exc:

        raise RuntimeError(
            f"[FAIL] Unable to obtain transformed feature names "
            f"for '{dataset_id}': {exc}"
        )

    transformed_feature_count = len(
        transformed_feature_names
    )

    if transformed_feature_count == 0:
        raise RuntimeError(
            f"[FAIL] Preprocessor produced zero transformed "
            f"features for '{dataset_id}'."
        )

    # ==========================================================================
    # 16.4.17 — STORE FITTED PREPROCESSOR
    # ==========================================================================

    TRAIN_PREPROCESSORS[
        dataset_id
    ] = preprocessor

    # ==========================================================================
    # 16.4.18 — STORE PREPROCESSOR FEATURE SCHEMA
    # ==========================================================================

    TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]["transformed_columns"] = (
        transformed_feature_names
    )

    # ==========================================================================
    # 16.4.19 — STORE METADATA
    # ==========================================================================

    PREPROCESSOR_METADATA[
        dataset_id
    ] = {

        "dataset_id":
            dataset_id,

        "training_rows":
            int(len(preprocessing_df)),

        "input_columns":
            int(len(model_columns)),

        "numeric_input_columns":
            int(len(numeric_columns)),

        "categorical_input_columns":
            int(len(categorical_columns)),

        "identifier_columns_excluded":
            int(len(identifier_columns)),

        "provenance_column_excluded":
            True,

        "target_column":
            target_column,

        "target_excluded_from_preprocessor":
            True,

        "generative_columns":
            int(len(generative_columns)),

        "transformed_columns":
            int(transformed_feature_count),

        "numeric_imputation":
            "median",

        "numeric_scaling":
            "StandardScaler",

        "categorical_imputation":
            "most_frequent",

        "categorical_encoding":
            "OneHotEncoder",

        "handle_unknown":
            "ignore",

        "fit_dataset":
            "training_only",

        "fit_scope":
            "training_features_only",
    }

    # ==========================================================================
    # 16.4.20 — TRANSFORM TRAINING FEATURES
    # ==========================================================================

    with warnings.catch_warnings():

        warnings.simplefilter("ignore")

        X_train_encoded = (
            preprocessor.transform(
                preprocessing_df
            )
        )

    if hasattr(
        X_train_encoded,
        "toarray"
    ):
        X_train_encoded = (
            X_train_encoded.toarray()
        )

    X_train_encoded = np.asarray(
        X_train_encoded,
        dtype=np.float32
    )

    # ==========================================================================
    # 16.4.21 — VERIFY TRANSFORMED SHAPE
    # ==========================================================================

    if X_train_encoded.shape[0] != len(
        preprocessing_df
    ):
        raise RuntimeError(
            f"[FAIL] Transformed row count mismatch "
            f"for '{dataset_id}'."
        )

    if X_train_encoded.shape[1] != (
        transformed_feature_count
    ):
        raise RuntimeError(
            f"[FAIL] Transformed feature count mismatch "
            f"for '{dataset_id}'."
        )

    # ==========================================================================
    # 16.4.22 — VERIFY FINITE VALUES
    # ==========================================================================

    if not np.isfinite(
        X_train_encoded
    ).all():
        raise RuntimeError(
            f"[FAIL] Non-finite values detected in transformed "
            f"training features for '{dataset_id}'."
        )

    # ==========================================================================
    # 16.4.23 — DATASET SUMMARY
    # ==========================================================================

    print(
        f"Input feature columns       : "
        f"{len(model_columns)}"
    )

    print(
        f"Numeric feature columns     : "
        f"{len(numeric_columns)}"
    )

    print(
        f"Categorical feature columns : "
        f"{len(categorical_columns)}"
    )

    print(
        f"Target excluded             : "
        f"{target_column}"
    )

    print(
        f"Identifiers excluded        : "
        f"{len(identifier_columns)}"
    )

    print(
        f"Provenance excluded         : "
        f"{PROVENANCE_COLUMN}"
    )

    print(
        f"Encoded output columns      : "
        f"{transformed_feature_count:,}"
    )

    print(
        f"Encoded dtype               : "
        f"{X_train_encoded.dtype}"
    )

    print(
        f"Encoded memory              : "
        f"{X_train_encoded.nbytes / (1024 ** 2):.2f} MB"
    )

    print(
        "[PASS] Dataset preprocessor validated."
    )


# ==============================================================================
# 16.5 — GLOBAL PREPROCESSOR VALIDATION
# ==============================================================================

print("\n" + "=" * 100)
print("GLOBAL PREPROCESSOR VALIDATION")
print("=" * 100)

assert set(
    TRAIN_PREPROCESSORS.keys()
) == set(DATASET_IDS), (
    "Not all configured datasets have fitted preprocessors."
)

assert set(
    PREPROCESSOR_METADATA.keys()
) == set(DATASET_IDS), (
    "Preprocessor metadata does not cover all datasets."
)


for dataset_id in DATASET_IDS:

    preprocessor = TRAIN_PREPROCESSORS[
        dataset_id
    ]

    preprocessing_config = (
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]
    )

    target = TARGET_COLUMNS[
        dataset_id
    ]

    identifiers = IDENTIFIER_COLUMNS.get(
        dataset_id,
        []
    )

    model_columns = list(
        preprocessing_config["all_columns"]
    )

    # --------------------------------------------------------------------------
    # Fitted-state validation
    # --------------------------------------------------------------------------

    assert hasattr(
        preprocessor,
        "transformers_"
    ), (
        f"Preprocessor for {dataset_id} is not fitted."
    )

    # --------------------------------------------------------------------------
    # Input-schema validation
    # --------------------------------------------------------------------------

    assert list(
        preprocessor.feature_names_in_
    ) == model_columns, (
        f"Fitted input schema mismatch for {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Target exclusion
    # --------------------------------------------------------------------------

    assert target not in model_columns, (
        f"Target leakage detected for {dataset_id}: {target}"
    )

    # --------------------------------------------------------------------------
    # Identifier exclusion
    # --------------------------------------------------------------------------

    leaked_identifiers = [
        column
        for column in identifiers
        if column in model_columns
    ]

    assert not leaked_identifiers, (
        f"Identifier leakage detected for "
        f"{dataset_id}: {leaked_identifiers}"
    )

    # --------------------------------------------------------------------------
    # Provenance exclusion
    # --------------------------------------------------------------------------

    assert PROVENANCE_COLUMN not in model_columns, (
        f"Provenance leakage detected for {dataset_id}."
    )

print(
    "[PASS] All training-only preprocessors validated."
)


# ==============================================================================
# 16.6 — CREATE PREPROCESSOR SUMMARY DATAFRAME
# ==============================================================================

PREPROCESSOR_SUMMARY_DF = pd.DataFrame(
    PREPROCESSOR_METADATA.values()
)


# ==============================================================================
# 16.7 — DISPLAY SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 16 PREPROCESSOR SUMMARY")
print("=" * 100)

display(
    PREPROCESSOR_SUMMARY_DF[
        [
            "dataset_id",
            "training_rows",
            "input_columns",
            "numeric_input_columns",
            "categorical_input_columns",
            "identifier_columns_excluded",
            "provenance_column_excluded",
            "target_column",
            "target_excluded_from_preprocessor",
            "generative_columns",
            "transformed_columns",
            "numeric_imputation",
            "numeric_scaling",
            "categorical_imputation",
            "categorical_encoding",
            "handle_unknown",
            "fit_dataset",
            "fit_scope",
        ]
    ]
)


# ==============================================================================
# 16.8 — FINAL STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 16 COMPLETE")
print("=" * 100)

print(
    f"Datasets processed         : {len(DATASET_IDS)}"
)

print(
    f"Fitted preprocessors       : "
    f"{len(TRAIN_PREPROCESSORS)}"
)

print(
    "\n[PASS] SECTION 16 — FIT TRAINING-ONLY PREPROCESSOR"
)

print(
    "[PASS] Preprocessors fitted using training features only."
)

print(
    "[PASS] Target excluded from generic preprocessing input."
)

print(
    "[PASS] Target retained in generative/native dataset schema."
)

print(
    "[PASS] Numeric median imputation configured."
)

print(
    "[PASS] Numeric StandardScaler configured."
)

print(
    "[PASS] Categorical most-frequent imputation configured."
)

print(
    "[PASS] One-hot encoding configured with unknown-category handling."
)

print(
    "[PASS] Identifier columns excluded from preprocessing."
)

print(
    "[PASS] Provenance column excluded from preprocessing."
)

print(
    "[PASS] Training transformations validated."
)

16. FIT TRAINING-ONLY PREPROCESSOR
[PASS] Required Section 16 objects are available.
[PASS] Preprocessor containers initialized.
[INFO] OneHotEncoder configuration: sparse_output=False, handle_unknown='ignore'

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Fitting feature preprocessor on 34,189 training rows...
[PASS] Preprocessor fitted using training features only.
Input feature columns       : 14
Numeric feature columns     : 6
Categorical feature columns : 8
Target excluded             : income
Identifiers excluded        : 0
Provenance excluded         : __original_row_id__
Encoded output columns      : 105
Encoded dtype               : float32
Encoded memory              : 13.69 MB
[PASS] Dataset preprocessor validated.

-------------------------------------------------------------------------------------

,dataset_id,training_rows,input_columns,numeric_input_columns,categorical_input_columns,identifier_columns_excluded,provenance_column_excluded,target_column,target_excluded_from_preprocessor,generative_columns,transformed_columns,numeric_imputation,numeric_scaling,categorical_imputation,categorical_encoding,handle_unknown,fit_dataset,fit_scope
0,adult_income,34189,14,6,8,0,True,income,True,15,105,median,StandardScaler,most_frequent,OneHotEncoder,ignore,training_only,training_features_only
1,bank_marketing,31647,16,7,9,0,True,y,True,17,51,median,StandardScaler,most_frequent,OneHotEncoder,ignore,training_only,training_features_only
2,diabetes_130us,71236,47,11,36,2,True,readmitted,True,48,2336,median,StandardScaler,most_frequent,OneHotEncoder,ignore,training_only,training_features_only



SECTION 16 COMPLETE
Datasets processed         : 3
Fitted preprocessors       : 3

[PASS] SECTION 16 — FIT TRAINING-ONLY PREPROCESSOR
[PASS] Preprocessors fitted using training features only.
[PASS] Target excluded from generic preprocessing input.
[PASS] Target retained in generative/native dataset schema.
[PASS] Numeric median imputation configured.
[PASS] Numeric StandardScaler configured.
[PASS] Categorical most-frequent imputation configured.
[PASS] One-hot encoding configured with unknown-category handling.
[PASS] Identifier columns excluded from preprocessing.
[PASS] Provenance column excluded from preprocessing.
[PASS] Training transformations validated.


In [42]:
# ==============================================================================
# SECTION 17 — TRANSFORM TRAIN / VALIDATION / TEST
# ==============================================================================
#
# PURPOSE
# -------
# Transform TRAIN / VALIDATION / TEST datasets using the SAME preprocessor
# fitted exclusively on TRAIN in Section 16.
#
# ARCHITECTURE
# ------------
#
# TRAIN / VALIDATION / TEST split dataframe
#              |
#              +--> target                 -> retained outside preprocessing
#              |
#              +--> __original_row_id__    -> audit / reproducibility only
#              |
#              +--> explicit identifiers   -> excluded from preprocessing
#              |
#              +--> preprocessing features -> passed to fitted preprocessor
#
# IMPORTANT
# ---------
# NO preprocessing is fitted in Section 17.
#
# The preprocessor fitted in Section 16 is reused unchanged for:
#
#     TRAIN
#     VALIDATION
#     TEST
#
# INPUTS
# ------
# DATASET_IDS
# TRAIN_DATASETS
# VALIDATION_DATASETS
# TEST_DATASETS
# TRAIN_PREPROCESSORS
# TRAIN_PREPROCESSING_COLUMNS
#
# OUTPUTS
# -------
# TRANSFORMED_TRAIN_DATASETS
# TRANSFORMED_VALIDATION_DATASETS
# TRANSFORMED_TEST_DATASETS
# TRANSFORMED_DATASET_SUMMARY
# TRANSFORMED_DATASET_SUMMARY_DF
#
# ==============================================================================

print("=" * 100)
print("SECTION 17 — TRANSFORM TRAIN / VALIDATION / TEST")
print("=" * 100)


# ==============================================================================
# 17.1 — REQUIRED OBJECT CHECKS
# ==============================================================================

required_objects = [
    "DATASET_IDS",
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
    "TRAIN_PREPROCESSORS",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
]

missing_objects = [
    obj
    for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "SECTION 17 CANNOT START.\n"
        "The following required objects are missing:\n"
        + "\n".join(f"  - {x}" for x in missing_objects)
        + "\n\n"
        "Run Sections 13, 15, and 16 successfully before Section 17."
    )

PROVENANCE_COLUMN = "__original_row_id__"

print("✓ Required Section 17 objects detected.")


# ==============================================================================
# 17.2 — VALIDATE IDENTIFIER POLICY
# ==============================================================================

for dataset_id in DATASET_IDS:

    if dataset_id not in IDENTIFIER_COLUMNS:
        raise RuntimeError(
            f"{dataset_id}: identifier policy is missing."
        )

    configured_identifiers = list(
        IDENTIFIER_COLUMNS.get(
            dataset_id,
            []
        )
    )

    configured_identifiers_from_section_15 = list(
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ].get(
            "identifier_columns",
            []
        )
    )

    if configured_identifiers != configured_identifiers_from_section_15:
        raise RuntimeError(
            f"{dataset_id}: identifier policy mismatch between "
            "IDENTIFIER_COLUMNS and TRAIN_PREPROCESSING_COLUMNS."
        )

print("✓ Identifier policy validated.")


# ==============================================================================
# 17.3 — RESET OUTPUT CONTAINERS
# ==============================================================================

TRANSFORMED_TRAIN_DATASETS = {}
TRANSFORMED_VALIDATION_DATASETS = {}
TRANSFORMED_TEST_DATASETS = {}

TRANSFORMED_DATASET_SUMMARY = []

print("✓ Transformation output containers initialized.")


# ==============================================================================
# 17.4 — TRANSFORMATION HELPER
# ==============================================================================

def transform_dataset_split(
    dataset_id,
    split_name,
    dataframe,
    preprocessor,
    expected_input_columns,
    expected_output_columns,
    target_column,
    identifier_columns,
):
    """
    Transform one dataset split using a preprocessor fitted ONLY on TRAIN.

    Only canonical preprocessing feature columns are supplied to the
    preprocessor.

    Target, explicit identifiers, and provenance are never passed to
    the preprocessor.
    """

    if dataframe is None:
        raise ValueError(
            f"{dataset_id} | {split_name}: dataframe is None."
        )

    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(
            f"{dataset_id} | {split_name}: dataframe is not a "
            "pandas DataFrame."
        )

    if preprocessor is None:
        raise ValueError(
            f"{dataset_id} | {split_name}: preprocessor is None."
        )

    # ==========================================================================
    # Validate target
    # ==========================================================================

    if target_column not in dataframe.columns:
        raise ValueError(
            f"{dataset_id} | {split_name}: target column "
            f"'{target_column}' is missing."
        )

    # ==========================================================================
    # Validate provenance
    # ==========================================================================

    if PROVENANCE_COLUMN not in dataframe.columns:
        raise ValueError(
            f"{dataset_id} | {split_name}: provenance column "
            f"'{PROVENANCE_COLUMN}' is missing."
        )

    # ==========================================================================
    # Validate explicit identifiers
    # ==========================================================================

    missing_identifiers = [
        column
        for column in identifier_columns
        if column not in dataframe.columns
    ]

    if missing_identifiers:
        raise ValueError(
            f"{dataset_id} | {split_name}: configured identifier "
            f"columns are missing: {missing_identifiers}"
        )

    # ==========================================================================
    # Validate preprocessing feature columns
    # ==========================================================================

    missing_columns = [
        column
        for column in expected_input_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{dataset_id} | {split_name}: required preprocessing "
            f"feature columns are missing:\n{missing_columns}"
        )

    # ==========================================================================
    # CRITICAL POLICY VALIDATION
    # ==========================================================================

    # Target must never enter generic preprocessing.
    if target_column in expected_input_columns:
        raise RuntimeError(
            f"{dataset_id} | {split_name}: target '{target_column}' "
            "is incorrectly included in preprocessing feature columns."
        )

    # Provenance must never enter preprocessing.
    if PROVENANCE_COLUMN in expected_input_columns:
        raise RuntimeError(
            f"{dataset_id} | {split_name}: provenance column "
            f"'{PROVENANCE_COLUMN}' entered preprocessing."
        )

    # Explicit identifiers must never enter preprocessing.
    leaked_identifiers = [
        column
        for column in identifier_columns
        if column in expected_input_columns
    ]

    if leaked_identifiers:
        raise RuntimeError(
            f"{dataset_id} | {split_name}: identifier leakage detected: "
            f"{leaked_identifiers}"
        )

    # ==========================================================================
    # Build exact preprocessing feature dataframe
    # ==========================================================================

    preprocessing_df = dataframe.loc[
        :,
        expected_input_columns
    ].copy()

    # ==========================================================================
    # Confirm exact column ordering
    # ==========================================================================

    if list(preprocessing_df.columns) != list(
        expected_input_columns
    ):
        raise RuntimeError(
            f"{dataset_id} | {split_name}: preprocessing feature "
            "column ordering mismatch."
        )

    # ==========================================================================
    # Validate row count
    # ==========================================================================

    input_rows = len(dataframe)

    if input_rows == 0:
        raise ValueError(
            f"{dataset_id} | {split_name}: dataframe contains zero rows."
        )

    # ==========================================================================
    # Transform using TRAIN-FITTED PREPROCESSOR
    # ==========================================================================

    transformed = preprocessor.transform(
        preprocessing_df
    )

    # ==========================================================================
    # Convert to RAM-efficient float32
    # ==========================================================================

    if hasattr(
        transformed,
        "toarray"
    ):
        transformed = transformed.toarray()

    transformed = np.asarray(
        transformed,
        dtype=np.float32
    )

    # ==========================================================================
    # Validate dimensionality
    # ==========================================================================

    if transformed.ndim != 2:
        raise ValueError(
            f"{dataset_id} | {split_name}: transformed output "
            "must be 2-D.\n"
            f"Observed shape: {transformed.shape}"
        )

    output_rows, output_columns = transformed.shape

    # ==========================================================================
    # Row preservation
    # ==========================================================================

    if output_rows != input_rows:
        raise ValueError(
            f"{dataset_id} | {split_name}: row count changed "
            "during transformation.\n"
            f"Input rows : {input_rows}\n"
            f"Output rows: {output_rows}"
        )

    # ==========================================================================
    # Feature dimensionality
    # ==========================================================================

    if output_columns != expected_output_columns:
        raise ValueError(
            f"{dataset_id} | {split_name}: transformed feature "
            "count mismatch.\n"
            f"Expected: {expected_output_columns}\n"
            f"Actual  : {output_columns}"
        )

    # ==========================================================================
    # Finite-value validation
    # ==========================================================================

    if not np.isfinite(
        transformed
    ).all():
        raise ValueError(
            f"{dataset_id} | {split_name}: transformed data "
            "contains NaN or infinite values."
        )

    # ==========================================================================
    # Memory
    # ==========================================================================

    memory_mb = (
        transformed.nbytes
        / (1024 ** 2)
    )

    # ==========================================================================
    # Return
    # ==========================================================================

    return (
        transformed,
        memory_mb,
    )


# ==============================================================================
# 17.5 — TRANSFORM ALL DATASETS
# ==============================================================================

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    # ==========================================================================
    # Retrieve training-fitted preprocessor
    # ==========================================================================

    if dataset_id not in TRAIN_PREPROCESSORS:
        raise KeyError(
            f"Training preprocessor not found for dataset: "
            f"{dataset_id}"
        )

    preprocessor = TRAIN_PREPROCESSORS[
        dataset_id
    ]

    # ==========================================================================
    # Verify fitted state
    # ==========================================================================

    if not hasattr(
        preprocessor,
        "transformers_"
    ):
        raise RuntimeError(
            f"{dataset_id}: preprocessor is not fitted."
        )

    # ==========================================================================
    # Retrieve canonical preprocessing schema
    # ==========================================================================

    if dataset_id not in TRAIN_PREPROCESSING_COLUMNS:
        raise KeyError(
            f"TRAIN_PREPROCESSING_COLUMNS does not contain "
            f"dataset '{dataset_id}'."
        )

    preprocessing_schema = (
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]
    )

    # ==========================================================================
    # Canonical preprocessing feature columns
    # ==========================================================================

    preprocessing_feature_columns = list(
        preprocessing_schema[
            "all_columns"
        ]
    )

    # ==========================================================================
    # Target
    # ==========================================================================

    target_column = preprocessing_schema[
        "target_column"
    ]

    # ==========================================================================
    # Explicit identifiers
    # ==========================================================================

    identifier_columns = list(
        preprocessing_schema.get(
            "identifier_columns",
            []
        )
    )

    # ==========================================================================
    # Generative columns
    # ==========================================================================

    generative_columns = list(
        preprocessing_schema[
            "generative_columns"
        ]
    )

    # ==========================================================================
    # Validate target policy
    # ==========================================================================

    if target_column not in generative_columns:
        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' is not present "
            "in generative columns."
        )

    if target_column in preprocessing_feature_columns:
        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' is present "
            "in preprocessing feature columns."
        )

    # ==========================================================================
    # Validate provenance policy
    # ==========================================================================

    if PROVENANCE_COLUMN in preprocessing_feature_columns:
        raise RuntimeError(
            f"{dataset_id}: provenance column entered "
            "preprocessing feature columns."
        )

    # ==========================================================================
    # Validate identifier policy
    # ==========================================================================

    leaked_identifiers = [
        column
        for column in identifier_columns
        if column in preprocessing_feature_columns
    ]

    if leaked_identifiers:
        raise RuntimeError(
            f"{dataset_id}: identifier leakage detected: "
            f"{leaked_identifiers}"
        )

    # ==========================================================================
    # Validate fitted preprocessor schema
    # ==========================================================================

    if not hasattr(
        preprocessor,
        "feature_names_in_"
    ):
        raise RuntimeError(
            f"{dataset_id}: fitted preprocessor does not expose "
            "feature_names_in_."
        )

    fitted_input_columns = list(
        preprocessor.feature_names_in_
    )

    if fitted_input_columns != (
        preprocessing_feature_columns
    ):
        raise RuntimeError(
            f"{dataset_id}: fitted preprocessor input schema "
            "does not match Section 15.\n"
            f"Expected: {preprocessing_feature_columns}\n"
            f"Fitted  : {fitted_input_columns}"
        )

    # ==========================================================================
    # Canonical transformed columns
    # ==========================================================================

    transformed_feature_names = list(
        preprocessing_schema[
            "transformed_columns"
        ]
    )

    expected_output_columns = len(
        transformed_feature_names
    )

    if expected_output_columns == 0:
        raise RuntimeError(
            f"{dataset_id}: transformed feature schema is empty."
        )

    # ==========================================================================
    # Display schema
    # ==========================================================================

    print(
        f"Preprocessing feature columns : "
        f"{len(preprocessing_feature_columns)}"
    )

    print(
        f"Target excluded               : "
        f"{target_column}"
    )

    print(
        f"Identifiers excluded          : "
        f"{len(identifier_columns)}"
    )

    print(
        f"Provenance excluded           : "
        f"{PROVENANCE_COLUMN}"
    )

    print(
        f"Output encoded features       : "
        f"{expected_output_columns}"
    )

    # ==========================================================================
    # Retrieve split datasets
    # ==========================================================================

    train_df = TRAIN_DATASETS[
        dataset_id
    ]

    validation_df = VALIDATION_DATASETS[
        dataset_id
    ]

    test_df = TEST_DATASETS[
        dataset_id
    ]

    print(
        f"Train rows                    : "
        f"{len(train_df):,}"
    )

    print(
        f"Validation rows               : "
        f"{len(validation_df):,}"
    )

    print(
        f"Test rows                     : "
        f"{len(test_df):,}"
    )

    # ==========================================================================
    # Validate split schemas
    # ==========================================================================

    for split_name, split_df in [
        ("Train", train_df),
        ("Validation", validation_df),
        ("Test", test_df),
    ]:

        if target_column not in split_df.columns:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: target "
                f"'{target_column}' is missing."
            )

        if PROVENANCE_COLUMN not in split_df.columns:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: provenance "
                "column is missing."
            )

        missing_features = [
            column
            for column in preprocessing_feature_columns
            if column not in split_df.columns
        ]

        if missing_features:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: missing preprocessing "
                f"features: {missing_features}"
            )

    print(
        "✓ Required target, provenance, and preprocessing "
        "feature columns verified for all three splits."
    )

    # ==========================================================================
    # Transform TRAIN
    # ==========================================================================

    print()
    print("Transforming TRAIN...")

    (
        transformed_train,
        train_memory_mb,
    ) = transform_dataset_split(
        dataset_id=dataset_id,
        split_name="Train",
        dataframe=train_df,
        preprocessor=preprocessor,
        expected_input_columns=preprocessing_feature_columns,
        expected_output_columns=expected_output_columns,
        target_column=target_column,
        identifier_columns=identifier_columns,
    )

    TRANSFORMED_TRAIN_DATASETS[
        dataset_id
    ] = transformed_train

    print(
        f"✓ Train transformed: "
        f"{transformed_train.shape} | "
        f"{train_memory_mb:.2f} MB"
    )

    # ==========================================================================
    # Transform VALIDATION
    # ==========================================================================

    print("Transforming VALIDATION...")

    (
        transformed_validation,
        validation_memory_mb,
    ) = transform_dataset_split(
        dataset_id=dataset_id,
        split_name="Validation",
        dataframe=validation_df,
        preprocessor=preprocessor,
        expected_input_columns=preprocessing_feature_columns,
        expected_output_columns=expected_output_columns,
        target_column=target_column,
        identifier_columns=identifier_columns,
    )

    TRANSFORMED_VALIDATION_DATASETS[
        dataset_id
    ] = transformed_validation

    print(
        f"✓ Validation transformed: "
        f"{transformed_validation.shape} | "
        f"{validation_memory_mb:.2f} MB"
    )

    # ==========================================================================
    # Transform TEST
    # ==========================================================================

    print("Transforming TEST...")

    (
        transformed_test,
        test_memory_mb,
    ) = transform_dataset_split(
        dataset_id=dataset_id,
        split_name="Test",
        dataframe=test_df,
        preprocessor=preprocessor,
        expected_input_columns=preprocessing_feature_columns,
        expected_output_columns=expected_output_columns,
        target_column=target_column,
        identifier_columns=identifier_columns,
    )

    TRANSFORMED_TEST_DATASETS[
        dataset_id
    ] = transformed_test

    print(
        f"✓ Test transformed: "
        f"{transformed_test.shape} | "
        f"{test_memory_mb:.2f} MB"
    )

    # ==========================================================================
    # DATASET-LEVEL FEATURE DIMENSION VALIDATION
    # ==========================================================================

    if (
        transformed_train.shape[1]
        != transformed_validation.shape[1]
        or
        transformed_train.shape[1]
        != transformed_test.shape[1]
    ):
        raise RuntimeError(
            f"{dataset_id}: train/validation/test transformed "
            "feature counts are inconsistent."
        )

    # ==========================================================================
    # ROW PRESERVATION
    # ==========================================================================

    if transformed_train.shape[0] != len(
        train_df
    ):
        raise RuntimeError(
            f"{dataset_id}: train row count was not preserved."
        )

    if transformed_validation.shape[0] != len(
        validation_df
    ):
        raise RuntimeError(
            f"{dataset_id}: validation row count was not preserved."
        )

    if transformed_test.shape[0] != len(
        test_df
    ):
        raise RuntimeError(
            f"{dataset_id}: test row count was not preserved."
        )

    # ==========================================================================
    # FINITE-VALUE VALIDATION
    # ==========================================================================

    for split_name, array in [
        ("Train", transformed_train),
        ("Validation", transformed_validation),
        ("Test", transformed_test),
    ]:

        if not np.isfinite(array).all():
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "non-finite values detected."
            )

    # ==========================================================================
    # DTYPE VALIDATION
    # ==========================================================================

    for split_name, array in [
        ("Train", transformed_train),
        ("Validation", transformed_validation),
        ("Test", transformed_test),
    ]:

        if array.dtype != np.float32:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: expected float32, "
                f"got {array.dtype}."
            )

    # ==========================================================================
    # MEMORY SUMMARY
    # ==========================================================================

    total_memory_mb = (
        train_memory_mb
        + validation_memory_mb
        + test_memory_mb
    )

    # ==========================================================================
    # STORE SUMMARY
    # ==========================================================================

    TRANSFORMED_DATASET_SUMMARY.append({

        "dataset_id":
            dataset_id,

        "train_rows":
            len(train_df),

        "validation_rows":
            len(validation_df),

        "test_rows":
            len(test_df),

        "input_preprocessing_features":
            len(preprocessing_feature_columns),

        "target_excluded":
            target_column,

        "transformed_features":
            expected_output_columns,

        "train_memory_mb":
            train_memory_mb,

        "validation_memory_mb":
            validation_memory_mb,

        "test_memory_mb":
            test_memory_mb,

        "total_memory_mb":
            total_memory_mb,

        "dtype":
            str(transformed_train.dtype),

        "status":
            "SUCCESS",
    })

    print()
    print(
        f"✓ {dataset_id} transformation PASSED."
    )

    print(
        f"  Total transformed memory: "
        f"{total_memory_mb:.2f} MB"
    )


# ==============================================================================
# 17.6 — CREATE SUMMARY DATAFRAME
# ==============================================================================

TRANSFORMED_DATASET_SUMMARY_DF = pd.DataFrame(
    TRANSFORMED_DATASET_SUMMARY
)

print()
print("=" * 100)
print("SECTION 17 SUMMARY")
print("=" * 100)

display(
    TRANSFORMED_DATASET_SUMMARY_DF
)


# ==============================================================================
# 17.7 — GLOBAL DATASET COVERAGE
# ==============================================================================

assert set(
    TRANSFORMED_TRAIN_DATASETS.keys()
) == set(DATASET_IDS), (
    "Transformed train dataset coverage mismatch."
)

assert set(
    TRANSFORMED_VALIDATION_DATASETS.keys()
) == set(DATASET_IDS), (
    "Transformed validation dataset coverage mismatch."
)

assert set(
    TRANSFORMED_TEST_DATASETS.keys()
) == set(DATASET_IDS), (
    "Transformed test dataset coverage mismatch."
)

print(
    "✓ All datasets transformed."
)


# ==============================================================================
# 17.8 — FEATURE DIMENSIONALITY VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    train_array = (
        TRANSFORMED_TRAIN_DATASETS[
            dataset_id
        ]
    )

    validation_array = (
        TRANSFORMED_VALIDATION_DATASETS[
            dataset_id
        ]
    )

    test_array = (
        TRANSFORMED_TEST_DATASETS[
            dataset_id
        ]
    )

    expected_features = len(
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]["transformed_columns"]
    )

    assert train_array.shape[1] == expected_features
    assert validation_array.shape[1] == expected_features
    assert test_array.shape[1] == expected_features

    assert (
        train_array.shape[1]
        ==
        validation_array.shape[1]
    )

    assert (
        train_array.shape[1]
        ==
        test_array.shape[1]
    )

print(
    "✓ Transformed feature dimensionality is consistent."
)


# ==============================================================================
# 17.9 — ROW PRESERVATION VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    assert (
        TRANSFORMED_TRAIN_DATASETS[
            dataset_id
        ].shape[0]
        ==
        len(TRAIN_DATASETS[dataset_id])
    )

    assert (
        TRANSFORMED_VALIDATION_DATASETS[
            dataset_id
        ].shape[0]
        ==
        len(VALIDATION_DATASETS[dataset_id])
    )

    assert (
        TRANSFORMED_TEST_DATASETS[
            dataset_id
        ].shape[0]
        ==
        len(TEST_DATASETS[dataset_id])
    )

print(
    "✓ Row counts preserved for all datasets and splits."
)


# ==============================================================================
# 17.10 — FINITE-VALUE VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    for array in [
        TRANSFORMED_TRAIN_DATASETS[
            dataset_id
        ],
        TRANSFORMED_VALIDATION_DATASETS[
            dataset_id
        ],
        TRANSFORMED_TEST_DATASETS[
            dataset_id
        ],
    ]:

        assert np.isfinite(array).all()

print(
    "✓ No NaN or infinite values detected."
)


# ==============================================================================
# 17.11 — DTYPE VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    for array in [
        TRANSFORMED_TRAIN_DATASETS[
            dataset_id
        ],
        TRANSFORMED_VALIDATION_DATASETS[
            dataset_id
        ],
        TRANSFORMED_TEST_DATASETS[
            dataset_id
        ],
    ]:

        assert array.dtype == np.float32

print(
    "✓ All transformed arrays are float32."
)


# ==============================================================================
# 17.12 — TRAINING-ONLY FIT VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    preprocessor = TRAIN_PREPROCESSORS[
        dataset_id
    ]

    if not hasattr(
        preprocessor,
        "transformers_"
    ):
        raise RuntimeError(
            f"{dataset_id}: preprocessor does not appear "
            "to be fitted."
        )

    expected_input_columns = list(
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]["all_columns"]
    )

    fitted_input_columns = list(
        preprocessor.feature_names_in_
    )

    if fitted_input_columns != (
        expected_input_columns
    ):
        raise RuntimeError(
            f"{dataset_id}: fitted preprocessor schema changed "
            "between Sections 16 and 17."
        )

print(
    "✓ All preprocessors were fitted before Section 17."
)

print(
    "✓ No preprocessing refitting performed in Section 17."
)


# ==============================================================================
# 17.13 — FINAL TARGET / IDENTIFIER / PROVENANCE POLICY VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]

    target_column = schema[
        "target_column"
    ]

    preprocessing_columns = list(
        schema["all_columns"]
    )

    identifier_columns = list(
        schema.get(
            "identifier_columns",
            []
        )
    )

    assert target_column not in (
        preprocessing_columns
    ), (
        f"{dataset_id}: target entered preprocessing."
    )

    assert PROVENANCE_COLUMN not in (
        preprocessing_columns
    ), (
        f"{dataset_id}: provenance entered preprocessing."
    )

    assert not any(
        col in preprocessing_columns
        for col in identifier_columns
    ), (
        f"{dataset_id}: identifier entered preprocessing."
    )

print(
    "✓ Target, identifiers, and provenance are excluded "
    "from preprocessing input."
)


# ==============================================================================
# 17.14 — FINAL PASS
# ==============================================================================

print()
print("=" * 100)
print("SECTION 17 — PASS")
print("=" * 100)

print()
print(
    "Training-fitted preprocessing was successfully applied to:"
)

print("  ✓ TRAIN")
print("  ✓ VALIDATION")
print("  ✓ TEST")

print()
print("Methodological confirmation:")

print(
    "  ✓ Preprocessors were fitted only on TRAIN."
)

print(
    "  ✓ Validation was transformed without refitting."
)

print(
    "  ✓ Test was transformed without refitting."
)

print(
    "  ✓ Target was excluded from preprocessing input."
)

print(
    "  ✓ Explicit identifiers were excluded."
)

print(
    "  ✓ __original_row_id__ was excluded."
)

print(
    "  ✓ Canonical preprocessing feature columns were used."
)

print(
    "  ✓ Row counts were preserved."
)

print(
    "  ✓ Output feature dimensionality was preserved."
)

print(
    "  ✓ No NaN or infinite values remain."
)

print(
    "  ✓ All transformed arrays are float32."
)

print()
print(
    "SECTION 17 COMPLETED SUCCESSFULLY."
)

SECTION 17 — TRANSFORM TRAIN / VALIDATION / TEST
✓ Required Section 17 objects detected.
✓ Identifier policy validated.
✓ Transformation output containers initialized.

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Preprocessing feature columns : 14
Target excluded               : income
Identifiers excluded          : 0
Provenance excluded           : __original_row_id__
Output encoded features       : 105
Train rows                    : 34,189
Validation rows               : 7,326
Test rows                     : 7,327
✓ Required target, provenance, and preprocessing feature columns verified for all three splits.

Transforming TRAIN...
✓ Train transformed: (34189, 105) | 13.69 MB
Transforming VALIDATION...
✓ Validation transformed: (7326, 105) | 2.93 MB
Transforming TEST...
✓ Test transformed: (7327, 105) | 2.

,dataset_id,train_rows,validation_rows,test_rows,input_preprocessing_features,target_excluded,transformed_features,train_memory_mb,validation_memory_mb,test_memory_mb,total_memory_mb,dtype,status
0,adult_income,34189,7326,7327,14,income,105,13.694172,2.934380,2.934780,19.563332,float32,SUCCESS
1,bank_marketing,31647,6782,6782,16,y,51,6.156910,1.319435,1.319435,8.795780,float32,SUCCESS
2,diabetes_130us,71236,15265,15265,47,readmitted,2336,634.793457,136.028442,136.028442,906.850342,float32,SUCCESS


✓ All datasets transformed.
✓ Transformed feature dimensionality is consistent.
✓ Row counts preserved for all datasets and splits.
✓ No NaN or infinite values detected.
✓ All transformed arrays are float32.
✓ All preprocessors were fitted before Section 17.
✓ No preprocessing refitting performed in Section 17.
✓ Target, identifiers, and provenance are excluded from preprocessing input.

SECTION 17 — PASS

Training-fitted preprocessing was successfully applied to:
  ✓ TRAIN
  ✓ VALIDATION
  ✓ TEST

Methodological confirmation:
  ✓ Preprocessors were fitted only on TRAIN.
  ✓ Validation was transformed without refitting.
  ✓ Test was transformed without refitting.
  ✓ Target was excluded from preprocessing input.
  ✓ Explicit identifiers were excluded.
  ✓ __original_row_id__ was excluded.
  ✓ Canonical preprocessing feature columns were used.
  ✓ Row counts were preserved.
  ✓ Output feature dimensionality was preserved.
  ✓ No NaN or infinite values remain.
  ✓ All transformed arrays 

In [43]:
# ==================================================================================================
# 18. CREATE NATIVE PROCESSED DATASETS
# ==================================================================================================

print("=" * 100)
print("18. CREATE NATIVE PROCESSED DATASETS")
print("=" * 100)


# ==================================================================================================
# 18.1 REQUIRED OBJECT VALIDATION
# ==================================================================================================

required_objects = [
    "DATASET_IDS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
    "TRAIN_PREPROCESSING_COLUMNS",
]

missing_objects = [
    obj
    for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 18 cannot proceed.\n"
        f"Missing required objects: {missing_objects}"
    )

print("✓ Required objects detected.")


# ==================================================================================================
# 18.2 REQUIRED LIBRARIES
# ==================================================================================================

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd


# ==================================================================================================
# 18.3 CANONICAL PROVENANCE COLUMN
# ==================================================================================================

PROVENANCE_COLUMN = "__original_row_id__"


# ==================================================================================================
# 18.4 DEFINE CANONICAL NATIVE OUTPUT DIRECTORY
# ==================================================================================================

if "NB02_DIRECTORIES" in globals():

    native_root = Path(
        NB02_DIRECTORIES["native"]
    )

else:

    native_root = (
        Path(PROJECT_ROOT)
        / "data"
        / "processed"
        / "notebook_02"
        / "native"
    )

native_root.mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"✓ Native output directory: {native_root}"
)


# ==================================================================================================
# 18.5 INITIALIZE NATIVE DATASET CONTAINER
# ==================================================================================================

NATIVE_FINAL_DATASETS = {}

print(
    "✓ Native dataset container initialized."
)


# ==================================================================================================
# 18.6 INITIALIZE NATIVE DATASET MANIFEST
# ==================================================================================================

NATIVE_DATASET_MANIFEST_RECORDS = []


# ==================================================================================================
# 18.7 SHA-256 HELPER
# ==================================================================================================

def _calculate_sha256(
    file_path,
    chunk_size=1024 * 1024
):

    sha256 = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file_handle:

        while True:

            chunk = file_handle.read(
                chunk_size
            )

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


# ==================================================================================================
# 18.8 CREATE NATIVE DATASETS
# ==================================================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    target = TARGET_COLUMNS[dataset_id]

    identifier_columns = IDENTIFIER_COLUMNS.get(
        dataset_id,
        []
    )


    # ----------------------------------------------------------------------------------------------
    # Obtain canonical Section 15 preprocessing schema
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in TRAIN_PREPROCESSING_COLUMNS:

        raise RuntimeError(
            f"{dataset_id}: preprocessing schema not found in "
            "TRAIN_PREPROCESSING_COLUMNS."
        )

    schema = TRAIN_PREPROCESSING_COLUMNS[dataset_id]


    # ----------------------------------------------------------------------------------------------
    # Canonical preprocessing feature columns
    # ----------------------------------------------------------------------------------------------

    preprocessing_columns = list(
        schema["all_columns"]
    )


    # ----------------------------------------------------------------------------------------------
    # Canonical generative columns
    #
    # Section 15 defines:
    #
    #   generative_columns = features + target
    #
    #   preprocessing columns = features only
    #
    # Native datasets must therefore use generative_columns.
    # ----------------------------------------------------------------------------------------------

    if "generative_columns" not in schema:

        raise RuntimeError(
            f"{dataset_id}: 'generative_columns' missing from "
            "TRAIN_PREPROCESSING_COLUMNS."
        )

    generative_columns = list(
        schema["generative_columns"]
    )


    print(
        f"Target                         : {target}"
    )

    print(
        f"Identifier columns             : {identifier_columns}"
    )

    print(
        f"Preprocessing feature columns  : "
        f"{len(preprocessing_columns)}"
    )

    print(
        f"Generative columns             : "
        f"{len(generative_columns)}"
    )


    # ----------------------------------------------------------------------------------------------
    # Validate generative schema
    # ----------------------------------------------------------------------------------------------

    if target not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target}' is missing from "
            "the canonical generative schema."
        )


    # Target must remain excluded from preprocessing input.

    if target in preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target}' incorrectly appears "
            "in preprocessing feature columns."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate preprocessing / generative relationship
    # ----------------------------------------------------------------------------------------------

    expected_generatives_from_preprocessing = (
        preprocessing_columns + [target]
    )

    if set(generative_columns) != set(
        expected_generatives_from_preprocessing
    ):

        raise RuntimeError(
            f"{dataset_id}: generative schema is inconsistent.\n"
            f"Preprocessing features: {preprocessing_columns}\n"
            f"Target: {target}\n"
            f"Generative columns: {generative_columns}"
        )


    # Preserve canonical schema order from Section 15.

    if generative_columns[-1] != target:

        raise RuntimeError(
            f"{dataset_id}: target '{target}' is not the final "
            "column in the canonical generative schema."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate identifier exclusion
    # ----------------------------------------------------------------------------------------------

    identifier_overlap = set(
        identifier_columns
    ).intersection(
        generative_columns
    )

    if identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: identifier columns appear in the "
            f"canonical generative schema: "
            f"{sorted(identifier_overlap)}"
        )


    # ----------------------------------------------------------------------------------------------
    # Validate provenance exclusion
    # ----------------------------------------------------------------------------------------------

    if PROVENANCE_COLUMN in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column '{PROVENANCE_COLUMN}' "
            "must not be part of the generative modeling schema."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate duplicate columns
    # ----------------------------------------------------------------------------------------------

    if len(generative_columns) != len(
        set(generative_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: duplicate columns found in "
            "canonical generative schema."
        )


    # ----------------------------------------------------------------------------------------------
    # Canonical expected native schema
    #
    # Provenance is retained for auditability, but is NOT a
    # generative modeling feature.
    # ----------------------------------------------------------------------------------------------

    expected_native_columns = [
        PROVENANCE_COLUMN
    ] + generative_columns


    # ----------------------------------------------------------------------------------------------
    # Get canonical split DataFrames
    # ----------------------------------------------------------------------------------------------

    split_sources = {

        "train":
            TRAIN_DATASETS[dataset_id],

        "validation":
            VALIDATION_DATASETS[dataset_id],

        "test":
            TEST_DATASETS[dataset_id],
    }

    NATIVE_FINAL_DATASETS[dataset_id] = {}


    # ----------------------------------------------------------------------------------------------
    # Create dataset-specific output directory
    # ----------------------------------------------------------------------------------------------

    dataset_native_root = (
        native_root / dataset_id
    )

    dataset_native_root.mkdir(
        parents=True,
        exist_ok=True
    )


    # ----------------------------------------------------------------------------------------------
    # Process each split
    # ----------------------------------------------------------------------------------------------

    for split_name, original_split in split_sources.items():

        print(
            f"\n  Split: {split_name}"
        )


        # ------------------------------------------------------------------------------------------
        # Validate provenance
        # ------------------------------------------------------------------------------------------

        if PROVENANCE_COLUMN not in original_split.columns:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                f"provenance column '{PROVENANCE_COLUMN}' "
                "was not found."
            )


        provenance = original_split[
            PROVENANCE_COLUMN
        ].copy()


        if provenance.isna().any():

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "provenance contains missing values."
            )


        if not provenance.is_unique:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "provenance IDs are not unique."
            )


        # ------------------------------------------------------------------------------------------
        # Validate all generative columns exist
        # ------------------------------------------------------------------------------------------

        missing_generative_columns = [

            col
            for col in generative_columns
            if col not in original_split.columns
        ]

        if missing_generative_columns:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "required generative columns are missing:\n"
                f"{missing_generative_columns}"
            )


        # ------------------------------------------------------------------------------------------
        # Construct native generative data
        # ------------------------------------------------------------------------------------------

        native_features = original_split[
            generative_columns
        ].copy()


        # ------------------------------------------------------------------------------------------
        # Verify provenance is NOT already present
        # ------------------------------------------------------------------------------------------

        if PROVENANCE_COLUMN in native_features.columns:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                f"provenance column '{PROVENANCE_COLUMN}' "
                "unexpectedly appears inside the generative schema."
            )


        # ------------------------------------------------------------------------------------------
        # Add provenance exactly once
        # ------------------------------------------------------------------------------------------

        native_features.insert(
            0,
            PROVENANCE_COLUMN,
            provenance.to_numpy(
                copy=True
            )
        )


        # ------------------------------------------------------------------------------------------
        # Validate exact native schema
        # ------------------------------------------------------------------------------------------

        if list(native_features.columns) != (
            expected_native_columns
        ):

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "native column schema/order mismatch.\n"
                f"Expected: {expected_native_columns}\n"
                f"Found:    {list(native_features.columns)}"
            )


        # ------------------------------------------------------------------------------------------
        # Validate target retention
        # ------------------------------------------------------------------------------------------

        if target not in native_features.columns:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                f"target '{target}' was not retained."
            )


        # ------------------------------------------------------------------------------------------
        # Validate target was copied unchanged
        # ------------------------------------------------------------------------------------------

        if not np.array_equal(
            native_features[target].to_numpy(),
            original_split[target].to_numpy()
        ):

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "target values changed during native dataset construction."
            )


        # ------------------------------------------------------------------------------------------
        # Validate identifier exclusion
        # ------------------------------------------------------------------------------------------

        identifier_leaks = [

            col
            for col in identifier_columns
            if col in native_features.columns
        ]

        if identifier_leaks:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "identifier leakage detected:\n"
                f"{identifier_leaks}"
            )


        # ------------------------------------------------------------------------------------------
        # Validate duplicate columns
        # ------------------------------------------------------------------------------------------

        if len(native_features.columns) != len(
            set(native_features.columns)
        ):

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "duplicate columns detected."
            )


        # ------------------------------------------------------------------------------------------
        # Validate row count
        # ------------------------------------------------------------------------------------------

        if len(native_features) != len(
            original_split
        ):

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "row count changed during native dataset construction."
            )


        # ------------------------------------------------------------------------------------------
        # Validate provenance alignment
        # ------------------------------------------------------------------------------------------

        if not np.array_equal(
            native_features[
                PROVENANCE_COLUMN
            ].to_numpy(),

            original_split[
                PROVENANCE_COLUMN
            ].to_numpy()
        ):

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "provenance alignment changed."
            )


        # ------------------------------------------------------------------------------------------
        # Store canonical in-memory native dataset
        # ------------------------------------------------------------------------------------------

        NATIVE_FINAL_DATASETS[
            dataset_id
        ][split_name] = native_features


        # ------------------------------------------------------------------------------------------
        # Persist native dataset to Drive
        # ------------------------------------------------------------------------------------------

        output_path = (
            dataset_native_root
            / f"{split_name}.csv"
        )

        native_features.to_csv(
            output_path,
            index=False
        )


        # ------------------------------------------------------------------------------------------
        # Verify physical file exists
        # ------------------------------------------------------------------------------------------

        if not output_path.exists():

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                f"native dataset was not persisted: "
                f"{output_path}"
            )


        if output_path.stat().st_size == 0:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "persisted native dataset is empty."
            )


        # ------------------------------------------------------------------------------------------
        # Calculate physical artifact hash
        # ------------------------------------------------------------------------------------------

        sha256_hash = _calculate_sha256(
            output_path
        )


        # ------------------------------------------------------------------------------------------
        # Immediate reload validation
        # ------------------------------------------------------------------------------------------

        reloaded_native = pd.read_csv(
            output_path
        )


        reload_validation = (

            list(reloaded_native.columns)
            == expected_native_columns

            and
            len(reloaded_native)
            == len(native_features)

            and
            PROVENANCE_COLUMN
            in reloaded_native.columns

            and
            target
            in reloaded_native.columns

            and
            all(
                col not in reloaded_native.columns
                for col in identifier_columns
            )

            and
            reloaded_native[
                PROVENANCE_COLUMN
            ].is_unique
        )


        if not reload_validation:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "physical native CSV reload validation failed."
            )


        # ------------------------------------------------------------------------------------------
        # Store manifest record
        # ------------------------------------------------------------------------------------------

        NATIVE_DATASET_MANIFEST_RECORDS.append(
            {

                "dataset_id":
                    dataset_id,

                "split":
                    split_name,

                "relative_path":
                    str(
                        output_path.relative_to(
                            native_root
                        )
                    ),

                "absolute_path":
                    str(
                        output_path
                    ),

                "rows":
                    len(native_features),

                "columns":
                    len(native_features.columns),

                "preprocessing_feature_columns":
                    len(preprocessing_columns),

                "generative_columns":
                    len(generative_columns),

                "target_column":
                    target,

                "provenance_column":
                    PROVENANCE_COLUMN,

                "identifier_columns":
                    json.dumps(
                        identifier_columns
                    ),

                "provenance_present":
                    PROVENANCE_COLUMN
                    in native_features.columns,

                "target_present":
                    target
                    in native_features.columns,

                "identifiers_excluded":
                    len(identifier_leaks) == 0,

                "file_exists":
                    output_path.exists(),

                "file_size_bytes":
                    output_path.stat().st_size,

                "sha256":
                    sha256_hash,

                "reload_validation":
                    reload_validation,

                "status":
                    "PASS",
            }
        )


        # ------------------------------------------------------------------------------------------
        # Display split result
        # ------------------------------------------------------------------------------------------

        print(
            f"    Rows                     : "
            f"{len(native_features):,}"
        )

        print(
            f"    Generative columns       : "
            f"{len(generative_columns):,}"
        )

        print(
            f"    Native columns           : "
            f"{len(native_features.columns):,}"
        )

        print(
            f"    Provenance               : "
            f"{PROVENANCE_COLUMN}"
        )

        print(
            f"    Target retained          : PASS"
        )

        print(
            f"    Identifier exclusion     : PASS"
        )

        print(
            f"    Row alignment            : PASS"
        )

        print(
            f"    Physical persistence     : PASS"
        )

        print(
            f"    Reload validation        : PASS"
        )


# ==================================================================================================
# 18.9 GLOBAL NATIVE DATASET VALIDATION
# ==================================================================================================

print("\n" + "=" * 100)
print("18.9 GLOBAL NATIVE DATASET VALIDATION")
print("=" * 100)


for dataset_id in DATASET_IDS:

    for split_name in [
        "train",
        "validation",
        "test",
    ]:

        native_df = NATIVE_FINAL_DATASETS[
            dataset_id
        ][split_name]

        target = TARGET_COLUMNS[
            dataset_id
        ]

        identifier_columns = IDENTIFIER_COLUMNS.get(
            dataset_id,
            []
        )

        schema = TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]

        preprocessing_columns = list(
            schema["all_columns"]
        )

        generative_columns = list(
            schema["generative_columns"]
        )


        # ------------------------------------------------------------------------------------------
        # Expected native structure
        # ------------------------------------------------------------------------------------------

        expected_column_count = (
            len(generative_columns) + 1
        )

        if len(native_df.columns) != (
            expected_column_count
        ):

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                f"expected {expected_column_count} native columns, "
                f"found {len(native_df.columns)}."
            )


        expected_columns = [
            PROVENANCE_COLUMN
        ] + generative_columns


        if list(native_df.columns) != (
            expected_columns
        ):

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "native column order/schema mismatch."
            )


        # ------------------------------------------------------------------------------------------
        # Target
        # ------------------------------------------------------------------------------------------

        if target not in native_df.columns:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "target missing from native dataset."
            )


        if target in preprocessing_columns:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "target incorrectly appears in preprocessing columns."
            )


        # ------------------------------------------------------------------------------------------
        # Identifiers
        # ------------------------------------------------------------------------------------------

        leaked_identifiers = [

            col
            for col in identifier_columns
            if col in native_df.columns
        ]

        if leaked_identifiers:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                f"identifier leakage: "
                f"{leaked_identifiers}"
            )


        # ------------------------------------------------------------------------------------------
        # Provenance
        # ------------------------------------------------------------------------------------------

        if PROVENANCE_COLUMN not in native_df.columns:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "provenance column missing."
            )


        if native_df[
            PROVENANCE_COLUMN
        ].isna().any():

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "provenance contains missing values."
            )


        if not native_df[
            PROVENANCE_COLUMN
        ].is_unique:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "provenance is not unique."
            )


print(
    "✓ Native dataset schema validation : PASS"
)

print(
    "✓ Generative schema validation      : PASS"
)

print(
    "✓ Target retention                  : PASS"
)

print(
    "✓ Target exclusion from preprocessor: PASS"
)

print(
    "✓ Identifier exclusion              : PASS"
)

print(
    "✓ Provenance validation              : PASS"
)

print(
    "✓ Column-order validation            : PASS"
)


# ==================================================================================================
# 18.10 CREATE AND PERSIST NATIVE DATASET MANIFEST
# ==================================================================================================

print("\n" + "=" * 100)
print("18.10 CREATE NATIVE DATASET MANIFEST")
print("=" * 100)


NATIVE_DATASET_MANIFEST = pd.DataFrame(
    NATIVE_DATASET_MANIFEST_RECORDS
)


if NATIVE_DATASET_MANIFEST.empty:

    raise RuntimeError(
        "Native dataset manifest is empty."
    )


expected_manifest_rows = (
    len(DATASET_IDS) * 3
)


if len(NATIVE_DATASET_MANIFEST) != (
    expected_manifest_rows
):

    raise RuntimeError(
        "Native dataset manifest row count mismatch.\n"
        f"Expected: {expected_manifest_rows}\n"
        f"Found:    {len(NATIVE_DATASET_MANIFEST)}"
    )


NATIVE_DATASET_MANIFEST_PATH = (
    native_root
    / "native_dataset_manifest.csv"
)


NATIVE_DATASET_MANIFEST.to_csv(
    NATIVE_DATASET_MANIFEST_PATH,
    index=False
)


if not NATIVE_DATASET_MANIFEST_PATH.exists():

    raise RuntimeError(
        "Native dataset manifest was not persisted."
    )


print(
    f"✓ Manifest saved: "
    f"{NATIVE_DATASET_MANIFEST_PATH}"
)


# ==================================================================================================
# 18.11 PHYSICAL NATIVE ARTIFACT VERIFICATION
# ==================================================================================================

print("\n" + "=" * 100)
print("18.11 PHYSICAL NATIVE ARTIFACT VERIFICATION")
print("=" * 100)


required_native_files = []


for dataset_id in DATASET_IDS:

    for split_name in [
        "train",
        "validation",
        "test",
    ]:

        required_native_files.append(
            native_root
            / dataset_id
            / f"{split_name}.csv"
        )


missing_native_files = [

    str(path)
    for path in required_native_files
    if not path.exists()
]


if missing_native_files:

    raise RuntimeError(
        "Missing persisted native dataset artifacts:\n"
        + "\n".join(
            f"  - {path}"
            for path in missing_native_files
        )
    )


if not NATIVE_DATASET_MANIFEST_PATH.exists():

    raise RuntimeError(
        "native_dataset_manifest.csv is missing."
    )


# ----------------------------------------------------------------------------------------------
# Verify manifest hashes and physical files
# ----------------------------------------------------------------------------------------------

for _, manifest_row in (
    NATIVE_DATASET_MANIFEST.iterrows()
):

    artifact_path = Path(
        manifest_row["absolute_path"]
    )


    if not artifact_path.exists():

        raise RuntimeError(
            f"Manifest artifact does not exist: "
            f"{artifact_path}"
        )


    actual_hash = _calculate_sha256(
        artifact_path
    )

    expected_hash = str(
        manifest_row["sha256"]
    )


    if actual_hash != expected_hash:

        raise RuntimeError(
            "SHA-256 mismatch for native artifact:\n"
            f"  {artifact_path}\n"
            f"  Expected: {expected_hash}\n"
            f"  Actual:   {actual_hash}"
        )


    reloaded_df = pd.read_csv(
        artifact_path
    )


    dataset_id = manifest_row[
        "dataset_id"
    ]

    split_name = manifest_row[
        "split"
    ]

    target = TARGET_COLUMNS[
        dataset_id
    ]

    identifier_columns = IDENTIFIER_COLUMNS.get(
        dataset_id,
        []
    )

    generative_columns = list(
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]["generative_columns"]
    )


    expected_columns = [
        PROVENANCE_COLUMN
    ] + generative_columns


    if list(reloaded_df.columns) != (
        expected_columns
    ):

        raise RuntimeError(
            f"{dataset_id} | {split_name}: "
            "physical native column order mismatch."
        )


    if len(reloaded_df) != int(
        manifest_row["rows"]
    ):

        raise RuntimeError(
            f"{dataset_id} | {split_name}: "
            "physical native row count mismatch."
        )


    if target not in reloaded_df.columns:

        raise RuntimeError(
            f"{dataset_id} | {split_name}: "
            "target missing from persisted native dataset."
        )


    if PROVENANCE_COLUMN not in reloaded_df.columns:

        raise RuntimeError(
            f"{dataset_id} | {split_name}: "
            "provenance missing from persisted native dataset."
        )


    if not reloaded_df[
        PROVENANCE_COLUMN
    ].is_unique:

        raise RuntimeError(
            f"{dataset_id} | {split_name}: "
            "persisted provenance is not unique."
        )


    leaked_identifiers = [

        col
        for col in identifier_columns
        if col in reloaded_df.columns
    ]


    if leaked_identifiers:

        raise RuntimeError(
            f"{dataset_id} | {split_name}: "
            f"identifier leakage in persisted dataset: "
            f"{leaked_identifiers}"
        )


print(
    f"✓ Physical native CSV files verified : "
    f"{len(required_native_files)} / "
    f"{len(required_native_files)}"
)

print(
    "✓ Native manifest verified            : PASS"
)

print(
    "✓ SHA-256 artifact verification       : PASS"
)

print(
    "✓ Physical reload verification        : PASS"
)


# ==================================================================================================
# 18.12 FINAL SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("SECTION 18 COMPLETE")
print("=" * 100)


for dataset_id in DATASET_IDS:

    print(
        f"\nDataset: {dataset_id}"
    )


    for split_name in [
        "train",
        "validation",
        "test",
    ]:

        native_df = NATIVE_FINAL_DATASETS[
            dataset_id
        ][split_name]


        print(
            f"  {split_name:10s} : "
            f"{native_df.shape[0]:,} rows × "
            f"{native_df.shape[1]:,} columns"
        )


print(
    "\nCanonical output:"
)

print(
    "  ✓ NATIVE_FINAL_DATASETS"
)


print(
    "\nPhysical output:"
)

print(
    f"  ✓ {native_root}"
)

print(
    f"  ✓ {NATIVE_DATASET_MANIFEST_PATH}"
)


print(
    "\nNative dataset policy:"
)

print(
    "  ✓ Original/native feature values preserved"
)

print(
    "  ✓ Target retained in generative dataset"
)

print(
    "  ✓ Target excluded from preprocessing input"
)

print(
    "  ✓ Explicit identifiers excluded"
)

print(
    "  ✓ Provenance retained for auditability"
)

print(
    "  ✓ Provenance excluded from generative schema"
)

print(
    "  ✓ Provenance is first column"
)

print(
    "  ✓ No encoding"
)

print(
    "  ✓ No scaling"
)

print(
    "  ✓ No refitting"
)

print(
    "  ✓ No transformation of native values"
)

print(
    "  ✓ Physical CSV persistence"
)

print(
    "  ✓ Immediate reload validation"
)

print(
    "  ✓ SHA-256 artifact verification"
)


print(
    "\nSECTION 18 — PASS"
)

print("=" * 100)

18. CREATE NATIVE PROCESSED DATASETS
✓ Required objects detected.
✓ Native output directory: /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native
✓ Native dataset container initialized.

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Target                         : income
Identifier columns             : []
Preprocessing feature columns  : 14
Generative columns             : 15

  Split: train
    Rows                     : 34,189
    Generative columns       : 15
    Native columns           : 16
    Provenance               : __original_row_id__
    Target retained          : PASS
    Identifier exclusion     : PASS
    Row alignment            : PASS
    Physical persistence     : PASS
    Reload validation        : PASS

  Split: validation
    Rows                     : 7,326
    Gener

In [44]:
# ==============================================================================
# SECTION 19 — CREATE FINAL ENCODED DATASETS
# ==============================================================================

# Canonical structure:
#
# ENCODED_FINAL_DATASETS[dataset_id]["train"]
# ENCODED_FINAL_DATASETS[dataset_id]["validation"]
# ENCODED_FINAL_DATASETS[dataset_id]["test"]
#
# Encoded datasets contain ONLY the transformed preprocessing features.
#
# They MUST NOT contain:
#   - __original_row_id__
#   - explicit identifiers
#   - raw target
#   - manually appended target
#
# Important architectural distinction:
#
#   Section 15:
#       preprocessing columns = FEATURES ONLY
#       generative columns    = FEATURES + TARGET
#
#   Section 16:
#       preprocessor fitted on TRAIN FEATURES ONLY
#
#   Section 17:
#       transformed arrays = ENCODED FEATURES ONLY
#
#   Section 18:
#       native datasets = NATIVE FEATURES + TARGET + PROVENANCE
#
# Therefore, Section 19 does NOT claim that the target is represented
# inside the transformed feature arrays.
#
# Any later requirement for an encoded target must be handled explicitly
# outside the generic feature preprocessor.
# ==============================================================================

print("=" * 100)
print("19. CREATE FINAL ENCODED DATASETS")
print("=" * 100)


# ==============================================================================
# 19.1 REQUIRED OBJECTS
# ==============================================================================

required_objects = [
    "DATASET_IDS",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TRANSFORMED_TRAIN_DATASETS",
    "TRANSFORMED_VALIDATION_DATASETS",
    "TRANSFORMED_TEST_DATASETS",
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 19 cannot start. Missing required objects:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
    )

print("✓ Required canonical objects detected.")


# ==============================================================================
# 19.2 REQUIRED LIBRARIES
# ==============================================================================

import numpy as np
import pandas as pd


# ==============================================================================
# 19.3 RESET CANONICAL DATASET-FIRST STRUCTURE
# ==============================================================================

ENCODED_FINAL_DATASETS = {}

ENCODED_DATASET_SUMMARY = []


# ==============================================================================
# 19.4 REQUIRED SPLIT STRUCTURE
# ==============================================================================

required_splits = [
    "train",
    "validation",
    "test",
]


# ==============================================================================
# 19.5 BUILD ENCODED DATASETS
# ==============================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"DATASET : {dataset_id}")
    print("-" * 100)


    # ------------------------------------------------------------------------------
    # Canonical preprocessing schema
    # ------------------------------------------------------------------------------

    if dataset_id not in TRAIN_PREPROCESSING_COLUMNS:

        raise RuntimeError(
            f"{dataset_id}: missing from "
            "TRAIN_PREPROCESSING_COLUMNS."
        )

    schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]


    required_schema_keys = [
        "all_columns",
        "numeric_columns",
        "categorical_columns",
        "target_column",
        "identifier_columns",
        "provenance_column",
        "transformed_columns",
        "generative_columns",
    ]


    missing_schema_keys = [
        key
        for key in required_schema_keys
        if key not in schema
    ]


    if missing_schema_keys:

        raise RuntimeError(
            f"{dataset_id}: preprocessing schema is incomplete.\n"
            f"Missing keys: {missing_schema_keys}"
        )


    # ------------------------------------------------------------------------------
    # Canonical preprocessing feature columns
    #
    # Section 15 defines "all_columns" as preprocessing FEATURES ONLY.
    # ------------------------------------------------------------------------------

    preprocessing_columns = list(
        schema["all_columns"]
    )


    # ------------------------------------------------------------------------------
    # Canonical generative columns
    #
    # Generative schema = preprocessing features + target.
    # ------------------------------------------------------------------------------

    generative_columns = list(
        schema["generative_columns"]
    )


    target_column = TARGET_COLUMNS[
        dataset_id
    ]


    identifier_columns = IDENTIFIER_COLUMNS.get(
        dataset_id,
        []
    )


    provenance_column = schema[
        "provenance_column"
    ]


    transformed_columns = list(
        schema["transformed_columns"]
    )


    expected_feature_count = len(
        transformed_columns
    )


    # ------------------------------------------------------------------------------
    # Validate target consistency
    # ------------------------------------------------------------------------------

    if target_column != schema[
        "target_column"
    ]:

        raise RuntimeError(
            f"{dataset_id}: target mismatch between "
            "TARGET_COLUMNS and TRAIN_PREPROCESSING_COLUMNS.\n"
            f"TARGET_COLUMNS: {target_column}\n"
            f"Schema target:  {schema['target_column']}"
        )


    # ------------------------------------------------------------------------------
    # CRITICAL TARGET POLICY
    #
    # Target MUST be excluded from preprocessing input.
    # Target MUST remain in the generative schema.
    # Target MUST NOT be claimed to be part of transformed features.
    # ------------------------------------------------------------------------------

    if target_column in preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "incorrectly appears in preprocessing feature columns."
        )


    if target_column not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "is missing from the canonical generative schema."
        )


    if target_column in transformed_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "incorrectly appears in transformed feature names."
        )


    # ------------------------------------------------------------------------------
    # Validate preprocessing/generative relationship
    # ------------------------------------------------------------------------------

    expected_generative_columns = (
        preprocessing_columns + [target_column]
    )


    if generative_columns != (
        expected_generative_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: generative schema does not equal "
            "preprocessing features + target.\n"
            f"Expected: {expected_generative_columns}\n"
            f"Found:    {generative_columns}"
        )


    # ------------------------------------------------------------------------------
    # Enforce identifier exclusion
    # ------------------------------------------------------------------------------

    identifier_overlap = set(
        identifier_columns
    ).intersection(
        preprocessing_columns
    )


    if identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: identifier columns appear in "
            f"preprocessing feature columns: "
            f"{sorted(identifier_overlap)}"
        )


    # ------------------------------------------------------------------------------
    # Enforce provenance exclusion
    # ------------------------------------------------------------------------------

    if provenance_column in preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column "
            f"'{provenance_column}' appears in "
            "preprocessing feature columns."
        )


    if provenance_column in transformed_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column "
            f"'{provenance_column}' appears in "
            "transformed feature names."
        )


    # ------------------------------------------------------------------------------
    # Validate transformed feature schema
    # ------------------------------------------------------------------------------

    if expected_feature_count == 0:

        raise RuntimeError(
            f"{dataset_id}: transformed feature schema is empty."
        )


    if len(transformed_columns) != len(
        set(transformed_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: duplicate transformed "
            "feature names detected."
        )


    print(
        f"Target                         : "
        f"{target_column}"
    )

    print(
        f"Preprocessing features         : "
        f"{len(preprocessing_columns)}"
    )

    print(
        f"Generative columns             : "
        f"{len(generative_columns)}"
    )

    print(
        f"Transformed encoded features   : "
        f"{expected_feature_count}"
    )

    print(
        "Target in preprocessing input  : NOT PRESENT"
    )

    print(
        "Target in generative schema    : PASS"
    )

    print(
        "Target in transformed features : NOT PRESENT"
    )

    print(
        "Provenance in transformed data : NOT PRESENT"
    )

    print(
        "Identifiers in transformed data: NOT PRESENT"
    )


    # ------------------------------------------------------------------------------
    # Dataset-first canonical structure
    # ------------------------------------------------------------------------------

    ENCODED_FINAL_DATASETS[
        dataset_id
    ] = {}


    # ------------------------------------------------------------------------------
    # Canonical transformed arrays
    # ------------------------------------------------------------------------------

    split_arrays = {

        "train":
            TRANSFORMED_TRAIN_DATASETS[
                dataset_id
            ],

        "validation":
            TRANSFORMED_VALIDATION_DATASETS[
                dataset_id
            ],

        "test":
            TRANSFORMED_TEST_DATASETS[
                dataset_id
            ],
    }


    # ------------------------------------------------------------------------------
    # Canonical original split data
    # ------------------------------------------------------------------------------

    original_split_data = {

        "train":
            TRAIN_DATASETS[
                dataset_id
            ],

        "validation":
            VALIDATION_DATASETS[
                dataset_id
            ],

        "test":
            TEST_DATASETS[
                dataset_id
            ],
    }


    # ------------------------------------------------------------------------------
    # Process each split
    # ------------------------------------------------------------------------------

    for split_name in required_splits:

        transformed_array = split_arrays[
            split_name
        ]

        original_df = original_split_data[
            split_name
        ]


        print(
            f"\n  Split: {split_name}"
        )


        # --------------------------------------------------------------------------
        # Validate transformed array type
        # --------------------------------------------------------------------------

        if not isinstance(
            transformed_array,
            np.ndarray
        ):

            raise TypeError(
                f"{dataset_id} / {split_name}: "
                "expected NumPy array, got "
                f"{type(transformed_array).__name__}"
            )


        # --------------------------------------------------------------------------
        # Validate dimensionality
        # --------------------------------------------------------------------------

        if transformed_array.ndim != 2:

            raise ValueError(
                f"{dataset_id} / {split_name}: "
                f"expected 2D array, got "
                f"ndim={transformed_array.ndim}"
            )


        # --------------------------------------------------------------------------
        # Validate transformed feature count
        # --------------------------------------------------------------------------

        if transformed_array.shape[1] != (
            expected_feature_count
        ):

            raise ValueError(
                f"{dataset_id} / {split_name}: "
                f"transformed feature count "
                f"{transformed_array.shape[1]} != expected "
                f"{expected_feature_count}"
            )


        # --------------------------------------------------------------------------
        # Validate row count
        # --------------------------------------------------------------------------

        if transformed_array.shape[0] != (
            len(original_df)
        ):

            raise ValueError(
                f"{dataset_id} / {split_name}: "
                f"transformed rows "
                f"{transformed_array.shape[0]} != "
                f"original split rows "
                f"{len(original_df)}"
            )


        # --------------------------------------------------------------------------
        # Validate source transformed array values
        # --------------------------------------------------------------------------

        chunk_size = 10000


        for start in range(
            0,
            transformed_array.shape[0],
            chunk_size
        ):

            stop = min(
                start + chunk_size,
                transformed_array.shape[0]
            )


            if not np.isfinite(
                transformed_array[
                    start:stop
                ]
            ).all():

                raise ValueError(
                    f"{dataset_id} / {split_name}: "
                    "non-finite values detected in canonical "
                    f"transformed array at rows "
                    f"{start}:{stop}"
                )


        # --------------------------------------------------------------------------
        # Convert canonical transformed representation to float32
        # --------------------------------------------------------------------------

        encoded_array = np.asarray(
            transformed_array,
            dtype=np.float32
        )


        # --------------------------------------------------------------------------
        # Validate float32 conversion
        # --------------------------------------------------------------------------

        if encoded_array.dtype != (
            np.float32
        ):

            raise RuntimeError(
                f"{dataset_id} / {split_name}: "
                f"encoded dtype is "
                f"{encoded_array.dtype}, "
                "expected float32."
            )


        # --------------------------------------------------------------------------
        # Validate encoded values
        # --------------------------------------------------------------------------

        finite_pass = True


        for start in range(
            0,
            encoded_array.shape[0],
            chunk_size
        ):

            stop = min(
                start + chunk_size,
                encoded_array.shape[0]
            )


            if not np.isfinite(
                encoded_array[
                    start:stop
                ]
            ).all():

                finite_pass = False

                raise ValueError(
                    f"{dataset_id} / {split_name}: "
                    "non-finite encoded values detected "
                    f"in rows {start}:{stop}"
                )


        # --------------------------------------------------------------------------
        # Store ONLY transformed preprocessing features
        # --------------------------------------------------------------------------

        ENCODED_FINAL_DATASETS[
            dataset_id
        ][split_name] = encoded_array


        # --------------------------------------------------------------------------
        # Structural validation
        # --------------------------------------------------------------------------

        stored_array = (
            ENCODED_FINAL_DATASETS[
                dataset_id
            ][split_name]
        )


        expected_shape = (

            len(original_df),

            expected_feature_count
        )


        if stored_array.shape != (
            expected_shape
        ):

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                f"stored encoded shape "
                f"{stored_array.shape} != expected "
                f"{expected_shape}"
            )


        # --------------------------------------------------------------------------
        # Verify exact correspondence with canonical transformed array
        # --------------------------------------------------------------------------

        expected_encoded_array = np.asarray(
            transformed_array,
            dtype=np.float32
        )


        if not np.array_equal(
            stored_array,
            expected_encoded_array
        ):

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                "encoded dataset is not identical to "
                "the canonical transformed representation "
                "after float32 conversion."
            )


        # --------------------------------------------------------------------------
        # Verify target remains outside encoded feature representation
        #
        # The encoded array is numerical and has no named columns,
        # so this policy is enforced at schema level.
        # --------------------------------------------------------------------------

        if target_column in preprocessing_columns:

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                "target is present in preprocessing schema."
            )


        if target_column in transformed_columns:

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                "target appears in transformed feature schema."
            )


        # --------------------------------------------------------------------------
        # Record summary
        # --------------------------------------------------------------------------

        ENCODED_DATASET_SUMMARY.append(
            {

                "dataset_id":
                    dataset_id,

                "split":
                    split_name,

                "rows":
                    len(original_df),

                "preprocessing_features":
                    len(preprocessing_columns),

                "generative_columns":
                    len(generative_columns),

                "encoded_features":
                    expected_feature_count,

                "dtype":
                    str(stored_array.dtype),

                "target_column":
                    target_column,

                "target_in_preprocessing_schema":
                    False,

                "target_in_generative_schema":
                    True,

                "target_in_transformed_schema":
                    False,

                "raw_target_manually_appended":
                    False,

                "identifier_columns_excluded":
                    len(identifier_overlap) == 0,

                "provenance_excluded":
                    provenance_column
                    not in transformed_columns,

                "transformed_encoded_consistent":
                    True,

                "finite_values":
                    finite_pass,

                "status":
                    "PASS",
            }
        )


        # --------------------------------------------------------------------------
        # Display split result
        # --------------------------------------------------------------------------

        print(
            f"    Shape                    : "
            f"{stored_array.shape}"
        )

        print(
            f"    Dtype                    : "
            f"{stored_array.dtype}"
        )

        print(
            "    Feature count            : PASS"
        )

        print(
            "    Row count                : PASS"
        )

        print(
            "    Finite values            : PASS"
        )

        print(
            "    Target manually appended : NO"
        )

        print(
            "    Transformed consistency  : PASS"
        )


# ==============================================================================
# 19.6 BUILD SUMMARY DATAFRAME
# ==============================================================================

ENCODED_DATASET_SUMMARY_DF = pd.DataFrame(
    ENCODED_DATASET_SUMMARY
)


if ENCODED_DATASET_SUMMARY_DF.empty:

    raise RuntimeError(
        "ENCODED_DATASET_SUMMARY_DF is empty."
    )


expected_summary_rows = (
    len(DATASET_IDS)
    * len(required_splits)
)


if len(
    ENCODED_DATASET_SUMMARY_DF
) != expected_summary_rows:

    raise RuntimeError(
        "Encoded dataset summary row count mismatch.\n"
        f"Expected: {expected_summary_rows}\n"
        f"Found:    "
        f"{len(ENCODED_DATASET_SUMMARY_DF)}"
    )


# ==============================================================================
# 19.7 FINAL STRUCTURE VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    if dataset_id not in (
        ENCODED_FINAL_DATASETS
    ):

        raise AssertionError(
            f"Missing dataset key: {dataset_id}"
        )


    dataset_container = (
        ENCODED_FINAL_DATASETS[
            dataset_id
        ]
    )


    if set(
        dataset_container.keys()
    ) != set(required_splits):

        raise AssertionError(
            f"{dataset_id}: incorrect split structure.\n"
            f"Found:    "
            f"{list(dataset_container.keys())}\n"
            f"Expected: "
            f"{required_splits}"
        )


# ==============================================================================
# 19.8 VERIFY NO TARGET / PROVENANCE / IDENTIFIER LEAKAGE
# ==============================================================================

for dataset_id in DATASET_IDS:

    schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]

    target_column = TARGET_COLUMNS[
        dataset_id
    ]

    identifier_columns = IDENTIFIER_COLUMNS.get(
        dataset_id,
        []
    )

    provenance_column = schema[
        "provenance_column"
    ]

    preprocessing_columns = list(
        schema["all_columns"]
    )

    transformed_columns = list(
        schema["transformed_columns"]
    )


    # ------------------------------------------------------------------------------
    # Target must be excluded from preprocessing schema
    # ------------------------------------------------------------------------------

    if target_column in preprocessing_columns:

        raise AssertionError(
            f"{dataset_id}: target column appears "
            "in preprocessing feature schema."
        )


    # ------------------------------------------------------------------------------
    # Target must exist in generative schema
    # ------------------------------------------------------------------------------

    if target_column not in schema[
        "generative_columns"
    ]:

        raise AssertionError(
            f"{dataset_id}: target column is absent "
            "from generative schema."
        )


    # ------------------------------------------------------------------------------
    # Target must NOT appear in transformed feature schema
    # ------------------------------------------------------------------------------

    if target_column in transformed_columns:

        raise AssertionError(
            f"{dataset_id}: target column appears "
            "in transformed feature schema."
        )


    # ------------------------------------------------------------------------------
    # Provenance must be excluded
    # ------------------------------------------------------------------------------

    if provenance_column in preprocessing_columns:

        raise AssertionError(
            f"{dataset_id}: provenance column appears "
            "in preprocessing schema."
        )


    if provenance_column in transformed_columns:

        raise AssertionError(
            f"{dataset_id}: provenance column appears "
            "in transformed feature schema."
        )


    # ------------------------------------------------------------------------------
    # Explicit identifiers must be excluded
    # ------------------------------------------------------------------------------

    identifier_leaks = [

        identifier
        for identifier in identifier_columns

        if (
            identifier in preprocessing_columns
            or identifier in transformed_columns
        )
    ]


    if identifier_leaks:

        raise AssertionError(
            f"{dataset_id}: identifier leakage detected: "
            f"{identifier_leaks}"
        )


    # ------------------------------------------------------------------------------
    # Encoded array-level validation
    # ------------------------------------------------------------------------------

    for split_name in required_splits:

        encoded_array = (
            ENCODED_FINAL_DATASETS[
                dataset_id
            ][split_name]
        )


        if not isinstance(
            encoded_array,
            np.ndarray
        ):

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                "encoded dataset must be NumPy ndarray."
            )


        if encoded_array.ndim != 2:

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                "encoded dataset must be 2-dimensional."
            )


        if encoded_array.shape[1] != len(
            transformed_columns
        ):

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                "encoded array contains unexpected "
                "feature count."
            )


        if encoded_array.dtype != (
            np.float32
        ):

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                f"expected float32, found "
                f"{encoded_array.dtype}"
            )


        if not np.isfinite(
            encoded_array
        ).all():

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                "encoded array contains NaN or infinite values."
            )


# ==============================================================================
# 19.9 CROSS-SPLIT ROW VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    expected_rows = {

        "train":
            len(
                TRAIN_DATASETS[
                    dataset_id
                ]
            ),

        "validation":
            len(
                VALIDATION_DATASETS[
                    dataset_id
                ]
            ),

        "test":
            len(
                TEST_DATASETS[
                    dataset_id
                ]
            ),
    }


    for split_name in required_splits:

        actual_rows = len(
            ENCODED_FINAL_DATASETS[
                dataset_id
            ][split_name]
        )


        if actual_rows != (
            expected_rows[
                split_name
            ]
        ):

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                f"encoded rows={actual_rows}, "
                f"expected="
                f"{expected_rows[split_name]}"
            )


# ==============================================================================
# 19.10 FINAL CONSISTENCY CHECK AGAINST TRANSFORMED ARRAYS
# ==============================================================================

transformed_sources = {

    "train":
        TRANSFORMED_TRAIN_DATASETS,

    "validation":
        TRANSFORMED_VALIDATION_DATASETS,

    "test":
        TRANSFORMED_TEST_DATASETS,
}


for dataset_id in DATASET_IDS:

    for split_name in required_splits:

        canonical_transformed = (
            transformed_sources[
                split_name
            ][dataset_id]
        )


        encoded_array = (
            ENCODED_FINAL_DATASETS[
                dataset_id
            ][split_name]
        )


        expected_encoded = np.asarray(
            canonical_transformed,
            dtype=np.float32
        )


        if not np.array_equal(
            encoded_array,
            expected_encoded
        ):

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                "final encoded dataset does not match "
                "the canonical transformed dataset."
            )


print(
    "✓ Encoded ↔ transformed consistency : PASS"
)


# ==============================================================================
# 19.11 FINAL DATASET SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("ENCODED DATASET SUMMARY")
print("=" * 100)


for dataset_id in DATASET_IDS:

    schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]

    print(
        f"\n{dataset_id}"
    )

    print(
        f"  Preprocessing features : "
        f"{len(schema['all_columns'])}"
    )

    print(
        f"  Generative columns     : "
        f"{len(schema['generative_columns'])}"
    )

    print(
        f"  Target                 : "
        f"{TARGET_COLUMNS[dataset_id]}"
    )

    print(
        "  Target encoded here    : NO"
    )


    for split_name in required_splits:

        array = (
            ENCODED_FINAL_DATASETS[
                dataset_id
            ][split_name]
        )


        print(
            f"  {split_name:<12}: "
            f"{array.shape} | "
            f"dtype={array.dtype}"
        )


# ==============================================================================
# 19.12 FINAL SECTION 19 STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 19 STATUS: PASS")
print("=" * 100)


print(
    "\n✓ Dataset-first encoded structure confirmed."
)

print(
    "✓ Only transformed preprocessing features are stored."
)

print(
    "✓ Target is excluded from preprocessing input."
)

print(
    "✓ Target is retained separately in the native generative dataset."
)

print(
    "✓ Target is NOT falsely claimed to be represented in "
    "the transformed feature arrays."
)

print(
    "✓ Raw target was NOT manually appended."
)

print(
    "✓ Provenance is NOT included in encoded datasets."
)

print(
    "✓ Explicit identifiers are NOT included in encoded datasets."
)

print(
    "✓ Encoded arrays are float32."
)

print(
    "✓ Encoded arrays are finite."
)

print(
    "✓ Encoded arrays match canonical transformed arrays."
)

print(
    "\nSTATUS: PASS"
)

19. CREATE FINAL ENCODED DATASETS
✓ Required canonical objects detected.

----------------------------------------------------------------------------------------------------
DATASET : adult_income
----------------------------------------------------------------------------------------------------
Target                         : income
Preprocessing features         : 14
Generative columns             : 15
Transformed encoded features   : 105
Target in preprocessing input  : NOT PRESENT
Target in generative schema    : PASS
Target in transformed features : NOT PRESENT
Provenance in transformed data : NOT PRESENT
Identifiers in transformed data: NOT PRESENT

  Split: train
    Shape                    : (34189, 105)
    Dtype                    : float32
    Feature count            : PASS
    Row count                : PASS
    Finite values            : PASS
    Target manually appended : NO
    Transformed consistency  : PASS

  Split: validation
    Shape                    : (7326

In [45]:
# ==============================================================================
# SECTION 20 — PERSIST PREPROCESSORS, SCHEMAS & METADATA
# ==============================================================================

# Canonical artifacts:
#
# preprocessors/
#   adult_income/
#       train_fitted_preprocessor.joblib
#   bank_marketing/
#       train_fitted_preprocessor.joblib
#   diabetes_130us/
#       train_fitted_preprocessor.joblib
#
# schemas/
#   adult_income/
#       preprocessing_schema.json
#   bank_marketing/
#       preprocessing_schema.json
#   diabetes_130us/
#       preprocessing_schema.json
#
# schemas/metadata/
#   adult_income_preprocessing_metadata.json
#   bank_marketing_preprocessing_metadata.json
#   diabetes_130us_preprocessing_metadata.json
#
# IMPORTANT ARCHITECTURE
# ----------------------
#
# preprocessing schema:
#
#   all_columns
#       = FEATURES ONLY
#
#   generative_columns
#       = FEATURES + TARGET
#
#   transformed_columns
#       = ENCODED FEATURES ONLY
#
# Target is therefore:
#
#   - retained in the generative/native schema
#   - excluded from the generic preprocessing input
#   - excluded from transformed feature arrays
#
# Provenance and explicit identifiers are excluded from model input.
#
# All preprocessors are fitted on TRAINING FEATURES ONLY.
# ==============================================================================

import os
import json
import joblib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd


print("=" * 100)
print("20. PERSIST PREPROCESSORS, SCHEMAS & METADATA")
print("=" * 100)


# ==============================================================================
# 20.1 REQUIRED OBJECTS
# ==============================================================================

required_objects = [
    "DATASET_IDS",
    "TRAIN_PREPROCESSORS",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TRAIN_PREPROCESSING_METADATA",
    "NB02_DIRECTORIES",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 20 cannot start. Missing required objects:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
    )

print("✓ Required canonical objects detected.")


# ==============================================================================
# 20.2 CANONICAL ROOTS
# ==============================================================================

preprocessors_root = Path(
    NB02_DIRECTORIES["preprocessors"]
)

schemas_root = Path(
    NB02_DIRECTORIES["schemas"]
)

metadata_root = (
    schemas_root / "metadata"
)


preprocessors_root.mkdir(
    parents=True,
    exist_ok=True
)

schemas_root.mkdir(
    parents=True,
    exist_ok=True
)

metadata_root.mkdir(
    parents=True,
    exist_ok=True
)


print(
    f"✓ Preprocessors root : {preprocessors_root}"
)

print(
    f"✓ Schemas root       : {schemas_root}"
)

print(
    f"✓ Metadata root      : {metadata_root}"
)


# ==============================================================================
# 20.3 RESET MANIFEST
# ==============================================================================

PREPROCESSOR_MANIFEST_RECORDS = []


# ==============================================================================
# 20.4 SAVE EACH DATASET
# ==============================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"PERSISTING : {dataset_id}")
    print("-" * 100)


    # ==========================================================================
    # 20.4.1 VALIDATE CANONICAL OBJECTS
    # ==========================================================================

    if dataset_id not in TRAIN_PREPROCESSORS:

        raise RuntimeError(
            f"{dataset_id}: missing from TRAIN_PREPROCESSORS."
        )


    if dataset_id not in TRAIN_PREPROCESSING_COLUMNS:

        raise RuntimeError(
            f"{dataset_id}: missing from "
            "TRAIN_PREPROCESSING_COLUMNS."
        )


    if dataset_id not in TRAIN_PREPROCESSING_METADATA:

        raise RuntimeError(
            f"{dataset_id}: missing from "
            "TRAIN_PREPROCESSING_METADATA."
        )


    preprocessor = (
        TRAIN_PREPROCESSORS[
            dataset_id
        ]
    )


    schema = (
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]
    )


    metadata = (
        TRAIN_PREPROCESSING_METADATA[
            dataset_id
        ]
    )


    # ==========================================================================
    # 20.4.2 DATASET CONFIGURATION
    # ==========================================================================

    target_column = TARGET_COLUMNS[
        dataset_id
    ]


    identifier_columns = IDENTIFIER_COLUMNS.get(
        dataset_id,
        []
    )


    provenance_column = (
        "__original_row_id__"
    )


    # ==========================================================================
    # 20.4.3 REQUIRED SCHEMA KEYS
    # ==========================================================================

    required_schema_keys = [

        "all_columns",

        "numeric_columns",

        "categorical_columns",

        "target_column",

        "identifier_columns",

        "provenance_column",

        "transformed_columns",

        "generative_columns",
    ]


    missing_schema_keys = [

        key
        for key in required_schema_keys

        if key not in schema
    ]


    if missing_schema_keys:

        raise RuntimeError(
            f"{dataset_id}: canonical preprocessing schema "
            f"is missing keys: {missing_schema_keys}"
        )


    # ==========================================================================
    # 20.4.4 CANONICAL COLUMN DEFINITIONS
    # ==========================================================================

    preprocessing_columns = list(
        schema["all_columns"]
    )


    numeric_columns = list(
        schema["numeric_columns"]
    )


    categorical_columns = list(
        schema["categorical_columns"]
    )


    generative_columns = list(
        schema["generative_columns"]
    )


    transformed_feature_names = list(
        schema["transformed_columns"]
    )


    # ==========================================================================
    # 20.4.5 PROVENANCE VALIDATION
    # ==========================================================================

    if schema[
        "provenance_column"
    ] != provenance_column:

        raise RuntimeError(
            f"{dataset_id}: unexpected provenance column.\n"
            f"Expected: {provenance_column}\n"
            f"Found:    {schema['provenance_column']}"
        )


    if provenance_column in preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column appears "
            "in preprocessing feature schema."
        )


    if provenance_column in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column appears "
            "in generative schema."
        )


    if provenance_column in transformed_feature_names:

        raise RuntimeError(
            f"{dataset_id}: provenance column appears "
            "in transformed feature schema."
        )


    # ==========================================================================
    # 20.4.6 TARGET VALIDATION
    # ==========================================================================

    if schema[
        "target_column"
    ] != target_column:

        raise RuntimeError(
            f"{dataset_id}: target mismatch.\n"
            f"TARGET_COLUMNS: {target_column}\n"
            f"Schema target:  {schema['target_column']}"
        )


    # --------------------------------------------------------------------------
    # CRITICAL:
    # Target MUST NOT be in preprocessing input.
    # --------------------------------------------------------------------------

    if target_column in preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "incorrectly appears in preprocessing feature columns."
        )


    # --------------------------------------------------------------------------
    # Target MUST be in generative schema.
    # --------------------------------------------------------------------------

    if target_column not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "is absent from generative schema."
        )


    # --------------------------------------------------------------------------
    # Target MUST NOT be in transformed features.
    # --------------------------------------------------------------------------

    if target_column in transformed_feature_names:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "incorrectly appears in transformed feature schema."
        )


    # ==========================================================================
    # 20.4.7 GENERATIVE SCHEMA VALIDATION
    # ==========================================================================

    expected_generative_columns = (
        preprocessing_columns
        + [target_column]
    )


    if generative_columns != (
        expected_generative_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: generative schema mismatch.\n"
            f"Expected: {expected_generative_columns}\n"
            f"Found:    {generative_columns}"
        )


    # ==========================================================================
    # 20.4.8 NUMERIC / CATEGORICAL SCHEMA VALIDATION
    # ==========================================================================

    if set(numeric_columns).intersection(
        categorical_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric and categorical "
            "feature schemas overlap."
        )


    if set(numeric_columns).union(
        categorical_columns
    ) != set(preprocessing_columns):

        raise RuntimeError(
            f"{dataset_id}: numeric + categorical columns "
            "do not exactly cover preprocessing features."
        )


    if target_column in numeric_columns:

        raise RuntimeError(
            f"{dataset_id}: target appears in numeric "
            "preprocessing columns."
        )


    if target_column in categorical_columns:

        raise RuntimeError(
            f"{dataset_id}: target appears in categorical "
            "preprocessing columns."
        )


    # ==========================================================================
    # 20.4.9 IDENTIFIER VALIDATION
    # ==========================================================================

    identifier_overlap = set(
        identifier_columns
    ).intersection(
        preprocessing_columns
    )


    if identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: identifiers appear in "
            f"preprocessing feature schema: "
            f"{sorted(identifier_overlap)}"
        )


    identifier_generative_overlap = set(
        identifier_columns
    ).intersection(
        generative_columns
    )


    if identifier_generative_overlap:

        raise RuntimeError(
            f"{dataset_id}: identifiers appear in "
            f"generative schema: "
            f"{sorted(identifier_generative_overlap)}"
        )


    # ==========================================================================
    # 20.4.10 TRANSFORMED FEATURE SCHEMA VALIDATION
    # ==========================================================================

    if not transformed_feature_names:

        raise RuntimeError(
            f"{dataset_id}: transformed feature schema is empty."
        )


    if len(
        transformed_feature_names
    ) != len(
        set(transformed_feature_names)
    ):

        raise RuntimeError(
            f"{dataset_id}: duplicate transformed "
            "feature names detected."
        )


    # ==========================================================================
    # 20.4.11 PREPROCESSOR OBJECT VALIDATION
    # ==========================================================================

    if not hasattr(
        preprocessor,
        "get_feature_names_out"
    ):

        raise RuntimeError(
            f"{dataset_id}: preprocessor does not expose "
            "get_feature_names_out()."
        )


    if not hasattr(
        preprocessor,
        "feature_names_in_"
    ):

        raise RuntimeError(
            f"{dataset_id}: fitted preprocessor does not "
            "expose feature_names_in_."
        )


    # --------------------------------------------------------------------------
    # Verify fitted input schema
    # --------------------------------------------------------------------------

    fitted_input_columns = list(
        preprocessor.feature_names_in_
    )


    if fitted_input_columns != (
        preprocessing_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: fitted preprocessor input schema "
            "does not match canonical preprocessing columns.\n"
            f"Expected: {preprocessing_columns}\n"
            f"Found:    {fitted_input_columns}"
        )


    # --------------------------------------------------------------------------
    # Verify target is absent from fitted input
    # --------------------------------------------------------------------------

    if target_column in fitted_input_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "is present in fitted preprocessor input."
        )


    # --------------------------------------------------------------------------
    # Verify identifiers absent from fitted input
    # --------------------------------------------------------------------------

    fitted_identifier_overlap = set(
        identifier_columns
    ).intersection(
        fitted_input_columns
    )


    if fitted_identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: identifier leakage detected "
            "in fitted preprocessor input: "
            f"{sorted(fitted_identifier_overlap)}"
        )


    # --------------------------------------------------------------------------
    # Verify provenance absent from fitted input
    # --------------------------------------------------------------------------

    if provenance_column in fitted_input_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance leakage detected "
            "in fitted preprocessor input."
        )


    # ==========================================================================
    # 20.4.12 PREPROCESSOR OUTPUT SCHEMA VALIDATION
    # ==========================================================================

    try:

        actual_transformed_feature_names = list(
            preprocessor.get_feature_names_out()
        )

    except Exception as exc:

        raise RuntimeError(
            f"{dataset_id}: unable to obtain transformed "
            f"feature names: {exc}"
        )


    if actual_transformed_feature_names != (
        transformed_feature_names
    ):

        raise RuntimeError(
            f"{dataset_id}: transformed feature schema mismatch "
            "between fitted preprocessor and canonical schema."
        )


    # ==========================================================================
    # 20.4.13 DATASET-SPECIFIC DIRECTORIES
    # ==========================================================================

    dataset_preprocessor_root = (
        preprocessors_root / dataset_id
    )


    dataset_schema_root = (
        schemas_root / dataset_id
    )


    dataset_preprocessor_root.mkdir(
        parents=True,
        exist_ok=True
    )


    dataset_schema_root.mkdir(
        parents=True,
        exist_ok=True
    )


    # ==========================================================================
    # 20.4.14 PREPROCESSOR ARTIFACT
    # ==========================================================================

    preprocessor_path = (
        dataset_preprocessor_root
        / "train_fitted_preprocessor.joblib"
    )


    joblib.dump(
        preprocessor,
        preprocessor_path
    )


    if not preprocessor_path.exists():

        raise RuntimeError(
            f"{dataset_id}: preprocessor artifact "
            "was not created."
        )


    if preprocessor_path.stat().st_size <= 0:

        raise RuntimeError(
            f"{dataset_id}: preprocessor artifact is empty."
        )


    # --------------------------------------------------------------------------
    # Immediate reload
    # --------------------------------------------------------------------------

    try:

        reloaded_preprocessor = joblib.load(
            preprocessor_path
        )

    except Exception as exc:

        raise RuntimeError(
            f"{dataset_id}: persisted preprocessor could "
            f"not be reloaded: {exc}"
        )


    if type(
        reloaded_preprocessor
    ) is not type(
        preprocessor
    ):

        raise RuntimeError(
            f"{dataset_id}: reloaded preprocessor type mismatch.\n"
            f"Original: {type(preprocessor).__name__}\n"
            f"Reloaded: {type(reloaded_preprocessor).__name__}"
        )


    # --------------------------------------------------------------------------
    # Reloaded input schema
    # --------------------------------------------------------------------------

    if list(
        reloaded_preprocessor.feature_names_in_
    ) != preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: reloaded preprocessor input "
            "schema mismatch."
        )


    # --------------------------------------------------------------------------
    # Reloaded output schema
    # --------------------------------------------------------------------------

    reloaded_feature_names = list(
        reloaded_preprocessor.get_feature_names_out()
    )


    if reloaded_feature_names != (
        transformed_feature_names
    ):

        raise RuntimeError(
            f"{dataset_id}: reloaded preprocessor "
            "feature schema mismatch."
        )


    # ==========================================================================
    # 20.4.15 CANONICAL PERSISTED SCHEMA
    # ==========================================================================

    training_rows = (

        int(
            len(
                TRAIN_PREPROCESSING_DATA[
                    dataset_id
                ]
            )
        )

        if "TRAIN_PREPROCESSING_DATA" in globals()

        else int(
            metadata.get(
                "training_rows",
                0
            )
        )
    )


    if training_rows <= 0:

        raise RuntimeError(
            f"{dataset_id}: invalid training row count "
            f"{training_rows}."
        )


    canonical_schema = {

        "dataset_id":
            dataset_id,

        "schema_version":
            "3.0",

        "fit_policy":
            "train_only",

        "modeling_schema": {

            # FEATURES ONLY
            "preprocessing_columns":
                preprocessing_columns,

            # Backward-compatible explicit name
            "all_columns":
                preprocessing_columns,

            "numeric_columns":
                numeric_columns,

            "categorical_columns":
                categorical_columns,

            # FEATURES + TARGET
            "generative_columns":
                generative_columns,

            "target_column":
                target_column,

            "identifier_columns_excluded":
                list(identifier_columns),

            "provenance_column":
                provenance_column,
        },

        "target_policy": {

            "retained_in_generative_schema":
                True,

            "excluded_from_preprocessor_input":
                True,

            "excluded_from_transformed_features":
                True,

            "raw_target_manually_appended":
                False,
        },

        "identifier_policy": {

            "excluded_from_preprocessor_input":
                True,

            "excluded_from_generative_schema":
                True,

            "identifier_columns":
                list(identifier_columns),
        },

        "provenance_policy": {

            "column":
                provenance_column,

            "excluded_from_preprocessor_input":
                True,

            "excluded_from_transformed_features":
                True,

            "retained_for_auditability":
                True,
        },

        "transformation_schema": {

            "transformed_feature_count":
                len(transformed_feature_names),

            "transformed_feature_names":
                transformed_feature_names,
        },

        "preprocessing": {

            "numeric": {

                "imputation":
                    "median",

                "scaling":
                    "standard_scaler",
            },

            "categorical": {

                "imputation":
                    "most_frequent",

                "encoding":
                    "one_hot",

                "handle_unknown":
                    "ignore",
            },

            "column_transformer": {

                "remainder":
                    "drop",

                "verbose_feature_names_out":
                    False,
            },
        },

        "training_rows":
            training_rows,

        "creation_timestamp_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    # ==========================================================================
    # 20.4.16 SAVE SCHEMA
    # ==========================================================================

    schema_path = (
        dataset_schema_root
        / "preprocessing_schema.json"
    )


    with open(
        schema_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            canonical_schema,
            f,
            indent=2
        )


    if not schema_path.exists():

        raise RuntimeError(
            f"{dataset_id}: schema artifact "
            "was not created."
        )


    # ==========================================================================
    # 20.4.17 RELOAD & VALIDATE SCHEMA
    # ==========================================================================

    try:

        with open(
            schema_path,
            "r",
            encoding="utf-8"
        ) as f:

            reloaded_schema = json.load(f)

    except Exception as exc:

        raise RuntimeError(
            f"{dataset_id}: persisted schema could "
            f"not be reloaded: {exc}"
        )


    if reloaded_schema.get(
        "dataset_id"
    ) != dataset_id:

        raise RuntimeError(
            f"{dataset_id}: persisted dataset_id mismatch."
        )


    if reloaded_schema.get(
        "schema_version"
    ) != "3.0":

        raise RuntimeError(
            f"{dataset_id}: persisted schema version mismatch."
        )


    if reloaded_schema.get(
        "fit_policy"
    ) != "train_only":

        raise RuntimeError(
            f"{dataset_id}: persisted schema does not "
            "declare train_only fitting."
        )


    persisted_modeling = (
        reloaded_schema[
            "modeling_schema"
        ]
    )


    # --------------------------------------------------------------------------
    # Preprocessing features
    # --------------------------------------------------------------------------

    if persisted_modeling[
        "preprocessing_columns"
    ] != preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: persisted preprocessing "
            "column schema mismatch."
        )


    if persisted_modeling[
        "all_columns"
    ] != preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: persisted all_columns "
            "schema mismatch."
        )


    # --------------------------------------------------------------------------
    # Generative columns
    # --------------------------------------------------------------------------

    if persisted_modeling[
        "generative_columns"
    ] != generative_columns:

        raise RuntimeError(
            f"{dataset_id}: persisted generative "
            "schema mismatch."
        )


    # --------------------------------------------------------------------------
    # Numeric / categorical
    # --------------------------------------------------------------------------

    if persisted_modeling[
        "numeric_columns"
    ] != numeric_columns:

        raise RuntimeError(
            f"{dataset_id}: persisted numeric "
            "schema mismatch."
        )


    if persisted_modeling[
        "categorical_columns"
    ] != categorical_columns:

        raise RuntimeError(
            f"{dataset_id}: persisted categorical "
            "schema mismatch."
        )


    # --------------------------------------------------------------------------
    # Target
    # --------------------------------------------------------------------------

    if persisted_modeling[
        "target_column"
    ] != target_column:

        raise RuntimeError(
            f"{dataset_id}: persisted target schema mismatch."
        )


    if target_column in persisted_modeling[
        "preprocessing_columns"
    ]:

        raise RuntimeError(
            f"{dataset_id}: persisted schema incorrectly "
            "contains target in preprocessing columns."
        )


    if target_column not in persisted_modeling[
        "generative_columns"
    ]:

        raise RuntimeError(
            f"{dataset_id}: persisted schema incorrectly "
            "omits target from generative columns."
        )


    # --------------------------------------------------------------------------
    # Identifier / provenance
    # --------------------------------------------------------------------------

    if persisted_modeling[
        "identifier_columns_excluded"
    ] != list(identifier_columns):

        raise RuntimeError(
            f"{dataset_id}: persisted identifier "
            "schema mismatch."
        )


    if persisted_modeling[
        "provenance_column"
    ] != provenance_column:

        raise RuntimeError(
            f"{dataset_id}: persisted provenance "
            "schema mismatch."
        )


    # --------------------------------------------------------------------------
    # Target policy
    # --------------------------------------------------------------------------

    persisted_target_policy = (
        reloaded_schema[
            "target_policy"
        ]
    )


    if not persisted_target_policy[
        "retained_in_generative_schema"
    ]:

        raise RuntimeError(
            f"{dataset_id}: persisted target policy "
            "does not retain target in generative schema."
        )


    if not persisted_target_policy[
        "excluded_from_preprocessor_input"
    ]:

        raise RuntimeError(
            f"{dataset_id}: persisted target policy "
            "does not exclude target from preprocessor input."
        )


    if not persisted_target_policy[
        "excluded_from_transformed_features"
    ]:

        raise RuntimeError(
            f"{dataset_id}: persisted target policy "
            "does not exclude target from transformed features."
        )


    # --------------------------------------------------------------------------
    # Transformation schema
    # --------------------------------------------------------------------------

    persisted_transformed_names = (
        reloaded_schema[
            "transformation_schema"
        ][
            "transformed_feature_names"
        ]
    )


    if persisted_transformed_names != (
        transformed_feature_names
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted transformed "
            "feature names mismatch."
        )


    persisted_feature_count = (
        reloaded_schema[
            "transformation_schema"
        ][
            "transformed_feature_count"
        ]
    )


    if persisted_feature_count != len(
        transformed_feature_names
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted transformed "
            "feature count mismatch."
        )


    # ==========================================================================
    # 20.4.18 CANONICAL METADATA
    # ==========================================================================

    canonical_metadata = dict(
        metadata
    )


    canonical_metadata.update({

        "dataset_id":
            dataset_id,

        "fit_dataset":
            "train_only",

        "fit_policy":
            "train_only",

        "training_rows":
            training_rows,

        "preprocessor_artifact":
            str(preprocessor_path),

        "schema_artifact":
            str(schema_path),

        "preprocessing_feature_count":
            len(preprocessing_columns),

        "generative_column_count":
            len(generative_columns),

        "target_column":
            target_column,

        "target_retained_in_generative_schema":
            True,

        "target_excluded_from_preprocessor_input":
            True,

        "target_excluded_from_transformed_features":
            True,

        "raw_target_manually_appended":
            False,

        "identifier_columns":
            list(identifier_columns),

        "identifiers_excluded":
            True,

        "provenance_column":
            provenance_column,

        "provenance_excluded_from_model_input":
            True,

        "transformed_feature_count":
            len(transformed_feature_names),

        "transformed_feature_names":
            transformed_feature_names,

        "creation_timestamp_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    })


    metadata_path = (
        metadata_root
        / f"{dataset_id}_preprocessing_metadata.json"
    )


    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            canonical_metadata,
            f,
            indent=2
        )


    if not metadata_path.exists():

        raise RuntimeError(
            f"{dataset_id}: metadata artifact "
            "was not created."
        )


    # ==========================================================================
    # 20.4.19 RELOAD & VALIDATE METADATA
    # ==========================================================================

    try:

        with open(
            metadata_path,
            "r",
            encoding="utf-8"
        ) as f:

            reloaded_metadata = json.load(f)

    except Exception as exc:

        raise RuntimeError(
            f"{dataset_id}: persisted metadata could "
            f"not be reloaded: {exc}"
        )


    if reloaded_metadata.get(
        "dataset_id"
    ) != dataset_id:

        raise RuntimeError(
            f"{dataset_id}: persisted metadata "
            "dataset_id mismatch."
        )


    if reloaded_metadata.get(
        "fit_dataset"
    ) != "train_only":

        raise RuntimeError(
            f"{dataset_id}: persisted metadata does "
            "not declare fit_dataset='train_only'."
        )


    if reloaded_metadata.get(
        "fit_policy"
    ) != "train_only":

        raise RuntimeError(
            f"{dataset_id}: persisted metadata does "
            "not declare fit_policy='train_only'."
        )


    if reloaded_metadata.get(
        "target_column"
    ) != target_column:

        raise RuntimeError(
            f"{dataset_id}: persisted metadata "
            "target mismatch."
        )


    if reloaded_metadata.get(
        "target_retained_in_generative_schema"
    ) is not True:

        raise RuntimeError(
            f"{dataset_id}: persisted metadata "
            "target retention policy failed."
        )


    if reloaded_metadata.get(
        "target_excluded_from_preprocessor_input"
    ) is not True:

        raise RuntimeError(
            f"{dataset_id}: persisted metadata "
            "target exclusion policy failed."
        )


    if reloaded_metadata.get(
        "target_excluded_from_transformed_features"
    ) is not True:

        raise RuntimeError(
            f"{dataset_id}: persisted metadata "
            "target transformed-feature exclusion failed."
        )


    if reloaded_metadata.get(
        "transformed_feature_count"
    ) != len(transformed_feature_names):

        raise RuntimeError(
            f"{dataset_id}: persisted metadata "
            "transformed feature count mismatch."
        )


    if reloaded_metadata.get(
        "transformed_feature_names"
    ) != transformed_feature_names:

        raise RuntimeError(
            f"{dataset_id}: persisted metadata "
            "transformed feature names mismatch."
        )


    # ==========================================================================
    # 20.4.20 MANIFEST RECORD
    # ==========================================================================

    PREPROCESSOR_MANIFEST_RECORDS.append({

        "dataset_id":
            dataset_id,

        "preprocessor_path":
            str(preprocessor_path),

        "schema_path":
            str(schema_path),

        "metadata_path":
            str(metadata_path),

        "fit_dataset":
            "train_only",

        "fit_policy":
            "train_only",

        "training_rows":
            training_rows,

        "preprocessing_feature_count":
            len(preprocessing_columns),

        "generative_column_count":
            len(generative_columns),

        "target_column":
            target_column,

        "target_excluded_from_preprocessor":
            True,

        "target_retained_in_generative_schema":
            True,

        "transformed_feature_count":
            len(transformed_feature_names),

        "preprocessor_exists":
            True,

        "preprocessor_reload":
            True,

        "preprocessor_input_schema_verified":
            True,

        "schema_exists":
            True,

        "schema_reload":
            True,

        "metadata_exists":
            True,

        "metadata_reload":
            True,

        "status":
            "PASS",
    })


    # ==========================================================================
    # 20.4.21 DISPLAY RESULT
    # ==========================================================================

    print(
        f"  Preprocessor : "
        f"{preprocessor_path}"
    )

    print(
        "    Persistence                  : PASS"
    )

    print(
        "    Reload                       : PASS"
    )

    print(
        "    Training-only input schema  : PASS"
    )

    print(
        "    Target excluded from input  : PASS"
    )

    print(
        "    Identifier exclusion         : PASS"
    )

    print(
        "    Provenance exclusion         : PASS"
    )


    print(
        f"  Schema       : "
        f"{schema_path}"
    )

    print(
        "    Persistence                  : PASS"
    )

    print(
        "    Reload                       : PASS"
    )

    print(
        "    Generative schema            : PASS"
    )

    print(
        "    Target policy                : PASS"
    )


    print(
        f"  Metadata     : "
        f"{metadata_path}"
    )

    print(
        "    Persistence                  : PASS"
    )

    print(
        "    Reload                       : PASS"
    )

    print(
        "    Train-only policy            : PASS"
    )

    print(
        "    Target policy                : PASS"
    )


# ==============================================================================
# 20.5 BUILD & SAVE MANIFEST
# ==============================================================================

PREPROCESSOR_MANIFEST_DF = pd.DataFrame(
    PREPROCESSOR_MANIFEST_RECORDS
)


if PREPROCESSOR_MANIFEST_DF.empty:

    raise RuntimeError(
        "Preprocessor manifest is empty."
    )


expected_manifest_rows = len(
    DATASET_IDS
)


if len(
    PREPROCESSOR_MANIFEST_DF
) != expected_manifest_rows:

    raise RuntimeError(
        "Preprocessor manifest row count mismatch.\n"
        f"Expected: {expected_manifest_rows}\n"
        f"Found:    "
        f"{len(PREPROCESSOR_MANIFEST_DF)}"
    )


manifest_path = (
    schemas_root
    / "preprocessor_manifest.csv"
)


PREPROCESSOR_MANIFEST_DF.to_csv(
    manifest_path,
    index=False
)


if not manifest_path.exists():

    raise RuntimeError(
        "Preprocessor manifest was not persisted."
    )


# ==============================================================================
# 20.6 FINAL PHYSICAL ARTIFACT VERIFICATION
# ==============================================================================

print("\n" + "=" * 100)
print("20.6 FINAL PHYSICAL ARTIFACT VERIFICATION")
print("=" * 100)


for dataset_id in DATASET_IDS:

    preprocessor_path = (
        preprocessors_root
        / dataset_id
        / "train_fitted_preprocessor.joblib"
    )


    schema_path = (
        schemas_root
        / dataset_id
        / "preprocessing_schema.json"
    )


    metadata_path = (
        metadata_root
        / f"{dataset_id}_preprocessing_metadata.json"
    )


    if not preprocessor_path.exists():

        raise RuntimeError(
            f"{dataset_id}: preprocessor artifact missing."
        )


    if not schema_path.exists():

        raise RuntimeError(
            f"{dataset_id}: schema artifact missing."
        )


    if not metadata_path.exists():

        raise RuntimeError(
            f"{dataset_id}: metadata artifact missing."
        )


    # --------------------------------------------------------------------------
    # Physical reload
    # --------------------------------------------------------------------------

    physical_preprocessor = joblib.load(
        preprocessor_path
    )


    with open(
        schema_path,
        "r",
        encoding="utf-8"
    ) as f:

        physical_schema = json.load(f)


    with open(
        metadata_path,
        "r",
        encoding="utf-8"
    ) as f:

        physical_metadata = json.load(f)


    # --------------------------------------------------------------------------
    # Physical preprocessor type
    # --------------------------------------------------------------------------

    if type(
        physical_preprocessor
    ) is not type(
        TRAIN_PREPROCESSORS[
            dataset_id
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}: final physical preprocessor "
            "type validation failed."
        )


    # --------------------------------------------------------------------------
    # Physical fitted input schema
    # --------------------------------------------------------------------------

    physical_input_columns = list(
        physical_preprocessor.feature_names_in_
    )


    expected_input_columns = list(
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]["all_columns"]
    )


    if physical_input_columns != (
        expected_input_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: final physical preprocessor "
            "input schema validation failed."
        )


    # --------------------------------------------------------------------------
    # Physical target exclusion
    # --------------------------------------------------------------------------

    physical_target = TARGET_COLUMNS[
        dataset_id
    ]


    if physical_target in (
        physical_input_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: target appears in final "
            "physical preprocessor input."
        )


    # --------------------------------------------------------------------------
    # Physical transformed feature schema
    # --------------------------------------------------------------------------

    physical_feature_names = list(
        physical_preprocessor.get_feature_names_out()
    )


    if physical_feature_names != (
        physical_schema[
            "transformation_schema"
        ][
            "transformed_feature_names"
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}: final physical feature "
            "schema validation failed."
        )


    # --------------------------------------------------------------------------
    # Physical persisted generative schema
    # --------------------------------------------------------------------------

    physical_generative_columns = (
        physical_schema[
            "modeling_schema"
        ][
            "generative_columns"
        ]
    )


    expected_generative_columns = (
        list(
            TRAIN_PREPROCESSING_COLUMNS[
                dataset_id
            ]["all_columns"]
        )
        + [physical_target]
    )


    if physical_generative_columns != (
        expected_generative_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: final physical generative "
            "schema validation failed."
        )


    # --------------------------------------------------------------------------
    # Physical target policy
    # --------------------------------------------------------------------------

    if physical_schema[
        "target_policy"
    ][
        "retained_in_generative_schema"
    ] is not True:

        raise RuntimeError(
            f"{dataset_id}: physical target retention "
            "policy validation failed."
        )


    if physical_schema[
        "target_policy"
    ][
        "excluded_from_preprocessor_input"
    ] is not True:

        raise RuntimeError(
            f"{dataset_id}: physical target exclusion "
            "policy validation failed."
        )


    # --------------------------------------------------------------------------
    # Physical fit policy
    # --------------------------------------------------------------------------

    if physical_schema[
        "fit_policy"
    ] != "train_only":

        raise RuntimeError(
            f"{dataset_id}: final physical schema "
            "train-only validation failed."
        )


    if physical_metadata.get(
        "fit_dataset"
    ) != "train_only":

        raise RuntimeError(
            f"{dataset_id}: final physical metadata "
            "train-only validation failed."
        )


    if physical_metadata.get(
        "fit_policy"
    ) != "train_only":

        raise RuntimeError(
            f"{dataset_id}: final physical metadata "
            "fit-policy validation failed."
        )


    if physical_metadata.get(
        "target_column"
    ) != physical_target:

        raise RuntimeError(
            f"{dataset_id}: final physical metadata "
            "target validation failed."
        )


    if physical_metadata.get(
        "target_excluded_from_preprocessor_input"
    ) is not True:

        raise RuntimeError(
            f"{dataset_id}: final physical metadata "
            "target exclusion validation failed."
        )


    print(
        f"✓ {dataset_id:<20} "
        "preprocessor + schema + metadata : PASS"
    )


# ==============================================================================
# 20.7 MANIFEST PHYSICAL VERIFICATION
# ==============================================================================

if not manifest_path.exists():

    raise RuntimeError(
        "Preprocessor manifest does not exist."
    )


reloaded_manifest = pd.read_csv(
    manifest_path
)


if len(
    reloaded_manifest
) != len(DATASET_IDS):

    raise RuntimeError(
        "Persisted preprocessor manifest "
        "row count mismatch."
    )


required_manifest_columns = [

    "dataset_id",

    "preprocessor_path",

    "schema_path",

    "metadata_path",

    "fit_dataset",

    "fit_policy",

    "training_rows",

    "preprocessing_feature_count",

    "generative_column_count",

    "target_column",

    "target_excluded_from_preprocessor",

    "target_retained_in_generative_schema",

    "transformed_feature_count",

    "preprocessor_exists",

    "preprocessor_reload",

    "preprocessor_input_schema_verified",

    "schema_exists",

    "schema_reload",

    "metadata_exists",

    "metadata_reload",

    "status",
]


missing_manifest_columns = [

    col
    for col in required_manifest_columns

    if col not in reloaded_manifest.columns
]


if missing_manifest_columns:

    raise RuntimeError(
        "Persisted preprocessor manifest is missing columns:\n"
        + "\n".join(
            f"  - {col}"
            for col in missing_manifest_columns
        )
    )


if not (
    reloaded_manifest[
        "status"
    ] == "PASS"
).all():

    raise RuntimeError(
        "Persisted preprocessor manifest contains "
        "non-PASS records."
    )


if not (
    reloaded_manifest[
        "fit_policy"
    ] == "train_only"
).all():

    raise RuntimeError(
        "Persisted manifest contains a non-train-only "
        "fit policy."
    )


if not (
    reloaded_manifest[
        "target_excluded_from_preprocessor"
    ].astype(bool)
).all():

    raise RuntimeError(
        "Persisted manifest contains a target "
        "preprocessor-input inclusion."
    )


if not (
    reloaded_manifest[
        "target_retained_in_generative_schema"
    ].astype(bool)
).all():

    raise RuntimeError(
        "Persisted manifest does not confirm target "
        "retention in generative schema."
    )


print(
    "✓ Preprocessor manifest physical verification : PASS"
)


# ==============================================================================
# 20.8 FINAL SECTION 20 STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 20 STATUS: PASS")
print("=" * 100)


print(
    "\n✓ Train-fitted preprocessors persisted."
)

print(
    "✓ Persisted preprocessors successfully reloaded."
)

print(
    "✓ Preprocessor input schema physically verified."
)

print(
    "✓ Target excluded from preprocessing input."
)

print(
    "✓ Generative schema persisted with target retained."
)

print(
    "✓ Explicit identifiers excluded from model input."
)

print(
    "✓ Provenance excluded from model input."
)

print(
    "✓ Canonical preprocessing schemas persisted."
)

print(
    "✓ Persisted schemas successfully reloaded."
)

print(
    "✓ Target policy persisted and physically verified."
)

print(
    "✓ Canonical preprocessing metadata persisted."
)

print(
    "✓ Persisted metadata successfully reloaded."
)

print(
    "✓ Train-only fitting policy physically verified."
)

print(
    "✓ Transformed feature schema physically verified."
)

print(
    "✓ Preprocessor manifest persisted and reloaded."
)

print(
    "\nSTATUS: PASS"
)

print("=" * 100)

20. PERSIST PREPROCESSORS, SCHEMAS & METADATA
✓ Required canonical objects detected.
✓ Preprocessors root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors
✓ Schemas root       : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas
✓ Metadata root      : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/metadata

----------------------------------------------------------------------------------------------------
PERSISTING : adult_income
----------------------------------------------------------------------------------------------------
  Preprocessor : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors/adult_income/train_fitted_preprocessor.joblib
    Persistence                  : PASS
    Reload                       : PASS
    Training-only input schema  : PASS
    Target excluded from input  : PASS
    Identifier exclusion         : PASS
    Provenance exclusion         : 

In [46]:
# ==============================================================================
# SECTION 21 — FEATURE MAPPING
# ==============================================================================
#
# Purpose:
#   Build an explicit mapping between:
#
#       1. Original PREPROCESSING FEATURES
#       2. Numeric/categorical source features
#       3. Fitted transformed features
#
# Frozen architecture:
#   - Native generative dataset contains features + target.
#   - Generic preprocessing input contains FEATURES ONLY.
#   - Target is NOT part of the fitted preprocessing input.
#   - Target is NOT part of the transformed feature matrix.
#   - Target is retained separately in the native generative dataset.
#
# Core policy:
#   - Uses ONLY the fitted TRAIN preprocessor.
#   - Never refits preprocessing.
#   - Explicit identifiers are excluded.
#   - __original_row_id__ is excluded.
#   - Target is excluded from preprocessing feature mapping.
#   - Mapping is derived from the ACTUAL fitted ColumnTransformer.
#   - ColumnTransformer output_indices_ is the positional source of truth.
#   - Transformer names are NOT hardcoded.
#   - Stored schema is used for validation, not reconstruction.
#
# Outputs:
#   feature_mapping.csv
#   feature_mapping_summary.csv
#   feature_mapping_metadata.json
#
# ==============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

print("=" * 100)
print("21. FEATURE MAPPING")
print("=" * 100)


# ==============================================================================
# 21.1 REQUIRED OBJECTS
# ==============================================================================

REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_PREPROCESSORS",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
    "NB02_DIRECTORIES",
]

missing_objects = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 21 cannot run.\n"
        f"Missing required objects: {missing_objects}"
    )

print("Required objects : PASS")


# ==============================================================================
# 21.2 FEATURE MAPPING ROOT
# ==============================================================================

FEATURE_MAPPING_ROOT = Path(
    NB02_DIRECTORIES["feature_mapping"]
)

FEATURE_MAPPING_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("\nFeature mapping root:")
print(f"  {FEATURE_MAPPING_ROOT}")


# ==============================================================================
# 21.3 RESET OUTPUT OBJECTS
# ==============================================================================

FEATURE_MAPPING_RECORDS = []
FEATURE_MAPPING_SUMMARY_RECORDS = []


# ==============================================================================
# 21.4 HELPER — RESOLVE COLUMN SPECIFICATION
# ==============================================================================

def _resolve_column_names(
    column_spec,
    feature_names_in
):
    """
    Resolve a fitted sklearn ColumnTransformer column specification
    into explicit source column names.

    Supports:
        - string column names
        - integer positions
        - lists / tuples / arrays
        - boolean masks
        - slices

    The fitted transformer is never modified.
    """

    feature_names_in = list(feature_names_in)

    # --------------------------------------------------------------------------
    # Slice
    # --------------------------------------------------------------------------

    if isinstance(column_spec, slice):

        return feature_names_in[column_spec]


    # --------------------------------------------------------------------------
    # Single string
    # --------------------------------------------------------------------------

    if isinstance(column_spec, str):

        return [column_spec]


    # --------------------------------------------------------------------------
    # Scalar integer
    # --------------------------------------------------------------------------

    if isinstance(
        column_spec,
        (int, np.integer)
    ):

        return [
            feature_names_in[int(column_spec)]
        ]


    # --------------------------------------------------------------------------
    # Array-like
    # --------------------------------------------------------------------------

    try:

        values = list(column_spec)

    except TypeError:

        raise RuntimeError(
            "Unable to resolve ColumnTransformer column specification: "
            f"{column_spec!r}"
        )


    if len(values) == 0:

        return []


    # --------------------------------------------------------------------------
    # Boolean mask
    # --------------------------------------------------------------------------

    if all(
        isinstance(
            value,
            (bool, np.bool_)
        )
        for value in values
    ):

        if len(values) != len(
            feature_names_in
        ):

            raise RuntimeError(
                "Boolean column mask length mismatch.\n"
                f"Mask length   : {len(values)}\n"
                f"Feature count : {len(feature_names_in)}"
            )

        return [
            feature_names_in[idx]
            for idx, selected in enumerate(values)
            if bool(selected)
        ]


    # --------------------------------------------------------------------------
    # Integer positions
    # --------------------------------------------------------------------------

    if all(
        isinstance(
            value,
            (int, np.integer)
        )
        for value in values
    ):

        return [
            feature_names_in[int(value)]
            for value in values
        ]


    # --------------------------------------------------------------------------
    # Explicit names
    # --------------------------------------------------------------------------

    return [
        str(value)
        for value in values
    ]


# ==============================================================================
# 21.5 HELPER — LOCATE TRANSFORMER BY SOURCE COLUMNS
# ==============================================================================

def _find_transformer_by_columns(
    column_transformer,
    expected_columns,
    feature_names_in
):
    """
    Locate the fitted ColumnTransformer branch whose input columns correspond
    exactly to expected_columns.

    Transformer names are deliberately ignored.
    """

    expected_columns = list(
        expected_columns
    )

    if not hasattr(
        column_transformer,
        "transformers_"
    ):

        raise RuntimeError(
            "Fitted ColumnTransformer does not expose transformers_."
        )

    matches = []

    for transformer_name, transformer, columns in (
        column_transformer.transformers_
    ):

        if transformer_name == "remainder":
            continue

        if transformer == "drop":
            continue

        if transformer == "passthrough":
            continue

        resolved_columns = _resolve_column_names(
            columns,
            feature_names_in
        )

        if resolved_columns == expected_columns:

            matches.append(
                (
                    transformer_name,
                    transformer,
                    resolved_columns
                )
            )

    if len(matches) == 0:

        raise RuntimeError(
            "Could not locate fitted transformer for expected columns.\n"
            f"Expected columns: {expected_columns}"
        )

    if len(matches) > 1:

        raise RuntimeError(
            "Multiple fitted transformers matched the same source columns.\n"
            f"Expected columns: {expected_columns}\n"
            f"Matches: {[match[0] for match in matches]}"
        )

    return matches[0]


# ==============================================================================
# 21.6 HELPER — FIND ONEHOTENCODER
# ==============================================================================

def _find_onehot_encoder(
    transformer
):
    """
    Locate the fitted OneHotEncoder inside a direct transformer or Pipeline.
    """

    if transformer.__class__.__name__ == "OneHotEncoder":

        return transformer


    if hasattr(
        transformer,
        "steps"
    ):

        for _, step in transformer.steps:

            if step.__class__.__name__ == "OneHotEncoder":

                return step


    return None


# ==============================================================================
# 21.7 HELPER — FIND FITTED SCALER
# ==============================================================================

def _find_scaler(
    transformer
):
    """
    Locate the fitted StandardScaler inside a direct transformer or Pipeline.
    """

    if transformer.__class__.__name__ == "StandardScaler":

        return transformer


    if hasattr(
        transformer,
        "steps"
    ):

        for _, step in transformer.steps:

            if step.__class__.__name__ == "StandardScaler":

                return step


    return None


# ==============================================================================
# 21.8 HELPER — TRANSFORMER OUTPUT POSITIONS
# ==============================================================================

def _get_output_positions(
    column_transformer,
    transformer_name,
    expected_count
):
    """
    Obtain the global transformed-feature positions associated with a
    ColumnTransformer branch.

    output_indices_ is the preferred source because it reflects the actual
    fitted ColumnTransformer output layout.
    """

    if not hasattr(
        column_transformer,
        "output_indices_"
    ):

        raise RuntimeError(
            "Fitted ColumnTransformer does not expose output_indices_."
        )

    output_indices = (
        column_transformer.output_indices_
    )

    if transformer_name not in output_indices:

        raise RuntimeError(
            "Transformer missing from output_indices_.\n"
            f"Transformer: {transformer_name}\n"
            f"Available: {list(output_indices.keys())}"
        )

    index_spec = output_indices[
        transformer_name
    ]

    if isinstance(
        index_spec,
        slice
    ):

        positions = list(
            range(
                index_spec.start or 0,
                index_spec.stop,
                index_spec.step or 1
            )
        )

    else:

        positions = list(
            np.asarray(
                index_spec
            ).astype(int)
        )


    if len(positions) != expected_count:

        raise RuntimeError(
            "Transformer output-position count mismatch.\n"
            f"Transformer : {transformer_name}\n"
            f"Expected    : {expected_count}\n"
            f"Actual      : {len(positions)}"
        )

    return positions


# ==============================================================================
# 21.9 HELPER — CATEGORY OUTPUT VALUES
# ==============================================================================

def _get_encoder_category_values(
    encoder,
    source_columns
):
    """
    Reconstruct the category represented by every fitted OneHotEncoder output.

    Uses fitted categories_ and drop_idx_.

    No category values are reconstructed from the raw datasets.
    """

    categories = getattr(
        encoder,
        "categories_",
        None
    )

    if categories is None:

        raise RuntimeError(
            "Fitted OneHotEncoder does not expose categories_."
        )


    if len(categories) != len(
        source_columns
    ):

        raise RuntimeError(
            "OneHotEncoder category/source-column mismatch.\n"
            f"Source columns : {len(source_columns)}\n"
            f"Category sets  : {len(categories)}"
        )


    drop_idx = getattr(
        encoder,
        "drop_idx_",
        None
    )


    category_records = []


    for feature_index, source_column in enumerate(
        source_columns
    ):

        source_categories = list(
            categories[feature_index]
        )

        if drop_idx is not None:

            feature_drop_idx = (
                drop_idx[feature_index]
            )

        else:

            feature_drop_idx = None


        for category_index, category_value in enumerate(
            source_categories
        ):

            if (
                feature_drop_idx is not None
                and
                category_index == feature_drop_idx
            ):

                continue

            category_records.append(
                {
                    "source_feature":
                        source_column,

                    "category":
                        category_value
                }
            )


    return category_records


# ==============================================================================
# 21.10 BUILD MAPPING DATASET BY DATASET
# ==============================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"BUILDING FEATURE MAPPING : {dataset_id}")
    print("-" * 100)


    # ==========================================================================
    # Retrieve canonical preprocessing schema
    # ==========================================================================

    schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]


    # --------------------------------------------------------------------------
    # IMPORTANT:
    # "all_columns" is the FEATURE-ONLY preprocessing input.
    # It does NOT contain the target.
    # --------------------------------------------------------------------------

    preprocessing_feature_columns = list(
        schema["all_columns"]
    )

    numeric_columns = list(
        schema["numeric_columns"]
    )

    categorical_columns = list(
        schema["categorical_columns"]
    )

    generative_columns = list(
        schema.get(
            "generative_columns",
            []
        )
    )

    target_column = TARGET_COLUMNS[
        dataset_id
    ]

    identifier_columns = list(
        IDENTIFIER_COLUMNS.get(
            dataset_id,
            []
        )
    )

    provenance_column = schema.get(
        "provenance_column",
        "__original_row_id__"
    )

    expected_transformed_columns = list(
        schema.get(
            "transformed_columns",
            []
        )
    )


    print(
        f"  Preprocessing features : "
        f"{len(preprocessing_feature_columns)}"
    )

    print(
        f"  Numeric features       : "
        f"{len(numeric_columns)}"
    )

    print(
        f"  Categorical features   : "
        f"{len(categorical_columns)}"
    )

    print(
        f"  Generative columns     : "
        f"{len(generative_columns)}"
    )

    print(
        f"  Expected transformed   : "
        f"{len(expected_transformed_columns)}"
    )


    # ==========================================================================
    # Validate frozen schema semantics
    # ==========================================================================

    # Target MUST be in generative schema.
    if target_column not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "is missing from generative schema."
        )


    # Target MUST NOT be in preprocessing input.
    if target_column in preprocessing_feature_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "incorrectly appears in preprocessing feature columns."
        )


    # Target MUST NOT be classified as numeric/categorical preprocessing input.
    if target_column in numeric_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "incorrectly appears in numeric preprocessing columns."
        )


    if target_column in categorical_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "incorrectly appears in categorical preprocessing columns."
        )


    # Identifiers MUST NOT be preprocessing inputs.
    identifier_overlap = (
        set(preprocessing_feature_columns)
        &
        set(identifier_columns)
    )

    if identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: identifier columns appear in preprocessing "
            f"feature columns: {sorted(identifier_overlap)}"
        )


    # Provenance MUST NOT be preprocessing input.
    if provenance_column in preprocessing_feature_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column appears in preprocessing "
            "feature columns."
        )


    # Numeric and categorical features must be disjoint.
    if set(numeric_columns).intersection(
        categorical_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric/categorical feature overlap detected."
        )


    # All preprocessing features must be classified.
    if (
        set(numeric_columns)
        |
        set(categorical_columns)
    ) != set(preprocessing_feature_columns):

        uncovered = sorted(
            set(preprocessing_feature_columns)
            -
            (
                set(numeric_columns)
                |
                set(categorical_columns)
            )
        )

        extra = sorted(
            (
                set(numeric_columns)
                |
                set(categorical_columns)
            )
            -
            set(preprocessing_feature_columns)
        )

        raise RuntimeError(
            f"{dataset_id}: preprocessing feature classification mismatch.\n"
            f"Unclassified : {uncovered}\n"
            f"Extra        : {extra}"
        )


    # Generative schema must equal feature-only preprocessing columns + target.
    expected_generative_columns = (
        list(preprocessing_feature_columns)
        +
        [target_column]
    )

    if (
        set(generative_columns)
        !=
        set(expected_generative_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: generative schema mismatch.\n"
            f"Expected feature + target : {expected_generative_columns}\n"
            f"Actual generative schema  : {generative_columns}"
        )


    # ==========================================================================
    # Retrieve fitted TRAIN preprocessor
    # ==========================================================================

    preprocessor = TRAIN_PREPROCESSORS[
        dataset_id
    ]

    if preprocessor is None:

        raise RuntimeError(
            f"{dataset_id}: TRAIN_PREPROCESSORS entry is None."
        )


    if not hasattr(
        preprocessor,
        "transformers_"
    ):

        raise RuntimeError(
            f"{dataset_id}: fitted ColumnTransformer not detected."
        )


    if not hasattr(
        preprocessor,
        "get_feature_names_out"
    ):

        raise RuntimeError(
            f"{dataset_id}: fitted preprocessor does not expose "
            "get_feature_names_out()."
        )


    if not hasattr(
        preprocessor,
        "feature_names_in_"
    ):

        raise RuntimeError(
            f"{dataset_id}: fitted preprocessor does not expose "
            "feature_names_in_."
        )


    feature_names_in = list(
        preprocessor.feature_names_in_
    )


    # ==========================================================================
    # Validate fitted input schema
    # ==========================================================================

    if feature_names_in != preprocessing_feature_columns:

        raise RuntimeError(
            f"{dataset_id}: fitted preprocessor input schema mismatch.\n"
            "Expected preprocessing features:\n"
            f"{preprocessing_feature_columns}\n\n"
            "Actual fitted input:\n"
            f"{feature_names_in}"
        )


    if target_column in feature_names_in:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "is present in fitted preprocessor input."
        )


    if any(
        column in feature_names_in
        for column in identifier_columns
    ):

        found = [
            column
            for column in identifier_columns
            if column in feature_names_in
        ]

        raise RuntimeError(
            f"{dataset_id}: identifier columns found in fitted "
            f"preprocessor input: {found}"
        )


    if provenance_column in feature_names_in:

        raise RuntimeError(
            f"{dataset_id}: provenance column found in fitted "
            "preprocessor input."
        )


    # ==========================================================================
    # Actual transformed feature names
    # ==========================================================================

    actual_transformed_columns = list(
        preprocessor.get_feature_names_out()
    )


    if len(
        actual_transformed_columns
    ) == 0:

        raise RuntimeError(
            f"{dataset_id}: fitted preprocessor returned no "
            "transformed feature names."
        )


    if expected_transformed_columns:

        if (
            actual_transformed_columns
            !=
            expected_transformed_columns
        ):

            raise RuntimeError(
                f"{dataset_id}: transformed feature schema mismatch.\n"
                "Stored preprocessing schema and fitted TRAIN "
                "preprocessor are not identical.\n\n"
                f"Expected:\n{expected_transformed_columns}\n\n"
                f"Actual:\n{actual_transformed_columns}"
            )

    else:

        expected_transformed_columns = (
            actual_transformed_columns.copy()
        )


    print(
        f"  Actual transformed    : "
        f"{len(actual_transformed_columns)}"
    )


    # ==========================================================================
    # Locate numeric transformer using actual source columns
    # ==========================================================================

    (
        numeric_transformer_name,
        numeric_transformer,
        numeric_source_columns
    ) = _find_transformer_by_columns(
        preprocessor,
        numeric_columns,
        feature_names_in
    )


    numeric_scaler = _find_scaler(
        numeric_transformer
    )


    if numeric_scaler is None:

        raise RuntimeError(
            f"{dataset_id}: fitted numeric transformer does not contain "
            "a StandardScaler."
        )


    print(
        f"  Numeric transformer   : "
        f"{numeric_transformer_name}"
    )


    # ==========================================================================
    # Locate categorical transformer using actual source columns
    # ==========================================================================

    (
        categorical_transformer_name,
        categorical_transformer,
        categorical_source_columns
    ) = _find_transformer_by_columns(
        preprocessor,
        categorical_columns,
        feature_names_in
    )


    categorical_encoder = _find_onehot_encoder(
        categorical_transformer
    )


    if categorical_encoder is None:

        raise RuntimeError(
            f"{dataset_id}: fitted categorical transformer does not contain "
            "a OneHotEncoder."
        )


    print(
        f"  Categorical transformer: "
        f"{categorical_transformer_name}"
    )

    print(
        f"  Categorical encoder    : "
        f"{type(categorical_encoder).__name__}"
    )


    # ==========================================================================
    # Obtain exact global output positions
    # ==========================================================================

    numeric_feature_names_local = list(
        numeric_transformer.get_feature_names_out(
            numeric_source_columns
        )
    )


    numeric_positions = _get_output_positions(
        preprocessor,
        numeric_transformer_name,
        len(numeric_feature_names_local)
    )


    categorical_feature_names_local = list(
        categorical_encoder.get_feature_names_out(
            categorical_source_columns
        )
    )


    categorical_positions = _get_output_positions(
        preprocessor,
        categorical_transformer_name,
        len(categorical_feature_names_local)
    )


    # ==========================================================================
    # Validate total branch coverage
    # ==========================================================================

    branch_positions = (
        numeric_positions
        +
        categorical_positions
    )


    if len(branch_positions) != len(
        set(branch_positions)
    ):

        raise RuntimeError(
            f"{dataset_id}: overlapping transformed positions detected "
            "between preprocessing branches."
        )


    if set(branch_positions) != set(
        range(
            len(actual_transformed_columns)
        )
    ):

        missing_positions = sorted(
            set(
                range(
                    len(actual_transformed_columns)
                )
            )
            -
            set(branch_positions)
        )

        extra_positions = sorted(
            set(branch_positions)
            -
            set(
                range(
                    len(actual_transformed_columns)
                )
            )
        )

        raise RuntimeError(
            f"{dataset_id}: preprocessing branch coverage mismatch.\n"
            f"Missing positions: {missing_positions}\n"
            f"Extra positions  : {extra_positions}"
        )


    # ==========================================================================
    # 21.11 NUMERIC FEATURE MAPPING
    # ==========================================================================

    if len(numeric_source_columns) != len(
        numeric_positions
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric source/output mismatch.\n"
            f"Source columns : {len(numeric_source_columns)}\n"
            f"Output columns : {len(numeric_positions)}"
        )


    for source_feature, transformed_position in zip(
        numeric_source_columns,
        numeric_positions
    ):

        transformed_feature = (
            actual_transformed_columns[
                transformed_position
            ]
        )


        # Target must never occur here.
        if source_feature == target_column:

            raise RuntimeError(
                f"{dataset_id}: target appeared in numeric "
                "feature mapping."
            )


        FEATURE_MAPPING_RECORDS.append(
            {
                "dataset_id":
                    dataset_id,

                "source_feature":
                    source_feature,

                "source_feature_type":
                    "numeric",

                "source_category":
                    None,

                "transformed_feature":
                    transformed_feature,

                "transformed_position":
                    int(transformed_position),

                "is_target":
                    False,

                "is_identifier":
                    False,

                "is_provenance":
                    False,

                "encoding":
                    "numeric_scaled",
            }
        )


    # ==========================================================================
    # 21.12 CATEGORICAL FEATURE MAPPING
    # ==========================================================================

    category_records = _get_encoder_category_values(
        categorical_encoder,
        categorical_source_columns
    )


    if len(category_records) != len(
        categorical_positions
    ):

        raise RuntimeError(
            f"{dataset_id}: categorical category/output mismatch.\n"
            f"Category records : {len(category_records)}\n"
            f"Output positions : {len(categorical_positions)}"
        )


    for category_record, transformed_position in zip(
        category_records,
        categorical_positions
    ):

        source_feature = category_record[
            "source_feature"
        ]

        category_value = category_record[
            "category"
        ]

        transformed_feature = (
            actual_transformed_columns[
                transformed_position
            ]
        )


        # Target must never occur here.
        if source_feature == target_column:

            raise RuntimeError(
                f"{dataset_id}: target appeared in categorical "
                "feature mapping."
            )


        FEATURE_MAPPING_RECORDS.append(
            {
                "dataset_id":
                    dataset_id,

                "source_feature":
                    source_feature,

                "source_feature_type":
                    "categorical",

                "source_category":
                    category_value,

                "transformed_feature":
                    transformed_feature,

                "transformed_position":
                    int(transformed_position),

                "is_target":
                    False,

                "is_identifier":
                    False,

                "is_provenance":
                    False,

                "encoding":
                    "one_hot",
            }
        )


    # ==========================================================================
    # 21.13 DATASET-LEVEL VALIDATION
    # ==========================================================================

    dataset_mapping = [
        record
        for record in FEATURE_MAPPING_RECORDS
        if record["dataset_id"] == dataset_id
    ]


    # --------------------------------------------------------------------------
    # Position uniqueness
    # --------------------------------------------------------------------------

    mapping_positions = [
        record["transformed_position"]
        for record in dataset_mapping
    ]


    if len(mapping_positions) != len(
        set(mapping_positions)
    ):

        duplicated_positions = sorted(
            pd.Series(mapping_positions)[
                pd.Series(
                    mapping_positions
                ).duplicated(
                    keep=False
                )
            ].unique()
        )

        raise RuntimeError(
            f"{dataset_id}: duplicate transformed positions detected: "
            f"{duplicated_positions}"
        )


    # --------------------------------------------------------------------------
    # Position coverage
    # --------------------------------------------------------------------------

    expected_positions = set(
        range(
            len(actual_transformed_columns)
        )
    )

    actual_positions = set(
        mapping_positions
    )


    if actual_positions != expected_positions:

        missing_positions = sorted(
            expected_positions
            -
            actual_positions
        )

        extra_positions = sorted(
            actual_positions
            -
            expected_positions
        )

        raise RuntimeError(
            f"{dataset_id}: transformed position coverage mismatch.\n"
            f"Missing positions : {missing_positions}\n"
            f"Extra positions   : {extra_positions}"
        )


    # --------------------------------------------------------------------------
    # Source feature coverage
    # --------------------------------------------------------------------------

    mapped_source_features = set(
        record["source_feature"]
        for record in dataset_mapping
    )


    if mapped_source_features != set(
        preprocessing_feature_columns
    ):

        missing_features = sorted(
            set(preprocessing_feature_columns)
            -
            mapped_source_features
        )

        extra_features = sorted(
            mapped_source_features
            -
            set(preprocessing_feature_columns)
        )

        raise RuntimeError(
            f"{dataset_id}: source feature coverage mismatch.\n"
            f"Missing : {missing_features}\n"
            f"Extra   : {extra_features}"
        )


    # --------------------------------------------------------------------------
    # Forbidden metadata validation
    # --------------------------------------------------------------------------

    forbidden = (
        set(identifier_columns)
        |
        {provenance_column}
        |
        {target_column}
    )


    forbidden_found = (
        mapped_source_features
        &
        forbidden
    )


    if forbidden_found:

        raise RuntimeError(
            f"{dataset_id}: forbidden metadata/target features "
            f"found in mapping: {sorted(forbidden_found)}"
        )


    # --------------------------------------------------------------------------
    # Target exclusion validation
    # --------------------------------------------------------------------------

    target_records = [
        record
        for record in dataset_mapping
        if record["source_feature"] == target_column
    ]


    if target_records:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "must NOT be represented in feature mapping."
        )


    # --------------------------------------------------------------------------
    # Mapping row count
    # --------------------------------------------------------------------------

    if len(dataset_mapping) != len(
        actual_transformed_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: mapping row count mismatch.\n"
            f"Mapping rows : {len(dataset_mapping)}\n"
            f"Transformed  : {len(actual_transformed_columns)}"
        )


    # --------------------------------------------------------------------------
    # Numeric/categorical type validation
    # --------------------------------------------------------------------------

    numeric_mapped_features = set(
        record["source_feature"]
        for record in dataset_mapping
        if record["source_feature_type"] == "numeric"
    )


    categorical_mapped_features = set(
        record["source_feature"]
        for record in dataset_mapping
        if record["source_feature_type"] == "categorical"
    )


    if numeric_mapped_features != set(
        numeric_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric feature mapping mismatch."
        )


    if categorical_mapped_features != set(
        categorical_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: categorical feature mapping mismatch."
        )


    # --------------------------------------------------------------------------
    # All mapping records must be preprocessing features only
    # --------------------------------------------------------------------------

    if any(
        record["is_target"]
        for record in dataset_mapping
    ):

        raise RuntimeError(
            f"{dataset_id}: mapping contains a target-marked feature."
        )


    if any(
        record["is_identifier"]
        for record in dataset_mapping
    ):

        raise RuntimeError(
            f"{dataset_id}: mapping contains an identifier."
        )


    if any(
        record["is_provenance"]
        for record in dataset_mapping
    ):

        raise RuntimeError(
            f"{dataset_id}: mapping contains provenance metadata."
        )


    # ==========================================================================
    # Dataset summary
    # ==========================================================================

    FEATURE_MAPPING_SUMMARY_RECORDS.append(
        {
            "dataset_id":
                dataset_id,

            "preprocessing_feature_count":
                len(preprocessing_feature_columns),

            "numeric_feature_count":
                len(numeric_columns),

            "categorical_feature_count":
                len(categorical_columns),

            "generative_column_count":
                len(generative_columns),

            "transformed_feature_count":
                len(actual_transformed_columns),

            "mapping_row_count":
                len(dataset_mapping),

            "target_column":
                target_column,

            "target_in_generative_schema":
                True,

            "target_in_feature_mapping":
                False,

            "target_in_preprocessor_input":
                False,

            "identifier_count_excluded":
                len(identifier_columns),

            "provenance_excluded":
                True,

            "position_coverage":
                True,

            "source_feature_coverage":
                True,

            "numeric_mapping_verified":
                True,

            "categorical_mapping_verified":
                True,

            "status":
                "PASS",
        }
    )


    print(
        f"  Mapping rows           : "
        f"{len(dataset_mapping)}"
    )

    print(
        "  Position coverage      : PASS"
    )

    print(
        "  Source feature coverage: PASS"
    )

    print(
        "  Target exclusion       : PASS"
    )

    print(
        "  Metadata exclusion     : PASS"
    )

    print(
        "  Numeric mapping        : PASS"
    )

    print(
        "  Categorical mapping    : PASS"
    )


# ==============================================================================
# 21.14 CREATE DATAFRAMES
# ==============================================================================

FEATURE_MAPPING_DF = pd.DataFrame(
    FEATURE_MAPPING_RECORDS
)

FEATURE_MAPPING_SUMMARY_DF = pd.DataFrame(
    FEATURE_MAPPING_SUMMARY_RECORDS
)


if FEATURE_MAPPING_DF.empty:

    raise RuntimeError(
        "FEATURE_MAPPING_DF is empty."
    )


if FEATURE_MAPPING_SUMMARY_DF.empty:

    raise RuntimeError(
        "FEATURE_MAPPING_SUMMARY_DF is empty."
    )


# ==============================================================================
# 21.15 GLOBAL VALIDATION
# ==============================================================================

expected_dataset_count = len(
    DATASET_IDS
)

actual_dataset_count = (
    FEATURE_MAPPING_DF[
        "dataset_id"
    ].nunique()
)


if actual_dataset_count != expected_dataset_count:

    raise RuntimeError(
        "Global dataset coverage mismatch.\n"
        f"Expected : {expected_dataset_count}\n"
        f"Actual   : {actual_dataset_count}"
    )


for dataset_id in DATASET_IDS:

    expected_count = len(
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ].get(
            "transformed_columns",
            []
        )
    )


    actual_count = int(
        (
            FEATURE_MAPPING_DF[
                "dataset_id"
            ]
            ==
            dataset_id
        ).sum()
    )


    if actual_count != expected_count:

        raise RuntimeError(
            f"{dataset_id}: final mapping count mismatch.\n"
            f"Expected : {expected_count}\n"
            f"Actual   : {actual_count}"
        )


# ==============================================================================
# 21.16 GLOBAL TARGET POLICY VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    dataset_rows = FEATURE_MAPPING_DF[
        FEATURE_MAPPING_DF["dataset_id"] == dataset_id
    ]


    target_column = TARGET_COLUMNS[
        dataset_id
    ]


    if target_column in set(
        dataset_rows["source_feature"]
    ):

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "incorrectly appears in feature mapping."
        )


    if dataset_rows["is_target"].any():

        raise RuntimeError(
            f"{dataset_id}: target-marked mapping rows detected."
        )


# ==============================================================================
# 21.17 SORT MAPPING
# ==============================================================================

FEATURE_MAPPING_DF = (
    FEATURE_MAPPING_DF
    .sort_values(
        by=[
            "dataset_id",
            "transformed_position"
        ]
    )
    .reset_index(
        drop=True
    )
)


FEATURE_MAPPING_SUMMARY_DF = (
    FEATURE_MAPPING_SUMMARY_DF
    .sort_values(
        by=[
            "dataset_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==============================================================================
# 21.18 SAVE FEATURE MAPPING
# ==============================================================================

FEATURE_MAPPING_CSV = (
    FEATURE_MAPPING_ROOT
    /
    "feature_mapping.csv"
)


FEATURE_MAPPING_SUMMARY_CSV = (
    FEATURE_MAPPING_ROOT
    /
    "feature_mapping_summary.csv"
)


FEATURE_MAPPING_METADATA_JSON = (
    FEATURE_MAPPING_ROOT
    /
    "feature_mapping_metadata.json"
)


FEATURE_MAPPING_DF.to_csv(
    FEATURE_MAPPING_CSV,
    index=False
)


FEATURE_MAPPING_SUMMARY_DF.to_csv(
    FEATURE_MAPPING_SUMMARY_CSV,
    index=False
)


# ==============================================================================
# 21.19 SAVE METADATA
# ==============================================================================

FEATURE_MAPPING_METADATA = {

    "artifact":
        "feature_mapping",

    "schema_version":
        "2.1",

    "creation_timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "dataset_ids":
        list(DATASET_IDS),

    "mapping_policy": {

        "fit_policy":
            "train_only",

        "preprocessor_source":
            "TRAIN_PREPROCESSORS",

        "mapping_source_of_truth":
            "fitted ColumnTransformer output_indices_",

        "source_schema":
            "TRAIN_PREPROCESSING_COLUMNS",

        "source_features_only":
            True,

        "target_excluded_from_feature_mapping":
            True,

        "target_excluded_from_preprocessor_input":
            True,

        "target_retained_in_native_generative_dataset":
            True,

        "identifier_columns_excluded":
            True,

        "provenance_column_excluded":
            True,

        "numeric_encoding":
            "verified fitted numeric transformer",

        "categorical_encoding":
            "verified fitted OneHotEncoder",

        "transformer_names_hardcoded":
            False,

        "transformer_names_inferred":
            True,

        "schema_used_for_validation":
            True,

        "schema_used_for_mapping_reconstruction":
            False,

        "mapping_represents":
            "preprocessing_feature_to_transformed_feature",
    },

    "artifacts": {

        "feature_mapping_csv":
            str(FEATURE_MAPPING_CSV),

        "feature_mapping_summary_csv":
            str(FEATURE_MAPPING_SUMMARY_CSV),

        "feature_mapping_metadata_json":
            str(FEATURE_MAPPING_METADATA_JSON),
    },

    "dataset_summary":
        FEATURE_MAPPING_SUMMARY_DF.to_dict(
            orient="records"
        ),
}


with open(
    FEATURE_MAPPING_METADATA_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FEATURE_MAPPING_METADATA,
        f,
        indent=2,
        ensure_ascii=False
    )


# ==============================================================================
# 21.20 ARTIFACT VALIDATION
# ==============================================================================

required_artifacts = {

    "feature_mapping":
        FEATURE_MAPPING_CSV,

    "feature_mapping_summary":
        FEATURE_MAPPING_SUMMARY_CSV,

    "feature_mapping_metadata":
        FEATURE_MAPPING_METADATA_JSON,
}


artifact_status = {}


for artifact_name, artifact_path in (
    required_artifacts.items()
):

    exists = artifact_path.exists()

    artifact_status[
        artifact_name
    ] = exists

    print(
        f"  {artifact_name:30s}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )


if not all(
    artifact_status.values()
):

    missing_artifacts = [
        name
        for name, exists
        in artifact_status.items()
        if not exists
    ]

    raise RuntimeError(
        "Section 21 artifact persistence failed.\n"
        f"Missing artifacts: {missing_artifacts}"
    )


# ==============================================================================
# 21.21 FINAL SECTION 21 VALIDATION
# ==============================================================================

print("\n" + "-" * 100)
print("SECTION 21 VALIDATION")
print("-" * 100)


print(
    f"Datasets mapped        : "
    f"{FEATURE_MAPPING_DF['dataset_id'].nunique()}"
)


print(
    f"Mapping rows           : "
    f"{len(FEATURE_MAPPING_DF)}"
)


print(
    f"Summary rows           : "
    f"{len(FEATURE_MAPPING_SUMMARY_DF)}"
)


print(
    f"Artifacts persisted    : "
    f"{all(artifact_status.values())}"
)


print(
    "\nFEATURE MAPPING CSV:"
)


print(
    f"  {FEATURE_MAPPING_CSV}"
)


print(
    "\nFEATURE MAPPING SUMMARY:"
)


print(
    f"  {FEATURE_MAPPING_SUMMARY_CSV}"
)


print(
    "\nFEATURE MAPPING METADATA:"
)


print(
    f"  {FEATURE_MAPPING_METADATA_JSON}"
)


print("\n" + "=" * 100)
print("SECTION 21 STATUS: PASS")
print("=" * 100)

21. FEATURE MAPPING
Required objects : PASS

Feature mapping root:
  /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/feature_mapping

----------------------------------------------------------------------------------------------------
BUILDING FEATURE MAPPING : adult_income
----------------------------------------------------------------------------------------------------
  Preprocessing features : 14
  Numeric features       : 6
  Categorical features   : 8
  Generative columns     : 15
  Expected transformed   : 105
  Actual transformed    : 105
  Numeric transformer   : numeric
  Categorical transformer: categorical
  Categorical encoder    : OneHotEncoder
  Mapping rows           : 105
  Position coverage      : PASS
  Source feature coverage: PASS
  Target exclusion       : PASS
  Metadata exclusion     : PASS
  Numeric mapping        : PASS
  Categorical mapping    : PASS

----------------------------------------------------------------------------------------

In [52]:
# ==================================================================================================
# SECTION 22 — SAVE SPLIT MANIFESTS AND VALIDATION REPORTS
# ==================================================================================================
#
# PURPOSE
# -------
# Persist and independently validate:
#
#   1. Canonical train / validation / test split manifests
#   2. Split provenance integrity
#   3. Preprocessing / generative schema policy
#   4. Persisted Section 20 preprocessing schemas
#   5. Persisted manifest physical files
#   6. Exact provenance-ID correspondence
#   7. Final validation report
#
# FROZEN ARCHITECTURE
# -------------------
# Native generative schema:
#       preprocessing features + target
#
# Generic preprocessing input:
#       preprocessing features ONLY
#
# Encoded feature matrix:
#       transformed preprocessing features ONLY
#
# Target:
#       retained in native generative dataset
#       excluded from generic preprocessing
#       excluded from transformed features
#
# Identifier / provenance:
#       excluded from preprocessing
#       excluded from generative schema
#       provenance retained for auditability
#
# Fit policy:
#       train_only
#
# ==================================================================================================

print("=" * 100)
print("22. SAVE SPLIT MANIFESTS AND VALIDATION REPORTS")
print("=" * 100)


# ==================================================================================================
# 22.0 — IMPORTS
# ==================================================================================================

import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd


# ==================================================================================================
# 22.1 — REQUIRED CANONICAL OBJECTS
# ==================================================================================================

REQUIRED_SECTION_22_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
    "SPLIT_MANIFESTS",
    "SPLIT_SUMMARY_DF",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TRAIN_PREPROCESSING_METADATA",
    "TRAIN_PREPROCESSORS",
    "TRANSFORMED_TRAIN_DATASETS",
    "TRANSFORMED_VALIDATION_DATASETS",
    "TRANSFORMED_TEST_DATASETS",
    "NATIVE_FINAL_DATASETS",
    "ENCODED_FINAL_DATASETS",
    "NB02_DIRECTORIES",
]


missing_section_22_objects = [
    name
    for name in REQUIRED_SECTION_22_OBJECTS
    if name not in globals()
]


if missing_section_22_objects:
    raise RuntimeError(
        "Required canonical objects are missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_section_22_objects
        )
    )


print("Required canonical objects : PASS")


# ==================================================================================================
# 22.2 — RESOLVE PROJECT ROOT AND MANIFEST DIRECTORY
# ==================================================================================================

# ----------------------------------------------------------------------------------------------
# Prefer canonical PROJECT_ROOT if already available.
# ----------------------------------------------------------------------------------------------

if "PROJECT_ROOT" in globals():

    PROJECT_ROOT_PATH = Path(
        PROJECT_ROOT
    )

else:

    # --------------------------------------------------------------------------
    # Fallback: derive project root from the persisted Notebook 02 paths.
    # --------------------------------------------------------------------------

    _candidate_paths = []

    for _key in [
        "schemas",
        "preprocessors",
        "native",
        "encoded",
        "feature_mapping",
    ]:

        if _key in NB02_DIRECTORIES:

            _candidate_paths.append(
                Path(
                    NB02_DIRECTORIES[_key]
                )
            )

    if not _candidate_paths:

        raise RuntimeError(
            "Unable to determine PROJECT_ROOT.\n"
            "PROJECT_ROOT is not defined and NB02_DIRECTORIES "
            "does not contain a usable Notebook 02 directory."
        )

    # Example:
    #
    # /content/.../SPP_GAN_Research/data/processed/notebook_02/schemas
    #
    # parents[3] -> SPP_GAN_Research
    #
    _schema_candidate = _candidate_paths[0]

    PROJECT_ROOT_PATH = (
        _schema_candidate
        .resolve()
        .parents[3]
    )


# ----------------------------------------------------------------------------------------------
# Validate project root
# ----------------------------------------------------------------------------------------------

if not PROJECT_ROOT_PATH.exists():

    raise FileNotFoundError(
        f"PROJECT_ROOT does not exist:\n"
        f"{PROJECT_ROOT_PATH}"
    )


# ----------------------------------------------------------------------------------------------
# Section 22 output directory
# ----------------------------------------------------------------------------------------------

MANIFEST_DIR = (
    PROJECT_ROOT_PATH
    / "results"
    / "raw_validation"
    / "notebook_02_manifests"
)

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("\nProject root:")
print(f"  {PROJECT_ROOT_PATH}")

print("\nManifest directory:")
print(f"  {MANIFEST_DIR}")


# ==================================================================================================
# 22.3 — GLOBAL POLICY DEFINITIONS
# ==================================================================================================

PROVENANCE_COLUMN = "__original_row_id__"

EXPECTED_SPLIT_NAMES = [
    "train",
    "validation",
    "test",
]

EXPECTED_SPLIT_FRACTIONS = {
    "train": 0.70,
    "validation": 0.15,
    "test": 0.15,
}


# ==================================================================================================
# 22.4 — HELPER: SHA-256
# ==================================================================================================

def section_22_sha256_file(file_path):

    sha256 = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            sha256.update(chunk)

    return sha256.hexdigest()


# ==================================================================================================
# 22.5 — HELPER: NORMALIZE PROVENANCE IDS
# ==================================================================================================

def normalize_provenance_ids(series):

    values = pd.to_numeric(
        series,
        errors="raise"
    )

    if values.isna().any():

        raise RuntimeError(
            "Provenance column contains NaN values."
        )

    if not (
        values == values.astype("int64")
    ).all():

        raise RuntimeError(
            "Provenance column contains non-integer values."
        )

    return values.astype("int64")


# ==================================================================================================
# 22.6 — HELPER: VALIDATE DATAFRAME PROVENANCE
# ==================================================================================================

def validate_dataframe_provenance(
    df,
    dataset_id,
    split_name
):

    if PROVENANCE_COLUMN not in df.columns:

        raise RuntimeError(
            f"{dataset_id} / {split_name}: "
            f"missing provenance column "
            f"'{PROVENANCE_COLUMN}'."
        )

    ids = normalize_provenance_ids(
        df[PROVENANCE_COLUMN]
    )

    if ids.duplicated().any():

        raise RuntimeError(
            f"{dataset_id} / {split_name}: "
            "duplicate provenance IDs detected."
        )

    return ids


# ==================================================================================================
# 22.7 — CANONICAL SPLIT PROVENANCE REGISTRY
# ==================================================================================================

print("\n" + "-" * 100)
print("22.7 CANONICAL SPLIT PROVENANCE REGISTRY")
print("-" * 100)


CANONICAL_SPLIT_ID_REGISTRY = {}


for dataset_id in DATASET_IDS:

    CANONICAL_SPLIT_ID_REGISTRY[
        dataset_id
    ] = {}

    train_ids = validate_dataframe_provenance(
        TRAIN_DATASETS[dataset_id],
        dataset_id,
        "train"
    )

    validation_ids = validate_dataframe_provenance(
        VALIDATION_DATASETS[dataset_id],
        dataset_id,
        "validation"
    )

    test_ids = validate_dataframe_provenance(
        TEST_DATASETS[dataset_id],
        dataset_id,
        "test"
    )

    split_id_sets = {
        "train": set(
            train_ids.tolist()
        ),
        "validation": set(
            validation_ids.tolist()
        ),
        "test": set(
            test_ids.tolist()
        ),
    }


    # ----------------------------------------------------------------------------------------------
    # Pairwise disjointness
    # ----------------------------------------------------------------------------------------------

    if (
        split_id_sets["train"]
        &
        split_id_sets["validation"]
    ):

        raise RuntimeError(
            f"{dataset_id}: train/validation "
            "provenance overlap detected."
        )


    if (
        split_id_sets["train"]
        &
        split_id_sets["test"]
    ):

        raise RuntimeError(
            f"{dataset_id}: train/test "
            "provenance overlap detected."
        )


    if (
        split_id_sets["validation"]
        &
        split_id_sets["test"]
    ):

        raise RuntimeError(
            f"{dataset_id}: validation/test "
            "provenance overlap detected."
        )


    CANONICAL_SPLIT_ID_REGISTRY[
        dataset_id
    ] = split_id_sets


print("Canonical split provenance registry : PASS")


# ==================================================================================================
# 22.8 — CANONICAL SPLIT ROW COUNTS
# ==================================================================================================

print("\n" + "-" * 100)
print("22.8 CANONICAL SPLIT ROW COUNTS")
print("-" * 100)


CANONICAL_SPLIT_SUMMARY_RECORDS = []


for dataset_id in DATASET_IDS:

    train_rows = len(
        TRAIN_DATASETS[dataset_id]
    )

    validation_rows = len(
        VALIDATION_DATASETS[dataset_id]
    )

    test_rows = len(
        TEST_DATASETS[dataset_id]
    )

    total_rows = (
        train_rows
        + validation_rows
        + test_rows
    )


    for split_name, expected_rows in {
        "train": train_rows,
        "validation": validation_rows,
        "test": test_rows,
    }.items():

        actual_rows = len(
            CANONICAL_SPLIT_ID_REGISTRY[
                dataset_id
            ][split_name]
        )

        if actual_rows != expected_rows:

            raise RuntimeError(
                f"{dataset_id} / {split_name}: "
                "provenance-ID count does not match "
                "dataframe row count."
            )


    CANONICAL_SPLIT_SUMMARY_RECORDS.append({

        "dataset_id": dataset_id,

        "train_rows": train_rows,

        "validation_rows": validation_rows,

        "test_rows": test_rows,

        "total_rows": total_rows,
    })


    print(
        f"{dataset_id:20s} | "
        f"train={train_rows:,} | "
        f"validation={validation_rows:,} | "
        f"test={test_rows:,} | "
        f"total={total_rows:,}"
    )


CANONICAL_SPLIT_SUMMARY_DF = pd.DataFrame(
    CANONICAL_SPLIT_SUMMARY_RECORDS
)


print("\nCanonical split row counts : PASS")


# ==================================================================================================
# 22.9 — VALIDATE SPLIT SUMMARY
# ==================================================================================================

print("\n" + "-" * 100)
print("22.9 VALIDATE SPLIT SUMMARY")
print("-" * 100)


if not isinstance(
    SPLIT_SUMMARY_DF,
    pd.DataFrame
):

    raise RuntimeError(
        "SPLIT_SUMMARY_DF is not a pandas DataFrame."
    )


required_split_summary_columns = {
    "dataset_id",
    "train_rows",
    "validation_rows",
    "test_rows",
}


missing_split_summary_columns = (
    required_split_summary_columns
    -
    set(SPLIT_SUMMARY_DF.columns)
)


if missing_split_summary_columns:

    raise RuntimeError(
        "SPLIT_SUMMARY_DF missing required columns:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in sorted(
                missing_split_summary_columns
            )
        )
    )


for dataset_id in DATASET_IDS:

    summary_rows = SPLIT_SUMMARY_DF[
        SPLIT_SUMMARY_DF[
            "dataset_id"
        ]
        ==
        dataset_id
    ]


    if len(summary_rows) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one "
            "row in SPLIT_SUMMARY_DF."
        )


    summary_row = summary_rows.iloc[0]

    expected_row = CANONICAL_SPLIT_SUMMARY_DF[
        CANONICAL_SPLIT_SUMMARY_DF[
            "dataset_id"
        ]
        ==
        dataset_id
    ].iloc[0]


    for column in [
        "train_rows",
        "validation_rows",
        "test_rows",
    ]:

        if int(
            summary_row[column]
        ) != int(
            expected_row[column]
        ):

            raise RuntimeError(
                f"{dataset_id}: SPLIT_SUMMARY_DF "
                f"mismatch in '{column}'."
            )


print("Split summary vs canonical data : PASS")


# ==================================================================================================
# 22.10 — PREPROCESSING / GENERATIVE SCHEMA POLICY
# ==================================================================================================

print("\n" + "-" * 100)
print("22.10 PREPROCESSING / GENERATIVE SCHEMA POLICY")
print("-" * 100)


SCHEMA_POLICY_RECORDS = []


for dataset_id in DATASET_IDS:

    schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]


    preprocessing_columns = list(
        schema["all_columns"]
    )

    numeric_columns = list(
        schema["numeric_columns"]
    )

    categorical_columns = list(
        schema["categorical_columns"]
    )

    generative_columns = list(
        schema["generative_columns"]
    )

    target_column = schema[
        "target_column"
    ]

    identifier_columns = list(
        schema["identifier_columns"]
    )

    provenance_column = schema[
        "provenance_column"
    ]


    # ----------------------------------------------------------------------------------------------
    # Target
    # ----------------------------------------------------------------------------------------------

    if target_column in preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "must NOT appear in preprocessing columns."
        )


    if target_column not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "must appear in generative columns."
        )


    expected_generative_columns = (
        preprocessing_columns
        +
        [target_column]
    )


    if generative_columns != expected_generative_columns:

        raise RuntimeError(
            f"{dataset_id}: generative columns must equal "
            "preprocessing columns + target."
        )


    # ----------------------------------------------------------------------------------------------
    # Numeric / categorical coverage
    # ----------------------------------------------------------------------------------------------

    if (
        set(numeric_columns)
        &
        set(categorical_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric and categorical "
            "columns overlap."
        )


    if (
        set(numeric_columns)
        |
        set(categorical_columns)
    ) != set(preprocessing_columns):

        raise RuntimeError(
            f"{dataset_id}: numeric + categorical columns "
            "do not cover preprocessing features exactly."
        )


    # ----------------------------------------------------------------------------------------------
    # Identifier
    # ----------------------------------------------------------------------------------------------

    if (
        set(identifier_columns)
        &
        set(preprocessing_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: identifier appears "
            "in preprocessing columns."
        )


    if (
        set(identifier_columns)
        &
        set(generative_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: identifier appears "
            "in generative columns."
        )


    # ----------------------------------------------------------------------------------------------
    # Provenance
    # ----------------------------------------------------------------------------------------------

    if provenance_column in preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column appears "
            "in preprocessing columns."
        )


    if provenance_column in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column appears "
            "in generative columns."
        )


    SCHEMA_POLICY_RECORDS.append({

        "dataset_id": dataset_id,

        "preprocessing_features": len(
            preprocessing_columns
        ),

        "numeric_features": len(
            numeric_columns
        ),

        "categorical_features": len(
            categorical_columns
        ),

        "generative_columns": len(
            generative_columns
        ),

        "target_column": target_column,

        "target_in_preprocessing": (
            target_column
            in preprocessing_columns
        ),

        "target_in_generative": (
            target_column
            in generative_columns
        ),

        "identifier_count": len(
            identifier_columns
        ),

        "provenance_column": provenance_column,

        "status": "PASS",
    })


    print(
        f"{dataset_id:20s} | "
        f"preprocessing={len(preprocessing_columns)} | "
        f"numeric={len(numeric_columns)} | "
        f"categorical={len(categorical_columns)} | "
        f"generative={len(generative_columns)} | "
        f"target={target_column}"
    )


SCHEMA_POLICY_DF = pd.DataFrame(
    SCHEMA_POLICY_RECORDS
)


print(
    "\nPreprocessing/generative schema policy : PASS"
)


# ==================================================================================================
# 22.11 — CREATE AND SAVE INDIVIDUAL SPLIT MANIFESTS
# ==================================================================================================

print("\n" + "-" * 100)
print("22.11 CREATE AND SAVE SPLIT MANIFESTS")
print("-" * 100)


MANIFEST_RECORDS = []


for dataset_id in DATASET_IDS:

    for split_name, split_df in [

        (
            "train",
            TRAIN_DATASETS[dataset_id]
        ),

        (
            "validation",
            VALIDATION_DATASETS[dataset_id]
        ),

        (
            "test",
            TEST_DATASETS[dataset_id]
        ),

    ]:

        ids = validate_dataframe_provenance(
            split_df,
            dataset_id,
            split_name
        )


        manifest_df = pd.DataFrame({

            "dataset_id": dataset_id,

            "split": split_name,

            PROVENANCE_COLUMN: ids.values,
        })


        manifest_path = (
            MANIFEST_DIR
            /
            f"{dataset_id}_{split_name}_manifest.csv"
        )


        manifest_df.to_csv(
            manifest_path,
            index=False
        )


        MANIFEST_RECORDS.append({

            "dataset_id": dataset_id,

            "split": split_name,

            "rows": len(
                manifest_df
            ),

            "path": str(
                manifest_path
            ),

            "sha256": section_22_sha256_file(
                manifest_path
            ),

            "status": "SAVED",
        })


        print(
            f"Saved: {dataset_id} | "
            f"{split_name:10s} | "
            f"{len(manifest_df):,} rows"
        )


MANIFEST_RECORD_DF = pd.DataFrame(
    MANIFEST_RECORDS
)


print("\nSplit manifests saved : PASS")


# ==================================================================================================
# 22.12 — COMBINED SPLIT MANIFEST
# ==================================================================================================

print("\n" + "-" * 100)
print("22.12 SAVE COMBINED SPLIT MANIFEST")
print("-" * 100)


COMBINED_MANIFEST_RECORDS = []


for dataset_id in DATASET_IDS:

    for split_name in EXPECTED_SPLIT_NAMES:

        ids = CANONICAL_SPLIT_ID_REGISTRY[
            dataset_id
        ][split_name]


        for provenance_id in sorted(ids):

            COMBINED_MANIFEST_RECORDS.append({

                "dataset_id": dataset_id,

                "split": split_name,

                PROVENANCE_COLUMN: provenance_id,
            })


COMBINED_SPLIT_MANIFEST_DF = pd.DataFrame(
    COMBINED_MANIFEST_RECORDS
)


COMBINED_MANIFEST_PATH = (
    MANIFEST_DIR
    /
    "combined_split_manifest.csv"
)


COMBINED_SPLIT_MANIFEST_DF.to_csv(
    COMBINED_MANIFEST_PATH,
    index=False
)


print(
    f"Combined manifest rows: "
    f"{len(COMBINED_SPLIT_MANIFEST_DF):,}"
)

print(
    f"Combined manifest path:\n"
    f"  {COMBINED_MANIFEST_PATH}"
)

print("Combined split manifest : PASS")


# ==================================================================================================
# 22.13 — VALIDATE COMBINED MANIFEST
# ==================================================================================================

print("\n" + "-" * 100)
print("22.13 VALIDATE COMBINED MANIFEST")
print("-" * 100)


expected_total_manifest_rows = sum(

    len(
        TRAIN_DATASETS[dataset_id]
    )

    +
    len(
        VALIDATION_DATASETS[dataset_id]
    )

    +
    len(
        TEST_DATASETS[dataset_id]
    )

    for dataset_id in DATASET_IDS
)


if len(
    COMBINED_SPLIT_MANIFEST_DF
) != expected_total_manifest_rows:

    raise RuntimeError(
        "Combined manifest row count mismatch.\n"
        f"Expected: {expected_total_manifest_rows:,}\n"
        f"Actual  : {len(COMBINED_SPLIT_MANIFEST_DF):,}"
    )


required_combined_manifest_columns = {
    "dataset_id",
    "split",
    PROVENANCE_COLUMN,
}


if not required_combined_manifest_columns.issubset(
    COMBINED_SPLIT_MANIFEST_DF.columns
):

    raise RuntimeError(
        "Combined manifest missing required columns."
    )


# ----------------------------------------------------------------------------------------------
# Validate uniqueness within each dataset/split
# ----------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    for split_name in EXPECTED_SPLIT_NAMES:

        subset = COMBINED_SPLIT_MANIFEST_DF[
            (
                COMBINED_SPLIT_MANIFEST_DF[
                    "dataset_id"
                ]
                ==
                dataset_id
            )
            &
            (
                COMBINED_SPLIT_MANIFEST_DF[
                    "split"
                ]
                ==
                split_name
            )
        ]


        if subset[
            PROVENANCE_COLUMN
        ].duplicated().any():

            raise RuntimeError(
                f"{dataset_id} / {split_name}: "
                "duplicate provenance IDs detected "
                "in combined manifest."
            )


print(
    f"Expected total rows : "
    f"{expected_total_manifest_rows:,}"
)

print(
    f"Actual total rows   : "
    f"{len(COMBINED_SPLIT_MANIFEST_DF):,}"
)

print("Combined manifest validation : PASS")


# ==================================================================================================
# 22.14 — EXACT MANIFEST / CANONICAL SPLIT MATCHING
# ==================================================================================================

print("\n" + "-" * 100)
print("22.14 EXACT MANIFEST / CANONICAL SPLIT MATCHING")
print("-" * 100)


for dataset_id in DATASET_IDS:

    for split_name in EXPECTED_SPLIT_NAMES:

        canonical_ids = (
            CANONICAL_SPLIT_ID_REGISTRY[
                dataset_id
            ][split_name]
        )


        manifest_ids = set(

            COMBINED_SPLIT_MANIFEST_DF.loc[

                (
                    COMBINED_SPLIT_MANIFEST_DF[
                        "dataset_id"
                    ]
                    ==
                    dataset_id
                )

                &

                (
                    COMBINED_SPLIT_MANIFEST_DF[
                        "split"
                    ]
                    ==
                    split_name
                ),

                PROVENANCE_COLUMN,

            ]
            .astype("int64")
            .tolist()
        )


        if manifest_ids != canonical_ids:

            missing_ids = (
                canonical_ids
                -
                manifest_ids
            )

            extra_ids = (
                manifest_ids
                -
                canonical_ids
            )


            raise RuntimeError(

                f"{dataset_id} / {split_name}: "
                "manifest provenance IDs do not exactly "
                "match canonical split.\n"
                f"Missing IDs: {len(missing_ids)}\n"
                f"Extra IDs  : {len(extra_ids)}"
            )


print(
    "Exact manifest/canonical split matching : PASS"
)


# ==================================================================================================
# 22.15 — SAVE MANIFEST METADATA
# ==================================================================================================

print("\n" + "-" * 100)
print("22.15 SAVE MANIFEST METADATA")
print("-" * 100)


manifest_metadata = {

    "section":
        "Notebook 02 - Section 22",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "dataset_ids":
        list(DATASET_IDS),

    "provenance_column":
        PROVENANCE_COLUMN,

    "split_names":
        EXPECTED_SPLIT_NAMES,

    "split_fractions":
        EXPECTED_SPLIT_FRACTIONS,

    "fit_policy":
        "train_only",

    "schema_policy": {

        "target_excluded_from_preprocessing":
            True,

        "target_retained_in_generative_schema":
            True,

        "target_excluded_from_transformed_features":
            True,

        "identifiers_excluded_from_preprocessing":
            True,

        "identifiers_excluded_from_generative_schema":
            True,

        "provenance_excluded_from_preprocessing":
            True,

        "provenance_excluded_from_generative_schema":
            True,

        "provenance_retained_for_auditability":
            True,
    },

    "manifest_files": [

        {

            "dataset_id":
                row["dataset_id"],

            "split":
                row["split"],

            "rows":
                int(row["rows"]),

            "path":
                row["path"],

            "sha256":
                row["sha256"],

        }

        for _, row
        in MANIFEST_RECORD_DF.iterrows()
    ],

    "combined_manifest": {

        "path":
            str(COMBINED_MANIFEST_PATH),

        "rows":
            int(
                len(
                    COMBINED_SPLIT_MANIFEST_DF
                )
            ),

        "sha256":
            section_22_sha256_file(
                COMBINED_MANIFEST_PATH
            ),
    },
}


MANIFEST_METADATA_PATH = (
    MANIFEST_DIR
    /
    "manifest_metadata.json"
)


with open(
    MANIFEST_METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    f"Manifest metadata saved:\n"
    f"  {MANIFEST_METADATA_PATH}"
)

print("Manifest metadata persistence : PASS")


# ==================================================================================================
# 22.16 — SAVE CANONICAL VALIDATION REPORT
# ==================================================================================================

print("\n" + "-" * 100)
print("22.16 SAVE CANONICAL VALIDATION REPORT")
print("-" * 100)


CANONICAL_VALIDATION_REPORT = {

    "section":
        "Notebook 02 - Section 22",

    "status":
        "PASS",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "datasets":
        list(DATASET_IDS),

    "total_manifest_rows":
        int(
            expected_total_manifest_rows
        ),

    "checks": {

        "required_objects":
            "PASS",

        "canonical_split_provenance":
            "PASS",

        "canonical_split_row_counts":
            "PASS",

        "split_summary_validation":
            "PASS",

        "preprocessing_generative_schema_policy":
            "PASS",

        "split_manifest_persistence":
            "PASS",

        "combined_manifest_validation":
            "PASS",

        "exact_manifest_canonical_matching":
            "PASS",
    },
}


CANONICAL_VALIDATION_REPORT_PATH = (
    MANIFEST_DIR
    /
    "section_22_canonical_validation_report.json"
)


with open(
    CANONICAL_VALIDATION_REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CANONICAL_VALIDATION_REPORT,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    f"Canonical validation report saved:\n"
    f"  {CANONICAL_VALIDATION_REPORT_PATH}"
)

print(
    "Canonical validation report : PASS"
)


# ==================================================================================================
# 22.17 — VALIDATE PERSISTED SECTION 20 PREPROCESSING SCHEMAS
# ==================================================================================================

print("\n" + "-" * 100)
print("22.17 VALIDATE PERSISTED SECTION 20 PREPROCESSING SCHEMAS")
print("-" * 100)


PERSISTED_SCHEMA_VALIDATION_RECORDS = []


for dataset_id in DATASET_IDS:

    schema_path = (
        Path(
            NB02_DIRECTORIES["schemas"]
        )
        /
        dataset_id
        /
        "preprocessing_schema.json"
    )


    if not schema_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: persisted preprocessing "
            f"schema not found:\n"
            f"{schema_path}"
        )


    with open(
        schema_path,
        "r",
        encoding="utf-8"
    ) as f:

        persisted_schema = json.load(f)


    # ----------------------------------------------------------------------------------------------
    # Exact Section 20 top-level contract
    # ----------------------------------------------------------------------------------------------

    required_top_level_keys = {

        "dataset_id",
        "schema_version",
        "fit_policy",
        "modeling_schema",
        "target_policy",
        "identifier_policy",
        "provenance_policy",
        "transformation_schema",
        "preprocessing",
        "training_rows",
        "creation_timestamp_utc",
    }


    missing_top_level_keys = (
        required_top_level_keys
        -
        set(
            persisted_schema.keys()
        )
    )


    if missing_top_level_keys:

        raise RuntimeError(

            f"{dataset_id}: persisted Section 20 "
            "schema is missing required top-level keys:\n"

            +
            "\n".join(
                f"  - {x}"
                for x
                in sorted(
                    missing_top_level_keys
                )
            )
        )


    # ----------------------------------------------------------------------------------------------
    # Identity / fit policy
    # ----------------------------------------------------------------------------------------------

    if (
        persisted_schema["dataset_id"]
        !=
        dataset_id
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted dataset_id mismatch."
        )


    if (
        persisted_schema["fit_policy"]
        !=
        "train_only"
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted fit_policy "
            "is not 'train_only'."
        )


    # ----------------------------------------------------------------------------------------------
    # Canonical Section 15 schema
    # ----------------------------------------------------------------------------------------------

    canonical_schema = (
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]
    )


    canonical_preprocessing_columns = list(
        canonical_schema["all_columns"]
    )

    canonical_numeric_columns = list(
        canonical_schema["numeric_columns"]
    )

    canonical_categorical_columns = list(
        canonical_schema["categorical_columns"]
    )

    canonical_generative_columns = list(
        canonical_schema["generative_columns"]
    )

    canonical_target = (
        canonical_schema["target_column"]
    )

    canonical_identifier_columns = list(
        canonical_schema["identifier_columns"]
    )

    canonical_provenance_column = (
        canonical_schema["provenance_column"]
    )


    # ----------------------------------------------------------------------------------------------
    # Persisted Section 20 modeling schema
    # ----------------------------------------------------------------------------------------------

    persisted_modeling_schema = (
        persisted_schema[
            "modeling_schema"
        ]
    )


    persisted_preprocessing_columns = list(
        persisted_modeling_schema[
            "preprocessing_columns"
        ]
    )


    persisted_all_columns = list(
        persisted_modeling_schema[
            "all_columns"
        ]
    )


    persisted_numeric_columns = list(
        persisted_modeling_schema[
            "numeric_columns"
        ]
    )


    persisted_categorical_columns = list(
        persisted_modeling_schema[
            "categorical_columns"
        ]
    )


    persisted_generative_columns = list(
        persisted_modeling_schema[
            "generative_columns"
        ]
    )


    persisted_target = (
        persisted_modeling_schema[
            "target_column"
        ]
    )


    persisted_identifier_columns = list(
        persisted_modeling_schema[
            "identifier_columns_excluded"
        ]
    )


    persisted_provenance_column = (
        persisted_modeling_schema[
            "provenance_column"
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # Exact preprocessing schema
    # ----------------------------------------------------------------------------------------------

    if (
        persisted_preprocessing_columns
        !=
        canonical_preprocessing_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted preprocessing "
            "feature schema does not match canonical schema."
        )


    if (
        persisted_all_columns
        !=
        canonical_preprocessing_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted "
            "modeling_schema.all_columns does not "
            "match canonical preprocessing columns."
        )


    # ----------------------------------------------------------------------------------------------
    # Numeric / categorical schema
    # ----------------------------------------------------------------------------------------------

    if (
        persisted_numeric_columns
        !=
        canonical_numeric_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted numeric feature "
            "schema does not match canonical schema."
        )


    if (
        persisted_categorical_columns
        !=
        canonical_categorical_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted categorical feature "
            "schema does not match canonical schema."
        )


    # ----------------------------------------------------------------------------------------------
    # Target
    # ----------------------------------------------------------------------------------------------

    if (
        persisted_target
        !=
        canonical_target
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted target mismatch."
        )


    if (
        persisted_target
        in
        persisted_preprocessing_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: target appears in persisted "
            "preprocessing columns."
        )


    # ----------------------------------------------------------------------------------------------
    # Generative schema
    # ----------------------------------------------------------------------------------------------

    if (
        persisted_generative_columns
        !=
        canonical_generative_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted generative schema "
            "does not match canonical schema."
        )


    expected_generative_columns = (
        persisted_preprocessing_columns
        +
        [persisted_target]
    )


    if (
        persisted_generative_columns
        !=
        expected_generative_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted generative schema "
            "must equal preprocessing columns + target."
        )


    # ----------------------------------------------------------------------------------------------
    # Identifier policy
    # ----------------------------------------------------------------------------------------------

    if (
        persisted_identifier_columns
        !=
        canonical_identifier_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted identifier exclusion "
            "schema does not match canonical schema."
        )


    if (
        set(persisted_identifier_columns)
        &
        set(persisted_preprocessing_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: identifier appears "
            "in preprocessing columns."
        )


    if (
        set(persisted_identifier_columns)
        &
        set(persisted_generative_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: identifier appears "
            "in generative columns."
        )


    # ----------------------------------------------------------------------------------------------
    # Provenance
    # ----------------------------------------------------------------------------------------------

    if (
        persisted_provenance_column
        !=
        canonical_provenance_column
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted provenance "
            "column mismatch."
        )


    if (
        persisted_provenance_column
        in
        persisted_preprocessing_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: provenance appears "
            "in preprocessing columns."
        )


    if (
        persisted_provenance_column
        in
        persisted_generative_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: provenance appears "
            "in generative columns."
        )


    # ----------------------------------------------------------------------------------------------
    # Target policy
    # ----------------------------------------------------------------------------------------------

    target_policy = (
        persisted_schema[
            "target_policy"
        ]
    )


    if (
        target_policy[
            "retained_in_generative_schema"
        ]
        is not True
    ):

        raise RuntimeError(
            f"{dataset_id}: target retention policy failed."
        )


    if (
        target_policy[
            "excluded_from_preprocessor_input"
        ]
        is not True
    ):

        raise RuntimeError(
            f"{dataset_id}: target exclusion from "
            "preprocessor input failed."
        )


    if (
        target_policy[
            "excluded_from_transformed_features"
        ]
        is not True
    ):

        raise RuntimeError(
            f"{dataset_id}: target exclusion from "
            "transformed features failed."
        )


    if (
        target_policy[
            "raw_target_manually_appended"
        ]
        is not False
    ):

        raise RuntimeError(
            f"{dataset_id}: raw target manually appended "
            "policy failed."
        )


    # ----------------------------------------------------------------------------------------------
    # Identifier policy
    # ----------------------------------------------------------------------------------------------

    identifier_policy = (
        persisted_schema[
            "identifier_policy"
        ]
    )


    if (
        identifier_policy[
            "excluded_from_preprocessor_input"
        ]
        is not True
    ):

        raise RuntimeError(
            f"{dataset_id}: identifier preprocessing "
            "exclusion failed."
        )


    if (
        identifier_policy[
            "excluded_from_generative_schema"
        ]
        is not True
    ):

        raise RuntimeError(
            f"{dataset_id}: identifier generative "
            "exclusion failed."
        )


    if list(
        identifier_policy[
            "identifier_columns"
        ]
    ) != canonical_identifier_columns:

        raise RuntimeError(
            f"{dataset_id}: identifier policy list mismatch."
        )


    # ----------------------------------------------------------------------------------------------
    # Provenance policy
    # ----------------------------------------------------------------------------------------------

    provenance_policy = (
        persisted_schema[
            "provenance_policy"
        ]
    )


    if (
        provenance_policy[
            "column"
        ]
        !=
        canonical_provenance_column
    ):

        raise RuntimeError(
            f"{dataset_id}: provenance policy "
            "column mismatch."
        )


    if (
        provenance_policy[
            "excluded_from_preprocessor_input"
        ]
        is not True
    ):

        raise RuntimeError(
            f"{dataset_id}: provenance preprocessing "
            "exclusion failed."
        )


    if (
        provenance_policy[
            "excluded_from_transformed_features"
        ]
        is not True
    ):

        raise RuntimeError(
            f"{dataset_id}: provenance transformed-feature "
            "exclusion failed."
        )


    if (
        provenance_policy[
            "retained_for_auditability"
        ]
        is not True
    ):

        raise RuntimeError(
            f"{dataset_id}: provenance auditability "
            "policy failed."
        )


    # ----------------------------------------------------------------------------------------------
    # Transformation schema
    # ----------------------------------------------------------------------------------------------

    transformation_schema = (
        persisted_schema[
            "transformation_schema"
        ]
    )


    persisted_transformed_feature_count = int(
        transformation_schema[
            "transformed_feature_count"
        ]
    )


    persisted_transformed_feature_names = list(
        transformation_schema[
            "transformed_feature_names"
        ]
    )


    if (
        persisted_transformed_feature_count
        !=
        len(
            persisted_transformed_feature_names
        )
    ):

        raise RuntimeError(
            f"{dataset_id}: transformed feature count "
            "does not match transformed feature names."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate against actual transformed training data
    # ----------------------------------------------------------------------------------------------

    canonical_transformed_train = (
        TRANSFORMED_TRAIN_DATASETS[
            dataset_id
        ]
    )


    if (
        canonical_transformed_train.shape[1]
        !=
        persisted_transformed_feature_count
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted transformed "
            "feature count does not match canonical "
            "transformed training data."
        )


    # ----------------------------------------------------------------------------------------------
    # Training rows
    # ----------------------------------------------------------------------------------------------

    persisted_training_rows = int(
        persisted_schema[
            "training_rows"
        ]
    )


    canonical_training_rows = len(
        TRAIN_DATASETS[
            dataset_id
        ]
    )


    if (
        persisted_training_rows
        !=
        canonical_training_rows
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted training "
            "row count mismatch."
        )


    # ----------------------------------------------------------------------------------------------
    # Record PASS
    # ----------------------------------------------------------------------------------------------

    PERSISTED_SCHEMA_VALIDATION_RECORDS.append({

        "dataset_id":
            dataset_id,

        "schema_version":
            persisted_schema[
                "schema_version"
            ],

        "fit_policy":
            persisted_schema[
                "fit_policy"
            ],

        "preprocessing_feature_count":
            len(
                persisted_preprocessing_columns
            ),

        "numeric_feature_count":
            len(
                persisted_numeric_columns
            ),

        "categorical_feature_count":
            len(
                persisted_categorical_columns
            ),

        "generative_column_count":
            len(
                persisted_generative_columns
            ),

        "target_column":
            persisted_target,

        "target_in_preprocessing":
            (
                persisted_target
                in
                persisted_preprocessing_columns
            ),

        "target_in_generative":
            (
                persisted_target
                in
                persisted_generative_columns
            ),

        "transformed_feature_count":
            persisted_transformed_feature_count,

        "training_rows":
            persisted_training_rows,

        "status":
            "PASS",
    })


    print(
        f"{dataset_id:20s} | "
        f"schema={persisted_schema['schema_version']} | "
        f"preprocessing={len(persisted_preprocessing_columns)} | "
        f"generative={len(persisted_generative_columns)} | "
        f"transformed={persisted_transformed_feature_count} | "
        f"target={persisted_target}"
    )


PERSISTED_SCHEMA_VALIDATION_DF = pd.DataFrame(
    PERSISTED_SCHEMA_VALIDATION_RECORDS
)


print(
    "\nPersisted Section 20 schemas : PASS"
)


# ==================================================================================================
# 22.18 — RELOAD AND VALIDATE PERSISTED MANIFEST FILES
# ==================================================================================================

print("\n" + "-" * 100)
print("22.18 RELOAD AND VALIDATE PERSISTED MANIFEST FILES")
print("-" * 100)


PERSISTED_MANIFEST_VALIDATION_RECORDS = []


for dataset_id in DATASET_IDS:

    for split_name in EXPECTED_SPLIT_NAMES:

        manifest_path = (
            MANIFEST_DIR
            /
            f"{dataset_id}_{split_name}_manifest.csv"
        )


        if not manifest_path.exists():

            raise FileNotFoundError(
                f"Persisted manifest not found:\n"
                f"{manifest_path}"
            )


        persisted_manifest_df = pd.read_csv(
            manifest_path
        )


        expected_manifest_columns = [
            "dataset_id",
            "split",
            PROVENANCE_COLUMN,
        ]


        if (
            list(
                persisted_manifest_df.columns
            )
            !=
            expected_manifest_columns
        ):

            raise RuntimeError(
                f"{dataset_id} / {split_name}: "
                "persisted manifest columns do not "
                "match expected schema."
            )


        if not (
            persisted_manifest_df[
                "dataset_id"
            ]
            ==
            dataset_id
        ).all():

            raise RuntimeError(
                f"{dataset_id} / {split_name}: "
                "manifest dataset_id mismatch."
            )


        if not (
            persisted_manifest_df[
                "split"
            ]
            ==
            split_name
        ).all():

            raise RuntimeError(
                f"{dataset_id} / {split_name}: "
                "manifest split mismatch."
            )


        expected_rows = len(
            CANONICAL_SPLIT_ID_REGISTRY[
                dataset_id
            ][split_name]
        )


        if len(
            persisted_manifest_df
        ) != expected_rows:

            raise RuntimeError(
                f"{dataset_id} / {split_name}: "
                "persisted manifest row count mismatch."
            )


        persisted_ids = normalize_provenance_ids(
            persisted_manifest_df[
                PROVENANCE_COLUMN
            ]
        )


        if persisted_ids.duplicated().any():

            raise RuntimeError(
                f"{dataset_id} / {split_name}: "
                "duplicate persisted provenance IDs."
            )


        persisted_id_set = set(
            persisted_ids.tolist()
        )


        canonical_id_set = (
            CANONICAL_SPLIT_ID_REGISTRY[
                dataset_id
            ][split_name]
        )


        if (
            persisted_id_set
            !=
            canonical_id_set
        ):

            raise RuntimeError(
                f"{dataset_id} / {split_name}: "
                "persisted manifest IDs do not exactly "
                "match canonical IDs."
            )


        PERSISTED_MANIFEST_VALIDATION_RECORDS.append({

            "dataset_id":
                dataset_id,

            "split":
                split_name,

            "rows":
                len(
                    persisted_manifest_df
                ),

            "sha256":
                section_22_sha256_file(
                    manifest_path
                ),

            "status":
                "PASS",
        })


PERSISTED_MANIFEST_VALIDATION_DF = pd.DataFrame(
    PERSISTED_MANIFEST_VALIDATION_RECORDS
)


print(
    "Persisted manifest physical validation : PASS"
)


# ==================================================================================================
# 22.19 — RELOAD AND VALIDATE COMBINED MANIFEST
# ==================================================================================================

print("\n" + "-" * 100)
print("22.19 RELOAD AND VALIDATE COMBINED MANIFEST")
print("-" * 100)


reloaded_combined_manifest = pd.read_csv(
    COMBINED_MANIFEST_PATH
)


expected_combined_columns = [
    "dataset_id",
    "split",
    PROVENANCE_COLUMN,
]


if (
    list(
        reloaded_combined_manifest.columns
    )
    !=
    expected_combined_columns
):

    raise RuntimeError(
        "Reloaded combined manifest schema mismatch."
    )


if (
    len(
        reloaded_combined_manifest
    )
    !=
    expected_total_manifest_rows
):

    raise RuntimeError(
        "Reloaded combined manifest row count mismatch."
    )


reloaded_combined_manifest[
    PROVENANCE_COLUMN
] = normalize_provenance_ids(
    reloaded_combined_manifest[
        PROVENANCE_COLUMN
    ]
)


for dataset_id in DATASET_IDS:

    for split_name in EXPECTED_SPLIT_NAMES:

        canonical_ids = (
            CANONICAL_SPLIT_ID_REGISTRY[
                dataset_id
            ][split_name]
        )


        persisted_ids = set(

            reloaded_combined_manifest.loc[

                (
                    reloaded_combined_manifest[
                        "dataset_id"
                    ]
                    ==
                    dataset_id
                )

                &

                (
                    reloaded_combined_manifest[
                        "split"
                    ]
                    ==
                    split_name
                ),

                PROVENANCE_COLUMN,

            ].tolist()
        )


        if (
            persisted_ids
            !=
            canonical_ids
        ):

            raise RuntimeError(
                f"{dataset_id} / {split_name}: "
                "reloaded combined manifest does not "
                "match canonical provenance IDs."
            )


print(
    "Reloaded combined manifest validation : PASS"
)


# ==================================================================================================
# 22.20 — VALIDATE PERSISTED MANIFEST METADATA
# ==================================================================================================

print("\n" + "-" * 100)
print("22.20 VALIDATE PERSISTED MANIFEST METADATA")
print("-" * 100)


if not MANIFEST_METADATA_PATH.exists():

    raise FileNotFoundError(
        f"Manifest metadata not found:\n"
        f"{MANIFEST_METADATA_PATH}"
    )


with open(
    MANIFEST_METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:

    reloaded_manifest_metadata = json.load(f)


if (
    reloaded_manifest_metadata[
        "dataset_ids"
    ]
    !=
    list(DATASET_IDS)
):

    raise RuntimeError(
        "Persisted manifest metadata "
        "dataset list mismatch."
    )


if (
    reloaded_manifest_metadata[
        "provenance_column"
    ]
    !=
    PROVENANCE_COLUMN
):

    raise RuntimeError(
        "Persisted manifest metadata "
        "provenance column mismatch."
    )


if (
    reloaded_manifest_metadata[
        "fit_policy"
    ]
    !=
    "train_only"
):

    raise RuntimeError(
        "Persisted manifest metadata "
        "fit policy mismatch."
    )


schema_policy = (
    reloaded_manifest_metadata[
        "schema_policy"
    ]
)


expected_schema_policy = {

    "target_excluded_from_preprocessing":
        True,

    "target_retained_in_generative_schema":
        True,

    "target_excluded_from_transformed_features":
        True,

    "identifiers_excluded_from_preprocessing":
        True,

    "identifiers_excluded_from_generative_schema":
        True,

    "provenance_excluded_from_preprocessing":
        True,

    "provenance_excluded_from_generative_schema":
        True,

    "provenance_retained_for_auditability":
        True,
}


if (
    schema_policy
    !=
    expected_schema_policy
):

    raise RuntimeError(
        "Persisted manifest metadata "
        "schema policy mismatch."
    )


print(
    "Persisted manifest metadata validation : PASS"
)


# ==================================================================================================
# 22.21 — SAVE FINAL SECTION 22 VALIDATION TABLE
# ==================================================================================================

print("\n" + "-" * 100)
print("22.21 SAVE FINAL SECTION 22 VALIDATION TABLE")
print("-" * 100)


SECTION_22_VALIDATION_RECORDS = []


for dataset_id in DATASET_IDS:

    canonical_row = (
        CANONICAL_SPLIT_SUMMARY_DF[
            CANONICAL_SPLIT_SUMMARY_DF[
                "dataset_id"
            ]
            ==
            dataset_id
        ]
        .iloc[0]
    )


    schema_row = (
        PERSISTED_SCHEMA_VALIDATION_DF[
            PERSISTED_SCHEMA_VALIDATION_DF[
                "dataset_id"
            ]
            ==
            dataset_id
        ]
        .iloc[0]
    )


    SECTION_22_VALIDATION_RECORDS.append({

        "dataset_id":
            dataset_id,

        "train_rows":
            int(
                canonical_row[
                    "train_rows"
                ]
            ),

        "validation_rows":
            int(
                canonical_row[
                    "validation_rows"
                ]
            ),

        "test_rows":
            int(
                canonical_row[
                    "test_rows"
                ]
            ),

        "preprocessing_features":
            int(
                schema_row[
                    "preprocessing_feature_count"
                ]
            ),

        "numeric_features":
            int(
                schema_row[
                    "numeric_feature_count"
                ]
            ),

        "categorical_features":
            int(
                schema_row[
                    "categorical_feature_count"
                ]
            ),

        "generative_columns":
            int(
                schema_row[
                    "generative_column_count"
                ]
            ),

        "target_column":
            schema_row[
                "target_column"
            ],

        "target_in_preprocessing":
            bool(
                schema_row[
                    "target_in_preprocessing"
                ]
            ),

        "target_in_generative":
            bool(
                schema_row[
                    "target_in_generative"
                ]
            ),

        "transformed_features":
            int(
                schema_row[
                    "transformed_feature_count"
                ]
            ),

        "schema_version":
            schema_row[
                "schema_version"
            ],

        "fit_policy":
            schema_row[
                "fit_policy"
            ],

        "status":
            "PASS",
    })


SECTION_22_VALIDATION_DF = pd.DataFrame(
    SECTION_22_VALIDATION_RECORDS
)


SECTION_22_VALIDATION_PATH = (
    MANIFEST_DIR
    /
    "section_22_validation_summary.csv"
)


SECTION_22_VALIDATION_DF.to_csv(
    SECTION_22_VALIDATION_PATH,
    index=False
)


print(
    f"Validation summary saved:\n"
    f"  {SECTION_22_VALIDATION_PATH}"
)

print(
    "Section 22 validation table : PASS"
)


# ==================================================================================================
# 22.22 — SAVE COMPLETE SECTION 22 REPORT
# ==================================================================================================

print("\n" + "-" * 100)
print("22.22 SAVE COMPLETE SECTION 22 REPORT")
print("-" * 100)


SECTION_22_REPORT = {

    "section":
        "Notebook 02 - Section 22",

    "status":
        "PASS",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "datasets":
        list(DATASET_IDS),

    "total_datasets":
        len(DATASET_IDS),

    "total_manifest_rows":
        int(
            expected_total_manifest_rows
        ),

    "provenance_column":
        PROVENANCE_COLUMN,

    "fit_policy":
        "train_only",

    "checks": {

        "required_canonical_objects":
            "PASS",

        "canonical_split_provenance_registry":
            "PASS",

        "canonical_split_row_counts":
            "PASS",

        "split_summary_validation":
            "PASS",

        "preprocessing_generative_schema_policy":
            "PASS",

        "split_manifest_persistence":
            "PASS",

        "combined_manifest_persistence":
            "PASS",

        "exact_manifest_canonical_matching":
            "PASS",

        "manifest_metadata_persistence":
            "PASS",

        "persisted_section_20_schema_validation":
            "PASS",

        "persisted_manifest_physical_validation":
            "PASS",

        "reloaded_combined_manifest_validation":
            "PASS",

        "persisted_manifest_metadata_validation":
            "PASS",

        "final_validation_table":
            "PASS",
    },


    "architecture": {

        "target_in_preprocessing":
            False,

        "target_in_generative_schema":
            True,

        "target_in_transformed_features":
            False,

        "raw_target_manually_appended":
            False,

        "identifiers_in_preprocessing":
            False,

        "identifiers_in_generative_schema":
            False,

        "provenance_in_preprocessing":
            False,

        "provenance_in_generative_schema":
            False,

        "provenance_retained_for_auditability":
            True,
    },


    "artifacts": {

        "manifest_directory":
            str(
                MANIFEST_DIR
            ),

        "combined_manifest":
            str(
                COMBINED_MANIFEST_PATH
            ),

        "manifest_metadata":
            str(
                MANIFEST_METADATA_PATH
            ),

        "canonical_validation_report":
            str(
                CANONICAL_VALIDATION_REPORT_PATH
            ),

        "section_22_validation_summary":
            str(
                SECTION_22_VALIDATION_PATH
            ),
    },
}


SECTION_22_REPORT_PATH = (
    MANIFEST_DIR
    /
    "section_22_final_report.json"
)


with open(
    SECTION_22_REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        SECTION_22_REPORT,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    f"Complete Section 22 report saved:\n"
    f"  {SECTION_22_REPORT_PATH}"
)

print(
    "Complete Section 22 report persistence : PASS"
)


# ==================================================================================================
# 22.23 — FINAL SECTION 22 INTEGRITY CHECK
# ==================================================================================================

print("=" * 100)
print("22.23 FINAL SECTION 22 INTEGRITY CHECK")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# Expected individual split manifest filenames
# --------------------------------------------------------------------------------------------------

EXPECTED_INDIVIDUAL_MANIFESTS = {
    f"{dataset_id}_{split_name}_manifest.csv"
    for dataset_id in DATASET_IDS
    for split_name in EXPECTED_SPLIT_NAMES
}


EXPECTED_INDIVIDUAL_MANIFEST_COUNT = (
    len(EXPECTED_INDIVIDUAL_MANIFESTS)
)


# --------------------------------------------------------------------------------------------------
# Validate each expected individual manifest physically
# --------------------------------------------------------------------------------------------------

MISSING_INDIVIDUAL_MANIFESTS = []

for manifest_name in sorted(
    EXPECTED_INDIVIDUAL_MANIFESTS
):

    manifest_path = (
        MANIFEST_DIR
        /
        manifest_name
    )

    if not manifest_path.exists():

        MISSING_INDIVIDUAL_MANIFESTS.append(
            manifest_name
        )


if MISSING_INDIVIDUAL_MANIFESTS:

    raise RuntimeError(
        "Missing expected individual split manifest files.\n"
        +
        "\n".join(
            f"  - {name}"
            for name in MISSING_INDIVIDUAL_MANIFESTS
        )
    )


print(
    f"Individual split manifests : PASS "
    f"({EXPECTED_INDIVIDUAL_MANIFEST_COUNT} files)"
)


# --------------------------------------------------------------------------------------------------
# Validate combined manifest separately
# --------------------------------------------------------------------------------------------------

if not COMBINED_MANIFEST_PATH.exists():

    raise RuntimeError(
        "Combined split manifest is missing:\n"
        f"{COMBINED_MANIFEST_PATH}"
    )


print(
    "Combined split manifest    : PASS"
)


# --------------------------------------------------------------------------------------------------
# Validate the exact individual manifest set
#
# IMPORTANT:
# MANIFEST_DIR contains other legitimate CSV validation artifacts from
# previous Notebook 02 sections. Therefore, do NOT count all *.csv files.
# Only the 9 canonical split manifest filenames are relevant here.
# --------------------------------------------------------------------------------------------------

ACTUAL_EXPECTED_MANIFEST_PATHS = {
    path.name
    for path in MANIFEST_DIR.iterdir()
    if (
        path.is_file()
        and
        path.name in EXPECTED_INDIVIDUAL_MANIFESTS
    )
}


if (
    ACTUAL_EXPECTED_MANIFEST_PATHS
    !=
    EXPECTED_INDIVIDUAL_MANIFESTS
):

    missing_manifests = (
        EXPECTED_INDIVIDUAL_MANIFESTS
        -
        ACTUAL_EXPECTED_MANIFEST_PATHS
    )

    raise RuntimeError(
        "Individual split manifest set mismatch.\n"
        f"Missing: {sorted(missing_manifests)}"
    )


# --------------------------------------------------------------------------------------------------
# Validate combined manifest is not accidentally counted as individual
# --------------------------------------------------------------------------------------------------

if (
    "combined_split_manifest.csv"
    in
    EXPECTED_INDIVIDUAL_MANIFESTS
):

    raise RuntimeError(
        "Internal integrity error: combined manifest "
        "must not be classified as an individual manifest."
    )


# --------------------------------------------------------------------------------------------------
# Validate required Section 22 artifacts
# --------------------------------------------------------------------------------------------------

REQUIRED_SECTION_22_ARTIFACTS = [

    MANIFEST_DIR
    /
    "manifest_metadata.json",

    MANIFEST_DIR
    /
    "section_22_canonical_validation_report.json",

    MANIFEST_DIR
    /
    "section_22_validation_summary.csv",

    MANIFEST_DIR
    /
    "section_22_final_report.json",

]


MISSING_REQUIRED_SECTION_22_ARTIFACTS = [

    path
    for path
    in REQUIRED_SECTION_22_ARTIFACTS

    if not path.exists()

]


if MISSING_REQUIRED_SECTION_22_ARTIFACTS:

    raise RuntimeError(
        "Required Section 22 artifacts are missing:\n"
        +
        "\n".join(
            f"  - {path}"
            for path
            in MISSING_REQUIRED_SECTION_22_ARTIFACTS
        )
    )


print(
    f"Required Section 22 artifacts : PASS "
    f"({len(REQUIRED_SECTION_22_ARTIFACTS)} files)"
)


# --------------------------------------------------------------------------------------------------
# Validate expected manifest count
# --------------------------------------------------------------------------------------------------

if (
    EXPECTED_INDIVIDUAL_MANIFEST_COUNT
    !=
    len(DATASET_IDS)
    *
    len(EXPECTED_SPLIT_NAMES)
):

    raise RuntimeError(
        "Unexpected expected-manifest count.\n"
        f"Expected formula: "
        f"{len(DATASET_IDS)} datasets × "
        f"{len(EXPECTED_SPLIT_NAMES)} splits\n"
        f"Actual count    : "
        f"{EXPECTED_INDIVIDUAL_MANIFEST_COUNT}"
    )


# --------------------------------------------------------------------------------------------------
# Final status
# --------------------------------------------------------------------------------------------------

SECTION_22_STATUS = "PASS"


print()
print("-" * 100)
print("SECTION 22 FINAL STATUS : PASS")
print("-" * 100)

print(
    f"Datasets             : {len(DATASET_IDS)}"
)

print(
    f"Expected split types : {len(EXPECTED_SPLIT_NAMES)}"
)

print(
    f"Individual manifests : "
    f"{EXPECTED_INDIVIDUAL_MANIFEST_COUNT}"
)

print(
    "Combined manifest    : 1"
)

print(
    f"Required artifacts   : "
    f"{len(REQUIRED_SECTION_22_ARTIFACTS)}"
)

print()
print("=" * 100)
print("22.23 FINAL SECTION 22 INTEGRITY CHECK : PASS")
print("=" * 100)


# ==================================================================================================
# 22.24 — FINAL SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("SECTION 22 COMPLETE")
print("=" * 100)


print(
    f"\nDatasets                         : "
    f"{len(DATASET_IDS)}"
)


print(
    f"Total manifest rows              : "
    f"{expected_total_manifest_rows:,}"
)


print(
    f"Individual manifest files        : "
    f"{EXPECTED_INDIVIDUAL_MANIFEST_COUNT}"
)


print(
    f"Combined manifest                : "
    f"{COMBINED_MANIFEST_PATH}"
)


print(
    f"Persisted schema validations     : "
    f"{len(PERSISTED_SCHEMA_VALIDATION_DF)}"
)


print(
    f"Persisted manifest validations   : "
    f"{len(PERSISTED_MANIFEST_VALIDATION_DF)}"
)


print(
    f"Validation summary               : "
    f"{SECTION_22_VALIDATION_PATH}"
)


print(
    f"Final Section 22 report          : "
    f"{SECTION_22_REPORT_PATH}"
)


print("\n" + "=" * 100)
print("SECTION 22 STATUS: PASS")
print("=" * 100)


# ==================================================================================================
# DISPLAY FINAL VALIDATION TABLE
# ==================================================================================================

display(
    SECTION_22_VALIDATION_DF
)

# ==================================================================================================
# 22.24 — FINAL SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("SECTION 22 COMPLETE")
print("=" * 100)


print(
    f"\nDatasets                         : "
    f"{len(DATASET_IDS)}"
)


print(
    f"Total manifest rows              : "
    f"{expected_total_manifest_rows:,}"
)


print(
    f"Individual manifest files        : "
    f"{expected_manifest_file_count}"
)


print(
    f"Combined manifest                : "
    f"{COMBINED_MANIFEST_PATH}"
)


print(
    f"Persisted schema validations     : "
    f"{len(PERSISTED_SCHEMA_VALIDATION_DF)}"
)


print(
    f"Persisted manifest validations   : "
    f"{len(PERSISTED_MANIFEST_VALIDATION_DF)}"
)


print(
    f"Validation summary               : "
    f"{SECTION_22_VALIDATION_PATH}"
)


print(
    f"Final Section 22 report          : "
    f"{SECTION_22_REPORT_PATH}"
)


print("\n" + "=" * 100)
print("SECTION 22 STATUS: PASS")
print("=" * 100)


# ==================================================================================================
# DISPLAY FINAL VALIDATION TABLE
# ==================================================================================================

display(
    SECTION_22_VALIDATION_DF
)

22. SAVE SPLIT MANIFESTS AND VALIDATION REPORTS
Required canonical objects : PASS

Project root:
  /content/drive/MyDrive/SPP_GAN_Research

Manifest directory:
  /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests

----------------------------------------------------------------------------------------------------
22.7 CANONICAL SPLIT PROVENANCE REGISTRY
----------------------------------------------------------------------------------------------------
Canonical split provenance registry : PASS

----------------------------------------------------------------------------------------------------
22.8 CANONICAL SPLIT ROW COUNTS
----------------------------------------------------------------------------------------------------
adult_income         | train=34,189 | validation=7,326 | test=7,327 | total=48,842
bank_marketing       | train=31,647 | validation=6,782 | test=6,782 | total=45,211
diabetes_130us       | train=71,236 | validation=15,265 | test=15

,dataset_id,train_rows,validation_rows,test_rows,preprocessing_features,numeric_features,categorical_features,generative_columns,target_column,target_in_preprocessing,target_in_generative,transformed_features,schema_version,fit_policy,status
0,adult_income,34189,7326,7327,14,6,8,15,income,False,True,105,3.0,train_only,PASS
1,bank_marketing,31647,6782,6782,16,7,9,17,y,False,True,51,3.0,train_only,PASS
2,diabetes_130us,71236,15265,15265,47,11,36,48,readmitted,False,True,2336,3.0,train_only,PASS



SECTION 22 COMPLETE

Datasets                         : 3
Total manifest rows              : 195,819
Individual manifest files        : 9
Combined manifest                : /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests/combined_split_manifest.csv
Persisted schema validations     : 3
Persisted manifest validations   : 9
Validation summary               : /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests/section_22_validation_summary.csv
Final Section 22 report          : /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests/section_22_final_report.json

SECTION 22 STATUS: PASS


,dataset_id,train_rows,validation_rows,test_rows,preprocessing_features,numeric_features,categorical_features,generative_columns,target_column,target_in_preprocessing,target_in_generative,transformed_features,schema_version,fit_policy,status
0,adult_income,34189,7326,7327,14,6,8,15,income,False,True,105,3.0,train_only,PASS
1,bank_marketing,31647,6782,6782,16,7,9,17,y,False,True,51,3.0,train_only,PASS
2,diabetes_130us,71236,15265,15265,47,11,36,48,readmitted,False,True,2336,3.0,train_only,PASS


In [53]:
# ==============================================================================
# SECTION 23 — RELOAD & VALIDATE PERSISTED ARTIFACTS
# ==============================================================================
#
# PURPOSE
# -------
# Independently reload and validate all persisted preprocessing artifacts
# produced by Notebook 02.
#
# ARCHITECTURAL RULES
# -------------------
# 1. No preprocessing is refitted.
# 2. No fit() or fit_transform() is called.
# 3. Persisted ColumnTransformer is the authoritative fitted object.
# 4. Only preprocessing FEATURES enter the fitted preprocessor.
# 5. TARGET is retained in the native generative schema.
# 6. TARGET is excluded from generic preprocessing input.
# 7. IDENTIFIERS are excluded from preprocessing and generative schema.
# 8. PROVENANCE is retained only for auditability.
# 9. Reloaded feature names must match persisted schema.
# 10. Reloaded transforms must reproduce persisted transformed dimensions.
#
# FROZEN ARCHITECTURE
# -------------------
#
# Native generative schema
#     ├── preprocessing features
#     └── target
#
# Fitted preprocessing
#     └── preprocessing features ONLY
#
# Encoded feature matrix
#     └── transformed preprocessing features ONLY
#
# ==============================================================================


import os
import json
import joblib
import numpy as np
import pandas as pd

from datetime import datetime, timezone


print("=" * 100)
print("SECTION 23 — RELOAD & VALIDATE PERSISTED ARTIFACTS")
print("=" * 100)


# ==============================================================================
# 23.01 — REQUIRED OBJECT VALIDATION
# ==============================================================================

print("\n" + "-" * 100)
print("23.01 — REQUIRED OBJECT VALIDATION")
print("-" * 100)


REQUIRED_OBJECTS = [
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
    "TRAIN_PREPROCESSORS",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TRAIN_PREPROCESSING_METADATA",
    "TRANSFORMED_TRAIN_DATASETS",
    "TRANSFORMED_VALIDATION_DATASETS",
    "TRANSFORMED_TEST_DATASETS",
    "NATIVE_FINAL_DATASETS",
    "IDENTIFIER_COLUMNS",
    "TARGET_COLUMNS",
    "NB02_DIRECTORIES",
]


MISSING_OBJECTS = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]


if MISSING_OBJECTS:
    raise RuntimeError(
        "Section 23 cannot continue.\n"
        f"Missing required objects: {MISSING_OBJECTS}"
    )


print("Required canonical objects:")

for name in REQUIRED_OBJECTS:
    print(f"  ✓ {name}")


# ==============================================================================
# 23.02 — CANONICAL DATASET REGISTRY VALIDATION
# ==============================================================================

print("\n" + "-" * 100)
print("23.02 — CANONICAL DATASET REGISTRY VALIDATION")
print("-" * 100)


DATASET_IDS = list(
    TRAIN_DATASETS.keys()
)


EXPECTED_DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]


if DATASET_IDS != EXPECTED_DATASET_IDS:
    raise RuntimeError(
        "Canonical dataset registry mismatch.\n"
        f"Expected: {EXPECTED_DATASET_IDS}\n"
        f"Actual  : {DATASET_IDS}"
    )


CANONICAL_CONTAINERS = {
    "TRAIN_DATASETS": TRAIN_DATASETS,
    "VALIDATION_DATASETS": VALIDATION_DATASETS,
    "TEST_DATASETS": TEST_DATASETS,
    "TRAIN_PREPROCESSORS": TRAIN_PREPROCESSORS,
    "TRAIN_PREPROCESSING_COLUMNS": TRAIN_PREPROCESSING_COLUMNS,
    "TRAIN_PREPROCESSING_METADATA": TRAIN_PREPROCESSING_METADATA,
    "TRANSFORMED_TRAIN_DATASETS": TRANSFORMED_TRAIN_DATASETS,
    "TRANSFORMED_VALIDATION_DATASETS": TRANSFORMED_VALIDATION_DATASETS,
    "TRANSFORMED_TEST_DATASETS": TRANSFORMED_TEST_DATASETS,
    "NATIVE_FINAL_DATASETS": NATIVE_FINAL_DATASETS,
}


for container_name, container in CANONICAL_CONTAINERS.items():

    container_ids = set(container.keys())
    expected_ids = set(DATASET_IDS)

    if container_ids != expected_ids:

        missing_ids = sorted(
            expected_ids - container_ids
        )

        extra_ids = sorted(
            container_ids - expected_ids
        )

        raise RuntimeError(
            f"{container_name} dataset registry mismatch.\n"
            f"Missing: {missing_ids}\n"
            f"Extra  : {extra_ids}"
        )

    print(f"  ✓ {container_name}")


print("\n✓ Canonical dataset registry validation: PASS")


# ==============================================================================
# 23.03 — ARTIFACT DIRECTORY VALIDATION
# ==============================================================================

print("\n" + "-" * 100)
print("23.03 — ARTIFACT DIRECTORY VALIDATION")
print("-" * 100)


PREPROCESSOR_ROOT = NB02_DIRECTORIES["preprocessors"]
SCHEMA_ROOT = NB02_DIRECTORIES["schemas"]


if not os.path.isdir(PREPROCESSOR_ROOT):
    raise RuntimeError(
        f"Preprocessor root does not exist:\n{PREPROCESSOR_ROOT}"
    )


if not os.path.isdir(SCHEMA_ROOT):
    raise RuntimeError(
        f"Schema root does not exist:\n{SCHEMA_ROOT}"
    )


METADATA_ROOT = os.path.join(
    SCHEMA_ROOT,
    "metadata",
)


if not os.path.isdir(METADATA_ROOT):
    raise RuntimeError(
        f"Metadata root does not exist:\n{METADATA_ROOT}"
    )


print(f"Preprocessor root : {PREPROCESSOR_ROOT}")
print(f"Schema root       : {SCHEMA_ROOT}")
print(f"Metadata root     : {METADATA_ROOT}")

print("  ✓ Artifact roots exist")


# ==============================================================================
# 23.04 — VALIDATION HELPERS
# ==============================================================================

def normalize_columns(value):

    if value is None:
        return []

    if isinstance(value, str):
        return [value]

    if isinstance(
        value,
        (list, tuple, np.ndarray, pd.Index, pd.Series)
    ):
        return [str(item) for item in list(value)]

    return [str(value)]


def normalize_single_column(value):

    values = normalize_columns(value)

    if not values:
        return None

    if len(values) > 1:
        raise RuntimeError(
            f"Expected a single column but received: {values}"
        )

    return values[0]


def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        return json.load(f)


def array_shape(value):

    return tuple(
        np.asarray(value).shape
    )


def recursive_find_field(
    obj,
    field_name,
):

    if isinstance(obj, dict):

        if field_name in obj:
            return obj[field_name]

        for value in obj.values():

            result = recursive_find_field(
                value,
                field_name,
            )

            if result is not None:
                return result

    elif isinstance(obj, list):

        for value in obj:

            result = recursive_find_field(
                value,
                field_name,
            )

            if result is not None:
                return result

    return None


def recursive_find_all_fields(
    obj,
    field_name,
):

    results = []

    if isinstance(obj, dict):

        if field_name in obj:
            results.append(obj[field_name])

        for value in obj.values():

            results.extend(
                recursive_find_all_fields(
                    value,
                    field_name,
                )
            )

    elif isinstance(obj, list):

        for value in obj:

            results.extend(
                recursive_find_all_fields(
                    value,
                    field_name,
                )
            )

    return results


def find_exact_serialized_list(
    obj,
    field_name,
    expected,
):

    expected = normalize_columns(expected)

    candidates = recursive_find_all_fields(
        obj,
        field_name,
    )

    for candidate in candidates:

        candidate_normalized = normalize_columns(
            candidate
        )

        if candidate_normalized == expected:
            return candidate_normalized

    return None


# ==============================================================================
# 23.05 — VALIDATION RESULT CONTAINER
# ==============================================================================

VALIDATION_RESULTS = []


# ==============================================================================
# 23.06 — DATASET-BY-DATASET VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "=" * 100)
    print(f"VALIDATING DATASET: {dataset_id}")
    print("=" * 100)

    dataset_status = True
    dataset_messages = []


    # ==========================================================================
    # 23.07 — ARTIFACT PATHS
    # ==========================================================================

    print("\n23.07 — ARTIFACT PATHS")


    preprocessor_path = os.path.join(
        PREPROCESSOR_ROOT,
        dataset_id,
        "train_fitted_preprocessor.joblib",
    )


    schema_path = os.path.join(
        SCHEMA_ROOT,
        dataset_id,
        "preprocessing_schema.json",
    )


    # IMPORTANT:
    # Section 20 persisted metadata as:
    #
    # schemas/metadata/<dataset_id>_preprocessing_metadata.json
    #
    metadata_path = os.path.join(
        METADATA_ROOT,
        f"{dataset_id}_preprocessing_metadata.json",
    )


    artifact_paths = {
        "preprocessor": preprocessor_path,
        "schema": schema_path,
        "metadata": metadata_path,
    }


    print(f"  Preprocessor : {preprocessor_path}")
    print(f"  Schema       : {schema_path}")
    print(f"  Metadata     : {metadata_path}")


    for artifact_name, artifact_path in artifact_paths.items():

        if os.path.isfile(artifact_path):

            print(
                f"  ✓ {artifact_name}: FOUND"
            )

        else:

            print(
                f"  ✗ {artifact_name}: MISSING"
            )

            dataset_status = False

            dataset_messages.append(
                f"{dataset_id}: missing "
                f"{artifact_name} artifact."
            )


    if not dataset_status:

        VALIDATION_RESULTS.append({
            "dataset_id": dataset_id,
            "status": "FAIL",
            "messages": dataset_messages,
        })

        continue


    # ==========================================================================
    # 23.08 — RELOAD TRAIN-FITTED PREPROCESSOR
    # ==========================================================================

    print("\n23.08 — RELOAD TRAIN-FITTED PREPROCESSOR")


    try:

        reloaded_preprocessor = joblib.load(
            preprocessor_path
        )


        if not hasattr(
            reloaded_preprocessor,
            "transform",
        ):
            raise RuntimeError(
                "Reloaded preprocessor does not expose transform()."
            )


        if not hasattr(
            reloaded_preprocessor,
            "feature_names_in_",
        ):
            raise RuntimeError(
                "Reloaded preprocessor does not expose "
                "feature_names_in_."
            )


        if not hasattr(
            reloaded_preprocessor,
            "transformers_",
        ):
            raise RuntimeError(
                "Reloaded preprocessor does not expose "
                "transformers_."
            )


        if not hasattr(
            reloaded_preprocessor,
            "get_feature_names_out",
        ):
            raise RuntimeError(
                "Reloaded preprocessor does not expose "
                "get_feature_names_out()."
            )


        print(
            f"  Reloaded type: "
            f"{type(reloaded_preprocessor).__name__}"
        )

        print("  ✓ Preprocessor reload: PASS")
        print("  ✓ Fitted-state validation: PASS")


        preprocessor_fitted_valid = True


    except Exception as exc:

        dataset_status = False
        preprocessor_fitted_valid = False
        reloaded_preprocessor = None

        dataset_messages.append(
            f"{dataset_id}: preprocessor reload failed — {exc}"
        )

        print(
            f"  ✗ Preprocessor reload: FAIL — {exc}"
        )


    # ==========================================================================
    # 23.09 — RELOAD PREPROCESSING SCHEMA
    # ==========================================================================

    print("\n23.09 — RELOAD PREPROCESSING SCHEMA")


    try:

        schema_data = load_json(
            schema_path
        )


        if not isinstance(
            schema_data,
            dict,
        ):
            raise TypeError(
                "Preprocessing schema must be a JSON object."
            )


        schema_valid = True

        print("  ✓ Schema JSON reload: PASS")


    except Exception as exc:

        dataset_status = False
        schema_valid = False
        schema_data = None

        dataset_messages.append(
            f"{dataset_id}: schema reload failed — {exc}"
        )

        print(
            f"  ✗ Schema JSON reload: FAIL — {exc}"
        )


    # ==========================================================================
    # 23.10 — RELOAD PREPROCESSING METADATA
    # ==========================================================================

    print("\n23.10 — RELOAD PREPROCESSING METADATA")


    try:

        metadata_data = load_json(
            metadata_path
        )


        if not isinstance(
            metadata_data,
            dict,
        ):
            raise TypeError(
                "Preprocessing metadata must be a JSON object."
            )


        metadata_valid = True

        print("  ✓ Metadata JSON reload: PASS")


    except Exception as exc:

        dataset_status = False
        metadata_valid = False
        metadata_data = None

        dataset_messages.append(
            f"{dataset_id}: metadata reload failed — {exc}"
        )

        print(
            f"  ✗ Metadata JSON reload: FAIL — {exc}"
        )


    # ==========================================================================
    # 23.11 — DATASET IDENTIFIER VALIDATION
    # ==========================================================================

    print("\n23.11 — DATASET IDENTIFIER VALIDATION")


    if schema_valid:

        serialized_schema_dataset_id = recursive_find_field(
            schema_data,
            "dataset_id",
        )

        schema_id_valid = (
            str(serialized_schema_dataset_id)
            == str(dataset_id)
        )

        print(
            f"  Schema dataset_id   : "
            f"{'PASS' if schema_id_valid else 'FAIL'}"
        )

        if not schema_id_valid:
            dataset_status = False

            dataset_messages.append(
                f"{dataset_id}: schema dataset_id mismatch."
            )


    if metadata_valid:

        serialized_metadata_dataset_id = recursive_find_field(
            metadata_data,
            "dataset_id",
        )

        metadata_id_valid = (
            str(serialized_metadata_dataset_id)
            == str(dataset_id)
        )

        print(
            f"  Metadata dataset_id : "
            f"{'PASS' if metadata_id_valid else 'FAIL'}"
        )

        if not metadata_id_valid:
            dataset_status = False

            dataset_messages.append(
                f"{dataset_id}: metadata dataset_id mismatch."
            )


    # ==========================================================================
    # 23.12 — LOAD CANONICAL STRUCTURED SCHEMA
    # ==========================================================================

    print("\n23.12 — LOAD CANONICAL STRUCTURED SCHEMA")


    preprocessing_schema = (
        TRAIN_PREPROCESSING_COLUMNS[dataset_id]
    )


    if not isinstance(
        preprocessing_schema,
        dict,
    ):
        raise RuntimeError(
            f"{dataset_id}: TRAIN_PREPROCESSING_COLUMNS must "
            "be a structured dictionary."
        )


    preprocessing_feature_columns = normalize_columns(
        preprocessing_schema.get(
            "all_columns"
        )
    )


    numeric_columns = normalize_columns(
        preprocessing_schema.get(
            "numeric_columns"
        )
    )


    categorical_columns = normalize_columns(
        preprocessing_schema.get(
            "categorical_columns"
        )
    )


    generative_columns = normalize_columns(
        preprocessing_schema.get(
            "generative_columns"
        )
    )


    target_column = normalize_single_column(
        preprocessing_schema.get(
            "target_column"
        )
    )


    identifier_columns = normalize_columns(
        preprocessing_schema.get(
            "identifier_columns"
        )
    )


    provenance_column = normalize_single_column(
        preprocessing_schema.get(
            "provenance_column"
        )
    )


    if not preprocessing_feature_columns:
        raise RuntimeError(
            f"{dataset_id}: preprocessing feature schema is empty."
        )


    if not generative_columns:
        raise RuntimeError(
            f"{dataset_id}: generative schema is empty."
        )


    print(
        f"  Preprocessing features : "
        f"{len(preprocessing_feature_columns)}"
    )

    print(
        f"  Generative columns     : "
        f"{len(generative_columns)}"
    )

    print(
        f"  Numeric features      : "
        f"{len(numeric_columns)}"
    )

    print(
        f"  Categorical features  : "
        f"{len(categorical_columns)}"
    )

    print(
        f"  Target                : "
        f"{target_column}"
    )


    # ==========================================================================
    # 23.13 — GENERATIVE / PREPROCESSING SCHEMA CONSISTENCY
    # ==========================================================================

    print(
        "\n23.13 — GENERATIVE / PREPROCESSING SCHEMA CONSISTENCY"
    )


    target_in_generative = (
        target_column is not None
        and target_column in generative_columns
    )


    target_excluded_from_preprocessing = (
        target_column is None
        or target_column not in preprocessing_feature_columns
    )


    expected_generative_columns = (
        preprocessing_feature_columns
        + (
            [target_column]
            if target_column is not None
            else []
        )
    )


    generative_schema_order_valid = (
        generative_columns
        == expected_generative_columns
    )


    print(
        f"  Target in generative schema       : "
        f"{'PASS' if target_in_generative else 'FAIL'}"
    )


    print(
        f"  Target excluded from preprocessing: "
        f"{'PASS' if target_excluded_from_preprocessing else 'FAIL'}"
    )


    print(
        f"  Generative schema consistency     : "
        f"{'PASS' if generative_schema_order_valid else 'FAIL'}"
    )


    if not target_in_generative:
        dataset_status = False

        dataset_messages.append(
            f"{dataset_id}: target is not present in "
            "generative schema."
        )


    if not target_excluded_from_preprocessing:
        dataset_status = False

        dataset_messages.append(
            f"{dataset_id}: target incorrectly appears in "
            "preprocessing feature schema."
        )


    if not generative_schema_order_valid:
        dataset_status = False

        dataset_messages.append(
            f"{dataset_id}: generative schema does not equal "
            "preprocessing features followed by target."
        )


    # ==========================================================================
    # 23.14 — CANONICAL SPLIT SCHEMA COVERAGE
    # ==========================================================================

    print(
        "\n23.14 — CANONICAL SPLIT SCHEMA COVERAGE"
    )


    split_schema_results = {}


    for split_name, split_df in [
        ("train", TRAIN_DATASETS[dataset_id]),
        ("validation", VALIDATION_DATASETS[dataset_id]),
        ("test", TEST_DATASETS[dataset_id]),
    ]:

        split_columns = list(
            split_df.columns
        )


        missing_columns = [
            column
            for column in preprocessing_feature_columns
            if column not in split_columns
        ]


        valid = (
            len(missing_columns) == 0
        )


        split_schema_results[split_name] = valid


        print(
            f"  {split_name.capitalize():<12} : "
            f"{'PASS' if valid else 'FAIL'}"
        )


        if not valid:

            dataset_status = False

            dataset_messages.append(
                f"{dataset_id}: missing preprocessing columns "
                f"in {split_name}: {missing_columns}"
            )


    # ==========================================================================
    # 23.15 — SERIALIZED PREPROCESSING SCHEMA VALIDATION
    # ==========================================================================

    print(
        "\n23.15 — SERIALIZED PREPROCESSING SCHEMA VALIDATION"
    )


    serialized_preprocessing_columns = None


    if schema_valid:

        serialized_preprocessing_columns = (
            find_exact_serialized_list(
                schema_data,
                "preprocessing_columns",
                preprocessing_feature_columns,
            )
        )


    serialized_preprocessing_valid = (
        serialized_preprocessing_columns is not None
    )


    print(
        f"  preprocessing_columns : "
        f"{'PASS' if serialized_preprocessing_valid else 'FAIL'}"
    )


    if not serialized_preprocessing_valid:

        dataset_status = False

        dataset_messages.append(
            f"{dataset_id}: persisted preprocessing_columns "
            "do not match canonical preprocessing features."
        )


    # --------------------------------------------------------------------------
    # Serialized all_columns
    # --------------------------------------------------------------------------

    serialized_all_columns = None


    if schema_valid:

        serialized_all_columns = (
            find_exact_serialized_list(
                schema_data,
                "all_columns",
                preprocessing_feature_columns,
            )
        )


    serialized_all_columns_valid = (
        serialized_all_columns is not None
    )


    print(
        f"  all_columns           : "
        f"{'PASS' if serialized_all_columns_valid else 'FAIL'}"
    )


    if not serialized_all_columns_valid:

        dataset_status = False

        dataset_messages.append(
            f"{dataset_id}: persisted all_columns do not "
            "match canonical preprocessing features."
        )


    # --------------------------------------------------------------------------
    # Serialized generative schema
    # --------------------------------------------------------------------------

    serialized_generative_columns = None


    if schema_valid:

        serialized_generative_columns = (
            find_exact_serialized_list(
                schema_data,
                "generative_columns",
                generative_columns,
            )
        )


    serialized_generative_valid = (
        serialized_generative_columns is not None
    )


    print(
        f"  generative_columns    : "
        f"{'PASS' if serialized_generative_valid else 'FAIL'}"
    )


    if not serialized_generative_valid:

        dataset_status = False

        dataset_messages.append(
            f"{dataset_id}: persisted generative_columns do "
            "not match canonical generative schema."
        )


    # ==========================================================================
    # 23.16 — FITTED PREPROCESSOR INPUT SCHEMA
    # ==========================================================================

    print(
        "\n23.16 — FITTED PREPROCESSOR INPUT SCHEMA"
    )


    fitted_input_columns = []


    if preprocessor_fitted_valid:

        fitted_input_columns = normalize_columns(
            reloaded_preprocessor.feature_names_in_
        )


        fitted_input_valid = (
            fitted_input_columns
            == preprocessing_feature_columns
        )


        print(
            f"  Canonical preprocessing features : "
            f"{len(preprocessing_feature_columns)}"
        )


        print(
            f"  Fitted preprocessor inputs       : "
            f"{len(fitted_input_columns)}"
        )


        print(
            f"  Exact fitted input schema        : "
            f"{'PASS' if fitted_input_valid else 'FAIL'}"
        )


        if not fitted_input_valid:

            missing_fitted = [
                column
                for column in preprocessing_feature_columns
                if column not in fitted_input_columns
            ]


            extra_fitted = [
                column
                for column in fitted_input_columns
                if column not in preprocessing_feature_columns
            ]


            print(
                f"    Missing: {missing_fitted}"
            )

            print(
                f"    Extra  : {extra_fitted}"
            )


            dataset_status = False

            dataset_messages.append(
                f"{dataset_id}: fitted preprocessor input "
                "schema does not exactly match canonical "
                "preprocessing features."
            )


    else:

        fitted_input_valid = False


    # ==========================================================================
    # 23.17 — TARGET / IDENTIFIER / PROVENANCE EXCLUSION
    # ==========================================================================

    print(
        "\n23.17 — TARGET / IDENTIFIER / PROVENANCE EXCLUSION"
    )


    target_not_in_fitted_input = (
        target_column is None
        or target_column not in fitted_input_columns
    )


    identifiers_not_in_fitted_input = all(
        column not in fitted_input_columns
        for column in identifier_columns
    )


    provenance_not_in_fitted_input = (
        provenance_column is None
        or provenance_column not in fitted_input_columns
    )


    print(
        f"  Target excluded from fitted input : "
        f"{'PASS' if target_not_in_fitted_input else 'FAIL'}"
    )


    print(
        f"  Identifiers excluded               : "
        f"{'PASS' if identifiers_not_in_fitted_input else 'FAIL'}"
    )


    print(
        f"  Provenance excluded                : "
        f"{'PASS' if provenance_not_in_fitted_input else 'FAIL'}"
    )


    if not target_not_in_fitted_input:
        dataset_status = False

        dataset_messages.append(
            f"{dataset_id}: target entered fitted "
            "preprocessor input."
        )


    if not identifiers_not_in_fitted_input:
        dataset_status = False

        dataset_messages.append(
            f"{dataset_id}: identifier entered fitted "
            "preprocessor input."
        )


    if not provenance_not_in_fitted_input:
        dataset_status = False

        dataset_messages.append(
            f"{dataset_id}: provenance entered fitted "
            "preprocessor input."
        )


    # ==========================================================================
    # 23.18 — TRANSFORMED FEATURE NAMES
    # ==========================================================================

    print(
        "\n23.18 — TRANSFORMED FEATURE NAMES"
    )


    reloaded_feature_names = []


    if preprocessor_fitted_valid:

        reloaded_feature_names = normalize_columns(
            reloaded_preprocessor.get_feature_names_out()
        )


        persisted_transformed_feature_names = None


        if schema_valid:

            persisted_transformed_feature_names = (
                find_exact_serialized_list(
                    schema_data,
                    "transformed_feature_names",
                    reloaded_feature_names,
                )
            )


        transformed_names_valid = (
            persisted_transformed_feature_names is not None
        )


        print(
            f"  Reloaded transformed features : "
            f"{len(reloaded_feature_names)}"
        )


        print(
            f"  Persisted transformed schema  : "
            f"{'PASS' if transformed_names_valid else 'FAIL'}"
        )


        if not transformed_names_valid:

            dataset_status = False

            dataset_messages.append(
                f"{dataset_id}: persisted transformed feature "
                "names do not match reloaded preprocessor."
            )


    else:

        transformed_names_valid = False


    # ==========================================================================
    # 23.19 — TRANSFORMED FEATURE COUNT
    # ==========================================================================

    print(
        "\n23.19 — TRANSFORMED FEATURE COUNT"
    )


    if preprocessor_fitted_valid:

        persisted_train_shape = array_shape(
            TRANSFORMED_TRAIN_DATASETS[dataset_id]
        )


        reloaded_feature_count = len(
            reloaded_feature_names
        )


        persisted_feature_count = (
            persisted_train_shape[1]
        )


        transformed_count_valid = (
            reloaded_feature_count
            == persisted_feature_count
        )


        print(
            f"  Reloaded count  : "
            f"{reloaded_feature_count}"
        )


        print(
            f"  Persisted count : "
            f"{persisted_feature_count}"
        )


        print(
            f"  Feature count   : "
            f"{'PASS' if transformed_count_valid else 'FAIL'}"
        )


        if not transformed_count_valid:

            dataset_status = False

            dataset_messages.append(
                f"{dataset_id}: transformed feature count mismatch."
            )


    else:

        transformed_count_valid = False


    # ==========================================================================
    # 23.20 — TRAIN-ONLY FITTING POLICY
    # ==========================================================================

    print(
        "\n23.20 — TRAIN-ONLY FITTING POLICY"
    )


    train_only_policy_valid = False


    if schema_valid:

        persisted_fit_policy = recursive_find_field(
            schema_data,
            "fit_policy",
        )


        if persisted_fit_policy is not None:

            train_only_policy_valid = (
                str(persisted_fit_policy).lower()
                == "train_only"
            )


    if not train_only_policy_valid and metadata_valid:

        persisted_fit_policy = recursive_find_field(
            metadata_data,
            "fit_policy",
        )


        if persisted_fit_policy is not None:

            train_only_policy_valid = (
                str(persisted_fit_policy).lower()
                == "train_only"
            )


    print(
        f"  Persisted fit_policy == train_only : "
        f"{'PASS' if train_only_policy_valid else 'FAIL'}"
    )


    if not train_only_policy_valid:

        dataset_status = False

        dataset_messages.append(
            f"{dataset_id}: persisted train-only fitting "
            "policy could not be validated."
        )


    # ==========================================================================
    # 23.21 — TARGET POLICY VALIDATION
    # ==========================================================================

    print(
        "\n23.21 — TARGET POLICY VALIDATION"
    )


    target_policy = (
        recursive_find_field(
            schema_data,
            "target_policy",
        )
        if schema_valid
        else None
    )


    if not isinstance(
        target_policy,
        dict,
    ):

        target_policy_valid = False

    else:

        target_retained_in_generative = (
            target_policy.get(
                "retained_in_generative_schema"
            )
            is True
        )


        target_excluded_from_preprocessor = (
            target_policy.get(
                "excluded_from_preprocessor_input"
            )
            is True
        )


        target_excluded_from_transformed = (
            target_policy.get(
                "excluded_from_transformed_features"
            )
            is True
        )


        raw_target_not_appended = (
            target_policy.get(
                "raw_target_manually_appended"
            )
            is False
        )


        target_policy_valid = all([
            target_retained_in_generative,
            target_excluded_from_preprocessor,
            target_excluded_from_transformed,
            raw_target_not_appended,
        ])


        print(
            f"  Retained in generative schema : "
            f"{'PASS' if target_retained_in_generative else 'FAIL'}"
        )


        print(
            f"  Excluded from preprocessor    : "
            f"{'PASS' if target_excluded_from_preprocessor else 'FAIL'}"
        )


        print(
            f"  Excluded from transformed     : "
            f"{'PASS' if target_excluded_from_transformed else 'FAIL'}"
        )


        print(
            f"  Raw target manually appended  : "
            f"{'PASS' if raw_target_not_appended else 'FAIL'}"
        )


    print(
        f"  Overall target policy          : "
        f"{'PASS' if target_policy_valid else 'FAIL'}"
    )


    if not target_policy_valid:

        dataset_status = False

        dataset_messages.append(
            f"{dataset_id}: persisted target policy "
            "is inconsistent with frozen architecture."
        )


    # ==========================================================================
    # 23.22 — RELOADED PREPROCESSOR TRANSFORMATION TEST
    # ==========================================================================

    print(
        "\n23.22 — RELOADED PREPROCESSOR TRANSFORMATION TEST"
    )


    reloaded_transform_valid = False


    if preprocessor_fitted_valid:

        try:

            # ------------------------------------------------------------------
            # ONLY preprocessing features are selected.
            #
            # Target, identifiers and provenance remain outside the
            # transformation input.
            # ------------------------------------------------------------------

            train_input = (
                TRAIN_DATASETS[dataset_id][
                    fitted_input_columns
                ]
            )


            validation_input = (
                VALIDATION_DATASETS[dataset_id][
                    fitted_input_columns
                ]
            )


            test_input = (
                TEST_DATASETS[dataset_id][
                    fitted_input_columns
                ]
            )


            # ------------------------------------------------------------------
            # Exact input schema
            # ------------------------------------------------------------------

            train_input_valid = (
                list(train_input.columns)
                == fitted_input_columns
            )


            validation_input_valid = (
                list(validation_input.columns)
                == fitted_input_columns
            )


            test_input_valid = (
                list(test_input.columns)
                == fitted_input_columns
            )


            print(
                "  Reload transformation input schema:"
            )


            print(
                f"    Train      : "
                f"{'PASS' if train_input_valid else 'FAIL'}"
            )


            print(
                f"    Validation : "
                f"{'PASS' if validation_input_valid else 'FAIL'}"
            )


            print(
                f"    Test       : "
                f"{'PASS' if test_input_valid else 'FAIL'}"
            )


            if not all([
                train_input_valid,
                validation_input_valid,
                test_input_valid,
            ]):

                raise RuntimeError(
                    "Canonical split DataFrames do not contain "
                    "the complete fitted preprocessing input schema."
                )


            # ------------------------------------------------------------------
            # CRITICAL:
            # transform() ONLY.
            #
            # NO fit()
            # NO fit_transform()
            # ------------------------------------------------------------------

            reloaded_train = (
                reloaded_preprocessor.transform(
                    train_input
                )
            )


            reloaded_validation = (
                reloaded_preprocessor.transform(
                    validation_input
                )
            )


            reloaded_test = (
                reloaded_preprocessor.transform(
                    test_input
                )
            )


            # ------------------------------------------------------------------
            # Shapes
            # ------------------------------------------------------------------

            reloaded_train_shape = array_shape(
                reloaded_train
            )


            reloaded_validation_shape = array_shape(
                reloaded_validation
            )


            reloaded_test_shape = array_shape(
                reloaded_test
            )


            persisted_train_shape = array_shape(
                TRANSFORMED_TRAIN_DATASETS[dataset_id]
            )


            persisted_validation_shape = array_shape(
                TRANSFORMED_VALIDATION_DATASETS[dataset_id]
            )


            persisted_test_shape = array_shape(
                TRANSFORMED_TEST_DATASETS[dataset_id]
            )


            # ------------------------------------------------------------------
            # Exact persisted-shape comparison
            # ------------------------------------------------------------------

            train_shape_match = (
                reloaded_train_shape
                == persisted_train_shape
            )


            validation_shape_match = (
                reloaded_validation_shape
                == persisted_validation_shape
            )


            test_shape_match = (
                reloaded_test_shape
                == persisted_test_shape
            )


            # ------------------------------------------------------------------
            # Row-count validation
            # ------------------------------------------------------------------

            train_expected_shape = (
                len(TRAIN_DATASETS[dataset_id]),
                persisted_train_shape[1],
            )


            validation_expected_shape = (
                len(VALIDATION_DATASETS[dataset_id]),
                persisted_validation_shape[1],
            )


            test_expected_shape = (
                len(TEST_DATASETS[dataset_id]),
                persisted_test_shape[1],
            )


            train_expected_match = (
                reloaded_train_shape
                == train_expected_shape
            )


            validation_expected_match = (
                reloaded_validation_shape
                == validation_expected_shape
            )


            test_expected_match = (
                reloaded_test_shape
                == test_expected_shape
            )


            reloaded_transform_valid = all([
                train_shape_match,
                validation_shape_match,
                test_shape_match,
                train_expected_match,
                validation_expected_match,
                test_expected_match,
            ])


            print(
                "\n  Reloaded transformation shapes:"
            )


            print(
                f"    Train      : "
                f"{'PASS' if train_shape_match else 'FAIL'} "
                f"{reloaded_train_shape}"
            )


            print(
                f"    Validation : "
                f"{'PASS' if validation_shape_match else 'FAIL'} "
                f"{reloaded_validation_shape}"
            )


            print(
                f"    Test       : "
                f"{'PASS' if test_shape_match else 'FAIL'} "
                f"{reloaded_test_shape}"
            )


            print(
                "\n  Persisted transformed shapes:"
            )


            print(
                f"    Train      : "
                f"{persisted_train_shape}"
            )


            print(
                f"    Validation : "
                f"{persisted_validation_shape}"
            )


            print(
                f"    Test       : "
                f"{persisted_test_shape}"
            )


            if not reloaded_transform_valid:

                dataset_status = False

                dataset_messages.append(
                    f"{dataset_id}: reloaded preprocessor does "
                    "not reproduce persisted transformed shapes."
                )


        except Exception as exc:

            reloaded_transform_valid = False
            dataset_status = False

            dataset_messages.append(
                f"{dataset_id}: reloaded transformation failed — {exc}"
            )


            print(
                f"\n  Reloaded transformation: FAIL — {exc}"
            )


    else:

        dataset_status = False

        dataset_messages.append(
            f"{dataset_id}: transformation test skipped because "
            "preprocessor reload failed."
        )


    # ==========================================================================
    # 23.23 — FINAL DATASET STATUS
    # ==========================================================================

    print("\n" + "-" * 100)
    print(f"23.23 — FINAL STATUS: {dataset_id}")
    print("-" * 100)


    final_dataset_status = (
        "PASS"
        if dataset_status
        else "FAIL"
    )


    print(
        f"Dataset validation status: "
        f"{final_dataset_status}"
    )


    if dataset_messages:

        print("\nValidation messages:")

        for message in dataset_messages:
            print(f"  • {message}")

    else:

        print(
            "  ✓ No validation errors detected."
        )


    VALIDATION_RESULTS.append({
        "dataset_id": dataset_id,
        "status": final_dataset_status,
        "messages": dataset_messages,
    })


# ==============================================================================
# 23.24 — CONSOLIDATED VALIDATION SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("23.24 — CONSOLIDATED VALIDATION SUMMARY")
print("=" * 100)


VALIDATION_SUMMARY_DF = pd.DataFrame(
    [
        {
            "dataset_id": result["dataset_id"],
            "status": result["status"],
            "n_messages": len(result["messages"]),
        }
        for result in VALIDATION_RESULTS
    ]
)


display(
    VALIDATION_SUMMARY_DF
)


# ==============================================================================
# 23.25 — GLOBAL VALIDATION STATUS
# ==============================================================================

all_datasets_pass = (
    len(VALIDATION_RESULTS)
    == len(DATASET_IDS)
    and
    all(
        result["status"] == "PASS"
        for result in VALIDATION_RESULTS
    )
)


SECTION_23_STATUS = (
    "PASS"
    if all_datasets_pass
    else "FAIL"
)


print("\n" + "=" * 100)
print("SECTION 23 FINAL STATUS")
print("=" * 100)


print(
    f"Datasets validated : "
    f"{len(VALIDATION_RESULTS)}"
)


print(
    f"Datasets passed    : "
    f"{sum(
        result['status'] == 'PASS'
        for result in VALIDATION_RESULTS
    )}"
)


print(
    f"Datasets failed    : "
    f"{sum(
        result['status'] == 'FAIL'
        for result in VALIDATION_RESULTS
    )}"
)


print(
    f"\nSECTION 23 STATUS: "
    f"{SECTION_23_STATUS}"
)


# ==============================================================================
# 23.26 — HARD FAILURE GUARD
# ==============================================================================

if SECTION_23_STATUS != "PASS":

    failed_datasets = [
        result["dataset_id"]
        for result in VALIDATION_RESULTS
        if result["status"] != "PASS"
    ]


    raise RuntimeError(
        "\n"
        + "=" * 100
        + "\n"
        + "SECTION 23 VALIDATION FAILED"
        + "\n"
        + "=" * 100
        + "\n"
        + f"Failed datasets: {failed_datasets}\n"
        + "\n"
        + "Do not proceed to downstream modeling until the "
          "persisted preprocessing artifacts are validated."
    )


# ==============================================================================
# 23.27 — VALIDATION MANIFEST
# ==============================================================================

SECTION_23_VALIDATION_MANIFEST = {

    "section":
        "23",

    "section_name":
        "Reload & Validate Persisted Artifacts",

    "status":
        SECTION_23_STATUS,

    "datasets":
        DATASET_IDS,

    "n_datasets":
        len(DATASET_IDS),

    "validated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "validation_scope": [

        "required canonical objects",

        "canonical dataset registry",

        "artifact existence",

        "preprocessor reload",

        "preprocessor fitted-state validation",

        "schema reload",

        "metadata reload",

        "dataset identifier consistency",

        "generative schema consistency",

        "preprocessing schema consistency",

        "canonical split schema coverage",

        "fitted preprocessor input schema",

        "target exclusion from preprocessing",

        "identifier exclusion",

        "provenance exclusion",

        "target policy validation",

        "transformed feature names",

        "transformed feature count",

        "train-only fitting policy",

        "reloaded train transformation",

        "reloaded validation transformation",

        "reloaded test transformation",

        "persisted-vs-reloaded transformed shape consistency",
    ],
}


print("\n" + "=" * 100)
print("23.27 — SECTION 23 VALIDATION MANIFEST")
print("=" * 100)


print(
    json.dumps(
        SECTION_23_VALIDATION_MANIFEST,
        indent=2,
        ensure_ascii=False,
    )
)


# ==============================================================================
# SECTION 23 COMPLETE
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 23 COMPLETE")
print("=" * 100)


print(
    f"FINAL STATUS: {SECTION_23_STATUS}"
)


if SECTION_23_STATUS == "PASS":

    print(
        "All persisted preprocessing artifacts have been "
        "reloaded and validated successfully."
    )

SECTION 23 — RELOAD & VALIDATE PERSISTED ARTIFACTS

----------------------------------------------------------------------------------------------------
23.01 — REQUIRED OBJECT VALIDATION
----------------------------------------------------------------------------------------------------
Required canonical objects:
  ✓ TRAIN_DATASETS
  ✓ VALIDATION_DATASETS
  ✓ TEST_DATASETS
  ✓ TRAIN_PREPROCESSORS
  ✓ TRAIN_PREPROCESSING_COLUMNS
  ✓ TRAIN_PREPROCESSING_METADATA
  ✓ TRANSFORMED_TRAIN_DATASETS
  ✓ TRANSFORMED_VALIDATION_DATASETS
  ✓ TRANSFORMED_TEST_DATASETS
  ✓ NATIVE_FINAL_DATASETS
  ✓ IDENTIFIER_COLUMNS
  ✓ TARGET_COLUMNS
  ✓ NB02_DIRECTORIES

----------------------------------------------------------------------------------------------------
23.02 — CANONICAL DATASET REGISTRY VALIDATION
----------------------------------------------------------------------------------------------------
  ✓ TRAIN_DATASETS
  ✓ VALIDATION_DATASETS
  ✓ TEST_DATASETS
  ✓ TRAIN_PREPROCESSORS
  ✓ TRAIN_PRE

,dataset_id,status,n_messages
0,adult_income,PASS,0
1,bank_marketing,PASS,0
2,diabetes_130us,PASS,0



SECTION 23 FINAL STATUS
Datasets validated : 3
Datasets passed    : 3
Datasets failed    : 0

SECTION 23 STATUS: PASS

23.27 — SECTION 23 VALIDATION MANIFEST
{
  "section": "23",
  "section_name": "Reload & Validate Persisted Artifacts",
  "status": "PASS",
  "datasets": [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
  ],
  "n_datasets": 3,
  "validated_at_utc": "2026-09-08T07:07:55.817226+00:00",
  "validation_scope": [
    "required canonical objects",
    "canonical dataset registry",
    "artifact existence",
    "preprocessor reload",
    "preprocessor fitted-state validation",
    "schema reload",
    "metadata reload",
    "dataset identifier consistency",
    "generative schema consistency",
    "preprocessing schema consistency",
    "canonical split schema coverage",
    "fitted preprocessor input schema",
    "target exclusion from preprocessing",
    "identifier exclusion",
    "provenance exclusion",
    "target policy validation",
    "transformed featu

In [54]:
# ==================================================================================================
# 24. FINAL INTEGRITY VERIFICATION — FROZEN NOTEBOOK 02 GATE
# ==================================================================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

print("=" * 100)
print("24. FINAL INTEGRITY VERIFICATION")
print("=" * 100)


# ==================================================================================================
# 24.01 CONFIGURATION
# ==================================================================================================

NB02_ROOT = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "notebook_02"
)

SCHEMA_ROOT = os.path.join(
    NB02_ROOT,
    "schemas"
)

PREPROCESSOR_ROOT = os.path.join(
    NB02_ROOT,
    "preprocessors"
)

METADATA_ROOT = os.path.join(
    SCHEMA_ROOT,
    "metadata"
)

FINAL_MANIFEST_PATH = os.path.join(
    SCHEMA_ROOT,
    "section_24_final_integrity_manifest.json"
)

FINAL_INTEGRITY_RECORDS = []


# ==================================================================================================
# 24.02 HELPERS
# ==================================================================================================

def add_check(name, passed, details=""):

    passed = bool(passed)

    FINAL_INTEGRITY_RECORDS.append({
        "check": name,
        "status": "PASS" if passed else "FAIL",
        "details": details
    })

    symbol = "✓" if passed else "✗"

    print(
        f"{symbol} {name:<50}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

    if not passed and details:
        print(f"    {details}")


def resolve_dataframe(obj):

    if isinstance(obj, pd.DataFrame):
        return obj

    if isinstance(obj, dict):

        for value in obj.values():

            if isinstance(value, pd.DataFrame):
                return value

    return None


def resolve_array(obj):

    if isinstance(obj, np.ndarray):
        return obj

    if isinstance(obj, dict):

        for value in obj.values():

            if isinstance(value, np.ndarray):
                return value

    return None


def file_exists(path):

    return os.path.isfile(path)


def normalize_identifier_list(value):

    if value is None:
        return []

    if isinstance(value, str):
        return [value]

    return list(value)


# ==================================================================================================
# 24.03 REQUIRED CANONICAL OBJECTS
# ==================================================================================================

REQUIRED_OBJECT_NAMES = [
    "DATASET_IDS",
    "RAW_DATASETS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
    "SPLIT_MANIFESTS",
    "SPLIT_SUMMARY_DF",
    "TRAIN_PREPROCESSORS",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TRAIN_PREPROCESSING_METADATA",
    "TRANSFORMED_TRAIN_DATASETS",
    "TRANSFORMED_VALIDATION_DATASETS",
    "TRANSFORMED_TEST_DATASETS",
    "NATIVE_FINAL_DATASETS",
    "ENCODED_FINAL_DATASETS",
    "NB02_DIRECTORIES",
]

missing_objects = [
    name
    for name in REQUIRED_OBJECT_NAMES
    if name not in globals()
]

add_check(
    "required_canonical_objects",
    len(missing_objects) == 0,
    (
        ""
        if not missing_objects
        else f"Missing objects: {missing_objects}"
    )
)


# ==================================================================================================
# 24.04 DATASET REGISTRY
# ==================================================================================================

dataset_count_ok = (
    isinstance(DATASET_IDS, (list, tuple))
    and len(DATASET_IDS) == 3
)

dataset_identifier_ok = (
    dataset_count_ok
    and len(DATASET_IDS) == len(set(DATASET_IDS))
    and all(
        isinstance(dataset_id, str)
        and dataset_id.strip()
        for dataset_id in DATASET_IDS
    )
)

add_check(
    "dataset_count",
    dataset_count_ok
)

add_check(
    "dataset_identifier_validity",
    dataset_identifier_ok
)

registry_ok = True

if dataset_count_ok:

    for dataset_id in DATASET_IDS:

        required_containers = [
            RAW_DATASETS,
            TARGET_COLUMNS,
            IDENTIFIER_COLUMNS,
            TRAIN_DATASETS,
            VALIDATION_DATASETS,
            TEST_DATASETS,
            SPLIT_MANIFESTS,
            TRAIN_PREPROCESSORS,
            TRAIN_PREPROCESSING_COLUMNS,
            TRAIN_PREPROCESSING_METADATA,
            TRANSFORMED_TRAIN_DATASETS,
            TRANSFORMED_VALIDATION_DATASETS,
            TRANSFORMED_TEST_DATASETS,
            NATIVE_FINAL_DATASETS,
            ENCODED_FINAL_DATASETS,
        ]

        if not all(
            dataset_id in container
            for container in required_containers
        ):
            registry_ok = False


add_check(
    "canonical_registry_consistency",
    registry_ok
)


# ==================================================================================================
# 24.05 RAW DATA / TARGET / IDENTIFIER VALIDATION
# ==================================================================================================

raw_ok = True
target_ok = True
identifier_policy_ok = True

for dataset_id in DATASET_IDS:

    raw_df = resolve_dataframe(
        RAW_DATASETS[dataset_id]
    )

    if raw_df is None or raw_df.empty:

        raw_ok = False
        continue

    target = TARGET_COLUMNS[dataset_id]

    if target not in raw_df.columns:
        target_ok = False

    identifiers = normalize_identifier_list(
        IDENTIFIER_COLUMNS.get(dataset_id, [])
    )

    missing_identifiers = [
        identifier
        for identifier in identifiers
        if identifier not in raw_df.columns
    ]

    if missing_identifiers:
        identifier_policy_ok = False


add_check(
    "raw_datasets_loaded",
    raw_ok
)

add_check(
    "targets_validated",
    target_ok
)

add_check(
    "identifier_policy_valid",
    identifier_policy_ok
)


# ==================================================================================================
# 24.06 SPLIT INTEGRITY
# ==================================================================================================

split_manifest_ok = True
split_summary_ok = isinstance(
    SPLIT_SUMMARY_DF,
    pd.DataFrame
)

split_rows_ok = True
split_provenance_ok = True
provenance_unique_ok = True
split_disjoint_ok = True
provenance_conservation_ok = True

for dataset_id in DATASET_IDS:

    train_df = resolve_dataframe(
        TRAIN_DATASETS[dataset_id]
    )

    validation_df = resolve_dataframe(
        VALIDATION_DATASETS[dataset_id]
    )

    test_df = resolve_dataframe(
        TEST_DATASETS[dataset_id]
    )

    raw_df = resolve_dataframe(
        RAW_DATASETS[dataset_id]
    )

    if dataset_id not in SPLIT_MANIFESTS:

        split_manifest_ok = False

    if any(
        df is None
        for df in [
            train_df,
            validation_df,
            test_df,
            raw_df
        ]
    ):

        split_rows_ok = False
        split_provenance_ok = False
        continue

    if (
        len(train_df)
        + len(validation_df)
        + len(test_df)
        != len(raw_df)
    ):

        split_rows_ok = False

    split_frames = [
        train_df,
        validation_df,
        test_df
    ]

    split_sets = []

    for split_df in split_frames:

        if "__original_row_id__" not in split_df.columns:

            split_provenance_ok = False
            continue

        provenance = split_df[
            "__original_row_id__"
        ]

        if provenance.isna().any():
            split_provenance_ok = False

        if not provenance.is_unique:
            provenance_unique_ok = False

        split_sets.append(
            set(provenance.tolist())
        )

    if len(split_sets) == 3:

        if (
            split_sets[0] & split_sets[1]
            or split_sets[0] & split_sets[2]
            or split_sets[1] & split_sets[2]
        ):

            split_disjoint_ok = False

        union_ids = (
            split_sets[0]
            | split_sets[1]
            | split_sets[2]
        )

        if "__original_row_id__" in raw_df.columns:

            raw_ids = set(
                raw_df[
                    "__original_row_id__"
                ].dropna().tolist()
            )

            if union_ids != raw_ids:
                provenance_conservation_ok = False

        else:

            if len(union_ids) != len(raw_df):
                provenance_conservation_ok = False


add_check(
    "split_manifest_coverage",
    split_manifest_ok
)

add_check(
    "split_summary_valid",
    split_summary_ok
)

add_check(
    "split_row_count_preservation",
    split_rows_ok
)

add_check(
    "split_provenance_present",
    split_provenance_ok
)

add_check(
    "provenance_uniqueness",
    provenance_unique_ok
)

add_check(
    "split_disjointness",
    split_disjoint_ok
)

add_check(
    "provenance_conservation",
    provenance_conservation_ok
)


# ==================================================================================================
# 24.07 PREPROCESSOR INTEGRITY
# ==================================================================================================

preprocessing_schema_ok = True
preprocessor_count_ok = (
    len(TRAIN_PREPROCESSORS) == len(DATASET_IDS)
)

preprocessors_fitted_ok = True
modeling_policy_ok = True
train_only_ok = True
preprocessor_input_schema_ok = True

for dataset_id in DATASET_IDS:

    preprocessor = TRAIN_PREPROCESSORS.get(
        dataset_id
    )

    schema_info = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]

    target = TARGET_COLUMNS[dataset_id]

    identifiers = normalize_identifier_list(
        IDENTIFIER_COLUMNS.get(dataset_id, [])
    )

    provenance_column = "__original_row_id__"

    forbidden_columns = set(
        [target, provenance_column]
        + identifiers
    )

    if preprocessor is None:

        preprocessors_fitted_ok = False
        modeling_policy_ok = False
        train_only_ok = False
        preprocessor_input_schema_ok = False
        continue

    if not hasattr(
        preprocessor,
        "feature_names_in_"
    ):

        preprocessors_fitted_ok = False
        train_only_ok = False
        preprocessor_input_schema_ok = False
        continue

    fitted_columns = list(
        preprocessor.feature_names_in_
    )

    # ----------------------------------------------------------------------------------------------
    # Correct canonical preprocessing feature schema
    # ----------------------------------------------------------------------------------------------

    expected_columns = list(
        schema_info["all_columns"]
    )

    if fitted_columns != expected_columns:

        preprocessor_input_schema_ok = False
        train_only_ok = False

        print(
            f"    Input schema mismatch: {dataset_id}"
        )

    # ----------------------------------------------------------------------------------------------
    # Target / identifier / provenance exclusion
    # ----------------------------------------------------------------------------------------------

    forbidden_present = [
        column
        for column in fitted_columns
        if column in forbidden_columns
    ]

    if forbidden_present:

        modeling_policy_ok = False

        print(
            f"    Modeling policy violation: "
            f"{dataset_id} | "
            f"forbidden={forbidden_present}"
        )

    # ----------------------------------------------------------------------------------------------
    # Train-only input schema
    # ----------------------------------------------------------------------------------------------

    train_df = resolve_dataframe(
        TRAIN_DATASETS[dataset_id]
    )

    if train_df is None:

        train_only_ok = False
        continue

    train_feature_columns = [
        column
        for column in train_df.columns
        if column not in forbidden_columns
    ]

    if fitted_columns != train_feature_columns:

        train_only_ok = False


# ----------------------------------------------------------------------------------------------
# Persisted schema directories
# ----------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    schema_path = os.path.join(
        SCHEMA_ROOT,
        dataset_id,
        "preprocessing_schema.json"
    )

    if not file_exists(schema_path):
        preprocessing_schema_ok = False


add_check(
    "preprocessing_schema_integrity",
    preprocessing_schema_ok
)

add_check(
    "training_preprocessor_count",
    preprocessor_count_ok
)

add_check(
    "preprocessors_fitted",
    preprocessors_fitted_ok
)

add_check(
    "modeling_column_policy",
    modeling_policy_ok
)

add_check(
    "train_only_preprocessing",
    train_only_ok
)

add_check(
    "preprocessor_input_schema",
    preprocessor_input_schema_ok
)


# ==================================================================================================
# 24.08 TRANSFORMED DATA INTEGRITY
# ==================================================================================================

transformed_feature_schema_ok = True
transformed_shape_ok = True
transformed_finite_ok = True
transformed_dtype_ok = True

for dataset_id in DATASET_IDS:

    preprocessor = TRAIN_PREPROCESSORS[
        dataset_id
    ]

    try:

        expected_features = list(
            preprocessor.get_feature_names_out()
        )

    except Exception:

        transformed_feature_schema_ok = False
        expected_features = []

    split_objects = [
        (
            TRANSFORMED_TRAIN_DATASETS,
            TRAIN_DATASETS
        ),
        (
            TRANSFORMED_VALIDATION_DATASETS,
            VALIDATION_DATASETS
        ),
        (
            TRANSFORMED_TEST_DATASETS,
            TEST_DATASETS
        )
    ]

    for transformed_container, native_container in split_objects:

        transformed = resolve_array(
            transformed_container[dataset_id]
        )

        native_df = resolve_dataframe(
            native_container[dataset_id]
        )

        if transformed is None:

            transformed_shape_ok = False
            continue

        if transformed.ndim != 2:

            transformed_shape_ok = False
            continue

        if (
            expected_features
            and transformed.shape[1]
            != len(expected_features)
        ):

            transformed_feature_schema_ok = False

        if native_df is not None:

            if transformed.shape[0] != len(native_df):

                transformed_shape_ok = False

        if not np.issubdtype(
            transformed.dtype,
            np.floating
        ):

            transformed_dtype_ok = False

        if not np.isfinite(transformed).all():

            transformed_finite_ok = False


add_check(
    "transformed_feature_schema",
    transformed_feature_schema_ok
)

add_check(
    "transformed_array_shapes",
    transformed_shape_ok
)

add_check(
    "transformed_values_finite",
    transformed_finite_ok
)

add_check(
    "transformed_numeric_dtype",
    transformed_dtype_ok
)


# ==================================================================================================
# 24.09 NATIVE FINAL DATASETS
# ==================================================================================================

native_structure_ok = True
native_rows_ok = True
native_dataframe_type_ok = True
native_target_ok = True
native_provenance_ok = True
native_identifier_ok = True
native_schema_ok = True

for dataset_id in DATASET_IDS:

    native_df = resolve_dataframe(
        NATIVE_FINAL_DATASETS[dataset_id]
    )

    train_df = resolve_dataframe(
        TRAIN_DATASETS[dataset_id]
    )

    schema_info = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]

    if native_df is None:

        native_structure_ok = False
        native_dataframe_type_ok = False
        continue

    if train_df is None:

        native_rows_ok = False
        native_schema_ok = False
        continue

    # ----------------------------------------------------------------------------------------------
    # Row count
    # ----------------------------------------------------------------------------------------------

    if len(native_df) != len(train_df):

        native_rows_ok = False

    # ----------------------------------------------------------------------------------------------
    # Provenance
    # ----------------------------------------------------------------------------------------------

    if "__original_row_id__" not in native_df.columns:

        native_provenance_ok = False

    # ----------------------------------------------------------------------------------------------
    # Target retention
    # ----------------------------------------------------------------------------------------------

    target = TARGET_COLUMNS[dataset_id]

    if target not in native_df.columns:

        native_target_ok = False

    # ----------------------------------------------------------------------------------------------
    # Identifier exclusion
    # ----------------------------------------------------------------------------------------------

    identifiers = normalize_identifier_list(
        IDENTIFIER_COLUMNS.get(dataset_id, [])
    )

    if any(
        identifier in native_df.columns
        for identifier in identifiers
    ):

        native_identifier_ok = False

    # ----------------------------------------------------------------------------------------------
    # Canonical native schema
    #
    # Native final dataset:
    #   provenance
    #   + generative columns
    #
    # Generative columns:
    #   preprocessing features
    #   + target
    #
    # Explicit identifiers are excluded.
    # ----------------------------------------------------------------------------------------------

    expected_native_columns = (
        ["__original_row_id__"]
        + list(schema_info["generative_columns"])
    )

    if list(native_df.columns) != expected_native_columns:

        native_schema_ok = False

        print(
            f"    Native schema mismatch: {dataset_id}"
        )


add_check(
    "native_dataset_structure",
    native_structure_ok
)

add_check(
    "native_row_count_preservation",
    native_rows_ok
)

add_check(
    "native_dataframe_type",
    native_dataframe_type_ok
)

add_check(
    "native_target_retention",
    native_target_ok
)

add_check(
    "native_provenance_retention",
    native_provenance_ok
)

add_check(
    "native_identifier_exclusion",
    native_identifier_ok
)

add_check(
    "native_schema_consistency",
    native_schema_ok
)


# ==================================================================================================
# 24.10 ENCODED FINAL DATASETS
# ==================================================================================================

encoded_structure_ok = True
encoded_rows_ok = True
encoded_target_ok = True
encoded_identifier_ok = True
encoded_provenance_ok = True
encoded_consistency_ok = True

for dataset_id in DATASET_IDS:

    encoded_obj = ENCODED_FINAL_DATASETS[
        dataset_id
    ]

    encoded_df = resolve_dataframe(
        encoded_obj
    )

    encoded_array = resolve_array(
        encoded_obj
    )

    train_df = resolve_dataframe(
        TRAIN_DATASETS[dataset_id]
    )

    transformed_train = resolve_array(
        TRANSFORMED_TRAIN_DATASETS[dataset_id]
    )

    if (
        encoded_df is None
        and encoded_array is None
    ):

        encoded_structure_ok = False
        continue

    if encoded_df is not None:

        if (
            train_df is not None
            and len(encoded_df) != len(train_df)
        ):

            encoded_rows_ok = False

        target = TARGET_COLUMNS[dataset_id]

        if target in encoded_df.columns:

            encoded_target_ok = False

        identifiers = normalize_identifier_list(
            IDENTIFIER_COLUMNS.get(
                dataset_id,
                []
            )
        )

        if any(
            identifier in encoded_df.columns
            for identifier in identifiers
        ):

            encoded_identifier_ok = False

        if "__original_row_id__" in encoded_df.columns:

            encoded_provenance_ok = False

    if encoded_array is not None:

        if transformed_train is not None:

            if encoded_array.shape != transformed_train.shape:

                encoded_consistency_ok = False


add_check(
    "encoded_dataset_structure",
    encoded_structure_ok
)

add_check(
    "encoded_row_count_preservation",
    encoded_rows_ok
)

add_check(
    "encoded_target_policy",
    encoded_target_ok
)

add_check(
    "encoded_identifier_exclusion",
    encoded_identifier_ok
)

add_check(
    "encoded_provenance_exclusion",
    encoded_provenance_ok
)

add_check(
    "encoded_transformed_consistency",
    encoded_consistency_ok
)


# ==================================================================================================
# 24.11 PERSISTED ARTIFACT INTEGRITY
# ==================================================================================================

persisted_preprocessors_ok = True
persisted_schemas_ok = True
persisted_metadata_ok = True

reload_preprocessor_ok = True
reload_schema_ok = True
reload_metadata_ok = True

serialized_schema_ok = True
serialized_metadata_ok = True

reloaded_input_schema_ok = True
reloaded_transformed_schema_ok = True

for dataset_id in DATASET_IDS:

    preprocessor_path = os.path.join(
        PREPROCESSOR_ROOT,
        dataset_id,
        "train_fitted_preprocessor.joblib"
    )

    schema_path = os.path.join(
        SCHEMA_ROOT,
        dataset_id,
        "preprocessing_schema.json"
    )

    # ----------------------------------------------------------------------------------------------
    # Correct Section 20 metadata location
    # ----------------------------------------------------------------------------------------------

    metadata_path = os.path.join(
        METADATA_ROOT,
        f"{dataset_id}_preprocessing_metadata.json"
    )

    # ----------------------------------------------------------------------------------------------
    # Artifact existence
    # ----------------------------------------------------------------------------------------------

    if not file_exists(preprocessor_path):
        persisted_preprocessors_ok = False

    if not file_exists(schema_path):
        persisted_schemas_ok = False

    if not file_exists(metadata_path):
        persisted_metadata_ok = False

    # ----------------------------------------------------------------------------------------------
    # Preprocessor reload
    # ----------------------------------------------------------------------------------------------

    if file_exists(preprocessor_path):

        try:

            reloaded_preprocessor = joblib.load(
                preprocessor_path
            )

            original_preprocessor = TRAIN_PREPROCESSORS[
                dataset_id
            ]

            if list(
                reloaded_preprocessor.feature_names_in_
            ) != list(
                original_preprocessor.feature_names_in_
            ):

                reloaded_input_schema_ok = False

            if list(
                reloaded_preprocessor.get_feature_names_out()
            ) != list(
                original_preprocessor.get_feature_names_out()
            ):

                reloaded_transformed_schema_ok = False

        except Exception as exc:

            reload_preprocessor_ok = False

            print(
                f"    Preprocessor reload error: "
                f"{dataset_id} | {exc}"
            )

    else:

        reload_preprocessor_ok = False

    # ----------------------------------------------------------------------------------------------
    # Schema reload
    # ----------------------------------------------------------------------------------------------

    if file_exists(schema_path):

        try:

            with open(
                schema_path,
                "r",
                encoding="utf-8"
            ) as f:

                schema = json.load(f)

            if not isinstance(schema, dict):

                reload_schema_ok = False
                serialized_schema_ok = False
                continue

            # ------------------------------------------------------------------
            # Required top-level schema sections
            # ------------------------------------------------------------------

            required_schema_sections = [
                "modeling_schema",
                "target_policy",
                "identifier_policy",
                "provenance_policy",
                "transformation_schema",
                "preprocessing",
                "training_rows",
            ]

            missing_sections = [
                key
                for key in required_schema_sections
                if key not in schema
            ]

            if missing_sections:

                serialized_schema_ok = False

                print(
                    f"    Missing schema sections: "
                    f"{dataset_id} | {missing_sections}"
                )

            else:

                preprocessor = TRAIN_PREPROCESSORS[
                    dataset_id
                ]

                fitted_columns = list(
                    preprocessor.feature_names_in_
                )

                transformed_columns = list(
                    preprocessor.get_feature_names_out()
                )

                modeling_schema = schema[
                    "modeling_schema"
                ]

                transformation_schema = schema[
                    "transformation_schema"
                ]

                # --------------------------------------------------------------
                # Preprocessing input schema
                # --------------------------------------------------------------

                serialized_input = list(
                    modeling_schema.get(
                        "preprocessing_columns",
                        []
                    )
                )

                expected_input = list(
                    TRAIN_PREPROCESSING_COLUMNS[
                        dataset_id
                    ]["all_columns"]
                )

                if serialized_input != expected_input:

                    serialized_schema_ok = False

                if fitted_columns != expected_input:

                    serialized_schema_ok = False

                # --------------------------------------------------------------
                # Generative schema
                # --------------------------------------------------------------

                serialized_generative = list(
                    modeling_schema.get(
                        "generative_columns",
                        []
                    )
                )

                expected_generative = list(
                    TRAIN_PREPROCESSING_COLUMNS[
                        dataset_id
                    ]["generative_columns"]
                )

                if (
                    serialized_generative
                    != expected_generative
                ):

                    serialized_schema_ok = False

                # --------------------------------------------------------------
                # Target policy
                # --------------------------------------------------------------

                target = TARGET_COLUMNS[
                    dataset_id
                ]

                if (
                    modeling_schema.get(
                        "target_column"
                    )
                    != target
                ):

                    serialized_schema_ok = False

                if (
                    target
                    not in serialized_generative
                ):

                    serialized_schema_ok = False

                if (
                    target
                    in serialized_input
                ):

                    serialized_schema_ok = False

                # --------------------------------------------------------------
                # Transformed schema
                # --------------------------------------------------------------

                serialized_transformed = list(
                    transformation_schema.get(
                        "transformed_feature_names",
                        []
                    )
                )

                serialized_count = (
                    transformation_schema.get(
                        "transformed_feature_count"
                    )
                )

                if (
                    serialized_transformed
                    != transformed_columns
                ):

                    serialized_schema_ok = False

                if (
                    serialized_count
                    != len(transformed_columns)
                ):

                    serialized_schema_ok = False

        except Exception as exc:

            reload_schema_ok = False
            serialized_schema_ok = False

            print(
                f"    Schema reload error: "
                f"{dataset_id} | {exc}"
            )

    else:

        reload_schema_ok = False
        serialized_schema_ok = False

    # ----------------------------------------------------------------------------------------------
    # Metadata reload
    # ----------------------------------------------------------------------------------------------

    if file_exists(metadata_path):

        try:

            with open(
                metadata_path,
                "r",
                encoding="utf-8"
            ) as f:

                metadata = json.load(f)

            if not isinstance(metadata, dict):

                reload_metadata_ok = False
                serialized_metadata_ok = False

            else:

                # --------------------------------------------------------------
                # Required metadata fields
                # --------------------------------------------------------------

                required_metadata_keys = [
                    "dataset_id",
                    "fit_policy",
                ]

                missing_metadata_keys = [
                    key
                    for key in required_metadata_keys
                    if key not in metadata
                ]

                if missing_metadata_keys:

                    serialized_metadata_ok = False

                # --------------------------------------------------------------
                # Dataset identity
                # --------------------------------------------------------------

                if metadata.get(
                    "dataset_id"
                ) != dataset_id:

                    serialized_metadata_ok = False

                # --------------------------------------------------------------
                # Train-only fitting policy
                # --------------------------------------------------------------

                if metadata.get(
                    "fit_policy"
                ) != "train_only":

                    serialized_metadata_ok = False

        except Exception as exc:

            reload_metadata_ok = False
            serialized_metadata_ok = False

            print(
                f"    Metadata reload error: "
                f"{dataset_id} | {exc}"
            )

    else:

        reload_metadata_ok = False
        serialized_metadata_ok = False


add_check(
    "persisted_preprocessors",
    persisted_preprocessors_ok
)

add_check(
    "persisted_preprocessing_schemas",
    persisted_schemas_ok
)

add_check(
    "persisted_preprocessing_metadata",
    persisted_metadata_ok
)

add_check(
    "persisted_preprocessor_reload",
    reload_preprocessor_ok
)

add_check(
    "persisted_schema_reload",
    reload_schema_ok
)

add_check(
    "persisted_metadata_reload",
    reload_metadata_ok
)

add_check(
    "serialized_schema_consistency",
    serialized_schema_ok
)

add_check(
    "serialized_metadata_consistency",
    serialized_metadata_ok
)

add_check(
    "reloaded_preprocessor_input_schema",
    reloaded_input_schema_ok
)

add_check(
    "reloaded_transformed_schema",
    reloaded_transformed_schema_ok
)


# ==================================================================================================
# 24.12 CROSS-OBJECT CONSISTENCY
# ==================================================================================================

artifact_completeness_ok = True
cross_object_rows_ok = True
feature_dimension_ok = True

for dataset_id in DATASET_IDS:

    train_df = resolve_dataframe(
        TRAIN_DATASETS[dataset_id]
    )

    native_df = resolve_dataframe(
        NATIVE_FINAL_DATASETS[dataset_id]
    )

    transformed_train = resolve_array(
        TRANSFORMED_TRAIN_DATASETS[dataset_id]
    )

    if train_df is None:

        cross_object_rows_ok = False
        continue

    if (
        native_df is None
        or len(native_df) != len(train_df)
    ):

        cross_object_rows_ok = False

    if (
        transformed_train is None
        or transformed_train.shape[0] != len(train_df)
    ):

        cross_object_rows_ok = False

    if transformed_train is not None:

        try:

            expected_dimension = len(
                TRAIN_PREPROCESSORS[
                    dataset_id
                ].get_feature_names_out()
            )

            if (
                transformed_train.shape[1]
                != expected_dimension
            ):

                feature_dimension_ok = False

        except Exception:

            feature_dimension_ok = False

    required_files = [
        os.path.join(
            PREPROCESSOR_ROOT,
            dataset_id,
            "train_fitted_preprocessor.joblib"
        ),
        os.path.join(
            SCHEMA_ROOT,
            dataset_id,
            "preprocessing_schema.json"
        ),
        os.path.join(
            METADATA_ROOT,
            f"{dataset_id}_preprocessing_metadata.json"
        )
    ]

    if not all(
        file_exists(path)
        for path in required_files
    ):

        artifact_completeness_ok = False


add_check(
    "artifact_completeness",
    artifact_completeness_ok
)

add_check(
    "cross_object_row_consistency",
    cross_object_rows_ok
)

add_check(
    "feature_dimension_consistency",
    feature_dimension_ok
)


# ==================================================================================================
# 24.13 FINAL CHECK SUMMARY
# ==================================================================================================

FINAL_INTEGRITY_DF = pd.DataFrame(
    FINAL_INTEGRITY_RECORDS
)

FAILED_CHECKS = FINAL_INTEGRITY_DF.loc[
    FINAL_INTEGRITY_DF["status"] == "FAIL",
    "check"
].tolist()

PASSED_COUNT = int(
    (
        FINAL_INTEGRITY_DF["status"] == "PASS"
    ).sum()
)

FAILED_COUNT = int(
    (
        FINAL_INTEGRITY_DF["status"] == "FAIL"
    ).sum()
)

TOTAL_CHECKS = len(
    FINAL_INTEGRITY_DF
)

OVERALL_FINAL_INTEGRITY_PASS = (
    FAILED_COUNT == 0
)

print()
print("=" * 100)
print("24.13 FINAL CHECK SUMMARY")
print("=" * 100)

print(
    f"Total checks : {TOTAL_CHECKS}"
)

print(
    f"Passed       : {PASSED_COUNT}"
)

print(
    f"Failed       : {FAILED_COUNT}"
)

if FAILED_CHECKS:

    print()
    print("FAILED CHECKS:")

    for name in FAILED_CHECKS:
        print(f"  - {name}")


# ==================================================================================================
# 24.14 SAVE FINAL INTEGRITY MANIFEST
# ==================================================================================================

FINAL_MANIFEST = {
    "section": "24",
    "title": "FINAL INTEGRITY VERIFICATION",
    "notebook": "02",
    "status": (
        "PASS"
        if OVERALL_FINAL_INTEGRITY_PASS
        else "FAIL"
    ),
    "notebook_02_status": (
        "FROZEN"
        if OVERALL_FINAL_INTEGRITY_PASS
        else "NOT_FROZEN"
    ),
    "total_checks": TOTAL_CHECKS,
    "passed_checks": PASSED_COUNT,
    "failed_checks": FAILED_COUNT,
    "failed_check_names": FAILED_CHECKS,
    "checks": FINAL_INTEGRITY_RECORDS,
}

os.makedirs(
    SCHEMA_ROOT,
    exist_ok=True
)

with open(
    FINAL_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print(
    f"✓ Final integrity manifest saved:"
)
print(
    f"  {FINAL_MANIFEST_PATH}"
)


# ==================================================================================================
# 24.15 VERIFY FINAL MANIFEST
# ==================================================================================================

FINAL_MANIFEST_RELOAD_OK = True

try:

    with open(
        FINAL_MANIFEST_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        FINAL_MANIFEST_RELOADED = json.load(f)

    FINAL_MANIFEST_RELOAD_OK = (
        FINAL_MANIFEST_RELOADED
        == FINAL_MANIFEST
    )

except Exception as exc:

    FINAL_MANIFEST_RELOAD_OK = False

    print(
        f"    Final manifest reload error: {exc}"
    )


add_check(
    "final_manifest_reload",
    FINAL_MANIFEST_RELOAD_OK
)


# ==================================================================================================
# 24.16 RECOMPUTE FINAL STATUS AFTER MANIFEST CHECK
# ==================================================================================================

# The manifest reload check is itself a registered integrity check.
# Recompute the complete final status.

FINAL_INTEGRITY_DF = pd.DataFrame(
    FINAL_INTEGRITY_RECORDS
)

FAILED_CHECKS = FINAL_INTEGRITY_DF.loc[
    FINAL_INTEGRITY_DF["status"] == "FAIL",
    "check"
].tolist()

PASSED_COUNT = int(
    (
        FINAL_INTEGRITY_DF["status"] == "PASS"
    ).sum()
)

FAILED_COUNT = int(
    (
        FINAL_INTEGRITY_DF["status"] == "FAIL"
    ).sum()
)

TOTAL_CHECKS = len(
    FINAL_INTEGRITY_DF
)

OVERALL_FINAL_INTEGRITY_PASS = (
    FAILED_COUNT == 0
)


# ==================================================================================================
# 24.17 UPDATE FINAL MANIFEST WITH COMPLETE CHECK SET
# ==================================================================================================

FINAL_MANIFEST = {
    "section": "24",
    "title": "FINAL INTEGRITY VERIFICATION",
    "notebook": "02",
    "status": (
        "PASS"
        if OVERALL_FINAL_INTEGRITY_PASS
        else "FAIL"
    ),
    "notebook_02_status": (
        "FROZEN"
        if OVERALL_FINAL_INTEGRITY_PASS
        else "NOT_FROZEN"
    ),
    "total_checks": TOTAL_CHECKS,
    "passed_checks": PASSED_COUNT,
    "failed_checks": FAILED_COUNT,
    "failed_check_names": FAILED_CHECKS,
    "checks": FINAL_INTEGRITY_RECORDS,
}

with open(
    FINAL_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ==================================================================================================
# 24.18 GLOBAL FINAL STATUS
# ==================================================================================================

print()
print("=" * 100)
print("24.18 GLOBAL FINAL STATUS")
print("=" * 100)

if OVERALL_FINAL_INTEGRITY_PASS:

    print("SECTION 24 STATUS : PASS")
    print("NOTEBOOK 02 STATUS: FROZEN")

    print()
    print(
        "NOTEBOOK 02 FREEZE GATE: PASSED"
    )

else:

    print("SECTION 24 STATUS : FAIL")
    print("NOTEBOOK 02 STATUS: NOT_FROZEN")

    print()
    print(
        "NOTEBOOK 02 FREEZE GATE: FAILED"
    )

    print()

    for name in FAILED_CHECKS:

        print(
            f"  - {name}"
        )


# ==================================================================================================
# 24.19 COMPLETION
# ==================================================================================================

print()
print("=" * 100)
print(
    "SECTION 24 — FINAL INTEGRITY VERIFICATION COMPLETE"
)
print("=" * 100)

print(
    f"Total final checks : {TOTAL_CHECKS}"
)

print(
    f"Passed             : {PASSED_COUNT}"
)

print(
    f"Failed             : {FAILED_COUNT}"
)

print()
print("=" * 100)
print("NOTEBOOK 02 FREEZE GATE")
print("=" * 100)

if OVERALL_FINAL_INTEGRITY_PASS:

    print(
        "✓ PASSED — Notebook 02 may be frozen."
    )

    print()
    print(
        "All canonical datasets, splits, preprocessing artifacts, "
        "transformed datasets, native datasets, encoded datasets, "
        "schemas, metadata, and persisted artifacts passed the "
        "final integrity verification."
    )

else:

    print(
        "✗ FAILED — Notebook 02 MUST NOT be frozen."
    )

    raise RuntimeError(
        "SECTION 24 FINAL INTEGRITY VERIFICATION FAILED.\n"
        f"Registered checks failed: {FAILED_CHECKS}\n"
        "Notebook 02 MUST NOT be frozen."
    )

24. FINAL INTEGRITY VERIFICATION
✓ required_canonical_objects                        : PASS
✓ dataset_count                                     : PASS
✓ dataset_identifier_validity                       : PASS
✓ canonical_registry_consistency                    : PASS
✓ raw_datasets_loaded                               : PASS
✓ targets_validated                                 : PASS
✓ identifier_policy_valid                           : PASS
✓ split_manifest_coverage                           : PASS
✓ split_summary_valid                               : PASS
✓ split_row_count_preservation                      : PASS
✓ split_provenance_present                          : PASS
✓ provenance_uniqueness                             : PASS
✓ split_disjointness                                : PASS
✓ provenance_conservation                           : PASS
✓ preprocessing_schema_integrity                    : PASS
✓ training_preprocessor_count                       : PASS
✓ preprocessors_fitted 

In [55]:
# ==============================================================================
# SECTION 25 — NOTEBOOK 02 FINAL COMPLETION GATE
# ==============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

print("=" * 100)
print("SECTION 25 — NOTEBOOK 02 FINAL COMPLETION GATE")
print("=" * 100)


# ==============================================================================
# 25.1 — REQUIRED CANONICAL OBJECTS
# ==============================================================================

REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
    "TRAIN_PREPROCESSING_DATA",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TRAIN_PREPROCESSING_METADATA",
    "TRAIN_PREPROCESSORS",
    "TRANSFORMED_TRAIN_DATASETS",
    "TRANSFORMED_VALIDATION_DATASETS",
    "TRANSFORMED_TEST_DATASETS",
    "NATIVE_FINAL_DATASETS",
    "ENCODED_FINAL_DATASETS",
    "NB02_DIRECTORIES",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
]

missing_objects = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]

print("\n[25.1] Required canonical objects")
print("-" * 100)

if missing_objects:

    print(
        f"  ✗ Missing canonical objects: "
        f"{missing_objects}"
    )

    raise RuntimeError(
        "Section 25 cannot continue because required "
        "canonical objects are missing."
    )

for name in REQUIRED_OBJECTS:
    print(f"  ✓ {name}")

print("  Required canonical objects : PASS")


# ==============================================================================
# 25.2 — SECTION 24 FINAL INTEGRITY GATE
# ==============================================================================

print("\n[25.2] Section 24 final integrity gate")
print("-" * 100)

if "OVERALL_FINAL_INTEGRITY_PASS" not in globals():

    raise RuntimeError(
        "OVERALL_FINAL_INTEGRITY_PASS not found. "
        "Run Section 24 before Section 25."
    )

section_24_pass = bool(
    OVERALL_FINAL_INTEGRITY_PASS
)

print(
    f"  Section 24 final integrity gate : "
    f"{'PASS' if section_24_pass else 'FAIL'}"
)

if not section_24_pass:

    raise RuntimeError(
        "Section 25 stopped because Section 24 "
        "final integrity verification failed."
    )


# ==============================================================================
# 25.3 — VERIFY SECTION 24 PERSISTED MANIFEST
# ==============================================================================

print("\n[25.3] Section 24 persisted manifest verification")
print("-" * 100)

schema_root = Path(
    NB02_DIRECTORIES["schemas"]
)

section_24_manifest_path = (
    schema_root
    / "section_24_final_integrity_manifest.json"
)

section_24_manifest_exists = (
    section_24_manifest_path.is_file()
    and section_24_manifest_path.stat().st_size > 0
)

print(
    f"  {'✓' if section_24_manifest_exists else '✗'} "
    f"Section 24 manifest : "
    f"{section_24_manifest_path}"
)

if not section_24_manifest_exists:

    raise RuntimeError(
        "Section 24 final integrity manifest is missing."
    )

with open(
    section_24_manifest_path,
    "r",
    encoding="utf-8"
) as f:

    section_24_manifest = json.load(f)

section_24_manifest_pass = (
    section_24_manifest.get("status") == "PASS"
    and
    section_24_manifest.get(
        "notebook_02_status"
    ) == "FROZEN"
    and
    int(
        section_24_manifest.get(
            "failed_checks",
            -1
        )
    ) == 0
)

print(
    f"  Manifest status     : "
    f"{section_24_manifest.get('status')}"
)

print(
    f"  Notebook 02 status  : "
    f"{section_24_manifest.get('notebook_02_status')}"
)

print(
    f"  Failed checks       : "
    f"{section_24_manifest.get('failed_checks')}"
)

print(
    f"\n  Section 24 manifest : "
    f"{'PASS' if section_24_manifest_pass else 'FAIL'}"
)

if not section_24_manifest_pass:

    raise RuntimeError(
        "Persisted Section 24 manifest does not confirm "
        "a PASS/FROZEN state."
    )


# ==============================================================================
# 25.4 — DATASET COMPLETENESS VERIFICATION
# ==============================================================================

print("\n[25.4] Dataset completeness verification")
print("-" * 100)

dataset_completion_records = []

for dataset_id in DATASET_IDS:

    print(f"\n  Dataset: {dataset_id}")

    # --------------------------------------------------------------------------
    # Canonical objects
    # --------------------------------------------------------------------------

    train_df = TRAIN_DATASETS[dataset_id]
    validation_df = VALIDATION_DATASETS[dataset_id]
    test_df = TEST_DATASETS[dataset_id]

    native_train = (
        NATIVE_FINAL_DATASETS[
            dataset_id
        ]["train"]
    )

    encoded_train = (
        ENCODED_FINAL_DATASETS[
            dataset_id
        ]["train"]
    )

    preprocessor = TRAIN_PREPROCESSORS[
        dataset_id
    ]

    schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]

    # --------------------------------------------------------------------------
    # Schema separation
    # --------------------------------------------------------------------------

    preprocessing_columns = list(
        schema["all_columns"]
    )

    numeric_columns = list(
        schema["numeric_columns"]
    )

    categorical_columns = list(
        schema["categorical_columns"]
    )

    generative_columns = list(
        schema["generative_columns"]
    )

    target_column = schema[
        "target_column"
    ]

    identifier_columns = list(
        IDENTIFIER_COLUMNS.get(
            dataset_id,
            []
        )
    )

    # --------------------------------------------------------------------------
    # Transformed feature schema
    #
    # The fitted preprocessor is the authoritative in-memory source.
    # --------------------------------------------------------------------------

    transformed_columns = list(
        preprocessor.get_feature_names_out()
    )

    # --------------------------------------------------------------------------
    # Row counts
    # --------------------------------------------------------------------------

    train_rows = len(train_df)
    validation_rows = len(validation_df)
    test_rows = len(test_df)

    original_rows = (
        train_rows
        + validation_rows
        + test_rows
    )

    # --------------------------------------------------------------------------
    # Expected native schema
    #
    # Native final:
    #   __original_row_id__
    #   +
    #   generative columns
    #
    # Generative:
    #   preprocessing features
    #   +
    #   target
    # --------------------------------------------------------------------------

    expected_native_columns = (
        ["__original_row_id__"]
        + generative_columns
    )

    # --------------------------------------------------------------------------
    # Expected encoded dimensions
    # --------------------------------------------------------------------------

    expected_encoded_columns = len(
        transformed_columns
    )

    native_shape = tuple(
        native_train.shape
    )

    encoded_shape = tuple(
        encoded_train.shape
    )

    # --------------------------------------------------------------------------
    # Target policy
    # --------------------------------------------------------------------------

    target_policy_pass = (
        target_column in generative_columns
        and
        target_column not in preprocessing_columns
        and
        target_column not in transformed_columns
    )

    # --------------------------------------------------------------------------
    # Identifier policy
    # --------------------------------------------------------------------------

    identifier_policy_pass = not any(
        identifier in generative_columns
        for identifier in identifier_columns
    )

    # --------------------------------------------------------------------------
    # Provenance policy
    # --------------------------------------------------------------------------

    provenance_policy_pass = (
        "__original_row_id__"
        not in preprocessing_columns
        and
        "__original_row_id__"
        not in transformed_columns
        and
        "__original_row_id__"
        in expected_native_columns
    )

    # --------------------------------------------------------------------------
    # Native shape
    # --------------------------------------------------------------------------

    native_pass = (
        native_shape[0] == train_rows
        and
        native_shape[1]
        == len(expected_native_columns)
    )

    # --------------------------------------------------------------------------
    # Native schema
    # --------------------------------------------------------------------------

    native_schema_pass = (
        list(native_train.columns)
        == expected_native_columns
    )

    # --------------------------------------------------------------------------
    # Encoded shape
    # --------------------------------------------------------------------------

    encoded_pass = (
        encoded_shape[0] == train_rows
        and
        encoded_shape[1]
        == expected_encoded_columns
    )

    # --------------------------------------------------------------------------
    # Dataset completion
    # --------------------------------------------------------------------------

    dataset_pass = all(
        [
            train_rows > 0,
            validation_rows > 0,
            test_rows > 0,
            len(preprocessing_columns) > 0,
            len(generative_columns) > 0,
            target_policy_pass,
            identifier_policy_pass,
            provenance_policy_pass,
            native_pass,
            native_schema_pass,
            encoded_pass,
        ]
    )

    identifier_display = (
        ", ".join(identifier_columns)
        if identifier_columns
        else "None"
    )

    # --------------------------------------------------------------------------
    # Report
    # --------------------------------------------------------------------------

    print(
        f"    Original rows       : "
        f"{original_rows:,}"
    )

    print(
        f"    Preprocessing cols  : "
        f"{len(preprocessing_columns)}"
    )

    print(
        f"    Generative cols     : "
        f"{len(generative_columns)}"
    )

    print(
        f"    Numeric columns     : "
        f"{len(numeric_columns)}"
    )

    print(
        f"    Categorical columns : "
        f"{len(categorical_columns)}"
    )

    print(
        f"    Transformed columns : "
        f"{len(transformed_columns)}"
    )

    print(
        f"    Target              : "
        f"{target_column}"
    )

    print(
        f"    Target in generative: "
        f"{'YES' if target_column in generative_columns else 'NO'}"
    )

    print(
        f"    Target in preproc.  : "
        f"{'YES' if target_column in preprocessing_columns else 'NO'}"
    )

    print(
        f"    Identifiers excluded: "
        f"{identifier_display}"
    )

    print(
        f"    Train / Val / Test  : "
        f"{train_rows:,} / "
        f"{validation_rows:,} / "
        f"{test_rows:,}"
    )

    print(
        f"    Native train shape  : "
        f"{native_shape}"
    )

    print(
        f"    Encoded train shape : "
        f"{encoded_shape}"
    )

    print(
        f"    Target policy       : "
        f"{'PASS' if target_policy_pass else 'FAIL'}"
    )

    print(
        f"    Native schema       : "
        f"{'PASS' if native_schema_pass else 'FAIL'}"
    )

    print(
        f"    Dataset status      : "
        f"{'PASS' if dataset_pass else 'FAIL'}"
    )

    dataset_completion_records.append(
        {
            "dataset_id": dataset_id,
            "original_rows": original_rows,
            "train_rows": train_rows,
            "validation_rows": validation_rows,
            "test_rows": test_rows,
            "preprocessing_columns":
                len(preprocessing_columns),
            "generative_columns":
                len(generative_columns),
            "numeric_columns":
                len(numeric_columns),
            "categorical_columns":
                len(categorical_columns),
            "transformed_columns":
                len(transformed_columns),
            "target_column":
                target_column,
            "target_in_generative_schema":
                target_column in generative_columns,
            "target_in_preprocessing_input":
                target_column in preprocessing_columns,
            "identifier_count":
                len(identifier_columns),
            "native_train_rows":
                native_shape[0],
            "native_train_columns":
                native_shape[1],
            "encoded_train_rows":
                encoded_shape[0],
            "encoded_train_columns":
                encoded_shape[1],
            "dataset_complete":
                dataset_pass,
        }
    )


DATASET_COMPLETENESS_DF = pd.DataFrame(
    dataset_completion_records
)

dataset_completeness_pass = bool(
    DATASET_COMPLETENESS_DF[
        "dataset_complete"
    ].all()
)

print(
    f"\n  Dataset completeness : "
    f"{'PASS' if dataset_completeness_pass else 'FAIL'}"
)

if not dataset_completeness_pass:

    failed_datasets = (
        DATASET_COMPLETENESS_DF.loc[
            ~DATASET_COMPLETENESS_DF[
                "dataset_complete"
            ],
            "dataset_id"
        ].tolist()
    )

    raise RuntimeError(
        "Dataset completeness verification failed: "
        f"{failed_datasets}"
    )


# ==============================================================================
# 25.5 — OUTPUT DIRECTORY VERIFICATION
# ==============================================================================

print("\n[25.5] Output directory verification")
print("-" * 100)

REQUIRED_DIRECTORY_KEYS = [
    "processed_root",
    "native",
    "encoded",
    "splits",
    "preprocessors",
    "schemas",
    "feature_mapping",
]

directory_records = []

for key in REQUIRED_DIRECTORY_KEYS:

    if key not in NB02_DIRECTORIES:

        directory_records.append(
            {
                "directory_key": key,
                "path": "",
                "exists": False,
            }
        )

        print(
            f"  ✗ {key:<25} : "
            f"NOT REGISTERED"
        )

        continue

    directory_path = Path(
        NB02_DIRECTORIES[key]
    )

    exists = (
        directory_path.exists()
        and directory_path.is_dir()
    )

    directory_records.append(
        {
            "directory_key": key,
            "path": str(directory_path),
            "exists": exists,
        }
    )

    print(
        f"  {'✓' if exists else '✗'} "
        f"{key:<25} : "
        f"{directory_path}"
    )


DIRECTORY_VERIFICATION_DF = pd.DataFrame(
    directory_records
)

directory_pass = bool(
    DIRECTORY_VERIFICATION_DF[
        "exists"
    ].all()
)

print(
    f"\n  Directory verification : "
    f"{'PASS' if directory_pass else 'FAIL'}"
)


# ==============================================================================
# 25.6 — PREPROCESSOR / SCHEMA ARTIFACT VERIFICATION
# ==============================================================================

print(
    "\n[25.6] Preprocessor and schema artifact verification"
)
print("-" * 100)

schema_root = Path(
    NB02_DIRECTORIES["schemas"]
)

preprocessor_root = Path(
    NB02_DIRECTORIES["preprocessors"]
)

metadata_root = (
    schema_root
    / "metadata"
)

preprocessor_artifact_records = []

for dataset_id in DATASET_IDS:

    preprocessor_path = (
        preprocessor_root
        / dataset_id
        / "train_fitted_preprocessor.joblib"
    )

    schema_path = (
        schema_root
        / dataset_id
        / "preprocessing_schema.json"
    )

    metadata_path = (
        metadata_root
        / f"{dataset_id}_preprocessing_metadata.json"
    )

    preprocessor_found = (
        preprocessor_path.is_file()
        and
        preprocessor_path.stat().st_size > 0
    )

    schema_found = (
        schema_path.is_file()
        and
        schema_path.stat().st_size > 0
    )

    metadata_found = (
        metadata_path.is_file()
        and
        metadata_path.stat().st_size > 0
    )

    artifacts_pass = (
        preprocessor_found
        and
        schema_found
        and
        metadata_found
    )

    print(f"\n  {dataset_id}")

    print(
        f"    {'✓' if preprocessor_found else '✗'} "
        f"Preprocessor : "
        f"{preprocessor_path}"
    )

    print(
        f"    {'✓' if schema_found else '✗'} "
        f"Schema       : "
        f"{schema_path}"
    )

    print(
        f"    {'✓' if metadata_found else '✗'} "
        f"Metadata     : "
        f"{metadata_path}"
    )

    print(
        f"    Status       : "
        f"{'PASS' if artifacts_pass else 'FAIL'}"
    )

    preprocessor_artifact_records.append(
        {
            "dataset_id": dataset_id,
            "preprocessor":
                preprocessor_found,
            "schema":
                schema_found,
            "metadata":
                metadata_found,
            "artifacts_complete":
                artifacts_pass,
        }
    )


PREPROCESSOR_ARTIFACTS_DF = pd.DataFrame(
    preprocessor_artifact_records
)

preprocessor_artifacts_pass = bool(
    PREPROCESSOR_ARTIFACTS_DF[
        "artifacts_complete"
    ].all()
)

print(
    f"\n  Preprocessor artifacts : "
    f"{'PASS' if preprocessor_artifacts_pass else 'FAIL'}"
)


# ==============================================================================
# 25.7 — SECTION 21 FEATURE MAPPING ARTIFACT VERIFICATION
# ==============================================================================

print(
    "\n[25.7] Section 21 feature mapping artifact verification"
)
print("-" * 100)

feature_mapping_root = Path(
    NB02_DIRECTORIES["feature_mapping"]
)

FEATURE_MAPPING_PATHS = [
    (
        "Section 21 | feature mapping",
        feature_mapping_root
        / "feature_mapping.csv",
    ),
    (
        "Section 21 | feature mapping summary",
        feature_mapping_root
        / "feature_mapping_summary.csv",
    ),
    (
        "Section 21 | feature mapping metadata",
        feature_mapping_root
        / "feature_mapping_metadata.json",
    ),
]

feature_mapping_records = []

for artifact_name, artifact_path in FEATURE_MAPPING_PATHS:

    found = (
        artifact_path.is_file()
        and
        artifact_path.stat().st_size > 0
    )

    print(
        f"  {'✓' if found else '✗'} "
        f"{artifact_name:<45} : "
        f"{'FOUND' if found else 'MISSING'}"
    )

    feature_mapping_records.append(
        {
            "artifact": artifact_name,
            "path": str(artifact_path),
            "exists": found,
        }
    )


FEATURE_MAPPING_ARTIFACTS_DF = pd.DataFrame(
    feature_mapping_records
)

feature_mapping_pass = bool(
    FEATURE_MAPPING_ARTIFACTS_DF[
        "exists"
    ].all()
)

print(
    f"\n  Feature mapping artifacts : "
    f"{'PASS' if feature_mapping_pass else 'FAIL'}"
)


# ==============================================================================
# 25.8 — SECTION 22 SPLIT MANIFEST VERIFICATION
# ==============================================================================

print(
    "\n[25.8] Section 22 split manifest verification"
)
print("-" * 100)

project_root = Path(
    PROJECT_ROOT
)

split_manifest_root = (
    project_root
    / "results"
    / "raw_validation"
    / "notebook_02_manifests"
)

split_manifest_path = (
    split_manifest_root
    / "split_manifest.csv"
)

combined_manifest_path = (
    split_manifest_root
    / "combined_split_manifest.csv"
)

split_manifest_found = (
    split_manifest_path.is_file()
    and
    split_manifest_path.stat().st_size > 0
)

combined_manifest_found = (
    combined_manifest_path.is_file()
    and
    combined_manifest_path.stat().st_size > 0
)

print(
    f"  Manifest root : "
    f"{split_manifest_root}"
)

print(
    f"  {'✓' if split_manifest_found else '✗'} "
    f"Section 22 | split manifest : "
    f"{'FOUND' if split_manifest_found else 'MISSING'}"
)

print(
    f"  {'✓' if combined_manifest_found else '✗'} "
    f"Section 22 | combined manifest : "
    f"{'FOUND' if combined_manifest_found else 'MISSING'}"
)

split_manifest_pass = (
    split_manifest_found
    and
    combined_manifest_found
)


# ==============================================================================
# 25.9 — SECTION 23 RELOAD VALIDATION
# ==============================================================================

print(
    "\n[25.9] Section 23 reload validation verification"
)
print("-" * 100)

# ------------------------------------------------------------------------------
# Do not invent an artifact path here.
#
# Section 23's authoritative completion evidence is the successful execution
# of Section 23 and its persisted artifacts. If a reload-validation CSV was
# explicitly created by the frozen Section 23 implementation, verify it.
# Otherwise, the Section 23 runtime PASS remains the source of truth.
# ------------------------------------------------------------------------------

section_23_pass = (
    globals().get(
        "SECTION_23_STATUS",
        None
    )
    == "PASS"
)

print(
    f"  Section 23 runtime status : "
    f"{'PASS' if section_23_pass else 'NOT VERIFIED'}"
)

if "SECTION_23_STATUS" in globals():

    print(
        f"  SECTION_23_STATUS : "
        f"{SECTION_23_STATUS}"
    )

else:

    print(
        "  NOTE: SECTION_23_STATUS was not found in memory."
    )


# ==============================================================================
# 25.10 — SECTION 24 FINAL INTEGRITY ARTIFACT
# ==============================================================================

print(
    "\n[25.10] Section 24 final integrity artifact"
)
print("-" * 100)

final_integrity_found = (
    section_24_manifest_path.is_file()
    and
    section_24_manifest_path.stat().st_size > 0
)

print(
    f"  {'✓' if final_integrity_found else '✗'} "
    f"Section 24 final integrity manifest : "
    f"{'FOUND' if final_integrity_found else 'MISSING'}"
)

print(
    f"  Path : "
    f"{section_24_manifest_path}"
)

final_integrity_pass = (
    final_integrity_found
    and
    section_24_manifest_pass
)


# ==============================================================================
# 25.11 — GLOBAL ARTIFACT COMPLETENESS GATE
# ==============================================================================

print(
    "\n[25.11] Global artifact completeness"
)
print("-" * 100)

artifact_completeness_pass = all(
    [
        directory_pass,
        preprocessor_artifacts_pass,
        feature_mapping_pass,
        split_manifest_pass,
        final_integrity_pass,
    ]
)

print(
    f"  Output directories       : "
    f"{'PASS' if directory_pass else 'FAIL'}"
)

print(
    f"  Preprocessor artifacts   : "
    f"{'PASS' if preprocessor_artifacts_pass else 'FAIL'}"
)

print(
    f"  Feature mapping          : "
    f"{'PASS' if feature_mapping_pass else 'FAIL'}"
)

print(
    f"  Split manifests          : "
    f"{'PASS' if split_manifest_pass else 'FAIL'}"
)

print(
    f"  Section 24 artifact      : "
    f"{'PASS' if final_integrity_pass else 'FAIL'}"
)

print(
    f"\n  Artifact completeness    : "
    f"{'PASS' if artifact_completeness_pass else 'FAIL'}"
)


# ==============================================================================
# 25.12 — FINAL NOTEBOOK STATUS
# ==============================================================================

notebook_02_complete = all(
    [
        section_24_pass,
        section_24_manifest_pass,
        section_23_pass,
        dataset_completeness_pass,
        artifact_completeness_pass,
    ]
)

print("\n" + "=" * 100)
print("NOTEBOOK 02 FINAL STATUS")
print("=" * 100)

print(
    f"  Section 23 status               : "
    f"{'PASS' if section_23_pass else 'FAIL'}"
)

print(
    f"  Section 24 integrity gate        : "
    f"{'PASS' if section_24_pass else 'FAIL'}"
)

print(
    f"  Section 24 persisted manifest    : "
    f"{'PASS' if section_24_manifest_pass else 'FAIL'}"
)

print(
    f"  Dataset completeness             : "
    f"{'PASS' if dataset_completeness_pass else 'FAIL'}"
)

print(
    f"  Artifact completeness            : "
    f"{'PASS' if artifact_completeness_pass else 'FAIL'}"
)

print(
    f"\n  NOTEBOOK 02 STATUS               : "
    f"{'COMPLETE' if notebook_02_complete else 'FAIL'}"
)


# ==============================================================================
# 25.13 — SAVE FINAL COMPLETION ARTIFACTS
# ==============================================================================

completion_root = schema_root

completion_root.mkdir(
    parents=True,
    exist_ok=True
)

completion_timestamp = (
    datetime.now(
        timezone.utc
    )
    .replace(microsecond=0)
    .isoformat()
)

NOTEBOOK_02_COMPLETION_SUMMARY = {
    "notebook": "02",
    "notebook_name":
        "Preprocessing, Encoding & Data Splits",
    "completion_timestamp_utc":
        completion_timestamp,

    "datasets_processed":
        len(DATASET_IDS),

    "section_23_status":
        section_23_pass,

    "section_24_final_integrity_pass":
        section_24_pass,

    "section_24_manifest_pass":
        section_24_manifest_pass,

    "dataset_completeness_pass":
        dataset_completeness_pass,

    "artifact_completeness_pass":
        artifact_completeness_pass,

    "notebook_02_complete":
        notebook_02_complete,

    "split_policy":
        "70/15/15 stratified",

    "random_seed":
        int(
            globals().get(
                "RANDOM_SEED",
                globals().get(
                    "MASTER_SEED",
                    2025
                )
            )
        ),

    "identifier_policy":
        "Explicit identifiers excluded from generative schema",

    "provenance_policy":
        "__original_row_id__ retained for native auditability and excluded from preprocessing/transformed features",

    "target_policy":
        "Target retained in native generative schema and excluded from generic preprocessing input and transformed feature matrix",

    "preprocessing_fit_policy":
        "Training split only; validation and test splits never refit",

    "section_24_manifest_path":
        str(section_24_manifest_path),

    "split_manifest_path":
        str(split_manifest_path),

    "combined_split_manifest_path":
        str(combined_manifest_path),
}

completion_metadata_path = (
    completion_root
    / "notebook_02_completion_metadata.json"
)

with open(
    completion_metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        NOTEBOOK_02_COMPLETION_SUMMARY,
        f,
        indent=2,
        ensure_ascii=False
    )

DATASET_COMPLETENESS_DF.to_csv(
    completion_root
    / "notebook_02_dataset_completion.csv",
    index=False
)

PREPROCESSOR_ARTIFACTS_DF.to_csv(
    completion_root
    / "notebook_02_preprocessor_artifacts.csv",
    index=False
)

FEATURE_MAPPING_ARTIFACTS_DF.to_csv(
    completion_root
    / "notebook_02_feature_mapping_artifacts.csv",
    index=False
)

DIRECTORY_VERIFICATION_DF.to_csv(
    completion_root
    / "notebook_02_directory_verification.csv",
    index=False
)


# ==============================================================================
# 25.14 — RELOAD COMPLETION METADATA
# ==============================================================================

completion_metadata_reload_ok = False

try:

    with open(
        completion_metadata_path,
        "r",
        encoding="utf-8"
    ) as f:

        reloaded_completion_metadata = json.load(f)

    completion_metadata_reload_ok = (
        reloaded_completion_metadata
        == NOTEBOOK_02_COMPLETION_SUMMARY
    )

except Exception as exc:

    print(
        f"  Completion metadata reload error: "
        f"{exc}"
    )

print(
    f"\n  Completion metadata reload : "
    f"{'PASS' if completion_metadata_reload_ok else 'FAIL'}"
)


# ==============================================================================
# 25.15 — HARD FINAL GATE
# ==============================================================================

FINAL_NOTEBOOK_02_GATE = all(
    [
        notebook_02_complete,
        completion_metadata_reload_ok,
    ]
)

print("\n" + "=" * 100)
print("NOTEBOOK 02 COMPLETION GATE")
print("=" * 100)

if FINAL_NOTEBOOK_02_GATE:

    print(
        "✓ PASSED — Notebook 02 is COMPLETE and FROZEN."
    )

    print()
    print(
        "All required canonical objects, datasets, "
        "splits, preprocessing artifacts, feature mappings, "
        "persisted artifacts, and final integrity evidence "
        "have passed the completion gate."
    )

    print()
    print(
        f"Completion metadata:"
    )

    print(
        f"  {completion_metadata_path}"
    )

else:

    print(
        "✗ FAILED — Notebook 02 MUST NOT be considered complete."
    )

    raise RuntimeError(
        "\n"
        + "=" * 100
        + "\n"
        + "NOTEBOOK 02 COMPLETION GATE FAILED"
        + "\n"
        + "=" * 100
        + "\n"
        + f"Section 23                : {section_23_pass}\n"
        + f"Section 24                : {section_24_pass}\n"
        + f"Section 24 manifest       : {section_24_manifest_pass}\n"
        + f"Dataset completeness      : {dataset_completeness_pass}\n"
        + f"Artifact completeness     : {artifact_completeness_pass}\n"
        + f"Metadata reload           : {completion_metadata_reload_ok}\n"
        + "\n"
        + "Notebook 02 MUST NOT be treated as complete."
    )

print()
print("=" * 100)
print("SECTION 25 — COMPLETE")
print("=" * 100)

SECTION 25 — NOTEBOOK 02 FINAL COMPLETION GATE

[25.1] Required canonical objects
----------------------------------------------------------------------------------------------------
  ✓ DATASET_IDS
  ✓ TRAIN_DATASETS
  ✓ VALIDATION_DATASETS
  ✓ TEST_DATASETS
  ✓ TRAIN_PREPROCESSING_DATA
  ✓ TRAIN_PREPROCESSING_COLUMNS
  ✓ TRAIN_PREPROCESSING_METADATA
  ✓ TRAIN_PREPROCESSORS
  ✓ TRANSFORMED_TRAIN_DATASETS
  ✓ TRANSFORMED_VALIDATION_DATASETS
  ✓ TRANSFORMED_TEST_DATASETS
  ✓ NATIVE_FINAL_DATASETS
  ✓ ENCODED_FINAL_DATASETS
  ✓ NB02_DIRECTORIES
  ✓ TARGET_COLUMNS
  ✓ IDENTIFIER_COLUMNS
  Required canonical objects : PASS

[25.2] Section 24 final integrity gate
----------------------------------------------------------------------------------------------------
  Section 24 final integrity gate : PASS

[25.3] Section 24 persisted manifest verification
----------------------------------------------------------------------------------------------------
  ✓ Section 24 manifest : /content/driv